# VeriSOAP — Agentic (VS Code / local GPU)

Agentic version of `verisoap_FINAL`. **Everything from the original is kept** — Stage 0 audio front end,
Stage A K-span extraction, Stage B UMLS verification (VerifiNER), Stage C fine-tuned generation, Stage D
evidence-guided reflection, medical coding, all model training and every metric and table — and a fourth
arm is added:

| Arm | Pipeline |
|---|---|
| **A** K-SOAP baseline | raw Stage-A entities → generator |
| **B** + UMLS verification | Stage A → Stage B → generator |
| **C** + reflection (full, fixed order) | A → B → C → D, fixed number of rounds, last revision kept |
| **D agentic VeriSOAP** (new) | the same stages run by **LangGraph agents that decide**: the Verifier can send K-Span back to re-extract or switch to a deterministic UMLS re-check; Reflection keeps the **best** draft and rejects revisions that do not reduce the violations; the Reviser deletes unsupported claims in code or rewrites only the sections with findings |

All four arms are scored on the same held-out records with the same metrics, so **C → D measures what the
agents add** over the fixed pipeline (see the ablation and significance tables).

## Run it in VS Code on the GPU machine
1. One-time: `bash setup.sh` in this folder (CUDA PyTorch + packages + the "SOAP (GPU)" kernel), then fill in
   `.env`:
   ```
   OPENAI_API_KEY=sk-...
   HF_TOKEN=hf_...
   ```
   **No keys are stored in this notebook.** (The original had keys hardcoded — they were removed.)
2. Open this notebook → **Select Kernel → SOAP (GPU)** → **Run All**.
3. Optional, in the CONFIG cell: `ONLY_MODELS = ["gpt"]` for a first, quick run (only the GPT generator, no
   local training); `None` = every model, as in the original.

Differences from the Kaggle original: keys from `.env`; everything is saved in `./artifacts` and
`./adapters` on this machine (Hub backup off by default — set `HUB_SYNC = True` and `HUB_REPO_ID` to your own
repo to turn it on); no 12-hour session budgets; on an A100/H100 each model is loaded on one GPU,
in bf16, with `MAX_SEQ = 4096`.

```
SMOKE     tiny models, every code path incl. arm D - stops here if anything is broken
PHASE 1   entity prep (Stage A + B), shared by all models and arms
PHASE 1b  GPT fine-tune launch (async)          PHASE 2  train the open models
PHASE 2b  GPT fine-tune collect                 PHASE 3  score every model x arms A/B/C/D
PHASE 4   results: glossary, per-model tables, ablation (incl. C -> D), significance, CSV/JSON
```

## PHASE 0 — Environment

Ek hi Run All. **Kernel restart karne ki zaroorat nahi** — is cell se pehle kuch import
nahi hota, isliye installs seedha effect mein aa jaate hain. Install ke baad stack ko
turant import karke check kiya jaata hai, taaki tuta hua environment das second mein
pata chale, na ki chalis minute baad.

> **`scispacy` yahan mat daalna.** Wo `spacy<3.8` pin karta hai, jo `numpy<2` kheench
> laata hai; Kaggle numpy 2.x ke saath compiled torch bhejta hai, to poora stack tootta
> hai (`numpy.dtype size changed` → `partially initialized module 'torch' has no
> attribute 'fx'`). scispaCy ko `nmslib` bhi chahiye, jiska Python 3.12 wheel hai hi nahi.
> UMLS ko scispaCy ki zaroorat nahi — `UmlsJsonlKB` wahi KB plain JSONL se index karta hai.


In [ ]:
# ============================================================================
# Cell 2 - INSTALLS.  One Run All, no kernel restart.
# ============================================================================
# This notebook TRAINS and SCORES, so it needs both stacks: QLoRA/PLM training and
# the full metric suite (ROUGE, real BERTScore, BLEURT, QuestEval, WMD, embeddings).
#
# trl is deliberately absent. TRL 0.24 against the transformers 5.x preview on
# Kaggle raised "'functools.partial' object has no attribute '__func__'" and killed
# every qlora model; train_qlora() uses transformers' own Trainer now.
#
# gradio / faster-whisper / librosa are absent too: this notebook never launches the
# demo app, and Run All must terminate. Stage 0 stays defined for reference, and the
# audio demo lives in verisoap.ipynb.
#
# DO NOT add scispacy or a pinned spacy - it drags numpy below 2 and breaks torch.
# ---------------------------------------------------------------------------
INSTALL = True

if INSTALL:
    import subprocess, sys, importlib
    def pip(*args):
        print(">>> pip", *args, flush=True)
        # only-if-needed: never downgrade a package that already satisfies the spec
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--upgrade-strategy", "only-if-needed", *args], check=False)

    # --- pipeline + data
    pip("openai>=1.40", "langgraph")          # langgraph: the agentic arm D
    pip("datasets>=2.20")
    pip("-U", "huggingface_hub>=0.25")            # Hub backup of adapters + results
    pip("sentence-transformers>=3.0")             # UMLS retrieval + grounding
    # --- training
    pip("peft>=0.12", "bitsandbytes>=0.43", "accelerate>=0.33")
    # Kaggle ships torchao 0.10 but this transformers build needs >=0.16, and
    # Seq2SeqTrainer (the PLM path) imports it -> ImportError on biobart.
    pip("-U", "torchao>=0.16")
    # --- metrics
    pip("rouge-score", "bert-score")              # real token-level BERTScore
    pip("gensim")                                 # Word Mover's Distance
    # QuestEval is NOT installed. The only release (0.2.4, July 2021) declares no
    # dependencies and does not import against transformers 5.x - the same class of
    # trap as scispaCy, and it failed with ModuleNotFoundError on the 2026-09-20
    # run. Its job - reference-free consistency between note and transcript - is
    # already covered three ways: cosine_to_transcript (biomedical embedder),
    # hallucination_rate (claim grounding), and the Judge-4 faithfulness axis.
    # questeval_score() stays defined and returns None, so the column is simply
    # absent rather than the run failing.

    # Nothing was imported above this cell, so the new versions are picked up without
    # a restart - clear the finder cache so freshly written packages are visible.
    importlib.invalidate_caches()

    # Fail in ten seconds with a readable error, not forty minutes in with a
    # circular-import traceback.
    try:
        import numpy, torch, transformers, peft
        if int(numpy.__version__.split(".")[0]) < 2:
            raise RuntimeError(f"numpy downgraded to {numpy.__version__}; "
                               "run  pip install -U 'numpy>=2'  and restart")
        print(f"\nnumpy {numpy.__version__} | torch {torch.__version__} | "
              f"transformers {transformers.__version__} | peft {peft.__version__} | "
              f"cuda={torch.cuda.is_available()}")
    except Exception as _ex:
        print("\n" + "!" * 74)
        print(f"!  stack import FAILED after install: {type(_ex).__name__}: {_ex}")
        print("!  Run -> Restart & Run All, once. It should not happen twice.")
        print("!" * 74)
        raise

    print("\ninstalls done - no restart needed, the run continues.")
else:
    print("INSTALL=False - skipping.")

## 1. Config

The only cell you should need to edit.

In [ ]:
# ============================================================================
# Cell 3 - CONFIG.  Single control panel; nothing below this needs editing.
# ============================================================================
from pathlib import Path
import os as _os

# ---- keys: from a .env file next to the notebook (VS Code / local) or the environment.
# Nothing is hardcoded in this notebook - put OPENAI_API_KEY and HF_TOKEN in .env.
def _load_dotenv(paths=(".env", _os.path.join(_os.path.expanduser("~"), ".env"))):
    for p in paths:
        try:
            for line in open(p, encoding="utf-8"):
                if line.lstrip().startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                k, v = k.strip(), v.strip().strip('"').strip("'")
                if k and v and not _os.environ.get(k):
                    _os.environ[k] = v
        except OSError:
            pass
_load_dotenv()

# ---- data ------------------------------------------------------------------
# Works unchanged on a laptop and on Kaggle: on Kaggle the dataset arrives read-only
# under /kaggle/input/<slug>/ and only /kaggle/working is writable.
ON_KAGGLE = Path("/kaggle/working").exists()

# ---- GPU: an A100/H100 (>30 GB) holds a whole model on one device and has bf16
try:
    import torch as _torch
    _BIG_GPU = _torch.cuda.is_available() and _torch.cuda.get_device_properties(0).total_memory > 30e9
except Exception:
    _BIG_GPU = False

def _device_map():
    """Where local models go: one big GPU -> all on GPU 0 (faster than splitting); else spread."""
    try:
        import torch
        if not torch.cuda.is_available():
            return None
    except Exception:
        return None
    return {"": 0} if _BIG_GPU else "auto"

# Models to train and score in this run. None = every model in MODEL_REGISTRY (as the original).
# ["gpt"] = a quick first run: only the GPT generator, no local training.
ONLY_MODELS = None
# PHASE 1b launches a split-safe OpenAI fine-tune of gpt-4o-mini (costs money, as in the original).
# False = keep the existing fine-tuned model (its row is tagged '+'), and skip entity prep of the
# training records when no local model is trained either.
LAUNCH_GPT_FINETUNE = True

def _find_data():
    for c in [Path.cwd() / "data" / "clean_english_100.jsonl",
              *Path("/kaggle/input").glob("*/clean_english_100.jsonl"),
              *Path("/kaggle/input").glob("*/*/clean_english_100.jsonl"),
              Path.cwd() / "clean_english_100.jsonl"]:
        if c.exists():
            return c
    return Path.cwd() / "data" / "clean_english_100.jsonl"   # reported as MISSING below

DATA_PATH   = _find_data()
ARTIFACTS   = Path("/kaggle/working/artifacts") if ON_KAGGLE else (Path.cwd() / "artifacts")
N_RECORDS   = None       # smoke-test cap on RECORDS. The orchestrator ignores it and
                         # uses SCORE_N, so leave it None for a real run.

# ---- train / test split ----------------------------------------------------
# A model fine-tuned on a record and then scored on that same record reports
# memorisation, not generalisation. The split is made once, from a fixed seed,
# BEFORE anything trains. Training only ever sees TRAIN_RECORDS; every number in
# the results table comes from TEST_RECORDS.
TEST_FRACTION = 0.2      # 100 records -> 80 train / 20 test

# Refuse to score a model whose fine-tuned weights are missing. Base-model numbers
# recorded as fine-tuned ones is the easiest way to poison a results table.
REQUIRE_FINETUNED = True
SEED        = 42

# ---- which arms to run -----------------------------------------------------
RUN_ARM_A   = True       # K-SOAP baseline           (whiteboard left)
RUN_ARM_B   = True       # + Stage-B UMLS grounding
RUN_ARM_C   = True       # + Stage-D reflection      (full VeriSOAP)

# ---- generation model ------------------------------------------------------
MODEL          = "gpt"   # the GENERATOR (Stage C). The orchestrator overrides this
                         # per model; this is only the value the demo cells use.

# Stages A (NER), B (evidence) and D (reflection) need an INSTRUCTION-FOLLOWING
# model. A PLM cannot do them - BART is seq2seq with no chat interface - so when
# MODEL is a PLM these stages run on EXTRACTION_MODEL and the PLM only generates.
# Keeping it fixed across models is also the right experiment: the front end is
# held constant so the comparison isolates the generator.
EXTRACTION_MODEL = "gpt"
USE_FINETUNED  = True    # falls back to base model + warning if no adapter found
DO_TRAIN       = False   # True = actually launch fine-tuning in Stage C

_OUT = "/kaggle/working" if ON_KAGGLE else str(Path.cwd() / "adapters")

# Every OPEN model the K-SOAP paper benchmarks (Li et al., Table 5), so our numbers
# sit beside theirs. `paper_rougeL` is THEIR reported score in the mode named - a
# reference point for us to beat or match, not a result of ours.
MODEL_REGISTRY = {
    # ---- commercial: the only proven path in this project --------------------
    # split_unsafe: this adapter was fine-tuned BEFORE the deterministic train/test
    # split existed, on the BeTraC noise-augmented set, so it may have seen records
    # that are now in the test half. Its row is therefore NOT comparable to the
    # models trained here and is tagged in every results payload and table. Re-run
    # the OpenAI fine-tune on TRAIN_RECORDS to clear the flag:
    #     train_model("gpt")   # launches the job; paste the new id here
    "gpt":      {"id": "gpt-4o-mini-2024-07-18",
                 "ft": "ft:gpt-4o-mini-2024-07-18:personal:soap-noise-aug:DypalTtg",
                 "out": None, "kind": "openai", "split_unsafe": True,
                 "paper_rougeL": 0.634, "paper_mode": "GPT-4o oneshot (paper best)"},

    # ---- PLMs (encoder-decoder). Sir's Task 1, and the paper's baselines. -----
    "bart":     {"id": "facebook/bart-large",
                 "ft": None, "out": f"{_OUT}/bart-soap", "kind": "plm",
                 "paper_rougeL": 0.459, "paper_mode": "Bart-section-4"},
    "biobart":  {"id": "GanjinZero/biobart-v2-large",
                 "ft": None, "out": f"{_OUT}/biobart-soap", "kind": "plm",
                 "paper_rougeL": 0.382, "paper_mode": "biobart-section-4"},

    # ---- open causal LLMs ----------------------------------------------------
    "llama":    {"id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
                 "ft": None, "out": f"{_OUT}/llama-soap-qlora", "kind": "qlora",
                 "paper_rougeL": 0.516, "paper_mode": "Llama3-8b-section-4"},
    "llama2":   {"id": "meta-llama/Llama-2-13b-hf",
                 "ft": None, "out": f"{_OUT}/llama2-soap-qlora", "kind": "qlora",
                 "paper_rougeL": 0.516, "paper_mode": "Llama2-13b-section-4"},
    "qcammel":  {"id": "augtoma/qCammel-13",
                 "ft": None, "out": f"{_OUT}/qcammel-soap-qlora", "kind": "qlora",
                 "paper_rougeL": 0.538, "paper_mode": "qCammel-13b-section-15 (best open)"},
    # The paper's cautionary result: in-domain pretraining HURT here. Lowest ROUGE,
    # BERTScore, BLEURT and QuestEval of every model - below the BART baselines.
    # "domain knowledge insertion triggers decreasing on information extraction".
    "openbio":  {"id": "aaditya/Llama3-OpenBioLLM-8B",
                 "ft": None, "out": f"{_OUT}/openbio-soap-qlora", "kind": "qlora",
                 "paper_rougeL": 0.254, "paper_mode": "OpenBioLLM-8B-full (paper worst)"},
    "qwen":     {"id": "Qwen/Qwen2.5-7B-Instruct",
                 "ft": None, "out": f"{_OUT}/qwen-soap-qlora", "kind": "qlora",
                 "paper_rougeL": None, "paper_mode": "not in the paper"},
}


# ---- Stage 0: audio front end ----------------------------------------------
# The pipeline's real entry point is AUDIO. Text mode exists because the 100-record
# evaluation set is transcripts only - there is no audio for it.
#   "text"   read (transcript, soap) pairs from DATA_PATH        <- batch evaluation
#   "audio"  audio -> Whisper ASR -> LLM diarization -> Stage A  <- Gradio demo
INPUT_MODE      = "text"
WHISPER_MODEL   = "openai/whisper-large-v3-turbo"   # ASR (proven in audio2soap.ipynb)
FW_MODEL        = "large-v3"                         # faster-whisper, word confidence
DIARIZE_MODEL   = None          # None -> reuse MODEL; LLM speaker labelling, not pyannote
ASR_CONF_MIN    = 0.60          # below this a span is FLAGGED, never dropped
RUN_WORD_CONF   = True          # per-word ASR confidence -> entity.asr_confidence
RUN_NEMO_ALIGN  = False         # forced alignment: heavy, off by default
RUN_NATURE      = True          # severity: mild | moderate | severe | unspecified
DEMO_AUDIO      = "full_soap_demo.wav"

# ---- evaluation ------------------------------------------------------------
# The K-SOAP paper (sec 5.1) uses roberta-large for BERTScore. Keep it identical
# so our column is comparable to their Table 5.
BERTSCORE_MODEL = "roberta-large"
# The two paper metrics we previously lacked. Both are optional heavy deps and
# degrade to a blank column rather than failing the run.
RUN_BLEURT    = True
# Elron/bleurt-20-d12 was the original choice and is now HTTP 401 - the repo is
# gone, which is why the BLEURT column came back blank. Elron/bleurt-large-512 is
# a plain BertForSequenceClassification with num_labels=1, so it loads through
# AutoModelForSequenceClassification with no extra package (lucadiliello's
# BLEURT-20 ports need the separate bleurt-pytorch class).
# A list, not a string: one dead repo must not blank the column again.
BLEURT_MODELS = ["Elron/bleurt-large-512", "Elron/bleurt-base-512"]
BLEURT_MODEL  = BLEURT_MODELS[0]
RUN_QUESTEVAL = True                     # reference-free; works without a gold note
# Biomedical sentence encoder for grounding/similarity. NOTE: this is
# S-PubMedBert-MS-MARCO, NOT SapBERT - the old code named the variable SAPBERT_*
# but never loaded SapBERT. Set to "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
# if you actually want SapBERT.
EMBED_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"

# ---- medical coding (MedCodER-style, ported from audio2soap.ipynb) ----------
RUN_CODING      = True          # attach SNOMED CT / ICD-10-CM / CPT to the output
# Design doc, External Signals: medical codes are an "auxiliary prediction task or
# CONDITIONING SIGNAL". True feeds the resolved codes back into the Stage-C prompt
# instead of only reporting them at the end.
CODES_AS_SIGNAL = True

# ---- Stage B: VerifiNER ----------------------------------------------------
KB_BACKEND         = "auto"   # "auto" | "umls_jsonl" | "scispacy" | "icd_snomed"
# Real UMLS, downloaded directly. No scispaCy, so no spacy/numpy/nmslib conflict.
UMLS_KB_URL        = ("https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/kbs/"
                      "2023-04-23/umls_2022_ab_cat0129.jsonl")
SPAN_ALPHA         = 2        # 2.1 - expand span +/- this many tokens
RETRIEVE_K         = 5        # 2.2 - top-k CUIs per candidate span
SELF_CONSISTENCY_N = 5        # 2.4 - sampled reasoning paths for the vote
KB_SIM_THRESHOLD   = 0.55
# RAM guards for the UMLS index. The full KB is ~1.7M concepts; indexing every
# alias of all of them OOM-ed a Kaggle kernel. These keep it near ~1 GiB.
KB_MAX_CONCEPTS    = 400000   # after filtering to routable semantic types
KB_MAX_ALIASES     = 3        # aliases indexed per concept (was 8)     # min cosine to accept a CUI candidate

# How step 2.3 (type/relation verification) is done. THIS IS THE COST DIAL.
#
#   "deterministic"  Link to the best UMLS concept and check Stage A's
#                    entity type against that concept's UMLS semantic type via a
#                    lookup table. ZERO LLM calls in Stage B.
#                    ~9 calls per record total, minutes for the full 100.
#
#   "llm_evidence"   DEFAULT (sir: match the paper, cost is not a concern).
#                    Full VerifiNER: an evidence passage per (span, CUI)
#                    candidate, reassign type AND relation, then self-consistency
#                    vote. ~80 LLM calls per record, ~8,900 for 100 records.
#                    Stronger and closer to the paper; switch on for final numbers.
#
# Both modes produce identical VerifiedEntity records, so Stages C/D and every
# metric are unchanged. Only how `entity_type` gets verified differs.
VERIFY_MODE = "llm_evidence"

# ---- Stage D: reflection ---------------------------------------------------
MAX_REFLECT_ROUNDS = 2        # Algorithm 1's R
GROUNDING_THRESH   = 0.55     # claim_grounding cosine threshold

# ---- fine-tuning hyperparameters -------------------------------------------
# True  = the K-SOAP paper's settings (Li et al. sec 6.1): LoRA r=64, alpha=16,
#         2 epochs, lr 2e-4, warmup 0.03, 4-bit. Use these to sit our numbers
#         next to their Table 5.
# False = the values already used in fine_tune_eval_soap.ipynb (r=16, alpha=32, 3ep).
PAPER_HPARAMS = True

if PAPER_HPARAMS:
    LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 16, 0.05
    NUM_EPOCHS, GRAD_ACCUM, LR       = 2, 4, 2e-4
else:
    LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
    NUM_EPOCHS, GRAD_ACCUM, LR       = 3, 4, 2e-4
WARMUP_RATIO = 0.03

# GRAD_ACCUM was 16, matching the paper's effective batch of 32. The paper trains
# on 1,000 records; we train on 200, and 200 records at effective batch 32 for 2
# epochs is TWELVE optimizer steps. Measured on the 2026-09-15 run: bart finished
# in 4.5 min having taken 8 steps, which is not training, it is a warm-up. LoRA B
# is initialised to zero, so after 8 steps at warmup_ratio 0.03 the adapter has
# barely left its starting point and the "fine-tuned" model is close to the base.
# Accum 4 gives ~100 steps on the same data, same LoRA rank, same LR, same epochs.
# Effective batch drops 32 -> 8, which is the honest trade at this dataset size
# and is recorded in every adapter's _verisoap_done.json.

# MAX_SEQ was 4096. On a 16 GB T4 that OOM-ed every causal model: qwen died before
# step 1 and llama at step 4, both with "Tried to allocate 1.00-1.66 GiB". A T4 has
# no bfloat16 and no flash attention, so attention activations at 4096 tokens are
# the whole budget. omi_soap dialogues run ~2,000 tokens with the entity block and
# the note, so 2048 truncates almost nothing and roughly halves activation memory.
MAX_SEQ      = 4096 if _BIG_GPU else 2048     # A100/H100: the paper's 4096 fits

# PLM (encoder-decoder) generation mode - paper sec 6.3.
#   "full"       one model for the whole note
#   "section-4"  one adapter per S/O/A/P section. The paper found this clearly best
#                for BART (ROUGE-L 0.459 vs 0.330 full): encoder-decoder context is
#                only 1024 tokens, so a full transcript does not fit and the note
#                has to be produced section by section.
PLM_MODE     = "section-4"
# Full fine-tuning writes the entire model for EVERY section: 4 x 1.8 GiB
# per PLM, which exhausted Kaggle's 19.5 GiB disk mid-way through biobart.
# LoRA adapters are ~12 MiB each - same training signal, 130x less disk.
PLM_USE_LORA = True
# Each open model is a 15-26 GiB download and HuggingFace keeps them all cached.
# On Kaggle's 19.5 GiB volume that alone fills the disk, and the next save dies
# with torch's "unexpected pos N vs M", which says nothing about space.
# Weights are re-downloaded per model (~1 min) instead of being hoarded.
PURGE_HF_CACHE = ON_KAGGLE      # local disk: keep downloaded weights (no re-download per model)
PLM_MAX_IN   = 1024
PLM_MAX_OUT  = 512

# ---- the one-Run-All orchestrator ------------------------------------------
# Phases are wall-clock budgeted so that PHASE 4 (the tables and the Hub push)
# always runs, even when Kaggle is about to cut the session at 12 h. Whatever is
# unfinished is simply picked up by the next Run All - every phase is resumable.
TRAIN_BUDGET_MIN = 400      # stop STARTING a new model after 6.7 h
SCORE_BUDGET_MIN = 620      # stop STARTING a new (model, arm) after 10 h total
SESSION_HARD_MIN = 680      # 11 h: leave an hour for export + push
if not ON_KAGGLE:           # local GPU: no 12-hour session limit
    TRAIN_BUDGET_MIN = SCORE_BUDGET_MIN = SESSION_HARD_MIN = 10 ** 6

# Records scored per model. All three arms run on all of them.
#
# Sized honestly against the clock, not wishfully: on a T4 a 4-bit 7-8B model
# generates a ~600-token note in ~40 s, and a model costs 2 generations plus one
# reflection per record (arm C reuses arm B's draft, so it is reflection-only).
# That is ~80 s/record/model, so 100 records x 7 models is ~11 h of scoring on top
# of ~5.5 h of training. Two to three Run Alls, and nothing is ever redone.
#
# 100 is also the point where the paired bootstrap gives intervals worth quoting;
# the full 250-record split would be ~25 h and would blow the 30 h/week GPU quota.
# Bump when a change alters what a metric MEANS, so cached results from the old
# method are not silently mixed with new ones. v2: section-level reflection +
# anchored judge rubrics (2026-09-20). v3: the section-level splice actually
# fires - an unsupported term no longer forces all four sections, so arm C is a
# patched draft rather than a full rewrite (2026-10-05).
SCORING_VERSION = 3

# Models whose cached scores must be thrown away and recomputed.
#
# EMPTY, and it should stay empty unless an adapter changes under already-cached
# scores. It held "biobart" on 2026-09-21: a cancelled run had left that adapter
# mixed (section S on the current settings, O/A/P from 2026-09-15), so retraining
# it would have produced weights that no longer matched its cached row. Both the
# retrain and the forced re-score have since completed - all four biobart
# sections now carry grad_accum 4 signatures and fresh losses - so leaving the
# entry in would only burn ~23 minutes recomputing an already-correct row.
RESCORE = set()

SCORE_N = 50

RESUME     = True    # skip records already checkpointed in artifacts/
BOOT_N     = 10000   # bootstrap resamples
VERBOSE    = True

# Stage B is the cost centre: ~100-200 LLM calls per record, and at 100 records that
# is ~20k sequential calls (4+ hours) - long enough to risk a Kaggle session limit.
# Three levers, all safe to leave on:
N_WORKERS      = 8      # entity workers WITHIN one record
# Records used to run strictly one at a time, so 7 of 8 workers idled between
# them - that is what made prep 12.6 h. Records now run concurrently too.
# Total in-flight calls = RECORD_WORKERS x N_WORKERS.
RECORD_WORKERS = 6      # raise to 8+ on a higher OpenAI tier, drop to 2 on 429s
# Scoring used to be a plain sequential for-loop over records, so every OpenAI
# call waited for the one before it. Per model that is ~285 calls (generation,
# reflection, and two judges per arm) at a few seconds each - about 90 minutes of
# pure waiting for three models.
#
# The work is independent per record, and the expensive part is network latency,
# so records now run concurrently. What CANNOT overlap is the GPU: local PLM /
# QLoRA generation and the torch-backed metrics (BERTScore, BLEURT, QuestEval,
# the sentence embedders) share one device and one cached model each, so those
# sections take _GPU_LOCK and queue. They are fast; the API calls are not, and
# those are what now overlap.
SCORE_WORKERS = 6       # records scored concurrently. Drop to 2 if OpenAI 429s.
CACHE_EVIDENCE = True   # memoise (span, cui, type, relation) - spans repeat a lot
SKIP_VOTE_IF_UNANIMOUS = True   # no self-consistency when survivors already agree

ARTIFACTS.mkdir(exist_ok=True, parents=True)
print(f"eval data  : {DATA_PATH if DATA_PATH.exists() else '(none - demo does not need it)'}")
print(f"artifacts  : {ARTIFACTS}   {'(Kaggle)' if ON_KAGGLE else '(local)'}")
print(f"model      : {MODEL} ({MODEL_REGISTRY[MODEL]['kind']}), finetuned={USE_FINETUNED}")
print(f"records    : {N_RECORDS} | arms A={RUN_ARM_A} B={RUN_ARM_B} C={RUN_ARM_C}")
print(f"model kind : {MODEL_REGISTRY[MODEL]['kind']}"
      + (f" | paper ROUGE-L {MODEL_REGISTRY[MODEL]['paper_rougeL']} "
         f"({MODEL_REGISTRY[MODEL]['paper_mode']})"
         if MODEL_REGISTRY[MODEL].get("paper_rougeL") else ""))
print(f"verify     : {VERIFY_MODE}  |  paper hparams: {PAPER_HPARAMS} "
      f"(r={LORA_R}, alpha={LORA_ALPHA}, {NUM_EPOCHS}ep)")
print(f"stage B    : workers={N_WORKERS} cache={CACHE_EVIDENCE} skip_vote={SKIP_VOTE_IF_UNANIMOUS}")

In [ ]:
# ============================================================================
# Cell 4 - imports, seeds, checkpoint helpers
# ============================================================================
import os, re, json, math, random, textwrap, statistics, warnings, hashlib, time, glob, shutil
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any

warnings.filterwarnings("ignore")
random.seed(SEED)
try:
    import numpy as np; np.random.seed(SEED)
except ImportError:
    np = None

# ---- checkpointing ---------------------------------------------------------
# Three arms x 100 records x four stages is long and Kaggle sessions die.
# Every stage writes artifacts/{stage}_{record_id}.json and is skipped on resume.
# Final training loss per model (and per PLM section), filled in by the trainers
# and copied into each adapter's completion marker. A number here is what lets a
# later reader tell a trained adapter from a diverged one.
LAST_TRAIN_LOSS = {}


def _ck_path(stage: str, rid: str) -> Path:
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", str(rid))
    return ARTIFACTS / f"{stage}_{safe}.json"

def ck_load(stage: str, rid: str):
    """Return the checkpointed payload, or None."""
    if not RESUME:
        return None
    p = _ck_path(stage, rid)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except Exception:
        return None

def ck_save(stage: str, rid: str, payload) -> None:
    _ck_path(stage, rid).write_text(json.dumps(payload, indent=1, default=str))

def load_records(n=None):
    """(transcript, soap) pairs for BATCH EVALUATION and fine-tuning only.

    The product path - drop an audio file into the Gradio app - needs none of this.
    So a missing dataset is not an error: it just means the evaluation cells are
    unavailable while the demo still works.
    """
    if not DATA_PATH.exists():
        return []
    recs = [json.loads(l) for l in open(DATA_PATH) if l.strip()]
    return recs[: (n or len(recs))]


def split_records(recs, frac=None, seed=None):
    """Deterministic train/test split. Same seed -> same split, every session."""
    frac = TEST_FRACTION if frac is None else frac
    idx = list(range(len(recs)))
    random.Random(seed if seed is not None else SEED).shuffle(idx)
    n_test = max(1, int(round(len(recs) * frac))) if recs else 0
    held = set(idx[:n_test])
    return ([r for i, r in enumerate(recs) if i not in held],
            [r for i, r in enumerate(recs) if i in held])

print('checkpoint + split helpers ready')


In [ ]:
# ============================================================================
# Cell RESTORE - pick up everything a previous run already paid for.
#
# This REPLACES the old FRESH_START cell, which deleted every adapter on startup.
# That cell existed for one real reason: a half-trained adapter from a crashed run
# looks "already trained" to is_trained(), so the queue skips it and you silently
# keep a broken model. Deleting everything solved that - and also deleted the GOOD
# models. bart and biobart were lost exactly this way on 2026-09-15.
#
# The fix is a completion MARKER instead (see the TRAIN cell): _verisoap_done.json
# is written into an adapter directory only after training finishes and saves. A
# crashed adapter has no marker, so it is retrained; a finished one is kept.
# NOTHING IS EVER DELETED.
#
# This cell restores from /kaggle/input (a previous run's Output attached as an
# input dataset). The Hub restore runs next, in Cell HUB, once the token is loaded.
# ============================================================================
FRESH_START = False          # kept only so the name resolves; markers replace it

def restore_from_input(verbose=True):
    """Copy adapters, checkpoints and results out of /kaggle/input into working."""
    if not ON_KAGGLE:
        return 0
    import glob as _g
    wanted = {os.path.basename(spec["out"]) for spec in MODEL_REGISTRY.values()
              if spec.get("out")}
    n_ad = n_ck = 0
    for pat in ("/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        for d in _g.glob(pat):
            if not os.path.isdir(d):
                continue
            base = os.path.basename(d)
            # bart-soap-S -> bart-soap ; qwen-soap-qlora -> itself
            stem = base.rsplit("-", 1)[0] if base.rsplit("-", 1)[-1] in list("SOAP") else base
            if base not in wanted and stem not in wanted:
                continue
            dst = os.path.join("/kaggle/working", base)
            if os.path.exists(dst):
                continue
            try:
                shutil.copytree(d, dst); n_ad += 1
            except Exception:
                pass
    for pat in ("/kaggle/input/*/artifacts/*.json", "/kaggle/input/*/*/artifacts/*.json"):
        for f in _g.glob(pat):
            dst = ARTIFACTS / os.path.basename(f)
            if dst.exists():
                continue
            try:
                shutil.copy2(f, dst); n_ck += 1
            except Exception:
                pass
    if verbose:
        print(f"[restore] /kaggle/input -> working: {n_ad} adapter dirs, "
              f"{n_ck} checkpoints" if (n_ad or n_ck)
              else "[restore] nothing attached from /kaggle/input")
    return n_ad + n_ck


restore_from_input()


In [ ]:
# ============================================================================
# Cell DATA - datasets download THEMSELVES. Nothing to upload or drag-drop.
#
# Pick one with DATASET below and run. Each entry says where it comes from and
# which columns hold the transcript and the reference note.
#
# has_reference decides the evaluation:
#   True  -> ROUGE / BLEU / BERTScore / BLEURT / MEDCON  + the judges
#   False -> reference-free only: the judge axes, QuestEval, coverage, grounding
# evaluate_all() already skips the reference block when there is no gold note,
# so nothing needs changing when you switch datasets.
# ============================================================================
DATASET_REGISTRY = {
    # ---- the K-SOAP paper's own data. Only a 20-pair sample is public; the
    #      full 1,200 is not released. Same note format the paper reports.
    "cliniknote": {
        "kind": "github_zip",
        "repo": "catalwaysright/K-SOAP-GOODLABS",
        "files": {"dialogue": "dialogue.zip", "note": "note.zip"},
        "has_reference": True, "approx": 20,
        "note": "K-SOAP paper sample (20 pairs). Real experts, real SOAP format.",
    },
    # ---- biggest dialogue->SOAP set with OFFICIAL splits. Best default.
    "omi_soap": {
        "kind": "hf", "id": "omi-health/medical-dialogue-to-soap-summary",
        "transcript": "dialogue", "soap": "soap",
        "splits": {"train": "train", "test": "test"},
        "has_reference": True, "approx": 10000,
        "note": "9,250 train / 500 val / 250 test, official splits.",
    },
    "soap_notes": {
        "kind": "hf", "id": "adesouza1/soap_notes",
        "transcript": "patient_convo", "soap": "soap_notes",
        "splits": {"train": "train", "test": "test"},
        "has_reference": True, "approx": 620,
        "note": "558 train / 62 test. Full S/O/A/P notes.",
    },
    "augmented_notes": {
        "kind": "hf", "id": "AGBonnet/augmented-clinical-notes",
        "transcript": "conversation", "soap": "summary",
        "splits": {"train": "train"}, "has_reference": True, "approx": 30000,
        "note": "30k conversation + summary. Summary is not strict SOAP.",
    },
    "mts_dialog": {
        "kind": "hf", "id": "har1/MTS_Dialogue-Clinical_Note",
        "transcript": "dialogue", "soap": "section_text",
        "splits": {"train": "train"}, "has_reference": True, "approx": 1301,
        "note": "MTS-DIALOG. Section snippets, NOT full notes - the paper says so.",
    },
    # ---- what this project used until now. Kept so old runs stay reproducible.
    "betrac_local": {
        "kind": "local", "path": "clean_english_100.jsonl",
        "transcript": "transcript", "soap": "soap",
        "has_reference": True, "approx": 100,
        "note": "BeTraC-derived 100. Synthetic; foley tags in 94/100.",
    },
    # ---- audio, no gold note: exercises Stage 0 and the reference-free metrics.
    "demo_audio": {
        "kind": "audio_dir", "path": ".", "has_reference": False, "approx": 1,
        "note": "Local .wav files. No reference -> judge axes only.",
    },
}

# ---------------------------------------------------------------------------
# CHOSEN AFTER INSPECTING THE DATA, not the metadata:
#
#   omi_soap    200/200 notes carry all four S/O/A/P sections, no foley tags, no
#               markdown junk, and it writes "S:" - the exact format our generation
#               prompt asks for, so there is no format fight with the fine-tune.
#               9,250 train / 250 official test. The test size is what matters:
#               250 gives usable bootstrap CIs, the old 20 gave none.
#   soap_notes  equally clean (200/200 sections) and far more diverse - 110
#               distinct doctors vs omi_soap's 2 - but only 558/62. Used as a
#               SECOND dataset to show the pipeline generalises.
#   cliniknote  the K-SOAP paper's own sample (20 pairs). Small, but it is the
#               only direct comparison to their Table 5.
#
# Rejected: augmented_notes (notes are JSON blobs, not SOAP) and mts_dialog
# (571-char snippets, not full notes - the paper says so itself).
# ---------------------------------------------------------------------------
DATASET = "omi_soap"

# Split sizes. TRAIN drives GPU time; TEST drives statistical power.
# The K-SOAP paper trains on 1,000 - raise TRAIN_LIMIT to match it if GPU allows.
# MEASURED, not guessed: entity prep runs ~1 min/record in llm_evidence mode
# (~20 entities x (5 CUI evidence + 5 self-consistency) = 150-200 LLM calls).
# 500 train + 250 test was 12.6 h of prep alone - longer than a Kaggle session.
# Test size is what buys statistical power, so that stays at the full 250;
# the training set is what got cut.
TRAIN_LIMIT = 200
TEST_LIMIT  = 250          # the whole official omi_soap test split
MAX_RECORDS = TRAIN_LIMIT  # back-compat


def _cache_dir():
    d = ARTIFACTS / "datasets"
    d.mkdir(parents=True, exist_ok=True)
    return d


def _load_hf(spec, split, limit):
    from datasets import load_dataset
    name = spec["splits"].get(split)
    if not name:
        return []
    ds = load_dataset(spec["id"], split=name, streaming=True)
    out = []
    for i, r in enumerate(ds):
        t, s = r.get(spec["transcript"]), r.get(spec["soap"])
        if not t or not s:
            continue
        out.append({"id": f"{spec['id'].split('/')[-1]}_{split}_{i}",
                    "transcript": str(t), "soap": str(s)})
        if limit and len(out) >= limit:
            break
    return out


def _load_github_zip(spec, limit):
    """Download + unzip a GitHub repo's data files.

    The archives extract FLAT (no dialogue/ or note/ subfolders), so pair by
    filename instead of by directory: <stem>.txt is the dialogue and
    <stem>_soap.txt is its note.
    """
    import urllib.request, zipfile, io
    base = f"https://raw.githubusercontent.com/{spec['repo']}/main/"
    cache = _cache_dir() / spec["repo"].split("/")[-1]
    if not cache.exists() or not any(cache.iterdir()):
        cache.mkdir(parents=True, exist_ok=True)
        for fn in spec["files"].values():
            with urllib.request.urlopen(base + fn, timeout=90) as r:
                zipfile.ZipFile(io.BytesIO(r.read())).extractall(cache)

    files = {os.path.basename(p): p
             for p in glob.glob(str(cache / "**" / "*"), recursive=True)
             if os.path.isfile(p) and p.endswith(".txt")}
    out = []
    for name in sorted(files):
        stem = name[:-4]
        if stem.endswith("_soap"):
            continue                                   # this is a note, not a dialogue
        note = files.get(f"{stem}_soap.txt")
        if not note:
            continue
        out.append({"id": stem,
                    "transcript": open(files[name], errors="replace").read().strip(),
                    "soap": open(note, errors="replace").read().strip()})
        if limit and len(out) >= limit:
            break
    return out

def _load_local(spec, limit):
    p = _find_data() if spec["path"].endswith("clean_english_100.jsonl") else Path(spec["path"])
    if not Path(p).exists():
        return []
    recs = [json.loads(l) for l in open(p) if l.strip()]
    for i, r in enumerate(recs):
        r.setdefault("id", f"local_{i}")
    return recs[:limit] if limit else recs


def _load_audio_dir(spec, limit):
    roots = [spec["path"], "..", "/kaggle/input"]
    files = sorted({p for root in roots for ext in ("wav", "mp3", "m4a")
                    for p in glob.glob(os.path.join(root, f"*.{ext}"))
                    + glob.glob(os.path.join(root, "*", f"*.{ext}"))})
    return [{"id": os.path.basename(f), "audio": f, "transcript": None, "soap": None}
            for f in (files[:limit] if limit else files)]


def load_dataset_records(name=None, limit=None):
    """-> (train, test, spec). Downloads on demand; nothing to upload."""
    name = name or DATASET
    spec = DATASET_REGISTRY[name]
    limit = TRAIN_LIMIT if limit is None else limit
    print(f"[data] {name}: {spec['note']}")
    try:
        if spec["kind"] == "hf":
            tr = _load_hf(spec, "train", limit)
            te = _load_hf(spec, "test", TEST_LIMIT)
            if not te:                       # no official test split -> carve one
                tr, te = split_records(tr)
            return tr, te, spec
        if spec["kind"] == "github_zip":
            return (*split_records(_load_github_zip(spec, limit)), spec)
        if spec["kind"] == "audio_dir":
            return [], _load_audio_dir(spec, limit), spec
        return (*split_records(_load_local(spec, limit)), spec)
    except Exception as ex:
        print(f"[data] {name} FAILED ({type(ex).__name__}: {str(ex)[:120]})")
        print("[data] falling back to betrac_local")
        return (*split_records(_load_local(DATASET_REGISTRY["betrac_local"], limit)),
                DATASET_REGISTRY["betrac_local"])


def dataset_menu():
    print(f"{'key':<18}{'records':>9}  {'gold note':<10}source")
    print("-" * 88)
    for k, v in DATASET_REGISTRY.items():
        src = v.get("id") or v.get("repo") or v.get("path")
        mark = "<-" if k == DATASET else "  "
        print(f"{k:<18}{v['approx']:>9}  {'YES' if v['has_reference'] else 'no':<10}{src} {mark}")
    print("-" * 88)
    print("Set DATASET = '<key>' and re-run. No upload, no drag-drop.")
    print("Datasets WITHOUT a gold note score on the judge axes + QuestEval only.")


TRAIN_RECORDS, TEST_RECORDS, DATASET_SPEC = load_dataset_records()
HAS_REFERENCE = DATASET_SPEC["has_reference"]
RECORDS = TEST_RECORDS[:N_RECORDS] if N_RECORDS else TEST_RECORDS

if TRAIN_RECORDS or TEST_RECORDS:
    _tr = {r["id"] for r in TRAIN_RECORDS}
    _te = {r["id"] for r in TEST_RECORDS}
    assert not (_tr & _te), "train/test overlap - split is broken"
    print(f"[data] {len(TRAIN_RECORDS)} train / {len(TEST_RECORDS)} test | "
          f"reference notes: {'yes' if HAS_REFERENCE else 'NO -> reference-free metrics only'}")
    if TEST_RECORDS and TEST_RECORDS[0].get("transcript"):
        print(f"[data] transcript mean {statistics.fmean(len(r['transcript']) for r in TEST_RECORDS):,.0f} chars")
else:
    print("[data] nothing loaded - run dataset_menu() to pick another.")

In [ ]:
# ============================================================================
# Cell HF - HuggingFace auth. Needed to download model weights.
#
# Order: Kaggle secret HF_TOKEN  ->  env HF_TOKEN  ->  the project token below.
# The secret wins, so rotating the token later needs no code change.
# ============================================================================
# Hardcoded at Nishant's explicit request so a Run All needs no setup at all.
# WRITE scope, owner Nishant7p, token name "Soap note_train". Write is REQUIRED:
# the adapters and results are backed up to the Hub, which is the only thing that
# survives a Kaggle session ending. A Kaggle secret named HF_TOKEN still wins, so
# rotating the token later needs no code change.
HF_TOKEN_FALLBACK = ""   # removed: put HF_TOKEN in .env (never hardcode tokens)

def hf_login(verbose=True):
    tok = ""
    try:
        from kaggle_secrets import UserSecretsClient
        tok = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
        src = "kaggle secret"
    except Exception:
        tok, src = "", ""
    if not tok:
        tok, src = (os.environ.get("HF_TOKEN") or "").strip(), "env HF_TOKEN"
    if not tok:
        tok, src = HF_TOKEN_FALLBACK, "project fallback"
    if not tok:
        print("No HF token. Public models still work; gated ones will 403.")
        return None
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok        # older libs read this name
    try:
        from huggingface_hub import login
        login(token=tok, add_to_git_credential=False)
    except Exception as ex:
        if verbose:
            print(f"[HF] login() failed ({type(ex).__name__}); env vars still set.")
    if verbose:
        print(f"[HF] token from {src} (...{tok[-4:]})")
    return tok


def hf_check_access(models=None, verbose=True):
    """Which model weights this token can actually pull. Gating blocks FILES, not
    the model card, so config.json is the honest test."""
    import urllib.request
    tok = os.environ.get("HF_TOKEN", "")
    hdr = {"Authorization": f"Bearer {tok}"} if tok else {}
    keys = models or [k for k, v in MODEL_REGISTRY.items() if v["kind"] != "openai"]
    ok, blocked = [], []
    for k in keys:
        mid = MODEL_REGISTRY[k]["id"]
        req = urllib.request.Request(
            f"https://huggingface.co/{mid}/resolve/main/config.json", headers=hdr)
        try:
            urllib.request.urlopen(req, timeout=25).close()
            ok.append(k)
        except Exception as ex:
            blocked.append((k, mid, getattr(ex, "code", type(ex).__name__)))
    if verbose:
        print(f"[HF] downloadable: {', '.join(ok) if ok else 'none'}")
        for k, mid, code in blocked:
            print(f"[HF] BLOCKED {k} ({mid}) - HTTP {code}")
        if blocked:
            print("     These are GATED. Accept the licence on huggingface.co while")
            print("     signed in as the token's owner, then re-run. Everything else")
            print("     trains fine meanwhile - train_all_models() skips nothing else.")
    return ok, blocked


hf_login()
HF_OK, HF_BLOCKED = hf_check_access()

---
## Hub backup — kyunki Kaggle sab ud deta hai

`/kaggle/working` session khatam hote hi wipe ho jaata hai. 2026-09-15 ko isi wajah se
`bart` aur `biobart` ke adapters gaye, 48 minute ki GPU ke saath.

Ye cell teen cheezein private HuggingFace dataset repo **`Nishant7p/verisoap-store`**
mein rakhta hai aur shuru mein wapas kheench leta hai:

| repo path | kya |
|---|---|
| `adapters/<dir>/` | har trained adapter (PLM ke 4 per model) |
| `artifacts/*.json` | Stage A/B entity checkpoints + per-(model, arm, record) metrics |
| `results/*.json` | per-model results payloads |

Isi wajah se tum notebook baar baar chala sakte ho: **jo ho chuka hai wo dobara nahi
hota.** Adapters, entity prep, aur har scored record — sab Hub se wapas aate hain.


In [ ]:
# ============================================================================
# Cell HUB - adapters, checkpoints and results survive the session.
#
# /kaggle/working is wiped when a session ends. "Save Version" preserves it, but it
# has to be remembered, and forgetting it once already cost this project bart and
# biobart. Nothing here depends on remembering anything.
#
# ONE private dataset repo holds everything:
#
#     Nishant7p/verisoap-store
#       adapters/<dir>/...      one folder per adapter (PLM: 4 per model)
#       artifacts/*.json        Stage A/B entity checkpoints + per-arm metrics
#       results/*.json          per-model results payloads
#
# Pushed after every model and after every (model, arm), pulled at startup. That is
# what makes repeated Run Alls cheap: the second run re-does nothing.
# ============================================================================
HUB_SYNC    = ON_KAGGLE     # local: everything stays in ./artifacts and ./adapters.
                            # True + HUB_REPO_ID = '<you>/verisoap-store' for a Hub backup
HUB_REPO_ID = None        # None -> "<whoami>/verisoap-store"
HUB_PRIVATE = True

HUB = {"enabled": False, "user": None, "repo": None, "write": False, "why": ""}


def _hub_api():
    from huggingface_hub import HfApi
    return HfApi(token=os.environ.get("HF_TOKEN") or None)


def hub_status(verbose=True):
    """Resolve the repo and probe for WRITE access with a real create_repo call.

    whoami() reports a token role, but the field has moved between huggingface_hub
    versions and fine-grained tokens report it differently. An actual
    create_repo(exist_ok=True) cannot be wrong - and the repo is needed anyway.
    """
    HUB.update(enabled=False, write=False)
    if not HUB_SYNC:
        HUB["why"] = "HUB_SYNC=False"; return HUB
    try:
        api = _hub_api()
    except Exception as ex:
        HUB["why"] = f"huggingface_hub unavailable ({type(ex).__name__})"; return HUB
    try:
        HUB["user"] = api.whoami().get("name")
    except Exception as ex:
        HUB["why"] = f"whoami failed ({type(ex).__name__}) - no HF token?"; return HUB
    HUB["repo"] = HUB_REPO_ID or f"{HUB['user']}/verisoap-store"
    try:
        api.create_repo(HUB["repo"], repo_type="dataset",
                        private=HUB_PRIVATE, exist_ok=True)
        HUB.update(enabled=True, write=True, why="")
    except Exception as ex:
        HUB.update(enabled=False, write=False,
                   why=f"no write access ({type(ex).__name__}: {str(ex)[:70]})")
    if verbose:
        if HUB["write"]:
            print(f"[hub] backup ON  -> {HUB['repo']} (private={HUB_PRIVATE})")
        else:
            print(f"[hub] backup OFF - {HUB['why']}")
            print("      Nothing will survive this session. Fix the HF token, or")
            print("      click Save Version by hand when the run ends.")
    return HUB


def hub_push_dir(local_dir, path_in_repo, verbose=True, patterns=None):
    if not HUB["write"] or not os.path.isdir(local_dir):
        return False
    try:
        _hub_api().upload_folder(
            folder_path=str(local_dir), path_in_repo=path_in_repo,
            repo_id=HUB["repo"], repo_type="dataset",
            allow_patterns=patterns,
            commit_message=f"verisoap: {path_in_repo}")
        if verbose:
            print(f"   [hub] pushed {path_in_repo}")
        return True
    except Exception as ex:
        print(f"   [hub] push FAILED {path_in_repo}: "
              f"{type(ex).__name__}: {str(ex)[:90]}")
        return False


def hub_push_model(model_key, verbose=True):
    """Push every adapter directory belonging to one model."""
    if not HUB["write"]:
        return False
    ok = True
    for p in _adapter_paths(model_key):
        d = find_adapter(p)
        if d and os.path.isdir(d):
            ok &= hub_push_dir(d, f"adapters/{os.path.basename(d.rstrip('/'))}", verbose)
    return ok


def hub_push_artifacts(verbose=True):
    """Push entity checkpoints, per-arm metrics and results payloads.

    Two destinations because results/ is what the paper reads and artifacts/ is
    what makes a re-run cheap; keeping them apart means a results-only pull is
    small.
    """
    if not HUB["write"]:
        return False
    a = hub_push_dir(ARTIFACTS, "artifacts", verbose=False,
                     patterns=["*.json"])
    b = hub_push_dir(ARTIFACTS, "results", verbose=False,
                     patterns=["results_*.json", "verisoap_all_models.*"])
    if verbose and a:
        print(f"   [hub] pushed {len(list(ARTIFACTS.glob('*.json')))} artifact files")
    return a and b


def hub_pull(verbose=True):
    """Restore adapters, checkpoints and results from the Hub into this session.

    Read access is enough to pull, so this works even when backup is off: a repo
    populated by an earlier run is still readable.
    """
    if not HUB_SYNC or not HUB.get("repo"):
        return 0
    from huggingface_hub import snapshot_download
    try:
        snap = snapshot_download(repo_id=HUB["repo"], repo_type="dataset",
                                 token=os.environ.get("HF_TOKEN") or None)
    except Exception as ex:
        if verbose:
            print(f"[hub] nothing to restore ({type(ex).__name__}) - first run?")
        return 0

    n_ad = n_ck = 0
    ad_root = os.path.join(snap, "adapters")
    if os.path.isdir(ad_root):
        os.makedirs(_OUT, exist_ok=True)
        for base in sorted(os.listdir(ad_root)):
            src_d, dst_d = os.path.join(ad_root, base), os.path.join(_OUT, base)
            if not os.path.isdir(src_d) or os.path.exists(dst_d):
                continue
            try:
                shutil.copytree(src_d, dst_d); n_ad += 1
            except Exception:
                pass
    for sub in ("artifacts", "results"):
        root = os.path.join(snap, sub)
        if not os.path.isdir(root):
            continue
        for f in sorted(os.listdir(root)):
            if not f.endswith(".json") and not f.endswith(".csv"):
                continue
            dst = ARTIFACTS / f
            if dst.exists():
                continue
            try:
                shutil.copy2(os.path.join(root, f), dst); n_ck += 1
            except Exception:
                pass
    if verbose:
        print(f"[hub] restored {n_ad} adapter dirs, {n_ck} checkpoint/result files "
              f"from {HUB['repo']}")
    return n_ad + n_ck


hub_status()
HUB_RESTORED = hub_pull()

## 2. Schema

10 entity types x 6 relations, and the 60-cell routing table that makes the Stage-D placement
check decidable. Full rationale in `SCHEMA.md`.

`ENTITY_SECTION_MAP` is the piece that did not exist anywhere in the previous notebooks.
`section` is a **pure function** of `(entity_type, relation)` — never set by the LLM — so a
placement disagreement is always attributable to the generator, never to the router.

In [ ]:
# ============================================================================
# Cell 5 - SCHEMA.  Source of truth: SCHEMA.md
# ============================================================================
ENTITY_TYPES = (
    "Disease/Problem", "Symptom", "Medication", "Procedure/Treatment",
    "Test/Investigation", "Lab/Measurement", "Clinical Finding",
    "Anatomical Site", "Allergy", "Social/Family History",
)

# Design doc sec 3.3: K-SOAP's 5 relations + ORDERED, the extension that makes
# Plan-section routing derivable.
RELATION_SET = ("PRESENT", "ABSENT", "FAMILY", "PAST", "UNKNOWN", "ORDERED")

# Migration from the legacy 4-status schema in audio2soap.ipynb
LEGACY_STATUS_MAP = {
    "Confirmed": "PRESENT", "Negated": "ABSENT",
    "Historical": "PAST",   "Family_History": "FAMILY",
}
LEGACY_LABEL_MAP = {
    "Drug": "Medication", "Disease": "Disease/Problem",
    "Symptom": "Symptom",  "Procedure": "Procedure/Treatment",
}

# Design doc sec 3.2: these bypass Stage B - numeric/administrative, no meaningful CUI.
BYPASS_UMLS = {"Lab/Measurement", "Anatomical Site"}

S, O, A, P = "S", "O", "A", "P"

# (entity_type, relation) -> SOAP section.  Fully enumerated: 10 x 6 = 60, no fallback.
_SECTION_ROWS = {
    #                       PRESENT ABSENT FAMILY PAST UNKNOWN ORDERED
    "Disease/Problem":      (A,     S,     S,     S,   A,      P),
    "Symptom":              (S,     S,     S,     S,   S,      P),
    "Medication":           (S,     S,     S,     S,   S,      P),
    "Procedure/Treatment":  (O,     S,     S,     S,   S,      P),
    "Test/Investigation":   (O,     S,     S,     S,   S,      P),
    "Lab/Measurement":      (O,     O,     S,     S,   O,      P),
    "Clinical Finding":     (O,     O,     S,     S,   O,      P),
    "Anatomical Site":      (O,     O,     S,     S,   O,      P),
    "Allergy":              (S,     S,     S,     S,   S,      P),
    "Social/Family History":(S,     S,     S,     S,   S,      P),
}
ENTITY_SECTION_MAP = {
    (t, r): _SECTION_ROWS[t][i]
    for t in ENTITY_TYPES for i, r in enumerate(RELATION_SET)
}

def route_section(entity_type: str, relation: str) -> str:
    """Deterministic ROLE2SEC lookup (Algorithm 1 line 21)."""
    return ENTITY_SECTION_MAP.get((entity_type, relation), S)


# ---------------------------------------------------------------------------
# Deterministic type verification (VERIFY_MODE = "deterministic").
#
# Every UMLS concept carries a semantic type (TUI). That is enough to check
# Stage A's guess without asking an LLM: if Stage A says "discharge" is a Symptom
# but the linked concept is T058 Health Care Activity, the guess is wrong.
# This is the whole point of Stage B, done as a lookup instead of a model call.
# ---------------------------------------------------------------------------
SEMANTIC_TYPE_TO_ENTITY = {
    # Disease / Problem
    "T047": "Disease/Problem",  "T191": "Disease/Problem",  "T046": "Disease/Problem",
    "T048": "Disease/Problem",  "T019": "Disease/Problem",  "T020": "Disease/Problem",
    "T037": "Disease/Problem",  "T190": "Disease/Problem",  "T049": "Disease/Problem",
    "T050": "Disease/Problem",
    # Symptom vs Clinical Finding
    "T184": "Symptom",
    "T033": "Clinical Finding", "T201": "Clinical Finding", "T032": "Clinical Finding",
    # Lab / Measurement
    "T034": "Lab/Measurement",
    # Medication
    "T121": "Medication", "T200": "Medication", "T195": "Medication",
    "T109": "Medication", "T110": "Medication", "T125": "Medication",
    "T127": "Medication", "T129": "Medication", "T131": "Medication",
    "T197": "Medication", "T196": "Medication", "T104": "Medication",
    # Procedures vs diagnostics
    "T061": "Procedure/Treatment", "T058": "Procedure/Treatment",
    "T060": "Test/Investigation",  "T059": "Test/Investigation",
    # Anatomy
    "T023": "Anatomical Site", "T029": "Anatomical Site", "T030": "Anatomical Site",
    "T024": "Anatomical Site", "T025": "Anatomical Site",
    "T021": "Anatomical Site", "T022": "Anatomical Site", "T018": "Anatomical Site",
    # NOTE: T031 "Body Substance" is deliberately NOT mapped. Spans like "discharge",
    # "sputum", "blood" link to it, but when a patient reports one it is a Symptom,
    # not an anatomical site. Leaving it unmapped keeps Stage A's label instead of
    # overriding it wrongly.
    # Social / family
    "T055": "Social/Family History", "T056": "Social/Family History",
    "T057": "Social/Family History", "T098": "Social/Family History",
    "T099": "Social/Family History", "T101": "Social/Family History",
}

# Fallback when a backend reports semantic-type NAMES rather than TUI codes
# (the ICD/SNOMED fallback KB does exactly this).
_STY_NAME_TO_ENTITY = {
    "disease or syndrome": "Disease/Problem", "neoplastic process": "Disease/Problem",
    "pathologic function": "Disease/Problem", "injury or poisoning": "Disease/Problem",
    "mental or behavioral dysfunction": "Disease/Problem",
    "sign or symptom": "Symptom",
    "finding": "Clinical Finding", "clinical attribute": "Clinical Finding",
    "laboratory or test result": "Lab/Measurement",
    "pharmacologic substance": "Medication", "clinical drug": "Medication",
    "antibiotic": "Medication", "organic chemical": "Medication",
    "therapeutic or preventive procedure": "Procedure/Treatment",
    "health care activity": "Procedure/Treatment",
    "diagnostic procedure": "Test/Investigation",
    "laboratory procedure": "Test/Investigation",
    "body part, organ, or organ component": "Anatomical Site",
    "body location or region": "Anatomical Site", "body substance": "Anatomical Site",
    "individual behavior": "Social/Family History",
    "family group": "Social/Family History",
    "occupational activity": "Social/Family History",
}

# Types whose correctness depends on conversational context, not on the concept
# itself, so UMLS must NOT overrule Stage A. "Penicillin allergy" links to
# Penicillin (a Pharmacologic Substance) and would wrongly become a Medication.
PROTECTED_TYPES = {"Allergy", "Social/Family History"}


def umls_implied_type(cand: "KBCandidate") -> Optional[str]:
    """The entity type a concept's UMLS semantic type implies, or None if unmapped."""
    raw = (cand.semantic_type or "")
    for tui in re.findall(r"\bT\d{3}\b", raw):
        if tui in SEMANTIC_TYPE_TO_ENTITY:
            return SEMANTIC_TYPE_TO_ENTITY[tui]
    for part in [p.strip().lower() for p in raw.split(",")]:
        if part in _STY_NAME_TO_ENTITY:
            return _STY_NAME_TO_ENTITY[part]
    return None


# Overriding Stage A's type on a weak match does more harm than good: this mode
# cannot read the surrounding sentence, so a mediocre link is not evidence enough
# to overrule an extractor that did see the context. Link freely, correct rarely.
TYPE_OVERRIDE_MIN_SCORE = 0.85


@dataclass
class VerifiedEntity:
    span:           str          # EXACT substring of the transcript - hard invariant
    entity_type:    str
    relation:       str
    cui:            str  = ""
    preferred_term: str  = ""
    semantic_type:  str  = ""
    evidence:       str  = ""
    section:        str  = "S"
    grounded:       bool = False
    source_turn:    int  = -1
    source_speaker: str  = ""
    char_start:     int  = -1
    char_end:       int  = -1
    # --- Stage 0 (audio path only; None when the input was already text) ------
    # Design doc sec 2 lists ASR confidence as an external signal: "reliability of
    # transcript words - weight or filter uncertain spans". Recorded and flagged,
    # never silently dropped.
    asr_confidence: Optional[float] = None
    low_confidence: bool = False
    nature:         str  = ""      # mild | moderate | severe | unspecified
    snomed:         Optional[dict] = None
    icd10_cm:       Optional[dict] = None

    def key(self) -> str:
        return f"{self.span.lower()}|{self.entity_type}|{self.relation}"


# ---- verification 1: the schema is total and self-consistent ---------------
assert len(ENTITY_TYPES) == 10, "expected 10 entity types"
assert len(RELATION_SET) == 6,  "expected 6 relations"
assert len(ENTITY_SECTION_MAP) == 60, f"expected 60 routing cells, got {len(ENTITY_SECTION_MAP)}"
for t in ENTITY_TYPES:
    for r in RELATION_SET:
        sec = route_section(t, r)
        assert sec in (S, O, A, P), f"({t},{r}) -> {sec!r} is not a SOAP section"
# every FAMILY / PAST cell must be Subjective - history of any kind is subjective
for t in ENTITY_TYPES:
    assert route_section(t, "FAMILY") == S, f"{t}+FAMILY should route to S"
    assert route_section(t, "PAST")   == S, f"{t}+PAST should route to S"
# the relation, not the span, decides Medication placement
assert route_section("Medication", "PRESENT") == S
assert route_section("Medication", "ORDERED") == P
assert route_section("Disease/Problem", "PRESENT") == A

print(f"schema OK - {len(ENTITY_TYPES)} types x {len(RELATION_SET)} relations "
      f"= {len(ENTITY_SECTION_MAP)} routing cells, all valid")
print("section distribution:", dict(Counter(ENTITY_SECTION_MAP.values())))

## 3. LLM backend

`_llm` / `_parse_json` / `_resolve_openai_key` are ported from `audio2soap.ipynb`. `_llm` is
widened to dispatch across OpenAI and local HF pipelines, since Stage B needs sampling
(`temperature`) for the self-consistency vote and Stage A/D need near-deterministic output.

In [ ]:
# ============================================================================
# Cell 6 - LLM backends.  _parse_json / _resolve_openai_key ported verbatim
#          from audio2soap.ipynb; _llm widened to dispatch across backends.
# ============================================================================
_OPENAI_CLIENT = None
_HF_PIPE = None
_HF_PIPE_HOLDER = {}   # so train_all_models() can release it between models

# ---------------------------------------------------------------------------
# PASTE THE OPENAI KEY HERE  <-- the one thing this notebook cannot fetch itself
#
# Kaggle secrets are attached PER NOTEBOOK. Nishant creates a fresh notebook for
# every run, so a secret added last time is simply not there, and the run dies at
# preflight. The HF token is hardcoded for exactly this reason; this is the same
# decision for the same reason.
#
# Fill this in ONCE in the .ipynb file on disk and it travels with every upload.
# A Kaggle secret named OPENAI_API_KEY still wins if one happens to be attached,
# so rotating the key later needs no code change.
# ---------------------------------------------------------------------------
OPENAI_KEY_FALLBACK = ""   # removed: put OPENAI_API_KEY in .env (never hardcode keys)


def _resolve_openai_key():
    """env var -> Kaggle secret -> the hardcoded fallback above."""
    key = (os.environ.get("OPENAI_API_KEY") or "").strip()
    src = "env OPENAI_API_KEY"
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = (UserSecretsClient().get_secret("OPENAI_API_KEY") or "").strip()
            src = "kaggle secret"
        except Exception:
            key = ""
    if not key:
        key, src = OPENAI_KEY_FALLBACK.strip(), "notebook fallback"
    if key and not _KEY_SRC.get("printed"):
        _KEY_SRC["printed"] = True
        print(f"[openai] key from {src} (...{key[-4:]})")
    if key:
        os.environ["OPENAI_API_KEY"] = key
    return key


_KEY_SRC = {}


_CLIENT_LOCK = __import__("threading").Lock()

def _openai():
    """Thread-safe singleton. Built without a lock, several record workers hit
    this at once and some raised 'No OPENAI_API_KEY' spuriously."""
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is not None:
        return _OPENAI_CLIENT
    with _CLIENT_LOCK:
        if _OPENAI_CLIENT is not None:
            return _OPENAI_CLIENT
        from openai import OpenAI
        key = _resolve_openai_key()
        if not key:
            raise RuntimeError(
                "No OPENAI_API_KEY. Put OPENAI_API_KEY=sk-... in the .env file next to this notebook."
            )
        _OPENAI_CLIENT = OpenAI(api_key=key)
    return _OPENAI_CLIENT


def _pipeline_spec():
    """Model for Stages A / B / D + judge. Always EXTRACTION_MODEL, never the model
    under test - otherwise each model would train on a different entity set and the
    comparison would measure extraction, not generation."""
    key = EXTRACTION_MODEL if EXTRACTION_MODEL in MODEL_REGISTRY else MODEL
    if MODEL_REGISTRY[key]["kind"] == "plm":
        key = MODEL
    return MODEL_REGISTRY[key], key


_HF_CACHE = {}          # (model_key, finetuned) -> pipeline

def _hf_pipe_for(key, finetuned=False):
    ck = (key, bool(finetuned))
    if ck not in _HF_CACHE:
        _HF_CACHE[ck] = _load_hf_pipeline(key, finetuned=finetuned)
        _HF_PIPE_HOLDER[ck] = _HF_CACHE[ck]
    return _HF_CACHE[ck]


def _llm(messages, max_tokens=2048, temperature=0.15, model=None):
    """Stages A / B / D and the judges.

    Deliberately uses the BASE checkpoint, never the fine-tuned one:
      * price - a fine-tuned gpt-4o-mini costs 2x base per token, and Stage B in
        llm_evidence mode is ~80 calls per record, so it doubles the whole bill
      * quality - the fine-tune was trained to emit SOAP notes; asking it for a
        JSON entity list is off-distribution
    Stage C is the only place fine-tuned weights belong - see _llm_generate().
    """
    spec, active = _pipeline_spec()
    if spec["kind"] == "openai":
        r = _with_retry(_openai().chat.completions.create,
                        model=model or spec["id"], messages=messages,
                        max_tokens=max_tokens, temperature=temperature, top_p=0.9)
        return r.choices[0].message.content or ""
    out = _hf_pipe_for(active, finetuned=False)(
        messages, max_new_tokens=max_tokens, do_sample=temperature > 0,
        temperature=max(temperature, 1e-5), top_p=0.9, repetition_penalty=1.15)
    return out[0]["generated_text"][-1]["content"]


def _llm_generate(messages, max_tokens=1800, temperature=0.15):
    """Stage C only. Uses MODEL with its FINE-TUNED weights when present.

    Kept separate from _llm on purpose: _llm routes to EXTRACTION_MODEL, so if
    generation went through it every model would emit notes written by GPT and
    the results table would be identical across models.
    """
    spec = MODEL_REGISTRY[MODEL]
    if spec["kind"] == "openai":
        mid = spec["ft"] if (USE_FINETUNED and spec.get("ft")) else spec["id"]
        r = _with_retry(_openai().chat.completions.create,
                        model=mid, messages=messages,
                        max_tokens=max_tokens, temperature=temperature, top_p=0.9)
        return r.choices[0].message.content or ""
    out = _hf_pipe_for(MODEL, finetuned=USE_FINETUNED)(
        messages, max_new_tokens=max_tokens, do_sample=temperature > 0,
        temperature=max(temperature, 1e-5), top_p=0.9, repetition_penalty=1.15)
    return out[0]["generated_text"][-1]["content"]


def _parse_json(raw, expect="list"):
    """Tolerant JSON extraction from LLM output. Ported verbatim from audio2soap.ipynb."""
    for fence in ("```json", "```"):
        if fence in raw:
            raw = raw.split(fence)[1].split("```")[0].strip()
            break
    search_order = [("{", "}"), ("[", "]")] if expect == "dict" else [("[", "]"), ("{", "}")]
    for opener, closer in search_order:
        start = raw.find(opener)
        if start == -1:
            continue
        depth, end = 0, -1
        for i, ch in enumerate(raw[start:], start):
            if ch == opener:
                depth += 1
            elif ch == closer:
                depth -= 1
            if depth == 0:
                end = i + 1
                break
        if end != -1:
            raw = raw[start:end]
            break
        if opener == "[" and start != -1:
            last_brace = raw.rfind("}")
            if last_brace > start:
                truncated = raw[start:last_brace + 1] + "]"
                try:
                    if isinstance(json.loads(truncated, strict=False), list):
                        raw = truncated
                        break
                except Exception:
                    last_comma = truncated.rfind(",")
                    if last_comma > start:
                        truncated2 = truncated[:last_comma] + "]"
                        try:
                            if isinstance(json.loads(truncated2, strict=False), list):
                                raw = truncated2
                                break
                        except Exception:
                            pass
    raw = re.sub(r",\s*([}\]])", r"\1", raw)
    _ESC = {"\n": "\\n", "\r": "\\r", "\t": "\\t"}
    result, in_str, esc_next = [], False, False
    for ch in raw:
        if esc_next:
            result.append(ch); esc_next = False; continue
        if ch == "\\":
            result.append(ch); esc_next = in_str; continue
        if ch == '"':
            in_str = not in_str
        if in_str and ch in _ESC:
            result.append(_ESC[ch]); continue
        result.append(ch)
    cleaned = "".join(result)
    try:
        parsed = json.loads(cleaned, strict=False)
    except Exception:
        for m in re.finditer(r'[\[{]', cleaned):
            try:
                parsed = json.loads(cleaned[m.start():], strict=False)
                if expect == "list" and isinstance(parsed, list):
                    return parsed
                if expect == "dict" and isinstance(parsed, dict):
                    return parsed
            except Exception:
                continue
        return [] if expect == "list" else {}
    if expect == "list":
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            for v in parsed.values():
                if isinstance(v, list) and v and isinstance(v[0], dict):
                    return v
        return []
    return parsed if isinstance(parsed, dict) else {}


_spec, _active = _pipeline_spec()
print(f"LLM backend ready: generator={MODEL} ({MODEL_REGISTRY[MODEL]['kind']})"
      + (f" | stages A/B/D run on {_active}" if _active != MODEL else ""))

---
## Stage 0 — Audio front end

**This is the real entry point.** Stages A–D consume a *speaker-labelled transcript*; Stage 0
is what produces one from a recording.

```
audio ──▶ Whisper large-v3-turbo ──▶ flat ASR text (no speaker labels)
                                          │
                                          ▼
                              diarize_asr_text()   LLM, 1-shot, adds DOCTOR:/PATIENT:
                                          │
                                          ▼
                             speaker-labelled transcript ──▶ Stage A
   from the same audio, in parallel:
     word_confidence()        faster-whisper large-v3 int8, per-word probability
     link_entity_confidence() attaches asr_confidence to every entity
     forced_align()           NeMo CTC (torchaudio fallback) - OFF by default
```

Diarization here is **LLM-based, not pyannote**: flat ASR text goes in, the model assigns
DOCTOR/PATIENT turns from conversational cues. That is what `audio2soap.ipynb` does and it is
ported as-is.

`INPUT_MODE="text"` skips all of this and reads the transcripts directly — which is what the
100-record evaluation uses, because there is no audio for those records.

**ASR confidence is a first-class signal**, per design doc §2 ("reliability of transcript
words — weight or filter uncertain spans"). Every entity carries `asr_confidence`; anything
below `ASR_CONF_MIN` is marked `low_confidence` and surfaced in the report. Nothing is
silently dropped — a mis-heard word should be visible, not deleted.

In [ ]:
# ============================================================================
# Cell S0.1 - ASR.  Whisper large-v3-turbo, ported from audio2soap.ipynb.
# ============================================================================
NEMO_CTC_MODEL = "stt_en_fastconformer_ctc_large"   # forced alignment
_ASR   = {"pipe": None}
_FW    = {"model": None}
_NEMO  = {"model": None}
_ALIGN_ENGINE = None


def _to_wav16(audio_path):
    """16 kHz mono wav via librosa/soundfile. Ported verbatim from audio2soap.ipynb."""
    import librosa, soundfile as sf, tempfile
    y, _ = librosa.load(audio_path, sr=16000, mono=True)
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(tmp, y, 16000)
    return tmp


def _load_asr():
    if _ASR["pipe"] is None:
        import torch
        from transformers import (AutoModelForSpeechSeq2Seq, AutoProcessor,
                                  pipeline as hf_pipeline)
        dev   = "cuda:0" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        print(f"[ASR] loading {WHISPER_MODEL} on {dev} ...")
        m = AutoModelForSpeechSeq2Seq.from_pretrained(
            WHISPER_MODEL, torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True)
        m.to(dev)
        proc = AutoProcessor.from_pretrained(WHISPER_MODEL)
        _ASR["pipe"] = hf_pipeline(
            "automatic-speech-recognition", model=m, return_timestamps=True,
            tokenizer=proc.tokenizer, feature_extractor=proc.feature_extractor,
            torch_dtype=dtype, device=dev)
    return _ASR["pipe"]


def transcribe_audio(audio_path: str) -> str:
    """audio -> flat ASR text (no speaker labels yet)."""
    t0 = time.time()
    out = _load_asr()(audio_path)
    text = (out.get("text", "") if isinstance(out, dict) else str(out)).strip()
    print(f"[ASR] {len(text)} chars in {time.time()-t0:.1f}s")
    if not text:
        raise RuntimeError("ASR produced an empty transcript.")
    return text


print("Stage 0 ASR ready |", WHISPER_MODEL)

In [ ]:
# ============================================================================
# Cell S0.2 - LLM diarization.  Prompt ported verbatim from audio2soap.ipynb.
#             Flat ASR text in -> DOCTOR:/PATIENT: labelled transcript out.
# ============================================================================
_DIAR_SYSTEM = ("You are an advanced AI model trained to analyze and structure the "
                "medical conversations between a doctor and a patient.")

_DIAR_USER = textwrap.dedent("""
You are given a two party conversation transcript between a Doctor and a Patient that was
generated by an ASR engine. The transcript does not contain speaker labels. Your task is to
correctly assign speaker roles as Doctor or Patient based on the context.

Guidelines for identifying the speaker of a given sentence:
1. Doctor's Role:
   * Asks diagnostic questions, gathers medical history, and provides medical insights.
   * Uses structured questioning (e.g., "What brought you in today?", "Have you
     experienced this before?").
   * Explains findings, gives instructions, orders tests, prescribes treatment.
2. Patient's Role:
   * Describes symptoms, answers the doctor's questions, shares personal health information.
   * Expresses concerns, uncertainty, or discomfort.

Rules:
- Preserve the wording EXACTLY. Do not paraphrase, correct, summarise or add content.
- Every line must begin with "DOCTOR:" or "PATIENT:".
- Start a new line each time the speaker changes.
- Merge consecutive sentences by the same speaker into one turn.

Return ONLY a JSON array with a single object: [{"text": "<the labelled transcript>"}]

TRANSCRIPT:
%s
""").strip()

_DIAR_EXAMPLE_IN = ("Good morning what brings you in today I've been having this cough for "
                    "about two weeks now is it a dry cough or are you bringing anything up "
                    "it's mostly dry especially at night")
_DIAR_EXAMPLE_OUT = json.dumps([{"text":
    "DOCTOR: Good morning, what brings you in today?\n"
    "PATIENT: I've been having this cough for about two weeks now.\n"
    "DOCTOR: Is it a dry cough or are you bringing anything up?\n"
    "PATIENT: It's mostly dry, especially at night."}])


def _diar_join(text: str) -> str:
    """Merge consecutive same-speaker turns and normalise the labels."""
    lines, out = [l.strip() for l in (text or "").split("\n") if l.strip()], []
    for ln in lines:
        m = re.match(r'^\**\s*(DOCTOR|PATIENT|DR|PT)\b\s*\**\s*[:\-]\s*(.*)$', ln, re.I)
        if not m:
            if out:
                out[-1][1] += " " + ln
            continue
        spk = "DOCTOR" if m.group(1).upper() in ("DOCTOR", "DR") else "PATIENT"
        utt = m.group(2).strip()
        if out and out[-1][0] == spk:
            out[-1][1] += " " + utt
        else:
            out.append([spk, utt])
    return "\n".join(f"{s}: {u}".strip() for s, u in out if u)


def diarize_asr_text(asr_text: str) -> str:
    """Flat ASR text -> speaker-labelled transcript. 1-shot, as in audio2soap.ipynb."""
    t0 = time.time()
    raw = _llm(
        [{"role": "system",    "content": _DIAR_SYSTEM},
         {"role": "user",      "content": _DIAR_USER % _DIAR_EXAMPLE_IN},
         {"role": "assistant", "content": _DIAR_EXAMPLE_OUT},
         {"role": "user",      "content": _DIAR_USER % asr_text}],
        max_tokens=4096, temperature=0.2, model=DIARIZE_MODEL,
    )
    parsed = _parse_json(raw, expect="list")
    text = ""
    if parsed and isinstance(parsed[0], dict):
        text = parsed[0].get("text", "") or ""
    if not text.strip():                       # model ignored the JSON envelope
        text = raw
    text = _diar_join(text)
    if not text.strip():
        raise RuntimeError("Diarization produced no labelled turns.")
    n_turns = len(text.split("\n"))
    print(f"[diar] {n_turns} turns in {time.time()-t0:.1f}s")
    return text


print("Stage 0 diarization ready (LLM-based, not pyannote)")

In [ ]:
# ============================================================================
# Cell S0.3 - word-level ASR confidence + optional forced alignment.
#             Ported from audio2soap.ipynb (wx_word_confidence, nemo_align_*).
# ============================================================================
def word_confidence(audio_path):
    """-> [{word, start, end, confidence}] via faster-whisper word timestamps."""
    from faster_whisper import WhisperModel
    import torch
    if _FW["model"] is None:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[faster-whisper] loading {FW_MODEL} (int8) on {dev} ...")
        _FW["model"] = WhisperModel(FW_MODEL, device=dev, compute_type="int8")
    segments, _ = _FW["model"].transcribe(audio_path, word_timestamps=True)
    words = [{"word": w.word, "start": w.start, "end": w.end,
              "confidence": round(float(w.probability), 4)}
             for seg in segments for w in (seg.words or [])]
    print(f"[faster-whisper] {len(words)} words with confidence")
    return words


def link_entity_confidence(entities: List[VerifiedEntity], words) -> List[VerifiedEntity]:
    """Mean per-word ASR confidence onto each entity. Extended from audio2soap.ipynb
    to also set the low_confidence flag (design doc sec 2: flag uncertain spans)."""
    wlist = [(re.sub(r"[^a-z0-9']", "", w["word"].lower()), w.get("confidence"))
             for w in words]
    for e in entities:
        toks  = set(re.findall(r"[a-z0-9']+", e.span.lower()))
        confs = [c for (wt, c) in wlist if wt and wt in toks and c is not None]
        if confs:
            e.asr_confidence = round(sum(confs) / len(confs), 4)
            e.low_confidence = e.asr_confidence < ASR_CONF_MIN
    return entities


def forced_align(audio_path, transcript=None):
    """CTC forced alignment -> ([{word,start,end,score}], text).
    NeMo first, torchaudio wav2vec2 MMS_FA fallback. OFF unless RUN_NEMO_ALIGN."""
    global _ALIGN_ENGINE
    try:
        import nemo.collections.asr as nemo_asr
        if _NEMO["model"] is None:
            print(f"[align] loading {NEMO_CTC_MODEL} ...")
            _NEMO["model"] = nemo_asr.models.ASRModel.from_pretrained(NEMO_CTC_MODEL)
        wav = _to_wav16(audio_path)
        hyp = _NEMO["model"].transcribe([wav], return_hypotheses=True)[0]
        _ALIGN_ENGINE = "NeMo CTC"
        words = [{"word": w, "start": None, "end": None, "score": None}
                 for w in (hyp.text or "").split()]
        return words, (hyp.text or "")
    except Exception as ex:
        print(f"[align] NeMo unavailable ({str(ex)[:60]}) -> torchaudio CTC aligner")
        try:
            import torch, torchaudio
            bundle = torchaudio.pipelines.MMS_FA
            model  = bundle.get_model()
            wav    = _to_wav16(audio_path)
            wf, sr = torchaudio.load(wav)
            with torch.inference_mode():
                emission, _ = model(wf)
            _ALIGN_ENGINE = "torchaudio CTC (wav2vec2 MMS_FA)"
            return [], (transcript or "")
        except Exception as ex2:
            print(f"[align] torchaudio fallback also failed ({str(ex2)[:60]})")
            _ALIGN_ENGINE = None
            return [], (transcript or "")


print(f"Stage 0 confidence ready | word_conf={RUN_WORD_CONF} align={RUN_NEMO_ALIGN}")

In [ ]:
# ============================================================================
# Cell S0.4 - severity / nature.  Ported from audio2soap.ipynb.
#             Orthogonal to `relation`: relation says WHETHER, nature says HOW BAD.
# ============================================================================
_NATURE_PROMPT = textwrap.dedent("""
For each clinical entity below, classify its SEVERITY/NATURE from the transcript as
exactly one of: mild, moderate, severe, unspecified.

Use "unspecified" when the transcript gives no severity information. Do not guess.

TRANSCRIPT:
{transcript}

ENTITIES:
{entities}

Return ONLY a JSON array of objects: [{"span": "...", "nature": "..."}]
""").strip()


def annotate_entity_nature(entities: List[VerifiedEntity], transcript: str):
    """Adds e.nature in {mild, moderate, severe, unspecified}. One LLM call total."""
    if not entities or not RUN_NATURE:
        for e in entities:
            e.nature = e.nature or "unspecified"
        return entities
    listing = "\n".join(f"- {e.span} [{e.entity_type}]" for e in entities)
    raw = _llm(
        [{"role": "system", "content": "Respond ONLY with a JSON array. No other text."},
         {"role": "user", "content": _NATURE_PROMPT
             .replace("{transcript}", transcript[:6000])
             .replace("{entities}", listing)}],
        max_tokens=800, temperature=0.0,
    )
    by_span = {}
    for it in _parse_json(raw, expect="list"):
        if isinstance(it, dict) and it.get("span"):
            n = str(it.get("nature", "")).strip().lower()
            by_span[str(it["span"]).strip().lower()] = (
                n if n in ("mild", "moderate", "severe", "unspecified") else "unspecified")
    for e in entities:
        e.nature = by_span.get(e.span.strip().lower(), "unspecified")
    return entities


print("Stage 0 severity annotation ready")

In [ ]:
# ============================================================================
# Cell S0.5 - audio -> transcript, the whole front end in one call.
# ============================================================================
def audio_to_transcript(audio_path: str, rid: str = ""):
    """-> (labelled_transcript, words, meta). Checkpointed like every other stage."""
    cached = ck_load("stage0", rid) if rid else None
    if cached is not None:
        return cached["transcript"], cached.get("words", []), cached.get("meta", {})

    t0 = time.time()
    asr_text   = transcribe_audio(audio_path)
    t_asr      = time.time() - t0

    t1 = time.time()
    transcript = diarize_asr_text(asr_text)
    t_diar     = time.time() - t1

    words = []
    if RUN_WORD_CONF:
        try:
            words = word_confidence(audio_path)
        except Exception as ex:
            print(f"[warn] word confidence unavailable ({type(ex).__name__}: {ex}) - "
                  f"entities will have asr_confidence=None")

    align_text = ""
    if RUN_NEMO_ALIGN:
        _w, align_text = forced_align(audio_path, transcript)

    meta = {
        "audio_path": audio_path, "asr_model": WHISPER_MODEL,
        "asr_seconds": round(t_asr, 2), "diar_seconds": round(t_diar, 2),
        "n_words": len(words), "align_engine": _ALIGN_ENGINE,
        "mean_confidence": (round(sum(w["confidence"] for w in words) / len(words), 4)
                            if words else None),
        "raw_asr_text": asr_text,
    }
    if rid:
        ck_save("stage0", rid, {"transcript": transcript, "words": words, "meta": meta})
    return transcript, words, meta


print("audio_to_transcript() ready - Stage 0 complete")

---
## Stage A — K-span extraction

*Algorithm 1 line 1.* `E_raw <- NER+RE(X)`, each `e = (span, entity_type, relation, turn_id, speaker)`.

`_NER_PROMPT_V2` extends the proven `_NER_PROMPT` from `audio2soap.ipynb` from 4 labels to 10
entity types and from 4 statuses to 6 relations. Four things are kept **verbatim** because they
were tuned and work: the exact-substring rule, the pertinent-negatives rule, the
"medication with dosage as one entity" rule, and the 15-25 entities/encounter calibration.

The **exact-substring invariant** is the one everything downstream depends on — offset
resolution, grounding, hallucination detection all assume it.

In [ ]:
# ============================================================================
# Cell 7 - Stage A prompt.  Extends _NER_PROMPT from audio2soap.ipynb
#          (4 labels -> 10 types, 4 statuses -> 6 relations).
# ============================================================================
_NER_PROMPT_V2 = textwrap.dedent("""
You are a clinical Named Entity Recognition engine. Extract ALL clinically
relevant medical entities from the transcript below.

ENTITY TYPES (use exactly one per entity):
  Disease/Problem       named diagnoses, conditions, disorders
  Symptom               patient-reported complaints
  Medication            drugs, with dose/route/frequency when contiguous
  Procedure/Treatment   therapeutic or surgical interventions
  Test/Investigation    diagnostic studies ordered or performed
  Lab/Measurement       quantitative values, vital signs
  Clinical Finding      examiner-observed signs
  Anatomical Site       body locations qualifying another entity
  Allergy               allergies and intolerances
  Social/Family History smoking, alcohol, occupation, relatives' conditions

RELATIONS (use exactly one per entity):
  PRESENT   asserted as currently true of the patient
  ABSENT    explicitly denied or ruled out (pertinent negative)
  FAMILY    true of a relative, NOT the patient
  PAST      resolved, historical, no longer active
  UNKNOWN   uncertain, hedged, or not specified
  ORDERED   the clinician is prescribing/ordering it going forward

RULES:
1. "span" MUST be an EXACT SUBSTRING copied verbatim from the transcript.
   Copy the words exactly as they appear. Do NOT combine or rephrase.
2. Extract PERTINENT NEGATIVES: when the patient denies a symptom
   (e.g., "Any fevers? No."), extract it with relation "ABSENT".
3. Extract vital signs and numeric results as Lab/Measurement.
   Extract examiner-observed signs as Clinical Finding.
4. Extract medications WITH dosage as a single entity when they appear
   together (e.g., "Lisinopril 20 milligrams daily" as one entity).
5. Use ORDERED - not PRESENT - when the DOCTOR is starting, prescribing or
   ordering something. "I'll start you on amoxicillin" -> Medication/ORDERED.
   "I've been taking amoxicillin" -> Medication/PRESENT. This distinction
   decides whether the entity lands in Plan or in Subjective.
6. Extract allergies as Allergy. "No known drug allergies" -> Allergy/ABSENT.
7. For family-member conditions use relation FAMILY, not PRESENT.
8. Use UNKNOWN for hedged statements ("might be viral", "not sure if I've had it").
9. Do NOT extract conversational filler, time phrases, or dosage fragments
   without a drug name. Bad: "six weeks ago", "twice a day", "20 milligrams".
10. Be thorough: 15-25 entities for a full encounter.

Return ONLY a JSON array. Each object: {"span": ..., "entity_type": ..., "relation": ...}

EXAMPLE 1:
Transcript: "Do you have chest pain? Yes, sharp pain. Any fevers? No."
Output:
[
  {"span": "chest pain", "entity_type": "Symptom", "relation": "PRESENT"},
  {"span": "sharp pain", "entity_type": "Symptom", "relation": "PRESENT"},
  {"span": "fevers", "entity_type": "Symptom", "relation": "ABSENT"}
]

EXAMPLE 2:
Transcript: "I take Metformin 500mg twice daily. My mother had breast cancer.
I'll order a CBC and start you on amoxicillin. Blood pressure is 148 over 92."
Output:
[
  {"span": "Metformin 500mg twice daily", "entity_type": "Medication", "relation": "PRESENT"},
  {"span": "breast cancer", "entity_type": "Disease/Problem", "relation": "FAMILY"},
  {"span": "CBC", "entity_type": "Test/Investigation", "relation": "ORDERED"},
  {"span": "amoxicillin", "entity_type": "Medication", "relation": "ORDERED"},
  {"span": "148 over 92", "entity_type": "Lab/Measurement", "relation": "PRESENT"}
]

TRANSCRIPT:
{transcript}

Output:
""").strip()
print("Stage A prompt ready |", len(_NER_PROMPT_V2), "chars")

In [ ]:
# ============================================================================
# Cell 8 - Stage A rule layer.  Ports _triple_negation / _self_negated from
#          audio2soap.ipynb, retargeted onto RELATION_SET.
# ============================================================================
_FAMILY         = re.compile(r"\b(mother|father|mom|dad|sister|brother|son|daughter|"
                             r"grandmother|grandfather|aunt|uncle|cousin|family history|"
                             r"runs in (?:the|my) family|parent)\b", re.I)
_HISTORICAL     = re.compile(r"\b(used to|history of|previously|in the past|years ago|"
                             r"back in|when I was|had (?:my|a).{0,20}(?:removed|out)|"
                             r"no longer|since resolved)\b", re.I)
_NEG_PRE        = re.compile(r"\b(no|not|denies|denied|without|negative for|never|"
                             r"free of|ruled out|absent)\s*$", re.I)
_NEG_POST       = re.compile(r"^\s*[:\-,]?\s*(no\b|none\b|negative\b|nope\b)", re.I)
# A denial may sit in the NEXT speaker turn, so allow one optional "SPEAKER:" prefix.
_SPK            = r"(?:\s*\n?\s*[A-Z][A-Za-z _-]{0,24}\s*:\s*)?"
_PATIENT_DENIAL = re.compile(rf"^\s*[?.,]?{_SPK}\s*(no|nope|none|not really|never|negative)\b", re.I)
# Q&A pertinent negative: doctor asks "Any X?", patient answers "No".
# The single most common way a negative is recorded in a real transcript, and the
# reason a purely pre-span negation window misses it.
_QA_DENIAL      = re.compile(rf"^[^.?!]{{0,25}}\?{_SPK}\s*(no|nope|none|not really|never|"
                             rf"negative|haven'?t|hasn'?t|don'?t|doesn'?t)\b", re.I)
_SELF_NEG_RE    = re.compile(r"^(no|not|without|denies|negative for)\s+", re.I)
_ORDER_CUE      = re.compile(r"\b(I'?ll (?:order|start|get|prescribe|send)|let'?s (?:get|do|start|order)|"
                             r"we'?ll (?:get|do|start|order)|I'?m going to (?:order|start|prescribe)|"
                             r"I want to (?:order|get|start)|going to put you on|start(?:ing)? you on|"
                             r"prescribe|refer(?:ral)? to|follow up in|come back in)\b", re.I)
_HEDGE          = re.compile(r"\b(might|maybe|possibly|probably|could be|not sure|"
                             r"unclear|suspect|likely|I think|perhaps)\b", re.I)


def _self_negated(span_text: str) -> bool:
    """True when the span carries its own negation ("no allergies"). Ported verbatim."""
    t = (span_text or "").strip()
    if not _SELF_NEG_RE.match(t):
        return False
    # "no" must be a whole word, so "nodule"/"normal"/"nonspecific" are safe
    return bool(_SELF_NEG_RE.sub("", t).strip())   # a bare "no" is not an entity


def assign_relation(span, transcript, llm_relation, entity_type, offset=None):
    """Deterministic override of the LLM's relation where the pattern is unambiguous.

    Extends _triple_negation from audio2soap.ipynb: same layered-negation voting,
    but retargeted onto RELATION_SET and with an ORDERED detector added, since
    ORDERED is what routes an entity into Plan rather than Subjective.
    """
    t_lower, e_lower = transcript.lower(), span.lower().strip()
    idx = offset if offset is not None and offset >= 0 else t_lower.find(e_lower)
    if idx == -1:
        return llm_relation if llm_relation in RELATION_SET else "PRESENT"

    pre  = t_lower[max(0, idx - 60):idx]
    post = t_lower[idx + len(e_lower): idx + len(e_lower) + 60]
    sent_start = max(t_lower.rfind(".", 0, idx), t_lower.rfind("?", 0, idx)) + 1
    sent_end   = t_lower.find(".", idx + len(e_lower))
    sent_end   = sent_end if sent_end != -1 else min(len(t_lower), idx + len(e_lower) + 100)
    sentence   = t_lower[sent_start:sent_end]
    fam_window = t_lower[max(0, idx - 40): idx + len(e_lower) + 40]

    # a self-negating span ("no known drug allergies") is ABSENT regardless of the LLM
    if _self_negated(span):
        return "ABSENT"
    # an explicit answer to a direct question is the strongest signal available -
    # it outranks the family/historical windows, which only look at surrounding words
    if _QA_DENIAL.search(post):
        return "ABSENT"
    if _FAMILY.search(fam_window) and entity_type in (
            "Disease/Problem", "Symptom", "Social/Family History"):
        return "FAMILY"
    # ORDERED must be tested before HISTORICAL: "let's start you on X" is forward-looking
    if _ORDER_CUE.search(pre[-70:]) and entity_type in (
            "Medication", "Test/Investigation", "Procedure/Treatment"):
        return "ORDERED"
    if _HISTORICAL.search(sentence):
        return "PAST"

    if entity_type == "Medication":
        if re.search(r"used\s+to\s+(?:be\s+on|take)", pre):
            return "PAST"
        if re.search(r"ran\s+out", sentence):
            return "PRESENT"
        if _NEG_PRE.search(pre):
            return "ABSENT"
        return llm_relation if llm_relation in RELATION_SET else "PRESENT"

    # three independent negation signals; two agreeing outvote the LLM
    layer1 = bool(_NEG_PRE.search(pre))
    layer2 = bool(_NEG_POST.search(post)) or bool(_PATIENT_DENIAL.search(post))
    layer3 = (llm_relation == "ABSENT")
    if sum([layer1, layer2, layer3]) >= 2 or layer1:
        return "ABSENT"
    if _HEDGE.search(sentence) and llm_relation not in ("ABSENT", "ORDERED"):
        return "UNKNOWN"
    return llm_relation if llm_relation in RELATION_SET else "PRESENT"


print("Stage A rule layer ready")

In [ ]:
# ============================================================================
# Cell 9 - extract_kspans().  Ports run_ner_spell + _enforce_schema from
#          audio2soap.ipynb, adding char-offset resolution and turn/speaker
#          attribution (Algorithm 1 line 1 carries turn_id and speaker).
# ============================================================================
_TURN_RE = re.compile(r"^\s*([A-Z][A-Za-z _-]{0,24})\s*:\s*(.*)$")

def split_turns(transcript: str):
    """-> [(turn_idx, speaker, utterance, char_offset_of_utterance)]"""
    turns, pos = [], 0
    for i, line in enumerate(transcript.split("\n")):
        m = _TURN_RE.match(line)
        if m:
            speaker, utt = m.group(1).strip().upper(), m.group(2)
            turns.append((len(turns), speaker, utt, pos + line.index(utt) if utt else pos))
        elif line.strip():
            turns.append((len(turns), "", line.strip(), pos))
        pos += len(line) + 1
    return turns


def _locate(span: str, transcript: str, used: set):
    """First unclaimed exact-substring occurrence. Returns (start, end) or (-1, -1).

    Case-insensitive fallback, but the STORED span is always re-sliced from the
    transcript so VerifiedEntity.span is a byte-exact substring either way.
    """
    for hay, needle in ((transcript, span), (transcript.lower(), span.lower())):
        start = hay.find(needle)
        while start != -1:
            if not any(start < e and start + len(needle) > s for s, e in used):
                return start, start + len(needle)
            start = hay.find(needle, start + 1)
    return -1, -1


def extract_kspans(transcript: str, rid: str = "") -> List[VerifiedEntity]:
    """Stage A. Returns entities with exact-substring spans and resolved offsets."""
    cached = ck_load("stageA", rid) if rid else None
    if cached is not None:
        return [VerifiedEntity(**e) for e in cached]

    raw = _llm(
        [{"role": "system", "content": "Respond ONLY with a JSON array. No other text."},
         {"role": "user", "content": _NER_PROMPT_V2.replace("{transcript}", transcript[:12000])}],
        max_tokens=3000, temperature=0.1,
    )
    items = _parse_json(raw, expect="list")

    turns = split_turns(transcript)
    used, out = set(), []
    for it in items:
        if not isinstance(it, dict):
            continue
        span = str(it.get("span") or it.get("text") or "").strip()
        if not span or len(span) < 2:
            continue

        etype = str(it.get("entity_type") or it.get("label") or "").strip()
        etype = LEGACY_LABEL_MAP.get(etype, etype)
        if etype not in ENTITY_TYPES:
            continue                                   # drop off-schema types

        rel = str(it.get("relation") or it.get("status") or "PRESENT").strip().upper()
        rel = LEGACY_STATUS_MAP.get(rel.title().replace(" ", "_"), rel)
        if rel not in RELATION_SET:
            rel = "PRESENT"

        start, end = _locate(span, transcript, used)
        if start == -1:
            continue                                   # INVARIANT: no exact match -> drop
        used.add((start, end))
        span = transcript[start:end]                   # re-slice: guarantees exact substring

        rel = assign_relation(span, transcript, rel, etype, offset=start)

        turn_id, speaker = -1, ""
        for ti, spk, _utt, toff in turns:
            if toff <= start:
                turn_id, speaker = ti, spk
            else:
                break

        out.append(VerifiedEntity(
            span=span, entity_type=etype, relation=rel,
            section=route_section(etype, rel),
            source_turn=turn_id, source_speaker=speaker,
            char_start=start, char_end=end,
        ))

    if rid:
        ck_save("stageA", rid, [asdict(e) for e in out])
    return out


print("extract_kspans() ready")

---
## Stage B — UMLS verification and linking

*Algorithm 1 lines 2-21.* This is the contribution. **UMLS appears zero times in the previous
codebase**, so everything here is new surface.

VerifiNER's three steps, adapted per design doc section 4:

| Step | What | Departure from the paper |
|---|---|---|
| 2.1 | expand span +/- `SPAN_ALPHA` tokens, enumerate sub-spans | none |
| 2.2 | top-k CUI retrieval | **embedding**, not string lookup — "tummy ache" will never string-match "Abdominal Pain" |
| 2.3 | LLM evidence bridging UMLS definition -> type | also reassigns **relation**, which the paper does not do |
| 2.4 | self-consistency vote when >1 candidate survives | none |

`KB_BACKEND="auto"` tries scispaCy's UMLS linker and falls back **loudly** to the ICD-10 +
SNOMED index from `audio2soap.ipynb`. A dependency failure then costs grounding quality
rather than the whole run.

In [ ]:
# ============================================================================
# Cell 10 - KB interface + two backends.
#   ScispacyUMLS : UMLS 2020AA subset, ~3M concepts, CUI + definition + TUI.
#   IcdSnomedKB  : fallback wrapping _retrieve_rerank/get_snomed from audio2soap.
# ============================================================================
@dataclass
class KBCandidate:
    cui: str; preferred_term: str; semantic_type: str
    definition: str = ""; score: float = 0.0


class KB:
    """link(span) -> List[KBCandidate], best first."""
    name = "base"
    def link(self, span: str, k: int = 5) -> List[KBCandidate]:
        raise NotImplementedError


class UmlsJsonlKB(KB):
    """UMLS without scispaCy, sized to fit in Kaggle RAM.

    The first version built a TfidfVectorizer over every alias of every concept.
    On the 59 MB test slice (154k concepts) that was fine; on the full 630 MB KB
    it is ~1.7M concepts x 8 aliases = ~6.5M strings, and the fitted vocabulary
    plus the sparse matrix OOM-ed a Kaggle kernel.

    Three changes keep memory flat:
      * HashingVectorizer - no vocabulary dict to fit or store, fixed n_features
      * fewer aliases per concept (KB_MAX_ALIASES)
      * a hard cap on concepts (KB_MAX_CONCEPTS), filtered to routable types first
    """
    name = "umls-jsonl"

    def __init__(self, tuis=None):
        import urllib.request
        from sklearn.feature_extraction.text import HashingVectorizer

        cache = ARTIFACTS / "umls_kb.jsonl"
        if not cache.exists():
            print("[UMLS] downloading knowledge base (~630 MB, one time) ...")
            last = [0]
            def _hook(b, bs, total):
                pct = min(100, int(b * bs * 100 / max(total, 1)))
                if pct >= last[0] + 20:
                    last[0] = pct; print(f"       {pct}%", flush=True)
            urllib.request.urlretrieve(UMLS_KB_URL, cache, reporthook=_hook)

        keep = set(tuis or SEMANTIC_TYPE_TO_ENTITY.keys())
        self.concepts, texts, owner = [], [], []
        skipped = 0
        with open(cache) as fh:
            for line in fh:
                if len(self.concepts) >= KB_MAX_CONCEPTS:
                    skipped += 1
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                if keep and not (set(d.get("types", [])) & keep):
                    continue                    # cannot be routed to any section
                idx = len(self.concepts)
                self.concepts.append((d["concept_id"], d.get("canonical_name", ""),
                                      ",".join(d.get("types", [])),
                                      (d.get("definition") or "")[:400]))
                for s in [d.get("canonical_name", "")] + \
                         list(d.get("aliases", []))[:KB_MAX_ALIASES]:
                    if s:
                        texts.append(s.lower()); owner.append(idx)

        self.owner = owner
        # stateless + fixed width: memory does not grow with vocabulary size
        self.vec = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 3),
                                     n_features=2 ** 18, alternate_sign=False,
                                     norm="l2", dtype="float32")
        self.matrix = self.vec.transform(texts)
        mb = (self.matrix.data.nbytes + self.matrix.indices.nbytes
              + self.matrix.indptr.nbytes) / 2**20
        print(f"[UMLS] {len(self.concepts):,} concepts | {len(texts):,} surface forms "
              f"| index {mb:,.0f} MiB"
              + (f" | {skipped:,} lines skipped at KB_MAX_CONCEPTS" if skipped else ""))
        del texts

    def link(self, span, k=5):
        from sklearn.metrics.pairwise import linear_kernel
        sims = linear_kernel(self.vec.transform([span.lower()]), self.matrix).ravel()
        if not sims.size:
            return []
        top = sims.argsort()[::-1][: k * 6]
        seen, out = set(), []
        for i in top:
            score = float(sims[i])
            if score < KB_SIM_THRESHOLD:
                break
            cui, name, types, defn = self.concepts[self.owner[i]]
            if cui in seen:
                continue
            seen.add(cui)
            out.append(KBCandidate(cui=cui, preferred_term=name, semantic_type=types,
                                   definition=defn, score=round(score, 4)))
            if len(out) >= k:
                break
        return out


class ScispacyUMLS(KB):
    name = "scispacy-umls"
    def __init__(self, k=30, threshold=0.5):
        import spacy
        from scispacy.linking import EntityLinker
        self.nlp = spacy.load("en_core_sci_sm")
        self.nlp.add_pipe("scispacy_linker", config={
            "resolve_abbreviations": True, "linker_name": "umls",
            "max_entities_per_mention": k, "threshold": threshold,
        })
        self.linker = self.nlp.get_pipe("scispacy_linker")
        print(f"  scispaCy UMLS linker up | {len(self.linker.kb.cui_to_entity):,} concepts")

    def link(self, span, k=5):
        doc = self.nlp(span)
        seen, out = set(), []
        # a span may produce several mentions; pool their candidates and rank globally
        for ent in doc.ents:
            for cui, score in ent._.kb_ents[:k]:
                if cui in seen:
                    continue
                seen.add(cui)
                e = self.linker.kb.cui_to_entity[cui]
                out.append(KBCandidate(
                    cui=cui, preferred_term=e.canonical_name,
                    semantic_type=", ".join(e.types),
                    definition=(e.definition or "")[:600], score=float(score),
                ))
        out.sort(key=lambda c: -c.score)
        return out[:k]


class IcdSnomedKB(KB):
    """Fallback. Reuses the bi-encoder + ICD-10 index shape from audio2soap.ipynb.

    Far smaller than UMLS - this weakens grounding and the notebook says so loudly.
    """
    name = "icd-snomed-fallback"
    def __init__(self):
        from sentence_transformers import SentenceTransformer, util as st_util
        self.util = st_util
        self.embedder = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")
        self.terms = _fallback_terms()
        self.emb = self.embedder.encode([t["term"] for t in self.terms],
                                        convert_to_tensor=True, show_progress_bar=False)
        print(f"  fallback KB up | {len(self.terms)} terms (vs ~3M in UMLS)")

    def link(self, span, k=5):
        q = self.embedder.encode(span, convert_to_tensor=True)
        hits = self.util.semantic_search(q, self.emb, top_k=k)[0]
        out = []
        for h in hits:
            t = self.terms[h["corpus_id"]]
            out.append(KBCandidate(
                cui=t["id"], preferred_term=t["term"], semantic_type=t["sty"],
                definition=t.get("definition", ""), score=float(h["score"]),
            ))
        return out


def _fallback_terms():
    """Minimal clinical vocabulary for the fallback path."""
    rows = [
        ("422587007", "Nausea", "Sign or Symptom"),
        ("422400008", "Vomiting", "Sign or Symptom"),
        ("25064002",  "Headache", "Sign or Symptom"),
        ("29857009",  "Chest pain", "Sign or Symptom"),
        ("386661006", "Fever", "Sign or Symptom"),
        ("43724002",  "Chills", "Sign or Symptom"),
        ("267036007", "Dyspnea", "Sign or Symptom"),
        ("21522001",  "Abdominal pain", "Sign or Symptom"),
        ("7180009",   "Meningitis", "Disease or Syndrome"),
        ("44054006",  "Type 2 diabetes mellitus", "Disease or Syndrome"),
        ("38341003",  "Hypertension", "Disease or Syndrome"),
        ("195967001", "Asthma", "Disease or Syndrome"),
        ("13645005",  "COPD", "Disease or Syndrome"),
        ("386013003", "Lisinopril", "Pharmacologic Substance"),
        ("372567009", "Metformin", "Pharmacologic Substance"),
        ("372687004", "Amoxicillin", "Pharmacologic Substance"),
        ("387207008", "Ibuprofen", "Pharmacologic Substance"),
        ("387517004", "Paracetamol", "Pharmacologic Substance"),
        ("399208008", "Chest X-ray", "Diagnostic Procedure"),
        ("277762005", "Complete blood count", "Laboratory Procedure"),
        ("277302009", "Lumbar puncture", "Diagnostic Procedure"),
        ("43396009",  "Hemoglobin A1c measurement", "Laboratory Procedure"),
        ("75367002",  "Blood pressure", "Clinical Attribute"),
        ("364075005", "Heart rate", "Clinical Attribute"),
        ("91936005",  "Penicillin allergy", "Pathologic Function"),
        ("77176002",  "Smoker", "Individual Behavior"),
        ("228273003", "Alcohol intake", "Individual Behavior"),
        ("80313002",  "Nuchal rigidity", "Finding"),
        ("48348007",  "Respiratory crackles", "Finding"),
        ("52101004",  "Present", "Finding"),
    ]
    return [{"id": f"S{i}:{r[0]}", "term": r[1], "sty": r[2]} for i, r in enumerate(rows)]


def build_kb() -> KB:
    """umls_jsonl is the default: real UMLS, no scispaCy, no nmslib, no numpy downgrade."""
    order = {"auto": ["umls_jsonl", "icd_snomed"],
             "umls_jsonl": ["umls_jsonl"],
             "scispacy": ["scispacy"],
             "icd_snomed": ["icd_snomed"]}[KB_BACKEND]
    for name in order:
        try:
            if name == "umls_jsonl":
                return UmlsJsonlKB()
            if name == "scispacy":
                return ScispacyUMLS(k=max(RETRIEVE_K * 4, 20),
                                    threshold=KB_SIM_THRESHOLD - 0.05)
            return IcdSnomedKB()
        except Exception as ex:
            if name == order[-1]:
                raise
            print("!" * 74)
            print(f"!  KB backend '{name}' unavailable: {type(ex).__name__}: {str(ex)[:90]}")
            print(f"!  falling back to '{order[order.index(name)+1]}'")
            if order[order.index(name) + 1] == "icd_snomed":
                print("!  That fallback is TINY. Grounding will be much worse - do not")
                print("!  report those as UMLS results. Check kb_health_report().")
            print("!" * 74)
    return IcdSnomedKB()


KB_INSTANCE = build_kb()
print("KB backend:", KB_INSTANCE.name)

In [ ]:
# ============================================================================
# Cell 11 - VerifiNER step 2.1: candidate span generation.
#   Expand the Stage-A span by +/- SPAN_ALPHA tokens and enumerate sub-spans.
#   VerifiNER's error analysis: most span errors are off by only 1-2 tokens.
# ============================================================================
def expand_candidate_spans(span: str, transcript: str, start: int, end: int,
                           alpha: int = None) -> List[Tuple[str, int, int]]:
    """-> [(candidate_span, start, end)], original first, deduped, all exact substrings."""
    alpha = SPAN_ALPHA if alpha is None else alpha
    if start < 0:
        return [(span, -1, -1)]

    # walk out whole words on each side, up to alpha of them
    left_ctx  = transcript[max(0, start - 60):start]
    right_ctx = transcript[end:end + 60]

    def word_starts(text, n):
        outs, i = [], len(text)
        while len(outs) < n and i > 0:
            j = text.rfind(" ", 0, i - 1)
            if j == -1:
                outs.append(0); break
            outs.append(j + 1); i = j + 1 if j + 1 < i else j
        return outs
    def word_ends(text, n):
        outs, i = [], 0
        while len(outs) < n and i < len(text):
            j = text.find(" ", i + 1)
            if j == -1:
                outs.append(len(text)); break
            outs.append(j); i = j + 1
        return outs

    l_offsets = [0] + [-(len(left_ctx) - s) for s in word_starts(left_ctx, alpha)]
    r_offsets = [0] + word_ends(right_ctx, alpha)

    seen, cands = set(), []
    for lo in sorted(set(l_offsets)):
        for ro in sorted(set(r_offsets)):
            s2, e2 = start + lo, end + ro
            if s2 < 0 or e2 > len(transcript) or s2 >= e2:
                continue
            cand = transcript[s2:e2].strip()
            if not cand or len(cand) > 120 or cand.lower() in seen:
                continue
            seen.add(cand.lower())
            cands.append((cand, s2, e2))

    # original span always first so it wins ties in the vote
    cands.sort(key=lambda c: (c[0].lower() != span.lower(), abs(len(c[0]) - len(span))))
    return cands[:12]


print("step 2.1 ready | SPAN_ALPHA =", SPAN_ALPHA)

In [ ]:
# ============================================================================
# Cell 12 - VerifiNER steps 2.3 + 2.4: evidence generation, type/relation
#           reassignment, and the self-consistency vote.
#   Departure from the paper (design doc sec 4, step 2.3): the evidence passage
#   bridges the UMLS definition to the entity type AND the relation. The paper
#   only verifies type.
# ============================================================================
_EVIDENCE_PROMPT = textwrap.dedent("""
You are verifying a clinical entity against a UMLS concept.

TRANSCRIPT CONTEXT:
{context}

CANDIDATE SPAN     : "{span}"
PROPOSED TYPE      : {etype}
PROPOSED RELATION  : {relation}

RETRIEVED UMLS CONCEPT:
  CUI            : {cui}
  Preferred term : {term}
  Semantic type  : {sty}
  Definition     : {definition}

Task:
1. Write ONE sentence of evidence stating whether the UMLS concept's definition and
   semantic type support the proposed ENTITY TYPE for this span in this context.
2. Decide the correct entity type from: {types}
   If the concept does not support ANY of them for this span, answer "NONE".
3. Decide the correct relation from: {relations}
   Base this on the TRANSCRIPT CONTEXT, not the UMLS definition. Use ORDERED if the
   clinician is prescribing/ordering it, FAMILY if it belongs to a relative, ABSENT if
   it is denied, PAST if resolved, UNKNOWN if hedged.

Return ONLY JSON:
{"evidence": "...", "entity_type": "...", "relation": "...", "supports": true/false}
""").strip()


def _local_window(transcript: str, start: int, end: int, chars: int = 320) -> str:
    if start < 0:
        return transcript[:chars]
    return transcript[max(0, start - chars // 2): min(len(transcript), end + chars // 2)]


# Spans repeat heavily across an encounter and across records ("blood pressure",
# "no known drug allergies"), so memoising the evidence call is the cheapest large win.
# Keyed on everything that changes the answer except the surrounding context, which
# only matters for relation - so cache entries carry the relation they were built for.
_EVIDENCE_CACHE: Dict[tuple, dict] = {}

def _evidence_cache_path():
    return ARTIFACTS / "evidence_cache.json"

def load_evidence_cache():
    p = _evidence_cache_path()
    if CACHE_EVIDENCE and p.exists():
        try:
            raw = json.loads(p.read_text())
            _EVIDENCE_CACHE.update({tuple(json.loads(k)): v for k, v in raw.items()})
            print(f"  evidence cache: {len(_EVIDENCE_CACHE):,} entries loaded")
        except Exception:
            pass
    return _EVIDENCE_CACHE

_CACHE_LOCK = __import__("threading").Lock()

def save_evidence_cache():
    """Snapshot under a lock first - several record workers mutate this dict
    concurrently, and iterating it while another thread writes raised
    'dictionary changed size during iteration' on ~40% of records."""
    if not (CACHE_EVIDENCE and _EVIDENCE_CACHE):
        return
    with _CACHE_LOCK:
        snap = dict(_EVIDENCE_CACHE)
    try:
        _evidence_cache_path().write_text(json.dumps(
            {json.dumps(list(k)): v for k, v in snap.items()}))
    except Exception:
        pass                      # a cache write must never fail a record


def generate_evidence(span, etype, relation, cand: "KBCandidate", context,
                      temperature=0.1) -> dict:
    """VerifiNER step 2.3. -> {evidence, entity_type, relation, supports}"""
    # only the deterministic pass is cacheable; the vote deliberately re-samples
    ck = (span.lower(), cand.cui, etype, relation) if temperature <= 0.1 else None
    if CACHE_EVIDENCE and ck is not None and ck in _EVIDENCE_CACHE:
        return _EVIDENCE_CACHE[ck]

    prompt = (_EVIDENCE_PROMPT
              .replace("{context}", context)
              .replace("{span}", span)
              .replace("{etype}", etype)
              .replace("{relation}", relation)
              .replace("{cui}", cand.cui)
              .replace("{term}", cand.preferred_term)
              .replace("{sty}", cand.semantic_type or "(none)")
              .replace("{definition}", (cand.definition or "(no definition in KB)")[:600])
              .replace("{types}", ", ".join(ENTITY_TYPES))
              .replace("{relations}", ", ".join(RELATION_SET)))
    raw = _llm(
        [{"role": "system", "content": "Respond ONLY with a JSON object. No other text."},
         {"role": "user", "content": prompt}],
        max_tokens=300, temperature=temperature,
    )
    d = _parse_json(raw, expect="dict")
    t_hat = str(d.get("entity_type", "")).strip()
    r_hat = str(d.get("relation", "")).strip().upper()
    out = {
        "evidence":    str(d.get("evidence", "")).strip()[:400],
        "entity_type": t_hat if t_hat in ENTITY_TYPES else ("NONE" if t_hat.upper() == "NONE" else etype),
        "relation":    r_hat if r_hat in RELATION_SET else relation,
        "supports":    bool(d.get("supports", True)) and t_hat.upper() != "NONE",
    }
    if CACHE_EVIDENCE and ck is not None:
        with _CACHE_LOCK:
            _EVIDENCE_CACHE[ck] = out
    return out


def self_consistency_vote(candidates: List[dict], context: str, n_paths: int = None) -> dict:
    """VerifiNER step 2.4 / Algorithm 1 line 18.

    Re-runs evidence generation at higher temperature over the surviving candidates
    and majority-votes on (span, cui, entity_type, relation).
    """
    n_paths = SELF_CONSISTENCY_N if n_paths is None else n_paths
    if len(candidates) == 1:
        return candidates[0]

    # If every survivor already agrees on the CUI, type and relation, the vote can
    # only re-elect that answer - so skip N_PATHS calls that cannot change anything.
    if SKIP_VOTE_IF_UNANIMOUS:
        sig = {(c["kb"].cui, c["entity_type"], c["relation"]) for c in candidates}
        if len(sig) == 1:
            return candidates[0]

    votes = Counter()
    detail = {}
    for path in range(n_paths):
        c = candidates[path % len(candidates)]
        res = generate_evidence(
            c["span"], c["entity_type"], c["relation"], c["kb"], context,
            temperature=0.7,                      # genuine diversity across paths
        )
        if not res["supports"]:
            continue
        key = (c["span"], c["kb"].cui, res["entity_type"], res["relation"])
        votes[key] += 1
        detail.setdefault(key, {**c, "evidence": res["evidence"],
                                "entity_type": res["entity_type"],
                                "relation": res["relation"]})
    if not votes:
        return candidates[0]
    return detail[votes.most_common(1)[0][0]]


load_evidence_cache()   # defined just above; the KB cell runs before this one
print("steps 2.3 / 2.4 ready | SELF_CONSISTENCY_N =", SELF_CONSISTENCY_N)

In [ ]:
# ============================================================================
# Cell 13 - verify_entities().  Algorithm 1 lines 2-21, orchestrated.
# ============================================================================
# A KB that throws on every call would otherwise look identical to a KB that simply
# finds no match: every entity comes back grounded=False and the run still produces a
# full results table. Count the failures so that case announces itself.
KB_HEALTH = {"link_calls": 0, "link_errors": 0, "last_error": None}


_KB_LOCK = __import__("threading").Lock()

def verify_type_deterministic(e: VerifiedEntity, cands: List["KBCandidate"]):
    """Step 2.3 without an LLM. Returns (candidate, entity_type, note).

    1. Prefer the best-scoring candidate whose UMLS semantic type AGREES with
       Stage A - that is confirmation.
    2. If none agrees, adopt the best candidate's implied type, but only when the
       match is strong. That is the correction Stage B exists to make.
    3. Otherwise link the concept and keep Stage A's type.
    """
    # Sort defensively: correctness must not depend on a backend returning
    # candidates pre-ranked.
    scored = sorted((c for c in cands if c.score >= KB_SIM_THRESHOLD),
                    key=lambda c: -c.score)
    if not scored:
        return None, e.entity_type, ""

    for c in scored:
        if umls_implied_type(c) == e.entity_type:
            return c, e.entity_type, f"UMLS {c.cui} ({c.preferred_term}) confirms {e.entity_type}."

    best = scored[0]
    implied = umls_implied_type(best)
    if (implied and implied != e.entity_type
            and e.entity_type not in PROTECTED_TYPES
            and best.score >= TYPE_OVERRIDE_MIN_SCORE):
        return best, implied, (f"UMLS {best.cui} ({best.preferred_term}) is a "
                               f"{best.semantic_type} concept, indicating {implied} rather "
                               f"than the extracted {e.entity_type}. Type corrected.")
    return best, e.entity_type, (f"UMLS {best.cui} ({best.preferred_term}) linked; "
                                 f"type {e.entity_type} kept.")


def _verify_one(e: VerifiedEntity, transcript: str) -> VerifiedEntity:
    """Algorithm 1 lines 3-19 for a single entity. Pure w.r.t. other entities,
    which is what makes the N_WORKERS fan-out safe."""
    # design doc sec 3.2: numeric / structural types bypass Stage B
    if e.entity_type in BYPASS_UMLS:
        e.section = route_section(e.entity_type, e.relation)
        return e

    context = _local_window(transcript, e.char_start, e.char_end)

    # 2.1 candidate spans
    span_cands = expand_candidate_spans(e.span, transcript, e.char_start, e.char_end)

    def _link(span_text):
        try:
            hits = KB_INSTANCE.link(span_text, k=RETRIEVE_K)
            with _KB_LOCK:
                KB_HEALTH["link_calls"] += 1
            return hits
        except Exception as ex:
            with _KB_LOCK:
                KB_HEALTH["link_calls"] += 1
                KB_HEALTH["link_errors"] += 1
                KB_HEALTH["last_error"] = f"{type(ex).__name__}: {ex}"
            return []

    # ---- deterministic mode: no LLM calls at all -------------------------
    if VERIFY_MODE == "deterministic":
        for cand_span, cs, ce in span_cands:
            hits = _link(cand_span)
            cand, new_type, note = verify_type_deterministic(e, hits)
            if cand is None:
                continue                      # try the next candidate span
            rel = assign_relation(cand_span, transcript, e.relation, new_type, offset=cs)
            return VerifiedEntity(
                span=cand_span, entity_type=new_type, relation=rel,
                cui=cand.cui, preferred_term=cand.preferred_term,
                semantic_type=cand.semantic_type, evidence=note,
                section=route_section(new_type, rel),                  # line 21
                grounded=True,
                source_turn=e.source_turn, source_speaker=e.source_speaker,
                char_start=cs, char_end=ce,
            )
        e.grounded = False                    # line 14 - fallback, flagged
        e.section  = route_section(e.entity_type, e.relation)
        return e

    # ---- llm_evidence mode: full VerifiNER 2.3 + 2.4 ---------------------
    surviving = []
    for cand_span, cs, ce in span_cands:
        kb_hits = _link(cand_span)
        for kb in kb_hits:
            if kb.score < KB_SIM_THRESHOLD:
                continue
            res = generate_evidence(cand_span, e.entity_type, e.relation, kb, context)
            if res["supports"]:
                surviving.append({
                    "span": cand_span, "char_start": cs, "char_end": ce, "kb": kb,
                    "entity_type": res["entity_type"], "relation": res["relation"],
                    "evidence": res["evidence"],
                })
        if surviving:
            break            # first span with any support wins; original is tried first

    if not surviving:
        # line 14 - fallback, flagged. Kept, not dropped.
        e.grounded = False
        e.section  = route_section(e.entity_type, e.relation)
        return e

    best = surviving[0] if len(surviving) == 1 else self_consistency_vote(surviving, context)

    rel = assign_relation(best["span"], transcript, best["relation"],
                          best["entity_type"], offset=best["char_start"])
    return VerifiedEntity(
        span=best["span"], entity_type=best["entity_type"], relation=rel,
        cui=best["kb"].cui, preferred_term=best["kb"].preferred_term,
        semantic_type=best["kb"].semantic_type, evidence=best["evidence"],
        section=route_section(best["entity_type"], rel),          # line 21
        grounded=True,
        source_turn=e.source_turn, source_speaker=e.source_speaker,
        char_start=best["char_start"], char_end=best["char_end"],
    )


def verify_entities(entities: List[VerifiedEntity], transcript: str,
                    rid: str = "") -> List[VerifiedEntity]:
    cached = ck_load("stageB", rid) if rid else None
    if cached is not None:
        return [VerifiedEntity(**e) for e in cached]

    if N_WORKERS > 1 and len(entities) > 1:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            # .map preserves input order, so entity order is stable across runs
            out = list(ex.map(lambda e: _verify_one(e, transcript), entities))
    else:
        out = [_verify_one(e, transcript) for e in entities]

    if rid:
        # Must persist: all three arms share this Stage-B result, and it is by far
        # the most expensive stage to recompute.
        ck_save("stageB", rid, [asdict(x) for x in out])
        save_evidence_cache()

    n_grounded = sum(1 for x in out if x.grounded)
    n_eligible = sum(1 for x in out if x.entity_type not in BYPASS_UMLS)
    if n_eligible and n_grounded == 0 and KB_HEALTH["link_errors"]:
        print("!" * 74)
        print(f"!  Stage B grounded 0 of {n_eligible} eligible entities, and KB.link() "
              f"raised {KB_HEALTH['link_errors']} time(s).")
        print(f"!  last error: {KB_HEALTH['last_error']}")
        print("!  These are NOT UMLS-grounded results. Fix the KB before reporting them.")
        print("!" * 74)
    return out


def kb_health_report():
    """Call after a batch: a high error rate means the results are ungrounded."""
    c, e = KB_HEALTH["link_calls"], KB_HEALTH["link_errors"]
    rate = (e / c * 100) if c else 0.0
    print(f"KB backend  : {KB_INSTANCE.name}")
    print(f"link calls  : {c:,}   errors: {e:,}  ({rate:.1f}%)")
    if e:
        print(f"last error  : {KB_HEALTH['last_error']}")
    if rate > 5:
        print("WARNING: >5% of KB lookups failed - treat grounding metrics as unreliable.")
    return KB_HEALTH


print(f"verify_entities() ready - Algorithm 1 lines 2-21 | VERIFY_MODE={VERIFY_MODE}")
if VERIFY_MODE == "deterministic":
    print("   Stage B makes ZERO LLM calls in this mode (UMLS semantic-type check).")
    print("   Set VERIFY_MODE=\"llm_evidence\" for full VerifiNER 2.3+2.4.")

### Stage B probe

*Verification step 3.* The whiteboard's own two examples must round-trip:
`meningitis -> (Disease/Problem, PRESENT)` and `no chills -> (Symptom, ABSENT)`.
Run this before any batch job — it is cheap and it catches a broken KB immediately.

In [ ]:
# ============================================================================
# Cell 14 - Stage B probe.  The whiteboard's examples must round-trip.
# ============================================================================
def probe_stage_b(verbose=True):
    probe_transcript = (
        "DOCTOR: What brings you in today?\n"
        "PATIENT: I have had a terrible headache and a stiff neck for two days.\n"
        "DOCTOR: Any chills?\n"
        "PATIENT: No.\n"
        "DOCTOR: I am concerned this could be meningitis. I'll order a lumbar puncture "
        "and start you on ceftriaxone.\n"
    )
    probes = [("meningitis", "Disease/Problem", "PRESENT"),
              ("chills",     "Symptom",         "ABSENT")]

    ents = extract_kspans(probe_transcript, rid="")
    ver  = verify_entities(ents, probe_transcript, rid="")

    if verbose:
        print(f"{'span':<28} {'type':<22} {'rel':<9} {'sec':<4} {'CUI':<12} grounded")
        print("-" * 92)
        for e in ver:
            print(f"{e.span[:27]:<28} {e.entity_type:<22} {e.relation:<9} "
                  f"{e.section:<4} {e.cui[:11]:<12} {e.grounded}")

    print("\n--- whiteboard round-trip ---")
    ok = True
    for want_span, want_type, want_rel in probes:
        hit = next((e for e in ver if want_span in e.span.lower()), None)
        if hit is None:
            print(f"  MISS  {want_span!r} was not extracted at all")
            ok = False
            continue
        good = (hit.entity_type == want_type and hit.relation == want_rel)
        ok &= good
        print(f"  {'PASS' if good else 'FAIL'}  {want_span!r} -> "
              f"({hit.entity_type}, {hit.relation}) sec={hit.section}"
              f"{'' if good else f'   expected ({want_type}, {want_rel})'}")
    return ok

# probe_stage_b()      # <-- uncomment once an API key / KB is available
print("probe_stage_b() ready - run it before any batch job")

---
## Stage C — Fine-tuned generation

*Algorithm 1 lines 22-23.* `prompt <- serialize(X, E_verified)` grouped by predicted section
with evidence attached, then `N_draft <- M(prompt)`.

`build_verified_keyword_block` is `build_keyword_section` from `audio2soap.ipynb` upgraded to
carry CUI, preferred term and relation, grouped by the section the router predicts. Arm A
passes `grouped=False` to get the flat K-SOAP keyword list — that is the baseline.

In [ ]:
# ============================================================================
# Cell 15 - prompt construction.  Extends build_keyword_section from audio2soap.
# ============================================================================
_SECTION_NAMES = {"S": "SUBJECTIVE", "O": "OBJECTIVE", "A": "ASSESSMENT", "P": "PLAN"}

def build_verified_keyword_block(entities: List[VerifiedEntity], grouped=True,
                                 with_evidence=False) -> str:
    """The K-span block injected into the generation prompt.

    grouped=False -> flat K-SOAP keyword list (Arm A baseline).
    grouped=True  -> grouped by routed section, with CUI + relation (Arms B/C).
    """
    if not entities:
        return "(no entities extracted)"

    if not grouped:
        return "\n".join(
            f"- {e.span} [{e.entity_type}]" for e in entities
        )

    lines, by_sec = [], defaultdict(list)
    for e in entities:
        by_sec[e.section].append(e)
    for sec in ("S", "O", "A", "P"):
        if not by_sec[sec]:
            continue
        lines.append(f"{_SECTION_NAMES[sec]}:")
        for e in by_sec[sec]:
            tag = f"{e.entity_type}, {e.relation}"
            if e.grounded and e.cui:
                tag += f", UMLS:{e.cui} = {e.preferred_term}"
            # External Signals (design doc): normalised codes as a conditioning signal,
            # not just an end-of-pipeline report.
            if CODES_AS_SIGNAL:
                if getattr(e, "snomed", None):
                    tag += f", SNOMED:{e.snomed['concept_id']} = {e.snomed['term']}"
                if getattr(e, "icd10_cm", None) and e.icd10_cm.get("code"):
                    tag += f", ICD-10:{e.icd10_cm['code']}"
            lines.append(f"  - \"{e.span}\" [{tag}]")
            if with_evidence and e.evidence:
                lines.append(f"      evidence: {e.evidence}")
        lines.append("")
    return "\n".join(lines).rstrip()


# ---------------------------------------------------------------------------
# OUTPUT FORMAT.  _SECTION_HELP is ported VERBATIM from audio2soap.ipynb - this is
# the note format the project has always produced, and it must not drift.
# ---------------------------------------------------------------------------
_SECTION_HELP = {
    "S": "S: subjective. Write it as short labelled lines in this order: Chief complaint; HPI "
         "(onset, site, character, severity out of 10, radiation, what makes it better or worse, "
         "and any medication already tried and whether it helped); Pertinent negatives (every "
         "symptom the patient denied); Past medical history (the patient's age if stated, "
         "every chronic condition named, every current medication BY NAME, allergies, previous "
         "surgeries or procedures, and any past cardiac or major event with its timing - write "
         "'none reported' where the patient denied them); Family history (relation, condition "
         "and age or timing if stated); Social history (occupation or studies, living "
         "situation, alcohol, tobacco including pack-years or years smoked if stated, "
         "recreational drugs). Include any concern the patient "
         "expresses about their own diagnosis. If the transcript is internally inconsistent, "
         "record the clinically coherent version (for example episodes lasting seconds that "
         "recur over three days, rather than 'constant and lasting seconds').",
    "O": "O: objective - vital signs and physical-examination findings stated in the transcript. "
         "If the transcript contains none, write exactly: No objective findings stated.",
    "A": "A: assessment - a neutral one-or-two-line summary of the confirmed problems AS NAMED "
         "in the transcript. Do NOT add your own diagnosis or differential diagnoses. If the "
         "doctor stated no assessment, write exactly: No assessment given.",
    "P": "P: plan - what the doctor said WILL or MAY be done: investigations, tests, referrals, "
         "medications, follow-up and conditional plans. Phrases such as 'we will do some "
         "investigations', 'we may need to start you on X' or 'we will get you an ECG' ARE the "
         "plan and must be recorded (state them as planned or conditional). Include any working "
         "diagnosis the doctor said they are investigating. Do NOT invent orders the doctor never "
         "mentioned. Only if the doctor said nothing at all about next steps, write exactly: "
         "No plan given.",
}


_SOAP_PROMPT = textwrap.dedent("""
You are a clinical scribe. Convert the doctor-patient transcript into a SOAP note.
Use ONLY information explicitly stated in the transcript. Never add findings,
diagnoses, differentials or recommendations of your own.

VERIFIED CLINICAL ENTITIES (extracted and knowledge-grounded from this transcript):
{keywords}

TRANSCRIPT:
{transcript}

Output these sections, each starting on its own line with the single-letter label:
{section_help}

Rules:
- You MUST output every one of these labels: S, O, A, P.
- Keep each section brief: at most 6 short lines.
- Plain text only. No markdown, no bold, no numbered headings.
- Every entity listed above must appear in the section it is grouped under.
- An entity marked ABSENT is a pertinent negative: record that it was DENIED,
  never as though the patient has it.
- An entity marked FAMILY belongs to a relative, never to the patient.
- An entity marked ORDERED is part of the plan going forward, not current history.
- UMLS / SNOMED / ICD-10 codes are grounding for wording ONLY. Use the normalised
  term when the patient's phrasing is colloquial ("tummy ache" -> abdominal pain).
  A code is NOT permission to state a diagnosis the clinician did not give, and
  code numbers must NEVER appear in the note text.
- Do NOT write any disclaimer or "AI Suggestion" here; that is produced separately.
""").strip()


def build_soap_prompt(transcript: str, entities: List[VerifiedEntity],
                      grouped=True, with_evidence=False) -> str:
    return (_SOAP_PROMPT
            .replace("{keywords}", build_verified_keyword_block(entities, grouped, with_evidence))
            .replace("{transcript}", transcript[:12000])
            .replace("{section_help}", "\n".join(_SECTION_HELP[k] for k in ("S", "O", "A", "P"))))


# Two heading styles must both parse:
#   "S:"                    - what our generation prompt asks for
#   "**1. Subjective**"     - what the reference notes in clean_english_100.jsonl use
# A fine-tuned model learns the training data's style, so a parser that only handles
# "S:" would silently collapse every entity into section S and break the placement check.
_SEC_RE = re.compile(
    r"^[ \t]*(?:[*#]{1,3}[ \t]*)?(?:\d+[.\)][ \t]*)?\*{0,2}[ \t]*"
    r"(?:(Assessment[ \t]+(?:and|&|/)[ \t]*Plan|Subjective|Objective|Assessment|Plan)"
    r"\b[ \t]*(?:\([^)]*\))?[ \t]*\*{0,2}[ \t]*[:.\)]?"
    r"|([SOAP])[ \t]*[:.\)])[ \t]*",          # a bare letter MUST carry a delimiter,
    re.M | re.I)                               # else "A patient presented..." matches

# 99/100 reference notes in clean_english_100.jsonl use a COMBINED "Assessment and
# Plan" heading and none has a standalone Plan heading. Each problem block inside it
# does carry explicit "**Assessment:**" / "**Plan:**" sub-labels, so the two are
# recoverable - without this, every Plan entity would score as misplaced-into-A.
_AP_SUB = re.compile(r"\*{0,2}(Assessment|Plan)\*{0,2}[ \t]*:[ \t]*\*{0,2}[ \t]*", re.I)


def _split_combined_ap(body: str) -> Tuple[str, str]:
    """A combined 'Assessment and Plan' body -> (assessment_text, plan_text)."""
    marks = [(m.group(1)[0].upper(), m.end(), m.start()) for m in _AP_SUB.finditer(body)]
    if not marks:
        return body, ""                       # no sub-labels: treat it all as Assessment
    a, p = [], []
    for i, (k, st, _) in enumerate(marks):
        end = marks[i + 1][2] if i + 1 < len(marks) else len(body)
        (a if k == "A" else p).append(body[st:end].strip())
    return "\n".join(a).strip(), "\n".join(p).strip()


def _sec_key(m) -> str:
    w = m.group(1) or m.group(2)
    return "AP" if re.match(r"assessment[ \t]+(?:and|&|/)", w, re.I) else w[0].upper()


def split_soap_sections(note: str) -> Dict[str, str]:
    """-> {"S":..., "O":..., "A":..., "P":...}.  Extends _parse_ksoap_sections.

    Handles all three styles seen in this project: "S:" (what our prompt asks for),
    "**1. Subjective**" (the reference notes), and "**3. Assessment and Plan**"
    (99/100 of them).
    """
    out = {k: "" for k in ("S", "O", "A", "P")}
    marks = [(_sec_key(m), m.end(), m.start()) for m in _SEC_RE.finditer(note or "")]
    for i, (sec, st, _) in enumerate(marks):
        end = marks[i + 1][2] if i + 1 < len(marks) else len(note)
        body = (note[st:end] or "").strip()
        if sec == "AP":
            a_txt, p_txt = _split_combined_ap(body)
            out["A"] = (out["A"] + "\n" + a_txt).strip()
            out["P"] = (out["P"] + "\n" + p_txt).strip()
        else:
            out[sec] = (out[sec] + "\n" + body).strip()
    if not any(out.values()):
        out["S"] = (note or "").strip()
    return out


print("Stage C prompts ready")

In [ ]:
# ============================================================================
# Cell 16 - OpenAI fine-tuning.  Flow from fine_tune_eval_soap.ipynb, polling
#           from check_ft_eta.py.  Only runs when DO_TRAIN and kind == "openai".
# ============================================================================
def build_openai_ft_jsonl(records, out_path, grouped=True):
    """(transcript, soap) pairs -> chat-format JSONL for the OpenAI FT endpoint."""
    n = 0
    with open(out_path, "w") as f:
        for r, ents in _prep_records(records, grouped, " openai"):
            f.write(json.dumps({"messages": [
                {"role": "system", "content": "You are an experienced clinician writing SOAP notes."},
                {"role": "user",   "content": build_soap_prompt(r["transcript"], ents, grouped)},
                {"role": "assistant", "content": r["soap"]},
            ]}) + "\n")
            n += 1
    print(f"wrote {n} examples -> {out_path}")
    return out_path


def launch_openai_ft(jsonl_path, base_model=None, suffix="verisoap"):
    client = _openai()
    up = client.files.create(file=open(jsonl_path, "rb"), purpose="fine-tune")
    job = client.fine_tuning.jobs.create(
        training_file=up.id, model=base_model or MODEL_REGISTRY["gpt"]["id"], suffix=suffix,
    )
    print(f"launched {job.id} | status={job.status}")
    print(f"   reconnect later with: poll_openai_ft('{job.id}')")
    return job.id


def poll_openai_ft(job_id, wait=False, every=60):
    """Read-only status + ETA. Ported from check_ft_eta.py."""
    client = _openai()
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"status={job.status} | model={job.fine_tuned_model or '(pending)'} | "
              f"trained_tokens={getattr(job, 'trained_tokens', None) or '(not reported)'}")
        for e in reversed(client.fine_tuning.jobs.list_events(job_id, limit=5).data):
            print("   ", e.message)
        if job.status in ("succeeded", "failed", "cancelled") or not wait:
            return job
        time.sleep(every)


if DO_TRAIN and MODEL_REGISTRY[MODEL]["kind"] == "openai":
    p = build_openai_ft_jsonl(TRAIN_RECORDS, ARTIFACTS / "openai_ft_train.jsonl")
    FT_JOB_ID = launch_openai_ft(p)
else:
    print("OpenAI FT helpers ready (DO_TRAIN=False -> not launching).")
    print("Existing proven model:", MODEL_REGISTRY["gpt"]["ft"])

In [ ]:
# ============================================================================
# Cell 17 - QLoRA fine-tuning.  Config carried VERBATIM from
#           fine_tune_eval_soap.ipynb - the values that have actually trained.
#
#   NOT the design doc's r=64/alpha=16/2ep. Reason in PROVENANCE.md: changing
#   hyperparameters and architecture in the same step makes a bad result
#   unattributable. LORA_R / LORA_ALPHA are CONFIG flags if you want to A/B.
#
#   WARNING: this has NEVER been run in this project. Expect to debug OOM on a T4.
# ============================================================================
# Llama-3's own chat template. aaditya/Llama3-OpenBioLLM-8B is a Llama-3-8B
# derivative but its tokenizer_config.json omits chat_template entirely, and
# transformers 5.x raises instead of falling back:
#     ValueError: Cannot use chat template functions because tokenizer.chat_template
#                 is not set
# That killed all 150 openbio scoring records on 2026-09-21 while training, which
# uses plain concatenated text, had succeeded. Supplying the base model's template
# makes openbio receive exactly the same prompt shape as qwen and llama, which is
# what keeps the comparison fair.
_LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    " + message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)

# Last-resort shape for a non-Llama tokenizer with no template of its own.
_GENERIC_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{{ message['role'] | upper }}: {{ message['content'] }}\n\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ 'ASSISTANT: ' }}{% endif %}"
)


def _ensure_chat_template(tok, model_id=""):
    """Give a tokenizer a chat template if its repo shipped none."""
    if getattr(tok, "chat_template", None):
        return False
    is_llama3 = "llama-3" in (model_id or "").lower() or "llama3" in (model_id or "").lower()
    tok.chat_template = _LLAMA3_CHAT_TEMPLATE if is_llama3 else _GENERIC_CHAT_TEMPLATE
    print(f"[chat] {model_id or 'tokenizer'} ships no chat_template - applied the "
          f"{'Llama-3' if is_llama3 else 'generic'} one")
    return True


def _load_hf_pipeline(which=None, finetuned=None):
    """Base or adapter-merged pipeline for a causal model."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
    spec = MODEL_REGISTRY[which or MODEL]
    DT = torch.bfloat16 if torch.cuda.is_available() and \
         torch.cuda.get_device_capability()[0] >= 8 else torch.float16

    tok = AutoTokenizer.from_pretrained(spec["id"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    _ensure_chat_template(tok, spec["id"])
    # 4-bit needs bitsandbytes, which needs CUDA. Without this guard the whole
    # generation path is untestable off-GPU - which is exactly why its bugs only
    # ever showed up on Kaggle.
    _quant = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=DT,
                                 bnb_4bit_use_double_quant=True)
              if torch.cuda.is_available() else None)
    model = AutoModelForCausalLM.from_pretrained(
        spec["id"], device_map=_device_map(),
        torch_dtype=DT, quantization_config=_quant)
    _ft = USE_FINETUNED if finetuned is None else finetuned
    adapter = find_adapter(spec.get("out")) if _ft else None
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
        print(f"loaded adapter: {adapter}")
    elif _ft:
        print(f"!! fine-tuned weights requested but none at {spec.get('out')} - using BASE model.")
        print("!! Results from this run are base-model results. Do not label them fine-tuned.")
    return pipeline("text-generation", model=model, tokenizer=tok)


def _fit_vocab(model, tok, tag=""):
    """Make the embedding matrix cover every id the tokenizer can emit.

    hf-internal-testing/tiny-random-BartForConditionalGeneration ships a 1024-row
    embedding together with the FULL 50265-token BartTokenizer. Encoding a real
    sentence produces ids past the end of that matrix, and CUDA answers with

        vectorized_gather_kernel: Assertion `ind >= 0 && ind < ind_dim_size' failed

    which poisons the entire CUDA context: every later call in the session dies
    too, including torch.cuda.manual_seed_all. One root cause, a screenful of
    unrelated-looking tracebacks.

    Real checkpoints match, so this is a no-op for them. It exists so a mismatch
    is corrected instead of taking the process down.
    """
    try:
        have = model.get_input_embeddings().weight.shape[0]
        need = max(len(tok), getattr(tok, "vocab_size", 0) or 0)
        if have < need:
            print(f"  [{tag}] embedding has {have} rows but the tokenizer emits up to "
                  f"{need} - resizing to match")
            model.resize_token_embeddings(need)
            return True
    except Exception as ex:
        print(f"  [{tag}] vocab check skipped ({type(ex).__name__}: {str(ex)[:60]})")
    return False


def _gpu_free():
    """(free, total) GiB PER DEVICE. mem_get_info() reports only the CURRENT
    device, which is why a two-GPU leak looked like a one-GPU leak."""
    try:
        import torch
        if not torch.cuda.is_available():
            return []
        out = []
        for d in range(torch.cuda.device_count()):
            with torch.cuda.device(d):
                f, t = torch.cuda.mem_get_info()
            out.append((d, f / 2**30, t / 2**30))
        return out
    except Exception:
        return []


def _cuda_purge():
    """empty_cache() frees only the CURRENT device. With device_map='auto' the
    model is sharded across both T4s, so freeing device 0 alone left device 1
    holding its shard."""
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            for d in range(torch.cuda.device_count()):
                with torch.cuda.device(d):
                    torch.cuda.empty_cache()
                    torch.cuda.ipc_collect()
    except Exception:
        pass


def _teardown_trainer(trainer, *extra):
    """Drop every GPU reference a Trainer holds.

    Setting trainer.model = None was not enough. Measured 2026-09-20: free VRAM
    fell 11.2 -> 8.8 -> 6.4 -> 5.1 GiB across four trained models, and scoring
    then OOM-ed on every single record with 2.81 MiB free. What was still held:

      model_wrapped   the DataParallel wrapper - a SECOND reference to the model
      optimizer       paged_adamw_8bit state, which is the big one
      lr_scheduler    holds the optimizer
      accelerator     keeps its own _models / _optimizers lists
      AcceleratorState a MODULE-LEVEL singleton that outlives the Trainer
      callback_handler holds .model too
    """
    import gc
    try:
        acc = getattr(trainer, "accelerator", None)
        if acc is not None and hasattr(acc, "free_memory"):
            acc.free_memory()
    except Exception:
        pass
    for attr in ("model_wrapped", "model", "optimizer", "lr_scheduler",
                 "accelerator", "callback_handler", "train_dataset",
                 "data_collator", "_train_dataloader"):
        try:
            setattr(trainer, attr, None)
        except Exception:
            pass
    try:
        from accelerate.state import AcceleratorState
        AcceleratorState._reset_state(reset_partial_state=True)
    except Exception:
        try:
            from accelerate.state import AcceleratorState
            AcceleratorState._reset_state()
        except Exception:
            pass
    del trainer
    for obj in extra:
        del obj
    gc.collect()
    _cuda_purge()


def train_qlora(records, grouped=True):
    """Causal-LM fine-tuning on transformers' own Trainer.

    TRL's SFTTrainer used to do this, but TRL 0.24 against the transformers 5.x
    preview on Kaggle raised
        AttributeError: 'functools.partial' object has no attribute '__func__'
    for every causal model, while the PLM path (Seq2SeqTrainer, shipped inside
    transformers) trained fine. We were using none of TRL's actual features -
    no packing, no chat templates, no DPO - just supervised text. Dropping it
    removes a dependency and a whole class of version churn.
    """
    import torch
    from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                              Trainer, TrainingArguments, DataCollatorForLanguageModeling)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from datasets import Dataset

    spec = MODEL_REGISTRY[MODEL]
    DT = torch.bfloat16 if torch.cuda.is_available() and \
         torch.cuda.get_device_capability()[0] >= 8 else torch.float16

    rows = [{"text": build_soap_prompt(r["transcript"], ents, grouped) + "\n" + r["soap"]}
            for r, ents in _prep_records(records, grouped, " qlora")]
    if not rows:
        print("[qlora] no training rows"); return None

    tok = AutoTokenizer.from_pretrained(spec["id"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    _ensure_chat_template(tok, spec["id"])

    def _tok(b):
        out = tok(b["text"], max_length=MAX_SEQ, truncation=True)
        out["labels"] = [ids[:] for ids in out["input_ids"]]   # plain LM objective
        return out

    ds = Dataset.from_list(rows).map(_tok, batched=True, remove_columns=["text"])

    _quant = None
    if torch.cuda.is_available():
        _quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
    else:
        print("[qlora] no CUDA - loading unquantised so the path stays runnable")
    model = AutoModelForCausalLM.from_pretrained(
        spec["id"], device_map=_device_map(),
        torch_dtype=DT, quantization_config=_quant)
    if _quant is not None:
        model = prepare_model_for_kbit_training(model)
    elif hasattr(model, "enable_input_require_grads"):
        # prepare_model_for_kbit_training does this for us on GPU. Without it,
        # gradient checkpointing detaches the embedding output and the backward
        # pass dies with "element 0 of tensors does not require grad".
        model.enable_input_require_grads()
    _fit_vocab(model, tok, "qlora")
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    model.config.use_cache = False

    trainer = Trainer(
        model=model, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
        args=TrainingArguments(
            output_dir=spec["out"], num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR, weight_decay=0.01, warmup_ratio=WARMUP_RATIO,
            lr_scheduler_type="cosine",
            optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
            gradient_checkpointing=True,
            # non-reentrant checkpointing keeps the autograd graph intact for
            # LoRA params; the reentrant default silently drops their grads
            gradient_checkpointing_kwargs={"use_reentrant": False},
            save_total_limit=1, logging_steps=5, report_to="none",
        ),
    )
    try:
        hist = trainer.train()
        trainer.save_model(spec["out"])
        tok.save_pretrained(spec["out"])
        if getattr(hist, "training_loss", None) is not None:
            LAST_TRAIN_LOSS[MODEL] = float(hist.training_loss)
            print(f"[qlora] final training loss {hist.training_loss:.3f}")
    finally:
        # An exception keeps its traceback, the traceback keeps this frame, and
        # this frame holds `model` and `trainer` - so an OOM here leaked the whole
        # 4-bit model into the NEXT model's training. Measured on 2026-09-15:
        # free VRAM fell 14.3 -> 12.3 -> 10.0 GiB across two failures, and openbio
        # then trained at 3 min/step under the pressure. Dropping the references
        # inside a finally: runs before the traceback is built.
        _teardown_trainer(trainer, model)
        for _d, _f, _t in _gpu_free():
            print(f"   [gpu{_d}] {_f:.1f} GiB free of {_t:.1f} GiB after teardown")
    for ck in glob.glob(os.path.join(spec["out"], "checkpoint-*")):
        shutil.rmtree(ck, ignore_errors=True)
    print("adapter saved ->", spec["out"])
    return spec["out"]


if DO_TRAIN and MODEL_REGISTRY[MODEL]["kind"] == "qlora":
    train_qlora(TRAIN_RECORDS)
else:
    print(f"QLoRA helpers ready (DO_TRAIN=False). "
          f"r={LORA_R} alpha={LORA_ALPHA} epochs={NUM_EPOCHS} accum={GRAD_ACCUM}")

### PLM fine-tuning (sir's Task 1)

BART and BioBART are **encoder-decoder** models with a 1024-token encoder, so a full
transcript does not fit. That is why the paper reports section-wise modes, and why
`Bart-section-4` (ROUGE-L 0.459) beats `Bart-full` (0.330).

Stage B gives us something the paper's baselines lacked: the verified entity block is a
compressed, clinically complete view of the conversation, so a short encoder still sees
what matters. Entities are placed **before** the transcript so they survive truncation.

In [ ]:
# ============================================================================
# Cell 17b - PLM (encoder-decoder) fine-tuning.  Sir's Task 1, and the K-SOAP
#            paper's own baselines: BART-large and BioBART.
#
# Why this needs its own path, not the QLoRA one:
#   BART is seq2seq (AutoModelForSeq2SeqLM), not causal, and its encoder holds
#   only 1024 tokens. Our transcripts run ~2,000 tokens, so a whole conversation
#   does NOT fit. That is exactly why the paper reports section-wise modes and
#   why Bart-section-4 (ROUGE-L 0.459) beats Bart-full (0.330) so decisively.
#
#   Stage B helps here in a way the paper's baselines did not have: the verified
#   entity block is a compressed, clinically-complete view of the transcript, so
#   a 1024-token encoder can see the content that matters even when the raw
#   conversation would overflow it.
# ============================================================================
_PLM = {}          # section -> (model, tokenizer);  "full" for whole-note mode
_PLM_SECTIONS = ("S", "O", "A", "P")


def _plm_input(transcript: str, entities, section: str = None) -> str:
    """Encoder input. Entities first - they survive truncation, the transcript may not."""
    ents = [e for e in entities if section is None or e.section == section] or entities
    block = build_verified_keyword_block(ents, grouped=(section is None))
    head = f"section: {section}\n" if section else ""
    return f"{head}entities:\n{block}\n\ntranscript:\n{transcript}"


def _plm_paths(spec):
    if PLM_MODE == "full":
        return {"full": spec["out"]}
    return {s: f"{spec['out']}-{s}" for s in _PLM_SECTIONS}


def _load_plm(section="full"):
    """Lazy-load a fine-tuned PLM, falling back to the base checkpoint."""
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import torch
    if section in _PLM:
        return _PLM[section]
    spec = MODEL_REGISTRY[MODEL]
    path = _plm_paths(spec).get(section, spec["out"])
    found = find_adapter(path) if USE_FINETUNED else None
    src = found or spec["id"]
    if src == spec["id"] and USE_FINETUNED:
        print(f"!! no fine-tuned PLM at {path} - using BASE {spec['id']}. "
              f"Do not label these results fine-tuned.")
    tok = AutoTokenizer.from_pretrained(src)
    if src != spec["id"] and os.path.exists(os.path.join(src, "adapter_config.json")):
        from peft import PeftModel
        mdl = PeftModel.from_pretrained(
            AutoModelForSeq2SeqLM.from_pretrained(spec["id"]), src)
    else:
        mdl = AutoModelForSeq2SeqLM.from_pretrained(src)
    mdl.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    _PLM[section] = (mdl, tok)
    return _PLM[section]


def generate_plm(transcript: str, entities) -> str:
    """PLM note generation. section-4 produces each section separately, then joins."""
    import torch
    sections = ["full"] if PLM_MODE == "full" else list(_PLM_SECTIONS)
    out = {}
    for sec in sections:
        mdl, tok = _load_plm(sec)
        text = _plm_input(transcript, entities, None if sec == "full" else sec)
        enc = tok(text, return_tensors="pt", truncation=True,
                  max_length=PLM_MAX_IN).to(mdl.device)
        with torch.no_grad():
            ids = mdl.generate(**enc, max_new_tokens=PLM_MAX_OUT,
                               num_beams=4, no_repeat_ngram_size=3)
        out[sec] = tok.decode(ids[0], skip_special_tokens=True).strip()
    if PLM_MODE == "full":
        return out["full"]
    _EMPTY = {"O": "No objective findings stated.", "A": "No assessment given.",
              "P": "No plan given.", "S": "No subjective information stated."}
    return "\n".join(f"{s}: {out[s] or _EMPTY[s]}" for s in _PLM_SECTIONS)


import random as _rnd
from concurrent.futures import ThreadPoolExecutor as _TPE, as_completed as _as_done

_RL = {"retries": 0}

def _with_retry(fn, *a, **kw):
    """Backoff on rate limits and transient network errors.

    Without this a single 429 killed a worker thread outright, and 32 concurrent
    calls will hit rate limits on most OpenAI tiers.
    """
    for attempt in range(6):
        try:
            return fn(*a, **kw)
        except Exception as ex:
            m = str(ex).lower()
            if attempt == 5 or not any(k in m for k in (
                    "rate limit", "429", "timeout", "timed out", "connection",
                    "overloaded", "503", "502", "temporarily")):
                raise
            _RL["retries"] += 1
            time.sleep(min(2 ** attempt, 30) + _rnd.random())


def _prep_records(records, grouped=True, label=""):
    """Stage A + B over a record list, RECORD_WORKERS records at a time.

    This is the slow phase - roughly a minute per record - so it reports
    progress, and every record is checkpointed individually, meaning an
    interrupted run resumes instead of restarting.
    """
    n, t0 = len(records), time.time()
    results, done = [None] * n, 0

    def one(i, r):
        ents = extract_kspans(r["transcript"], rid=str(r["id"]))
        if grouped:
            ents = verify_entities(ents, r["transcript"], rid=str(r["id"]))
        return i, r, ents

    with _TPE(max_workers=max(1, RECORD_WORKERS)) as pool:
        futs = {pool.submit(one, i, r): i for i, r in enumerate(records)}
        for fut in _as_done(futs):
            done += 1
            try:
                i, r, ents = fut.result()
                results[i] = (r, ents)
            except Exception as ex:
                print(f"    [prep{label}] record {futs[fut]} FAILED "
                      f"({type(ex).__name__}: {str(ex)[:70]})", flush=True)
            if done == 1 or done % 10 == 0 or done == n:
                el = time.time() - t0
                print(f"    [prep{label}] {done}/{n}  {el/60:.1f} min elapsed  "
                      f"~{(el/done)*(n-done)/60:.0f} min left"
                      + (f"  [{_RL['retries']} retries]" if _RL["retries"] else ""),
                      flush=True)
    return [x for x in results if x]


def train_plm(records, grouped=True):
    """Fine-tune a PLM. section-4 trains one model PER SECTION, as the paper does.

    Targets come from split_soap_sections(reference), which already handles the
    combined 'Assessment and Plan' heading that 99/100 reference notes use.
    """
    from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                              Seq2SeqTrainer, Seq2SeqTrainingArguments,
                              DataCollatorForSeq2Seq)
    from datasets import Dataset

    spec = MODEL_REGISTRY[MODEL]
    if spec["kind"] != "plm":
        raise ValueError(f"{MODEL} is kind={spec['kind']}, not a PLM.")

    # entities once per record, reused across all four section models
    prepared = [(r, ents, split_soap_sections(r["soap"]))
                for r, ents in _prep_records(records, grouped, " plm")]

    targets = _plm_paths(spec)
    for sec, out_dir in targets.items():
        rows = []
        for r, ents, secs in prepared:
            tgt = r["soap"] if sec == "full" else secs.get(sec, "")
            if not tgt.strip():
                continue                    # this note omits the section - skip it
            rows.append({"src": _plm_input(r["transcript"], ents,
                                           None if sec == "full" else sec),
                         "tgt": tgt})
        if not rows:
            print(f"[PLM] section {sec}: no training rows, skipped")
            continue

        tok = AutoTokenizer.from_pretrained(spec["id"])
        mdl = AutoModelForSeq2SeqLM.from_pretrained(spec["id"])
        _fit_vocab(mdl, tok, "PLM")
        if PLM_USE_LORA:
            from peft import LoraConfig, get_peft_model
            mdl = get_peft_model(mdl, LoraConfig(
                r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                bias="none", task_type="SEQ_2_SEQ_LM",
                target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]))
            mdl.print_trainable_parameters()

        def _tok(b):
            x = tok(b["src"], max_length=PLM_MAX_IN, truncation=True)
            y = tok(text_target=b["tgt"], max_length=PLM_MAX_OUT, truncation=True)
            x["labels"] = y["input_ids"]
            return x

        ds = Dataset.from_list(rows).map(_tok, batched=True,
                                         remove_columns=["src", "tgt"])
        trainer = Seq2SeqTrainer(
            model=mdl, train_dataset=ds,
            data_collator=DataCollatorForSeq2Seq(tok, model=mdl),
            args=Seq2SeqTrainingArguments(
                output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
                per_device_train_batch_size=2, gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=LR, warmup_ratio=WARMUP_RATIO, weight_decay=0.01,
                lr_scheduler_type="cosine", save_total_limit=1,
                predict_with_generate=False, logging_steps=5, report_to="none",
            ),
        )
        print(f"[PLM] training section {sec}: {len(rows)} examples -> {out_dir}")
        try:
            hist = trainer.train()
            trainer.save_model(out_dir)
            tok.save_pretrained(out_dir)
            loss = getattr(hist, "training_loss", None)
            if loss is not None:
                LAST_TRAIN_LOSS[f"{MODEL}:{sec}"] = float(loss)
                print(f"[PLM] section {sec} final training loss {loss:.3f}")
        finally:
            # Four section models load in sequence; without a full teardown the
            # previous one is still resident when the next is built.
            _teardown_trainer(trainer, mdl)
        for ck in glob.glob(os.path.join(out_dir, "checkpoint-*")):
            shutil.rmtree(ck, ignore_errors=True)   # Trainer keeps a full copy
    print("PLM training done ->", list(targets.values()))
    return targets


if DO_TRAIN and MODEL_REGISTRY[MODEL]["kind"] == "plm":
    train_plm(TRAIN_RECORDS)
else:
    print(f"PLM helpers ready | mode={PLM_MODE} | in={PLM_MAX_IN} out={PLM_MAX_OUT} tok")
    print("   PLM models in the registry:",
          [k for k, v in MODEL_REGISTRY.items() if v["kind"] == "plm"])

In [ ]:
# ============================================================================
# Cell 18 - generate_soap().  Algorithm 1 lines 22-23.
# ============================================================================
def _mk_for(model_key: str, stage: str) -> str:
    """The checkpoint key for ANY model, not just the current global MODEL.

    _mk() reads the global, which is right inside the generation path but wrong
    for arm_done()/_collect_arm(), which are handed a model_key. Those two used
    to build the key by hand and so missed the SCORING_VERSION prefix entirely:
    run_batch wrote <model>__v2__metrics_<arm>, they looked for
    <model>__metrics_<arm>, and every arm reported zero results. One function,
    used everywhere, so the two can never drift again.
    """
    if stage == "stageD" or stage.startswith(("metrics_", "note_")):
        return f"{model_key}__v{SCORING_VERSION}__{stage}"
    return f"{model_key}__{stage}"


def _mk(stage: str) -> str:
    """Namespace a checkpoint stage by the CURRENT generator model.

    Stage A and Stage B are deliberately NOT namespaced: the front end is held
    constant across the whole results table, so the verified entities are computed
    once and reused by every model. That is the correct experiment and it is where
    all the OpenAI money goes.

    Everything downstream of the generator MUST be namespaced. Without this, the
    first model scored writes gen*/stageD/metrics_* checkpoints under a bare key,
    and every later model loads them back - so all seven models report byte
    identical notes and identical metrics. The results table looks plausible and
    is entirely wrong.
    """
    # SCORING_VERSION invalidates only what a method change actually affects.
    # v2 (2026-09-20): reflection now revises single sections instead of the whole
    # note, and both judges got written level anchors. Generations from arms A and
    # B are untouched by either change, so they keep their unversioned keys and
    # are NOT recomputed - only stageD, the notes and the metrics are.
    return _mk_for(MODEL, stage)


def generate_soap(transcript: str, entities: List[VerifiedEntity],
                  grouped=True, with_evidence=False, rid="", tag="") -> str:
    ck = _mk(f"gen{tag}")
    cached = ck_load(ck, rid) if rid else None
    if cached is not None:
        return cached["note"]

    if MODEL_REGISTRY[MODEL]["kind"] == "plm":
        # encoder-decoder path: no chat prompt, and in section-4 mode the note is
        # produced one section at a time (paper sec 6.3).
        # One GPU and one cached model per section, so record workers queue here.
        with _GPU_LOCK:
            note = generate_plm(transcript, entities)
    elif MODEL_REGISTRY[MODEL]["kind"] != "openai":
        with _GPU_LOCK:                      # local causal model, same reason
            note = _llm_generate(
                [{"role": "system",
                  "content": "You are an experienced clinician writing SOAP notes."},
                 {"role": "user", "content": build_soap_prompt(transcript, entities,
                                                               grouped, with_evidence)}],
                max_tokens=1800, temperature=0.15,
            )
    else:
        note = _llm_generate(
            [{"role": "system",
              "content": "You are an experienced clinician writing SOAP notes."},
             {"role": "user", "content": build_soap_prompt(transcript, entities,
                                                           grouped, with_evidence)}],
            max_tokens=1800, temperature=0.15,
        )
    note = (note or "").strip()
    if rid:
        ck_save(ck, rid, {"note": note})
    return note


print("generate_soap() ready")

---
## Stage D — Evidence-guided self-verification

*Algorithm 1 lines 24-39.* The whiteboard's three reflection checks become four, because
"3. Additional" splits into completeness and relation consistency.

| # | Check | Algorithm 1 | Possible without Stage B? |
|---|---|---|---|
| 1 | Completeness | lines 25-28 | no — needs `E_verified` |
| 2 | Placement | lines 29-30 | no — needs `ENTITY_SECTION_MAP` |
| 3 | Hallucination | line 31 | no — needs `grounded_spans(E_verified)` as the set to subtract |
| 4 | Relation consistency | design doc 5.2 | no |

This is what makes reflection *evidence-guided* rather than a yes/no checklist: the checks run
in **code** and their structured findings are fed back in as evidence, instead of asking the
model to rediscover its own mistakes unaided. Only the affected section is regenerated
(line 37).

In [ ]:
# ============================================================================
# Cell 19 - the four Stage-D checks.
# ============================================================================
def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())

def _mentions(hay: str, needle: str) -> bool:
    """Token-overlap containment - tolerant of inflection and reordering."""
    h, n = _norm(hay), _norm(needle)
    if not n.strip():
        return False
    if n.strip() in h:
        return True
    toks = [t for t in n.split() if len(t) > 3]
    if not toks:
        return n.strip() in h
    return sum(1 for t in toks if t in h) / len(toks) >= 0.7


def locate_entity_in_note(e: VerifiedEntity, note: str):
    """-> (present, actual_section). Algorithm 1 line 26."""
    secs = split_soap_sections(note)
    surface = [e.span] + ([e.preferred_term] if e.preferred_term else [])
    for sec in ("S", "O", "A", "P"):
        if any(_mentions(secs[sec], s) for s in surface):
            return True, sec
    return False, None


def check_completeness(entities, note):
    """Check 1 - Algorithm 1 lines 25-28."""
    missing = []
    for e in entities:
        if not e.grounded:
            continue
        present, _ = locate_entity_in_note(e, note)
        if not present:
            missing.append(e)
    return missing


def check_placement(entities, note):
    """Check 2 - Algorithm 1 lines 29-30. -> [(entity, expected, actual)]"""
    bad = []
    for e in entities:
        present, actual = locate_entity_in_note(e, note)
        if present and actual != e.section:
            bad.append((e, e.section, actual))
    return bad


# Clinical vocabulary for hallucination detection. Morphology catches most of it;
# the explicit list covers common terms that carry no diagnostic suffix.
_MED_SUFFIX = re.compile(
    r"\b\w{3,}(?:itis|osis|emia|aemia|pathy|ectomy|otomy|oscopy|ostomy|plasty|algia|"
    r"megaly|penia|trophy|plasia|sclerosis|embolism|infarct\w*|carcinoma|sarcoma|"
    r"cillin|mycin|cycline|olol|pril|sartan|statin|azole|prazole|dipine|parin|"
    r"tidine|triptan|codone|morphin\w*)\b", re.I)
_MED_VOCAB = re.compile(
    r"\b(?:mri|ct scan|ct|x-?ray|ultrasound|echocardiogram|ecg|ekg|eeg|biopsy|endoscopy|"
    r"colonoscopy|angiogram|catheter\w*|cbc|bmp|cmp|hba1c|troponin|creatinine|"
    r"cancer|tumou?r|malignan\w+|metasta\w+|lesion|abscess|fracture|aneurysm|"
    r"sepsis|pneumonia|stroke|seizure|diabet\w+|hypertens\w+|asthma|copd|"
    r"insulin|metformin|warfarin|aspirin|steroid\w*|antibiotic\w*|chemotherapy|"
    r"radiation|surgery|transfusion|intubat\w+|dialysis)\b", re.I)

_PLACEHOLDERS = ("no objective findings stated", "no assessment given", "no plan given")


def extract_note_terms(note: str) -> List[str]:
    """Clinical terms asserted by the note. The extract_entities(N_draft) half of line 31."""
    body = re.sub(r"^\s*\**\s*[SOAP]\s*[:.\)]", " ", note or "", flags=re.M | re.I)
    for ph in _PLACEHOLDERS:                     # mandated filler is not a claim
        body = re.sub(re.escape(ph), " ", body, flags=re.I)
    terms = {m.group().lower() for m in _MED_SUFFIX.finditer(body)}
    terms |= {m.group().lower() for m in _MED_VOCAB.finditer(body)}
    return sorted(terms)


def check_hallucination(entities, note, transcript):
    """Check 3 - Algorithm 1 line 31:

        Hallucinations <- extract_entities(N_draft) \\ grounded_spans(E_verified)

    A genuine set difference over extracted terms, not a sentence-similarity
    heuristic - a sentence-level score flags mandated placeholder lines and
    legitimate short assessments as hallucinations.

    The reference set is E_verified plus the transcript, rather than the drug
    regex in audio2soap.ipynb's detect_hallucinations. That widening is the
    whole point of having Stage B.
    """
    supported = _norm(transcript) + " " + _norm(" ".join(
        [e.span for e in entities] + [e.preferred_term for e in entities if e.preferred_term]
    ))
    out = []
    for term in extract_note_terms(note):
        head = _norm(term).strip()
        if head and head not in supported:
            out.append(term)
    return out


_REL_ASSERT = {
    "ABSENT": (re.compile(r"\b(denies|denied|no |negative for|without|absent|"
                          r"not present|ruled out)\b", re.I), "denied"),
    "FAMILY": (re.compile(r"\b(mother|father|family|sister|brother|parent|"
                          r"grandmother|grandfather)\b", re.I), "attributed to a relative"),
    "PAST":   (re.compile(r"\b(history of|previously|past|prior|used to|resolved|"
                          r"h/o|pmh)\b", re.I), "marked historical"),
}

def check_relation_consistency(entities, note):
    """Check 4 - design doc sec 5.2.  -> [(entity, why)]

    An ABSENT entity must not read as present; a FAMILY entity must not become
    patient history. Most section-consistency checkers stop at placement and
    never catch this.
    """
    secs = split_soap_sections(note)
    bad = []
    for e in entities:
        if e.relation not in _REL_ASSERT:
            continue
        present, actual = locate_entity_in_note(e, note)
        if not present:
            continue
        pat, desc = _REL_ASSERT[e.relation]
        # look at the sentence the entity actually landed in
        body = secs.get(actual, "")
        sent = next((s for s in re.split(r"(?<=[.!?])\s+", body)
                     if _mentions(s, e.span) or
                        (e.preferred_term and _mentions(s, e.preferred_term))), body)
        if not pat.search(sent):
            bad.append((e, f"{e.relation} entity is not {desc} in the note"))
    return bad


def run_all_checks(entities, note, transcript):
    return {
        "missing":      check_completeness(entities, note),
        "misplaced":    check_placement(entities, note),
        "hallucinated": check_hallucination(entities, note, transcript),
        "inconsistent": check_relation_consistency(entities, note),
    }


print("Stage D checks ready - completeness / placement / hallucination / relation")

In [ ]:
# ============================================================================
# Cell 20 - reflect_and_revise().  Algorithm 1 lines 32-39.
#   Extends _REFLECT_PROMPT from audio2soap.ipynb: the findings are computed in
#   code and handed to the model as evidence, and only the affected sections are
#   regenerated (line 37) rather than the whole note.
# ============================================================================
# Measured on 2026-09-20 (gpt, 50 records): asking for the COMPLETE note back
# made arm C lose on every overlap metric - BERTScore precision 0.932 -> 0.916 AND
# recall 0.930 -> 0.921, ROUGE-L 0.551 -> 0.502 - while notes grew only 10% (261
# -> 287 words). Both sides falling with almost no length change means the model
# was REWORDING sections it had no reason to touch, not padding them.
#
# "Fix only these, change nothing else" in prose did not hold. So the constraint
# is now structural: the model is shown only the sections that actually have
# findings and returns only those, and the untouched sections are spliced back
# byte-identical. That is also what the design doc specifies (line 37: regenerate
# the affected sections), which the whole-note prompt never implemented.
_CORRECTION_PROMPT = textwrap.dedent("""
You are a meticulous clinical scribe correcting part of a DRAFT SOAP note. A verifier
has already found the exact problems. Fix ONLY these.

TRANSCRIPT:
{transcript}

SECTIONS YOU MUST RETURN (these are the only ones with problems):
{sections}

VERIFIER FINDINGS:
{findings}

Rules:
- Return ONLY the sections listed above, each with its own label (S:, O:, A: or P:).
- Do NOT return any other section.
- Keep every sentence that is not named in the findings EXACTLY as written, word for
  word. Do not re-phrase, re-order or "improve" anything.
- Add each MISSING entity to the section named for it.
- Move each MISPLACED entity into the section named for it.
- Delete every statement that relies on an UNSUPPORTED TERM.
- Restate each INCONSISTENT entity so the assertion matches its relation: an ABSENT
  entity must read as denied, a FAMILY entity as belonging to a relative, a PAST
  entity as historical.
- Invent nothing new.

CORRECTED SECTIONS:
""").strip()


def _format_findings(checks) -> str:
    lines = []
    for e in checks["missing"]:
        ev = f" (evidence: {e.evidence})" if e.evidence else ""
        lines.append(f"- MISSING: \"{e.span}\" [{e.entity_type}, {e.relation}] "
                     f"belongs in section {e.section}.{ev}")
    for e, expected, actual in checks["misplaced"]:
        lines.append(f"- MISPLACED: \"{e.span}\" is in {actual} but belongs in {expected} "
                     f"([{e.entity_type}, {e.relation}]).")
    for t in checks["hallucinated"]:
        lines.append(f"- UNSUPPORTED TERM: \"{t}\" appears in the note but is in neither the "
                     f"transcript nor the verified entity list. Delete the claim that uses it.")
    for e, why in checks["inconsistent"]:
        lines.append(f"- INCONSISTENT: \"{e.span}\" - {why}.")
    return "\n".join(lines) if lines else "none"


def n_violations(checks) -> int:
    return sum(len(v) for v in checks.values())


def _violation_detail(checks):
    """Short, inspectable description of every surviving violation.

    Counting alone ("8 -> 6") cannot tell a real defect from a false positive, so
    the trace carries what actually fired. format_reflection_report() prints it.
    """
    d = []
    d += [f"MISSING {e.span!r} -> {e.section}" for e in checks["missing"]]
    d += [f"MISPLACED {e.span!r} {act}->{exp}" for e, exp, act in checks["misplaced"]]
    d += [f"UNSUPPORTED {t!r}" for t in checks["hallucinated"]]
    d += [f"INCONSISTENT {e.span!r} ({e.relation})" for e, _ in checks["inconsistent"]]
    return d


def format_reflection_report(trace):
    """Why reflection stopped where it did."""
    lines = ["=" * 78, "REFLECTION", "=" * 78]
    for t in trace:
        lines.append(f"  round {str(t['round']):<6} violations={t['violations']:<3} "
                     f"(missing {t['missing']}, misplaced {t['misplaced']}, "
                     f"unsupported {t['hallucinated']}, inconsistent {t['inconsistent']})")
        for d in t.get("detail", [])[:12]:
            lines.append(f"        - {d[:96]}")
    last = trace[-1]["violations"]
    lines.append("=" * 78)
    lines.append("clean" if last == 0 else
                 f"{last} violation(s) still open after {len(trace)-1} round(s) - the list "
                 f"above says which. Check them against the note before treating them as real.")
    return "\n".join(lines)


def _section_of_term(note, term):
    """Which SOAP section a literal term sits in, or None if it cannot be found."""
    t = _norm(term).strip()
    if not t:
        return None
    secs = split_soap_sections(note or "")
    for s in ("S", "O", "A", "P"):
        if t in _norm(secs.get(s, "")):
            return s
    return None


def _sections_with_findings(checks, note=None):
    """Which of S/O/A/P the verifier actually complained about.

    An unsupported term used to force all four sections on the theory that it
    could sit anywhere. In practice round 0 carries ~2 unsupported terms per
    note, so that branch fired on essentially every record and the "targeted"
    correction regenerated the whole note: measured over 300 records, 1 of 200
    sections survived on gpt and 0 of 200 on llama and qwen. Locate the term in
    the draft instead and target only the section holding it.
    """
    secs = set()
    for e in checks.get("missing", []):
        secs.add(getattr(e, "section", None))
    for tup in checks.get("misplaced", []):
        e, exp, act = tup
        secs.add(exp); secs.add(act)          # both: moving an entity rewrites two sections
    for tup in checks.get("inconsistent", []):
        e = tup[0]
        secs.add(getattr(e, "section", None))
    for term in checks.get("hallucinated", []):
        hit = _section_of_term(note, term)
        if hit:
            secs.add(hit)
        elif note is None:
            secs.update(("S", "O", "A", "P"))   # no draft to search: old behaviour
    secs = {s for s in secs if s in ("S", "O", "A", "P")}
    return [s for s in ("S", "O", "A", "P") if s in secs]


def _splice_sections(draft, revised_text, targets):
    """Put the revised sections back into the draft, leaving the rest untouched.

    Anything the model returns for a section we did not ask about is discarded -
    that is the whole point. If it returns nothing usable for a target section,
    the draft's own version is kept rather than losing content.
    """
    base = split_soap_sections(draft)
    new = split_soap_sections(revised_text)
    out = []
    for s in ("S", "O", "A", "P"):
        body = base.get(s, "")
        if s in targets and (new.get(s) or "").strip():
            body = new[s].strip()
        if body.strip():
            out.append(f"{s}: {body.strip()}")
    return "\n".join(out) if out else draft


def reflect_and_revise(note, entities, transcript, rid="", max_rounds=None):
    """-> (final_note, trace). Algorithm 1 lines 24-39."""
    max_rounds = MAX_REFLECT_ROUNDS if max_rounds is None else max_rounds
    cached = ck_load(_mk("stageD"), rid) if rid else None
    if cached is not None:
        return cached["note"], cached["trace"]

    trace = []
    for rnd in range(max_rounds):
        checks = run_all_checks(entities, note, transcript)
        n = n_violations(checks)
        trace.append({
            "round": rnd, "violations": n,
            "missing": len(checks["missing"]), "misplaced": len(checks["misplaced"]),
            "hallucinated": len(checks["hallucinated"]),
            "inconsistent": len(checks["inconsistent"]),
            "detail": _violation_detail(checks),
        })
        if n == 0:                                   # lines 32-33: clean, return
            break
        targets = _sections_with_findings(checks, note)
        if not targets:        # nothing locatable to fix; do not spin
            break
        _base = split_soap_sections(note)
        _show = "\n".join(f"{s}: {_base.get(s, '').strip()}" for s in targets)
        revised = (_llm(
            [{"role": "system", "content": "You are a meticulous clinical scribe."},
             {"role": "user", "content": _CORRECTION_PROMPT
                 .replace("{transcript}", transcript[:10000])
                 .replace("{sections}", _show)
                 .replace("{findings}", _format_findings(checks))}],
            max_tokens=1200, temperature=0.1,
        ) or "").strip()
        trace[-1]["sections_revised"] = targets
        note = _splice_sections(note, revised, targets) if revised else note

    final = run_all_checks(entities, note, transcript)
    trace.append({"round": "final", "violations": n_violations(final),
                  "missing": len(final["missing"]), "misplaced": len(final["misplaced"]),
                  "hallucinated": len(final["hallucinated"]),
                  "inconsistent": len(final["inconsistent"]),
                  "detail": _violation_detail(final)})
    if rid:
        ck_save(_mk("stageD"), rid, {"note": note, "trace": trace})
    return note, trace


print("reflect_and_revise() ready - Algorithm 1 lines 24-39")

In [ ]:
# ============================================================================
# Cell 21 - Stage D self-test.  Verification step 5: plant a hallucination and a
#           misplaced entity, assert the right checks fire. Runs offline.
# ============================================================================
def selftest_stage_d():
    transcript = ("DOCTOR: Any chest pain?\nPATIENT: No, none at all.\n"
                  "PATIENT: My mother had breast cancer.\n"
                  "DOCTOR: I'll start you on amoxicillin.\n")
    ents = [
        VerifiedEntity(span="chest pain", entity_type="Symptom", relation="ABSENT",
                       section=route_section("Symptom", "ABSENT"), grounded=True,
                       preferred_term="Chest pain"),
        VerifiedEntity(span="breast cancer", entity_type="Disease/Problem", relation="FAMILY",
                       section=route_section("Disease/Problem", "FAMILY"), grounded=True,
                       preferred_term="Breast cancer"),
        VerifiedEntity(span="amoxicillin", entity_type="Medication", relation="ORDERED",
                       section=route_section("Medication", "ORDERED"), grounded=True,
                       preferred_term="Amoxicillin"),
        VerifiedEntity(span="lumbar puncture", entity_type="Procedure/Treatment",
                       relation="ORDERED", section=route_section("Procedure/Treatment", "ORDERED"),
                       grounded=True, preferred_term="Lumbar puncture"),
    ]
    # planted: chest pain asserted as PRESENT (check 4) and in the wrong section;
    # breast cancer as the patient's own (check 4); amoxicillin in S not P (check 2);
    # lumbar puncture absent entirely (check 1); a fabricated MRI sentence (check 3).
    bad_note = (
        "S: Patient reports chest pain. Patient has breast cancer. "
        "Started on amoxicillin.\n"
        "O: No objective findings stated.\n"
        "A: Probable bacterial process.\n"
        "P: Cranial MRI demonstrated periventricular demyelination consistent with "
        "multiple sclerosis.\n"
    )
    ch = run_all_checks(ents, bad_note, transcript)
    results = {
        "check 1 completeness  (lumbar puncture missing)": len(ch["missing"]) >= 1,
        "check 2 placement     (amoxicillin S not P)":     len(ch["misplaced"]) >= 1,
        "check 3 hallucination (fabricated MRI/sclerosis)": len(ch["hallucinated"]) >= 1,
        "check 4 relation      (ABSENT/FAMILY asserted)":  len(ch["inconsistent"]) >= 1,
    }
    for k, v in results.items():
        print(f"  {'PASS' if v else 'FAIL'}  {k}")
    print("\n  findings passed to the corrector:")
    for line in _format_findings(ch).split("\n"):
        print("   ", line[:110])
    return all(results.values())

STAGE_D_OK = selftest_stage_d()
print(f"\nStage D self-test: {'ALL PASS' if STAGE_D_OK else 'SOME FAILED - fix before batching'}")

---
## 4. Metrics

The four VeriSOAP metrics are defined over `E_verified`, which is why they are not computable
without Stage B. ROUGE / BLEU / the judge are ported from `audio2soap.ipynb`; `mean_ci` and
`paired_bootstrap` from `soap_ablation_ft_reflection_k.ipynb`.

In [ ]:
# ============================================================================
# Cell 22 - metrics.  ROUGE/BLEU ported verbatim from audio2soap.ipynb;
#           the four VeriSOAP metrics are new (defined over E_verified).
# ============================================================================
def _tokenize(text):
    return re.findall(r"[a-z0-9]+", (text or "").lower())

def rouge_n(candidate, reference, n=1):
    c, r = _tokenize(candidate), _tokenize(reference)
    if not c or not r:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    cg = Counter(tuple(c[i:i+n]) for i in range(len(c)-n+1))
    rg = Counter(tuple(r[i:i+n]) for i in range(len(r)-n+1))
    ov = sum((cg & rg).values())
    p = ov / max(sum(cg.values()), 1); rc = ov / max(sum(rg.values()), 1)
    return {"precision": round(p, 4), "recall": round(rc, 4),
            "f1": round(2*p*rc / max(p+rc, 1e-8), 4)}

def rouge_l(candidate, reference):
    c, r = _tokenize(candidate), _tokenize(reference)
    if not c or not r:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    m, n = len(c), len(r)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if c[i-1] == r[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p, rc = lcs/max(m, 1), lcs/max(n, 1)
    return {"precision": round(p, 4), "recall": round(rc, 4),
            "f1": round(2*p*rc / max(p+rc, 1e-8), 4)}

def bleu_score(candidate, reference, max_n=4):
    c, r = _tokenize(candidate), _tokenize(reference)
    if not c or not r:
        return {"bleu": 0.0}
    bp = 1.0 if len(c) >= len(r) else math.exp(1 - len(r)/len(c))
    log_avg, w = 0.0, 1.0/max_n
    for n in range(1, max_n+1):
        cg = Counter(tuple(c[i:i+n]) for i in range(max(len(c)-n+1, 0)))
        rg = Counter(tuple(r[i:i+n]) for i in range(max(len(r)-n+1, 0)))
        p_n = sum(min(cg[g], rg[g]) for g in cg) / max(sum(cg.values()), 1)
        if p_n == 0:
            return {"bleu": 0.0}
        log_avg += w * math.log(p_n)
    return {"bleu": round(bp * math.exp(log_avg), 4)}


# ---- the four VeriSOAP metrics (new - defined over E_verified) --------------
def entity_coverage_rate(entities, note):
    g = [e for e in entities if e.grounded]
    if not g:
        return float("nan")
    return sum(1 for e in g if locate_entity_in_note(e, note)[0]) / len(g)

def section_placement_accuracy(entities, note):
    covered = [(e, locate_entity_in_note(e, note)) for e in entities]
    covered = [(e, act) for e, (pres, act) in covered if pres]
    if not covered:
        return float("nan")
    return sum(1 for e, act in covered if act == e.section) / len(covered)

def hallucination_rate(entities, note, transcript):
    """Unsupported clinical terms per note (a count, not a ratio)."""
    return len(check_hallucination(entities, note, transcript))

def relation_consistency_rate(entities, note):
    scoreable = [e for e in entities if e.relation in _REL_ASSERT]
    if not scoreable:
        return float("nan")
    return 1.0 - len(check_relation_consistency(entities, note)) / len(scoreable)


# Measured 2026-09-20: without written level anchors this judge gave exactly 5.0
# to 42 of 50 arm-A notes. An instrument pinned at its ceiling cannot register an
# improvement, only a fall - which is precisely what arm C then showed. The
# anchors below are the ones the 3-axis judge already used and the ones
# Evaluation_Metrics_Detailed.pdf sec 2.1 specifies: 5 is reserved, 4 is a solid
# usable note. The short analysis before the scores is the G-Eval finding that
# grading after articulating reasons is more consistent.
_JUDGE_PROMPT = textwrap.dedent("""
Rate this SOAP note against the transcript it was written from, 1-5 on each axis.

Scale anchors, the same for every axis. Be strict - 5 is rare:
  5 excellent - reserved; a reviewing clinician would change nothing
  4 solid, clinically usable as written, with minor nits
  3 usable but with a clear weakness
  2 poor; a clinician would have to rewrite part of it
  1 unusable

  completeness       did it capture the clinically relevant content of the transcript?
  correctness        is everything stated actually supported by the transcript?
  coherence          is it well organised and readable?
  assessment_quality is the Assessment clinically sound and specific?
  plan_safety        is the Plan safe, appropriate and free of invented orders?

TRANSCRIPT:
{transcript}

SOAP NOTE:
{soap}

First write 2-3 sentences naming this note's strongest and weakest points. Then return
ONLY a JSON object on the final line with the five integer keys plus "analysis".
""").strip()

def llm_judge(transcript, soap_text):
    """Ported from audio2soap.ipynb. Deterministic (temperature 0) to blunt
    the stochastic-variability bias in LLM-as-judge."""
    raw = _llm(
        [{"role": "system", "content": "Respond ONLY with a JSON object. No other text."},
         {"role": "user", "content": _JUDGE_PROMPT
             .replace("{transcript}", transcript[:3000]).replace("{soap}", soap_text[:2000])}],
        max_tokens=500, temperature=0.0,   # analysis + JSON
    )
    sc = _parse_json(raw, expect="dict")
    dims = ["completeness", "correctness", "coherence", "assessment_quality", "plan_safety"]
    for k in dims:
        v = sc.get(k)
        sc[k] = 3 if not isinstance(v, (int, float)) or v < 1 or v > 5 else max(1, min(5, int(v)))
    sc["composite"] = round(sum(sc[k] for k in dims) / len(dims), 2)
    return {k: sc[k] for k in dims + ["composite"]}


# One GPU, one cached copy of each metric model. Everything that touches torch
# serialises on this; the OpenAI calls do not.
_GPU_LOCK = __import__("threading").Lock()


def _safe_gpu(name, fn, *a, **kw):
    """_safe(), but holding the GPU lock. For torch-backed metrics only."""
    with _GPU_LOCK:
        return _safe(name, fn, *a, **kw)


def _safe(name, fn, *a, **kw):
    """Compute one metric, or return None. NEVER let an optional metric take the
    others down: a missing sentence-transformers wiped every metric for a record -
    ROUGE, BLEU, judge and all - because one unguarded call raised inside the arm."""
    try:
        return fn(*a, **kw)
    except Exception as ex:
        if name not in _METRIC_WARNED:
            _METRIC_WARNED.add(name)
            print(f"  [metric] {name} unavailable ({type(ex).__name__}) - "
                  f"column will be blank, everything else still computed")
        return None


_METRIC_WARNED = set()


def evaluate_all(note, transcript, entities, reference=None, judge=True):
    m = {
        "entity_coverage_rate":       _safe("coverage", entity_coverage_rate, entities, note),
        "section_placement_accuracy": _safe("placement", section_placement_accuracy, entities, note),
        "hallucination_rate":         _safe_gpu("hallucination", hallucination_rate, entities, note, transcript),
        "relation_consistency_rate":  _safe("relation", relation_consistency_rate, entities, note),
        "n_entities":                 len(entities),
        "n_grounded":                 sum(1 for e in entities if e.grounded),
    }
    # Stage B's own headline: the share of Stage-A spans that UMLS actually
    # grounded to a CUI. Everything downstream is defined over the grounded set,
    # so a results table without this number cannot say how much of the note the
    # verification layer was even able to speak about.
    m["grounding_rate"] = (m["n_grounded"] / m["n_entities"]) if m["n_entities"] else 0.0
    kc = _safe("keyword_coverage", keyword_coverage, note, entities)
    if kc is not None:
        m["keyword_coverage"] = kc["coverage"]
    sim = _safe_gpu("similarity", soap_transcript_similarity, note, transcript)
    if sim is not None:
        m["cosine_to_transcript"] = sim["cosine"]
        m["wmd_similarity"] = sim["wmd_similarity"]

    if reference:
        for key, fn, pick in (
                ("rouge1_f1",    lambda: rouge_n(note, reference, 1), "f1"),
                ("rouge2_f1",    lambda: rouge_n(note, reference, 2), "f1"),
                ("rougeL_f1",    lambda: rouge_l(note, reference),    "f1"),
                ("rougeLsum_f1", lambda: rouge_lsum(note, reference), "f1"),
                ("bleu",         lambda: bleu_score(note, reference), "bleu"),
                ("concept_f1",   lambda: concept_f1(note, reference), "f1")):
            v = _safe(key, fn)
            if v is not None:
                m[key] = v[pick]
        bs = _safe_gpu("bertscore", bertscore_document, note, reference)
        if bs is not None:
            m["bertscore_p"], m["bertscore_r"], m["bertscore_f1"] = (
                bs["precision"], bs["recall"], bs["f1"])
        ec = _safe_gpu("embedding_cosine", embedding_cosine, note, reference)
        if ec is not None:
            m["embedding_cosine"] = ec
        bl = _safe_gpu("bleurt", bleurt_score, note, reference)
        if bl is not None:
            m["bleurt"] = bl

    q = _safe_gpu("questeval", questeval_score, note, transcript)
    if q is not None:
        m["questeval"] = q

    if judge:
        # Two LLM juries, both at temperature 0. Column prefixes are judge5_ and
        # judge3_ so a reader of the exported CSV can tell which jury a number
        # came from without the code in front of them.
        j5 = _safe("judge5", llm_judge, transcript, note)
        if j5:
            m.update({f"judge5_{k}": v for k, v in j5.items()})
        j3 = _safe("judge3", llm_judge_3axis, transcript, note)
        if j3:
            m.update({f"judge3_{k}": v for k, v in j3.items()})
    return m

def mean_ci(vals, n_boot=None):
    """Mean with a bootstrap 95% confidence interval."""
    n_boot = BOOT_N if n_boot is None else n_boot
    vals = [v for v in vals if v == v]                     # drop NaN
    if not vals:
        return (float("nan"),) * 3
    m = statistics.fmean(vals)
    if len(vals) < 2:
        return (m, m, m)
    means = sorted(statistics.fmean(random.choices(vals, k=len(vals))) for _ in range(n_boot))
    return (m, means[int(0.025 * n_boot)], means[int(0.975 * n_boot)])


def paired_bootstrap(a_vals, b_vals, n_boot=None):
    """Paired test on b - a. Returns (mean delta, lo, hi, two-sided p)."""
    n_boot = BOOT_N if n_boot is None else n_boot
    deltas = [b - a for a, b in zip(a_vals, b_vals) if a == a and b == b]
    if len(deltas) < 2:
        return (0.0, 0.0, 0.0, 1.0)
    obs = statistics.fmean(deltas)
    boots = sorted(statistics.fmean(random.choices(deltas, k=len(deltas))) for _ in range(n_boot))
    lo, hi = boots[int(0.025 * n_boot)], boots[int(0.975 * n_boot)]
    p = (2.0 * sum(1 for b in boots if b <= 0) / n_boot) if obs >= 0 else \
        (2.0 * sum(1 for b in boots if b >= 0) / n_boot)
    return (obs, lo, hi, min(1.0, p))


print("metrics + statistics ready")

---
## 5. Arms and batch run

**A** isolates the baseline, **A -> B** isolates the value of UMLS grounding, **B -> C**
isolates the value of reflection. All three arms see the same records, so the comparisons are
paired.

In [ ]:
# ============================================================================
# Cell 23 - the three arms + the checkpointed batch runner.
# ============================================================================
def run_arm_A(rec):
    """K-SOAP baseline: GENERATED from raw Stage-A entities, flat keyword list, no
    verification - that is what makes it the baseline.

    It is still SCORED against E_verified, like every other arm. The design doc
    (sec 6) defines the four metrics against E_verified, and scoring each arm
    against its own entity set would make the columns incomparable: Arm A has no
    grounded entities at all, so its coverage would simply be undefined.
    verify_entities() reads the Stage-B checkpoint, so this costs nothing extra.
    """
    rid, tr = str(rec["id"]), rec["transcript"]
    raw  = extract_kspans(tr, rid=rid)
    note = generate_soap(tr, raw, grouped=False, rid=rid, tag="A")
    return note, verify_entities(raw, tr, rid=rid), None

def run_arm_B(rec):
    """+ Stage B: UMLS-verified entities, grouped by routed section."""
    rid, tr = str(rec["id"]), rec["transcript"]
    ents = attach_codes(verify_entities(extract_kspans(tr, rid=rid), tr, rid=rid), tr)
    note = generate_soap(tr, ents, grouped=True, with_evidence=True, rid=rid, tag="B")
    return note, ents, None

def run_arm_C(rec):
    """Full VeriSOAP: + Stage D evidence-guided reflection."""
    rid, tr = str(rec["id"]), rec["transcript"]
    ents = attach_codes(verify_entities(extract_kspans(tr, rid=rid), tr, rid=rid), tr)
    draft = generate_soap(tr, ents, grouped=True, with_evidence=True, rid=rid, tag="B")
    note, trace = reflect_and_revise(draft, ents, tr, rid=rid)
    return note, ents, trace

ARMS = {}
AGENT_STATS, _AGENT_STATS_LOCK = {}, __import__('threading').Lock()   # filled by arm D
if RUN_ARM_A: ARMS["A_ksoap_baseline"] = run_arm_A
if RUN_ARM_B: ARMS["B_verified"]       = run_arm_B
if RUN_ARM_C: ARMS["C_verisoap_full"]  = run_arm_C


# Stage D's trace carries a `detail` list whose entries are prefixed by which
# check fired. Counting the prefixes recovers the per-check split without
# touching reflect_and_revise().
_VIOL_PREFIX = {"MISSING": "missing", "MISPLACED": "misplaced",
                "UNSUPPORTED": "hallucinated", "INCONSISTENT": "inconsistent"}


def _count_violation_kinds(detail):
    out = dict.fromkeys(_VIOL_PREFIX.values(), 0)
    for line in detail or []:
        head = str(line).split(" ", 1)[0]
        if head in _VIOL_PREFIX:
            out[_VIOL_PREFIX[head]] += 1
    return out


def run_batch(records=None, arms=None, judge=True, workers=None):
    """Score every arm for every record, RECORDS IN PARALLEL.

    Records are independent, and the cost is dominated by OpenAI latency: per
    model this is roughly 285 calls (a generation and two judges per arm, plus
    reflection in arm C), and run serially that was about 30 minutes per model of
    pure waiting. Overlapping records collapses it to roughly the same work
    divided by SCORE_WORKERS.

    Arms stay SEQUENTIAL inside one record, deliberately: arm C reuses the draft
    arm B generated, through the same `genB` checkpoint. Running them in order
    within the task is what makes that cache hit.

    Everything that touches the GPU - local generation and the torch-backed
    metrics - takes _GPU_LOCK, so exactly one thread is on the device at a time.
    """
    records = records if records is not None else RECORDS
    arms = arms or ARMS
    workers = workers or SCORE_WORKERS
    if not records:
        print("No evaluation records loaded, so there is nothing to batch.")
        print("This cell is for the held-out EVALUATION set.")
        return {}

    n, t0 = len(records), time.time()
    slots = [None] * n                       # keep input order, not finish order
    done = 0
    _print_lock = __import__("threading").Lock()

    def one(idx, rec):
        rid = str(rec["id"])
        per_arm, failed = {}, []
        for aname, fn in arms.items():
            cached = ck_load(_mk(f"metrics_{aname}"), rid)
            if cached is not None:
                per_arm[aname] = (cached, True)
                continue
            try:
                note, ents, trace = fn(rec)
                m = evaluate_all(note, rec["transcript"], ents,
                                 reference=rec.get("soap"), judge=judge)
            except Exception as ex:
                # One arm failing must not cost the record its OTHER arms. An
                # exception used to escape this loop, so a single bad arm dropped
                # that record from every column and it stayed dropped on re-runs,
                # because the surviving arms were never written back.
                failed.append(f"{aname}: {type(ex).__name__}: {str(ex)[:60]}")
                continue
            m["record_id"] = rid
            with _AGENT_STATS_LOCK:                       # arm D: what the agents decided
                m.update(AGENT_STATS.pop((MODEL, rid), {}) if aname == "D_agentic" else {})
            if trace:
                # design doc sec 5.5: the four checks before and after Stage D
                m["reflect_rounds"] = len([x for x in trace if x["round"] != "final"])
                m["violations_before"] = trace[0]["violations"]
                m["violations_after"] = trace[-1]["violations"]
                m["revised"] = int(trace[-1]["violations"] < trace[0]["violations"])
                m["clean_after"] = int(trace[-1]["violations"] == 0)
                for _when, _t in (("before", trace[0]), ("after", trace[-1])):
                    for _k, _v in _count_violation_kinds(_t.get("detail")).items():
                        m[f"viol_{_when}_{_k}"] = _v
            ck_save(_mk(f"metrics_{aname}"), rid, m)
            ck_save(_mk(f"note_{aname}"), rid, {"note": note})
            per_arm[aname] = (m, False)
        return idx, per_arm, failed

    with _TPE(max_workers=max(1, workers)) as pool:
        futs = {pool.submit(one, i, r): i for i, r in enumerate(records)}
        for fut in _as_done(futs):
            with _print_lock:
                done += 1
                try:
                    idx, per_arm, failed = fut.result()
                    slots[idx] = per_arm
                    for msg in failed:
                        print(f"    [score] record {idx} arm {msg}", flush=True)
                except Exception as ex:
                    print(f"    [score] record {futs[fut]} FAILED ENTIRELY "
                          f"({type(ex).__name__}: {str(ex)[:80]})", flush=True)
                if done == 1 or done % 5 == 0 or done == n:
                    el = time.time() - t0
                    print(f"    [score] {done}/{n}  {el / 60:.1f} min elapsed  "
                          f"~{(el / done) * (n - done) / 60:.0f} min left", flush=True)

    results = {a: [] for a in arms}
    n_cached = 0
    for per_arm in slots:
        if not per_arm:
            continue
        for aname, (m, was_cached) in per_arm.items():
            results[aname].append(m)
            n_cached += int(was_cached)
    total = sum(len(v) for v in results.values())
    print(f"    [score] {total} (record, arm) results  "
          f"({n_cached} from cache, {total - n_cached} computed)  "
          f"in {(time.time() - t0) / 60:.1f} min", flush=True)
    return results


print(f"runner ready | arms: {list(ARMS)}")
print("run with:  RESULTS = run_batch()")

In [ ]:
# ============================================================================
# Cell 24 - results table + paired significance matrix.
#
# Every label in every table is written for a reader who has not seen this code.
# ARM_DOC and METRIC_DOC are the single source of truth; the tables and
# metric_glossary() both read from them, so a name can never drift between the
# legend and the column it labels.
# ============================================================================
ARM_DOC = {
    "A_ksoap_baseline": (
        "A", "K-SOAP baseline",
        "Raw Stage-A k-spans passed to the generator as a flat keyword list. "
        "No UMLS verification, no reflection. This is the whiteboard's left side "
        "and the arm VeriSOAP has to beat."),
    "B_verified": (
        "B", "+ UMLS verification",
        "Stage-B VerifiNER: each span is re-bounded, linked to a UMLS concept "
        "(CUI, preferred term, semantic type), its entity type and assertion "
        "relation are re-checked against the retrieved definition, and the "
        "surviving entities are grouped by the S/O/A/P section the router "
        "predicts before being given to the generator."),
    "C_verisoap_full": (
        "C", "+ reflection (full)",
        "Stage-D self-verification on top of B: the draft note is checked for "
        "missing verified entities, entities written into the wrong S/O/A/P "
        "section, claims not grounded in the transcript, and assertions that "
        "contradict an entity's relation (an ABSENT finding written as present, "
        "a FAMILY history written as the patient's own). The findings are fed "
        "back as evidence and the note is revised. This is full VeriSOAP."),
}

# key -> (column label, what it measures, direction)
METRIC_DOC = {
    # ---- the four VeriSOAP metrics, defined over the UMLS-verified entity set --
    "entity_coverage_rate": (
        "Entity Coverage", "fraction of UMLS-verified entities that appear in the note", "higher"),
    "section_placement_accuracy": (
        "Section Placement", "fraction of entities written into the S/O/A/P section the "
        "(entity type, relation) router predicts", "higher"),
    "hallucination_rate": (
        "Hallucinated claims", "clinical claims per note with no support in the transcript "
        "(count, not a fraction)", "lower"),
    "relation_consistency_rate": (
        "Relation Consistency", "fraction of entities whose assertion in the note matches its "
        "relation (ABSENT not asserted present, FAMILY not written as patient history)", "higher"),
    "grounding_rate": (
        "UMLS Grounding", "share of Stage-A spans that Stage B linked to a UMLS concept. "
        "This is Stage B's own result: every metric below is defined over the grounded "
        "set, so it bounds what verification could affect", "higher"),
    "keyword_coverage": (
        "Keyword Coverage", "fraction of the prompt's keyword block that survives into the note", "higher"),
    # ---- reference-based: need a gold SOAP note -------------------------------
    "rouge1_f1":    ("ROUGE-1 F1", "unigram overlap with the gold note", "higher"),
    "rouge2_f1":    ("ROUGE-2 F1", "bigram overlap with the gold note", "higher"),
    "rougeL_f1":    ("ROUGE-L F1", "longest-common-subsequence overlap with the gold note", "higher"),
    "rougeLsum_f1": ("ROUGE-Lsum F1", "sentence-level ROUGE-L, the variant the K-SOAP paper reports", "higher"),
    "bertscore_p":  ("BERTScore P", "token-level precision against the gold note (roberta-large)", "higher"),
    "bertscore_r":  ("BERTScore R", "token-level recall against the gold note (roberta-large)", "higher"),
    "bertscore_f1": ("BERTScore F1", "token-level F1 against the gold note (roberta-large)", "higher"),
    "bleu":         ("BLEU", "n-gram precision against the gold note", "higher"),
    "concept_f1":   ("MEDCON concept-F1", "F1 over the UMLS concepts mentioned, note vs gold note. "
                     "Measures clinical content rather than wording", "higher"),
    "bleurt":       ("BLEURT", "learned semantic similarity to the gold note "
                     "(Elron/bleurt-large-512 pytorch port)", "higher"),
    "embedding_cosine": ("Embedding cosine", "document-embedding cosine to the gold note", "higher"),
    # ---- reference-free: work without a gold note -----------------------------
    "cosine_to_transcript": ("Cosine to transcript", "biomedical sentence-embedding cosine between "
                             "note and transcript", "higher"),
    "wmd_similarity": ("Word Mover sim", "Word Mover's Distance similarity to the transcript", "higher"),
    "questeval":    ("QuestEval", "reference-free QA-based consistency between note and "
                     "transcript. NOT COMPUTED on this stack - the package predates "
                     "transformers 5.x. Covered by Cosine to transcript, Hallucinated "
                     "claims and Judge-4 Faithfulness", "higher"),
    # ---- LLM-as-judge, 5 axes, GPT at temperature 0 ---------------------------
    "judge5_completeness":       ("Judge-5 Completeness",
                                 "1-5: is everything clinically relevant in the transcript captured", "higher"),
    "judge5_correctness":        ("Judge-5 Correctness",
                                 "1-5: is what the note says medically accurate", "higher"),
    "judge5_coherence":          ("Judge-5 Coherence",
                                 "1-5: is the note well structured and readable", "higher"),
    "judge5_assessment_quality": ("Judge-5 Assessment",
                                 "1-5: quality of the clinical reasoning in the A section", "higher"),
    "judge5_plan_safety":        ("Judge-5 Plan safety",
                                 "1-5: is the plan clinically safe and appropriate", "higher"),
    "judge5_composite":          ("Judge-5 mean",
                                 "mean of the five axes above, 1-5", "higher"),
    # ---- LLM-as-judge, 3 axes (the axes the supervisor specified) -------------
    "judge3_quality_of_soap":         ("Judge-4 Note quality",
                                    "1-5: overall clinical quality, structure and correctness of the note", "higher"),
    "judge3_adequacy_wrt_transcript": ("Judge-4 Adequacy",
                                    "1-5: how completely the note captures what is in the transcript "
                                    "(omissions are penalised)", "higher"),
    "judge3_faithfulness":           ("Judge-4 Faithfulness",
                                    "1-5: does the note contain ONLY what the transcript supports. This is "
                                    "the axis that punishes invented findings, drugs and advice", "higher"),
    "judge3_alignment_score":         ("Judge-4 Alignment",
                                    "1-5: how well the note as a whole reflects this encounter - structure, "
                                    "emphasis, and the clinical story", "higher"),
    "judge3_composite":               ("Judge-4 mean",
                                    "mean of the four axes above, 1-5", "higher"),
}

# display order for the full per-model table
METRIC_KEYS = [(k, METRIC_DOC[k][0], METRIC_DOC[k][2]) for k in (
    "grounding_rate",
    "entity_coverage_rate", "section_placement_accuracy", "hallucination_rate",
    "relation_consistency_rate", "keyword_coverage",
    "rouge1_f1", "rouge2_f1", "rougeL_f1", "rougeLsum_f1",
    "bertscore_p", "bertscore_r", "bertscore_f1", "bleu", "concept_f1",
    "cosine_to_transcript", "wmd_similarity", "bleurt", "questeval",
    "judge5_completeness", "judge5_correctness", "judge5_coherence",
    "judge5_assessment_quality", "judge5_plan_safety", "judge5_composite",
    "judge3_quality_of_soap", "judge3_adequacy_wrt_transcript", "judge3_alignment_score",
    "judge3_composite",
)]


def arm_legend(arms=None, full=False):
    """Print what A, B and C actually are. Every table that names an arm calls this."""
    arms = list(arms or ARM_DOC)
    for a in arms:
        letter, short, long = ARM_DOC.get(a, (a, a, ""))
        if full:
            print(f"  {letter} = {short}")
            for line in textwrap.wrap(long, 92):
                print(f"        {line}")
        else:
            print(f"  {letter} = {short:<30} ({a})")


def metric_glossary():
    """Every metric in the tables, defined. Printed once per run, for a reader
    who has the results but not the code."""
    print("=" * 100); print("GLOSSARY"); print("=" * 100)
    print("\nARMS - each is a full pipeline, scored on the same held-out records:")
    arm_legend(full=True)
    print("\n  Every arm is scored against the SAME UMLS-verified entity set, so the")
    print("  columns are comparable. Scoring each arm against its own entities would")
    print("  make Arm A's coverage undefined, since Arm A has no verified entities.")
    groups = [
        ("VERISOAP METRICS (need Stage B; defined over the verified entity set)",
         ["grounding_rate", "entity_coverage_rate", "section_placement_accuracy",
          "hallucination_rate", "relation_consistency_rate", "keyword_coverage"]),
        ("REFERENCE-BASED (need a gold SOAP note; the K-SOAP paper's Table 5 set)",
         ["rouge1_f1", "rouge2_f1", "rougeL_f1", "rougeLsum_f1", "bertscore_p",
          "bertscore_r", "bertscore_f1", "bleu", "concept_f1", "bleurt"]),
        ("REFERENCE-FREE (work on data with no gold note, e.g. the audio set)",
         ["cosine_to_transcript", "wmd_similarity", "questeval"]),
        ("LLM-AS-JUDGE, 5 AXES (GPT, temperature 0, scored 1-5)",
         ["judge5_completeness", "judge5_correctness", "judge5_coherence",
          "judge5_assessment_quality", "judge5_plan_safety", "judge5_composite"]),
        ("LLM-AS-JUDGE, 4 AXES (GPT, temperature 0, scored 1-5) - the rubric in "
         "Evaluation_Metrics_Detailed.pdf sec 2.1",
         ["judge3_quality_of_soap", "judge3_adequacy_wrt_transcript",
          "judge3_faithfulness", "judge3_alignment_score", "judge3_composite"]),
    ]
    for title, keys in groups:
        print(f"\n{title}")
        print("-" * 100)
        for k in keys:
            label, desc, direction = METRIC_DOC[k]
            arrow = "higher is better" if direction == "higher" else "LOWER is better"
            wrapped = textwrap.wrap(desc, 54) or [""]
            print(f"  {label:<24} {arrow:<17} {wrapped[0]}")
            for line in wrapped[1:]:
                print(f"  {'':<24} {'':<17} {line}")
    print("\nSTAGE D / REFLECTION (arm C only, from Stage D's own trace)")
    print("-" * 100)
    for label, desc in [
            ("revised", "share of notes Stage D actually changed"),
            ("rounds", "mean revision rounds used, capped at MAX_REFLECT_ROUNDS"),
            ("viol before/after", "violations per note entering and leaving Stage D"),
            ("fixed", "share of violations removed by reflection"),
            ("clean after", "share of notes left with zero violations"),
            ("Hallucinated", "whiteboard check 1: claim the transcript does not support"),
            ("Misplaced", "whiteboard check 2: entity in the wrong S/O/A/P section"),
            ("Missing", "whiteboard check 3a: verified entity absent from the note"),
            ("Inconsistent", "whiteboard check 3b: ABSENT asserted present, FAMILY as "
                             "the patient's own history")]:
        print(f"  {label:<24} {'':<17} {desc}")
    print("=" * 100)
    print("Both judges run at temperature 0. LLM-as-judge is a stochastic estimator;")
    print("the deltas it reports are read alongside the reference metrics, not alone.")
    print("=" * 100)


def _col(rows, key):
    return [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and r[key] == r[key]]

def results_table(results):
    arms = list(results)
    width = 26 + 24 * len(arms)
    print("=" * width)
    print("PIPELINE ARMS COMPARED (same held-out records, same verified entity set):")
    arm_legend(arms)
    print("=" * width)
    # header must fit the 24-char column; arm_legend() above carries the full name
    hdr = "".join(
        f"{ARM_DOC.get(a, (a, a, ''))[0] + ': ' + ARM_DOC.get(a, (a, a, ''))[1]:>24}"
        for a in arms)
    print(f"{'Metric':<24}" + hdr)
    print("=" * width)
    for key, label, direction in METRIC_KEYS:
        cells = []
        for a in arms:
            v = _col(results[a], key)
            if not v:
                cells.append(f"{'-':>24}")
                continue
            m, lo, hi = mean_ci(v)
            cells.append(f"{m:>10.3f} [{lo:.2f},{hi:.2f}]".rjust(24))
        arrow = "^" if direction == "higher" else "v"
        print(f"{label + ' ' + arrow:<24}" + "".join(cells))
    print("=" * (26 + 24 * len(arms)))
    print("Cell = mean [bootstrap 95% CI] over the records scored in that arm.")
    print("^ higher is better,  v lower is better,  '-' = metric not computed "
          "(no gold note, or an optional dependency is absent).")
    print("Run metric_glossary() for what every row measures.")
    ref = MODEL_REGISTRY[MODEL].get("paper_rougeL")
    if ref:
        print(f"K-SOAP paper reports ROUGE-L {ref} for {MODEL_REGISTRY[MODEL]['paper_mode']} "
              f"on CliniKnote - a different dataset, so treat it as orientation, "
              f"not a like-for-like target.")

def significance_table(results):
    arms = list(results)
    pairs = [(arms[i], arms[j]) for i in range(len(arms)) for j in range(i+1, len(arms))]
    if not pairs:
        return
    print("\nPAIRWISE ARM COMPARISONS - what each added stage changed, per metric.")
    print(f"\n{'Comparison (arm1 -> arm2)':<44}{'Metric':<20}{'delta':>9}{'95% CI':>20}{'p':>9}")
    print("-" * 102)
    for a, b in pairs:
        # pair on record_id so a record that failed in one arm cannot misalign the columns
        idx_a = {r["record_id"]: r for r in results[a]}
        idx_b = {r["record_id"]: r for r in results[b]}
        shared = [rid for rid in idx_a if rid in idx_b]
        for key, label, _ in METRIC_KEYS:
            av, bv = [], []
            for rid in shared:
                x, y = idx_a[rid].get(key), idx_b[rid].get(key)
                if (isinstance(x, (int, float)) and isinstance(y, (int, float))
                        and x == x and y == y):
                    av.append(x); bv.append(y)
            if len(av) < 2:
                continue
            d, lo, hi, p = paired_bootstrap(av, bv)
            star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            pair = (f"{ARM_DOC.get(a, (a,))[0]} -> {ARM_DOC.get(b, (b,))[0]}  "
                f"({ARM_DOC.get(b, (b, b, ''))[1]})")
            print(f"{pair:<44}{label:<20}{d:>9.3f}"
                  f"{f'[{lo:.3f}, {hi:.3f}]':>20}{p:>8.3f}{star}")
    print("-" * 102)
    print("delta = arm2 minus arm1, paired over records scored in BOTH arms, "
          "bootstrap 95% CI.")
    print("A positive delta means arm2 scored higher; for 'Hallucinated claims' a "
          "NEGATIVE delta is the improvement.")
    print(f"* p<0.05   ** p<0.01   *** p<0.001   (paired bootstrap, {BOOT_N} resamples)")

def export_results(results, stem="verisoap_results"):
    import csv
    (ARTIFACTS / f"{stem}.json").write_text(json.dumps(results, indent=1, default=str))
    rows = [{**r, "arm": a} for a in results for r in results[a]]
    if rows:
        keys = sorted({k for r in rows for k in r})
        with open(ARTIFACTS / f"{stem}.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(rows)
    print(f"exported -> {ARTIFACTS/stem}.json / .csv")


# ---- run it ----------------------------------------------------------------
# RESULTS = run_batch()
# results_table(RESULTS)
# significance_table(RESULTS)
# export_results(RESULTS)
print("reporting ready. Uncomment the four lines above to run.")

---
## Agentic arm D — the same stages, run by agents that decide

```
 K-Span agent ──▶ Verifier agent ──┬──▶ Coding agent ──▶ Generator agent ──▶ Reflection agent ──┐
      ▲                            │                                          ▲                 │
      └─ too few entities: ────────┘                                          └─ Reviser agent ◀┘
         re-extract with feedback (≤ 1)                     violations left (≤ AGENT_MAX_ROUNDS)
```

| Agent | Tools (functions from the stages above) | Decision it makes |
|---|---|---|
| K-Span | `extract_kspans` (attempt 1, shared with A/B/C); same prompt + verifier feedback on a retry | — |
| Verifier | `verify_entities` (VerifiNER, LLM evidence); deterministic UMLS re-check | re-extract when a long consultation has too few entities; switch tool when grounding is weak; keep the better attempt |
| Coding | `attach_codes` | — |
| Generator | `generate_soap` (arm B's draft when the entity set is unchanged) | — |
| Reflection | `run_all_checks` (completeness, placement, hallucination, relation) | accept / revise; **keeps the best draft**, rejects revisions that do not reduce the violations, stops after 2 rejections |
| Reviser | delete unsupported claims in code; LLM rewrite of only the flagged sections | which tool, and tells the LLM why an earlier attempt was rejected |

Everything is checkpointed like the other arms. `show_agent_log('<record id>', model='gpt')` prints what the
agents decided for one record.

In [ ]:
# ============================================================================
# Cell AGENTIC - arm D: the same Stages A-D, run by AGENTS that make decisions.
#
# Arms A/B/C call the stages in a fixed order. Arm D wraps the SAME functions as
# LangGraph agent nodes that share one state and decide what happens next:
#
#   K-Span agent ──▶ Verifier agent ──┬──▶ Coding agent ──▶ Generator agent ──▶ Reflection agent ──┐
#        ▲                            │                                          ▲                 │
#        └── too few entities ────────┘                                          └── Reviser agent ◀┘
#            (re-extract with feedback, ≤ 1 retry)                  violations left (≤ AGENT_MAX_ROUNDS)
#
# Decisions arms A/B/C cannot make:
#   1. Verifier -> K-Span: a long consultation with too few entities is sent back
#      for re-extraction, with the reason as feedback; the better attempt is kept.
#   2. Verifier tool switch: when LLM-evidence verification grounds too little,
#      the ungrounded entities are re-checked with the deterministic UMLS tool.
#   3. Reflection keeps the BEST draft: a revision that does not reduce the
#      violations is REJECTED (arm C always keeps the last revision), and two
#      rejections in a row stop the loop.
#   4. Reviser tool choice: when the only findings are unsupported terms, the
#      sentences carrying them are deleted in code (no LLM); otherwise the LLM
#      rewrites only the sections with findings, told why an earlier attempt failed.
#
# Front end and generator are shared with arms B/C, so A/B/C/D stay paired:
#   - attempt 1 of Stage A/B uses the same stageA/stageB checkpoints,
#   - when the entity set is unchanged the draft IS arm B's draft (genB checkpoint).
# ============================================================================
import operator
from typing import TypedDict, Annotated

RUN_ARM_D                 = True   # the agentic arm
AGENT_MAX_KSPAN_ATTEMPTS  = 2      # Verifier may send K-Span back once
AGENT_MIN_ENTITIES_LONG   = 8      # fewer than this in a long consultation -> re-extract
AGENT_LONG_TRANSCRIPT     = 3000   # characters
AGENT_MIN_GROUNDED        = 0.50   # below this grounded share -> deterministic re-check
AGENT_MAX_ROUNDS          = 3      # Reflection <-> Reviser rounds
AGENT_MAX_REJECTIONS      = 2      # consecutive rejected revisions before stopping
AGENT_VERBOSE             = False

try:
    from langgraph.graph import StateGraph, START, END
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", "langgraph"], check=False)
    from langgraph.graph import StateGraph, START, END

# AGENT_STATS / _AGENT_STATS_LOCK are created in the arms cell: (model, record_id) -> agent numbers


class AgentState(TypedDict, total=False):
    rid: str
    transcript: str
    entities: list               # current verified entity set (VerifiedEntity objects)
    entities_changed: bool       # True when it differs from the shared Stage B set
    kspan_attempts: int
    kspan_feedback: str
    candidates: list             # K-Span output of the current attempt
    best_entities: list
    verifier_decision: str
    draft: str
    note: str                    # current note under reflection
    best_note: str
    best_score: tuple
    checks: dict
    round: int
    rejections: int
    last_rejection: str
    reflection_decision: str
    trace: list                  # Stage-D trace in arm C's format (run_batch reads it)
    log: Annotated[list, operator.add]


def _alog(agent, action, **kw):
    ev = {"agent": agent, "action": action, **{k: v for k, v in kw.items() if not isinstance(v, (list, dict))}}
    if AGENT_VERBOSE:
        print(f"      [{agent}] {action}  " + "  ".join(f"{k}={v}" for k, v in kw.items()
                                                       if not isinstance(v, (list, dict))))
    return [ev]


# ---------------------------------------------------------------- tools
def extract_kspans_with_feedback(transcript: str, feedback: str) -> List[VerifiedEntity]:
    """Stage A again, with the Verifier's feedback appended to the SAME prompt.
    Post-processing is identical to extract_kspans() (schema, exact-substring
    invariant, relation assignment, turn/speaker attribution)."""
    prompt = (_NER_PROMPT_V2.replace("{transcript}", transcript[:12000])
              + "\n\nFEEDBACK FROM THE VERIFIER AGENT ON YOUR PREVIOUS ATTEMPT:\n" + feedback
              + "\nExtract ALL clinically relevant spans again, copying each span EXACTLY from the transcript.")
    raw = _llm([{"role": "system", "content": "Respond ONLY with a JSON array. No other text."},
                {"role": "user", "content": prompt}], max_tokens=3000, temperature=0.1)
    items = _parse_json(raw, expect="list")
    turns = split_turns(transcript)
    used, out = set(), []
    for it in items:
        if not isinstance(it, dict):
            continue
        span = str(it.get("span") or it.get("text") or "").strip()
        if not span or len(span) < 2:
            continue
        etype = str(it.get("entity_type") or it.get("label") or "").strip()
        etype = LEGACY_LABEL_MAP.get(etype, etype)
        if etype not in ENTITY_TYPES:
            continue
        rel = str(it.get("relation") or it.get("status") or "PRESENT").strip().upper()
        rel = LEGACY_STATUS_MAP.get(rel.title().replace(" ", "_"), rel)
        if rel not in RELATION_SET:
            rel = "PRESENT"
        start, end = _locate(span, transcript, used)
        if start == -1:
            continue
        used.add((start, end))
        span = transcript[start:end]
        rel = assign_relation(span, transcript, rel, etype, offset=start)
        turn_id, speaker = -1, ""
        for ti, spk, _utt, toff in turns:
            if toff <= start:
                turn_id, speaker = ti, spk
            else:
                break
        out.append(VerifiedEntity(span=span, entity_type=etype, relation=rel,
                                  section=route_section(etype, rel),
                                  source_turn=turn_id, source_speaker=speaker,
                                  char_start=start, char_end=end))
    return out


def verify_deterministic_tool(e: VerifiedEntity, transcript: str) -> VerifiedEntity:
    """The verifier's second tool: UMLS semantic-type check with no LLM call
    (the 'deterministic' branch of _verify_one, usable while VERIFY_MODE stays as set)."""
    if e.entity_type in BYPASS_UMLS:
        return e
    for cand_span, cs, ce in expand_candidate_spans(e.span, transcript, e.char_start, e.char_end):
        try:
            hits = KB_INSTANCE.link(cand_span, k=RETRIEVE_K)
        except Exception:
            hits = []
        cand, new_type, note = verify_type_deterministic(e, hits)
        if cand is None:
            continue
        rel = assign_relation(cand_span, transcript, e.relation, new_type, offset=cs)
        return VerifiedEntity(span=cand_span, entity_type=new_type, relation=rel,
                              cui=cand.cui, preferred_term=cand.preferred_term,
                              semantic_type=cand.semantic_type,
                              evidence=note + " [agent: deterministic re-check]",
                              section=route_section(new_type, rel), grounded=True,
                              source_turn=e.source_turn, source_speaker=e.source_speaker,
                              char_start=cs, char_end=ce)
    return e


def _grounded_share(ents):
    elig = [e for e in ents if e.entity_type not in BYPASS_UMLS]
    return (sum(1 for e in elig if e.grounded) / len(elig)) if elig else 1.0


def delete_unsupported_tool(note: str, terms) -> Optional[str]:
    """Reviser tool without an LLM: drop the sentences that carry unsupported terms.
    Returns None when that would empty a section (then the LLM reviser is used)."""
    secs = split_soap_sections(note)
    nterms = [_norm(t).strip() for t in terms if _norm(t).strip()]
    changed = False
    for s in ("S", "O", "A", "P"):
        body = secs.get(s, "")
        if not body:
            continue
        sents = [x for x in re.split(r"(?<=[.!?])\s+|\n+", body) if x.strip()]
        keep = [x for x in sents if not any(t in _norm(x) for t in nterms)]
        if len(keep) != len(sents):
            if not keep:
                return None
            secs[s] = " ".join(keep)
            changed = True
    if not changed:
        return None
    return "\n".join(f"{s}: {secs[s].strip()}" for s in ("S", "O", "A", "P") if secs.get(s, "").strip())


def _score_checks(checks):
    """Lower is better: (all violations, unsupported terms)."""
    return (n_violations(checks), len(checks["hallucinated"]))


def _trace_entry(rnd, checks):
    return {"round": rnd, "violations": n_violations(checks),
            "missing": len(checks["missing"]), "misplaced": len(checks["misplaced"]),
            "hallucinated": len(checks["hallucinated"]),
            "inconsistent": len(checks["inconsistent"]),
            "detail": _violation_detail(checks)}


# ---------------------------------------------------------------- agents
def kspan_agent(st: AgentState):
    rid, tr = st["rid"], st["transcript"]
    attempt = int(st.get("kspan_attempts") or 0) + 1
    if attempt == 1:
        cands = extract_kspans(tr, rid=rid)                      # shared with arms A/B/C
        return {"candidates": cands, "kspan_attempts": attempt,
                "log": _alog("K-Span", "attempt 1 (shared Stage A)", entities=len(cands))}
    ck = ck_load("agentD_stageA", rid) if rid else None
    if ck is not None:
        cands = [VerifiedEntity(**e) for e in ck]
    else:
        cands = extract_kspans_with_feedback(tr, st.get("kspan_feedback", ""))
        if rid:
            ck_save("agentD_stageA", rid, [asdict(e) for e in cands])
    return {"candidates": cands, "kspan_attempts": attempt,
            "log": _alog("K-Span", f"attempt {attempt} with verifier feedback", entities=len(cands))}


def verifier_agent(st: AgentState):
    rid, tr, attempt = st["rid"], st["transcript"], st["kspan_attempts"]
    cands = st.get("candidates") or []
    if attempt == 1:
        ents = verify_entities(cands, tr, rid=rid)               # shared Stage B checkpoint
        changed = False
    else:
        ck = ck_load("agentD_stageB", rid) if rid else None
        ents = [VerifiedEntity(**e) for e in ck] if ck is not None else verify_entities(cands, tr, rid="")
        if rid and ck is None:
            ck_save("agentD_stageB", rid, [asdict(e) for e in ents])
        changed = True
    log = _alog("Verifier", f"verified attempt {attempt}", entities=len(ents),
                grounded=f"{_grounded_share(ents):.0%}")

    # tool switch: weak LLM-evidence grounding -> deterministic UMLS re-check of the ungrounded ones
    if VERIFY_MODE == "llm_evidence" and ents and _grounded_share(ents) < AGENT_MIN_GROUNDED:
        ck = ck_load(f"agentD_regrounded{attempt}", rid) if rid else None
        if ck is not None:
            new = [VerifiedEntity(**e) for e in ck]
        else:
            new = [e if e.grounded else verify_deterministic_tool(e, tr) for e in ents]
            if rid:
                ck_save(f"agentD_regrounded{attempt}", rid, [asdict(e) for e in new])
        gained = sum(1 for a, b in zip(ents, new) if b.grounded and not a.grounded)
        if gained:
            ents, changed = new, True
        log += _alog("Verifier", "tool switch: deterministic UMLS re-check", newly_grounded=gained)

    # keep the better of the attempts (more grounded entities, then more entities)
    best = st.get("best_entities")
    score = lambda es: (sum(1 for e in es if e.grounded), len(es))
    if best is not None and score(best) >= score(ents):
        ents, changed = best, bool(st.get("entities_changed"))
        log += _alog("Verifier", "earlier attempt kept (it was better)")

    # decision: send K-Span back?
    too_few = len(tr) > AGENT_LONG_TRANSCRIPT and len(ents) < AGENT_MIN_ENTITIES_LONG
    if too_few and attempt < AGENT_MAX_KSPAN_ATTEMPTS:
        fb = (f"Only {len(ents)} clinical entities were found in a long consultation "
              f"({len(tr)} characters); a full encounter usually has 15-25. Symptoms, findings, "
              f"vitals, tests, medications with dosage, diagnoses, family and past history and "
              f"pertinent negatives must all be extracted.")
        return {"entities": ents, "best_entities": ents, "entities_changed": changed,
                "verifier_decision": "retry", "kspan_feedback": fb,
                "log": log + _alog("Verifier", "decision: RE-EXTRACT (too few entities)")}
    return {"entities": ents, "best_entities": ents, "entities_changed": changed,
            "verifier_decision": "accept", "log": log + _alog("Verifier", "decision: accept")}


def coding_agent(st: AgentState):
    ents = attach_codes(st["entities"], st["transcript"])
    return {"entities": ents, "log": _alog("Coding", "SNOMED / ICD-10 attached",
                                           coded=sum(1 for e in ents if e.snomed))}


def generator_agent(st: AgentState):
    rid, tr, ents = st["rid"], st["transcript"], st["entities"]
    tag = "D" if st.get("entities_changed") else "B"            # unchanged entities -> arm B's draft
    draft = generate_soap(tr, ents, grouped=True, with_evidence=True, rid=rid, tag=tag)
    return {"draft": draft, "note": draft, "round": 0, "rejections": 0, "trace": [],
            "best_note": draft, "best_score": None, "last_rejection": "",
            "log": _alog("Generator", f"draft (gen{tag})", words=len(draft.split()))}


def reflection_agent(st: AgentState):
    ents, tr, note, rnd = st["entities"], st["transcript"], st["note"], int(st.get("round") or 0)
    checks = run_all_checks(ents, note, tr)
    score = _score_checks(checks)
    trace = list(st.get("trace") or []) + [_trace_entry(rnd, checks)]
    best_note, best_score = st.get("best_note") or note, st.get("best_score")
    rejections, last_rej = int(st.get("rejections") or 0), ""
    if best_score is None or score < tuple(best_score):
        best_note, best_score, rejections = note, score, 0
        verdict = "accepted"
    else:
        rejections += 1
        last_rej = (f"the previous correction left {score[0]} problems "
                    f"({score[1]} unsupported terms) instead of fewer than {best_score[0]}")
        verdict = "REJECTED (kept the better draft)"
    best_checks = checks if best_note is note else run_all_checks(ents, best_note, tr)
    targets = _sections_with_findings(best_checks, best_note)
    if best_score[0] == 0:
        decision, why = "accept", "clean"
    elif rnd >= AGENT_MAX_ROUNDS:
        decision, why = "accept", "round limit"
    elif rejections >= AGENT_MAX_REJECTIONS:
        decision, why = "accept", f"{rejections} rejected revisions in a row"
    elif not targets:
        decision, why = "accept", "nothing locatable to fix"
    else:
        decision, why = "revise", f"{best_score[0]} violation(s) in {''.join(targets)}"
    return {"checks": best_checks, "best_note": best_note, "best_score": best_score,
            "rejections": rejections, "last_rejection": last_rej, "trace": trace,
            "reflection_decision": decision,
            "log": _alog("Reflection", f"round {rnd}: {score[0]} violations -> {verdict}; "
                                       f"decision {decision.upper()} ({why})")}


def reviser_agent(st: AgentState):
    tr, note, checks = st["transcript"], st["best_note"], st["checks"]
    rnd = int(st.get("round") or 0) + 1
    only_unsupported = checks["hallucinated"] and not (checks["missing"] or checks["misplaced"]
                                                       or checks["inconsistent"])
    if only_unsupported and not st.get("last_rejection"):
        revised = delete_unsupported_tool(note, checks["hallucinated"])
        if revised:
            return {"note": revised, "round": rnd,
                    "log": _alog("Reviser", f"round {rnd}: deleted unsupported claims (no LLM)",
                                 terms=len(checks["hallucinated"]))}
    targets = _sections_with_findings(checks, note)
    base = split_soap_sections(note)
    show = "\n".join(f"{s}: {base.get(s, '').strip()}" for s in targets)
    findings = _format_findings(checks)
    if st.get("last_rejection"):
        findings += f"\n- NOTE: an earlier correction was rejected because {st['last_rejection']}. " \
                    f"Change only what the findings name."
    revised = (_llm([{"role": "system", "content": "You are a meticulous clinical scribe."},
                     {"role": "user", "content": _CORRECTION_PROMPT
                         .replace("{transcript}", tr[:10000])
                         .replace("{sections}", show)
                         .replace("{findings}", findings)}],
                    max_tokens=1200, temperature=0.1) or "").strip()
    new = _splice_sections(note, revised, targets) if revised else note
    return {"note": new, "round": rnd,
            "log": _alog("Reviser", f"round {rnd}: rewrote sections {''.join(targets)}")}


def _route_verifier(st):
    return "kspan_agent" if st.get("verifier_decision") == "retry" else "coding_agent"

def _route_reflection(st):
    return "reviser_agent" if st.get("reflection_decision") == "revise" else END


_g = StateGraph(AgentState)
for _name, _fn in [("kspan_agent", kspan_agent), ("verifier_agent", verifier_agent),
                   ("coding_agent", coding_agent), ("generator_agent", generator_agent),
                   ("reflection_agent", reflection_agent), ("reviser_agent", reviser_agent)]:
    _g.add_node(_name, _fn)
_g.add_edge(START, "kspan_agent")
_g.add_edge("kspan_agent", "verifier_agent")
_g.add_conditional_edges("verifier_agent", _route_verifier, ["kspan_agent", "coding_agent"])
_g.add_edge("coding_agent", "generator_agent")
_g.add_edge("generator_agent", "reflection_agent")
_g.add_conditional_edges("reflection_agent", _route_reflection, ["reviser_agent", END])
_g.add_edge("reviser_agent", "reflection_agent")
VERISOAP_AGENTS = _g.compile()


def run_arm_D(rec):
    """Agentic VeriSOAP: the agent graph above. Same return shape as arms A/B/C."""
    rid, tr = str(rec["id"]), rec["transcript"]
    cached = ck_load(_mk("note_agentD_full"), rid) if rid else None
    if cached is not None:
        ents = [VerifiedEntity(**e) for e in cached["entities"]]
        stats = cached.get("stats", {})
        with _AGENT_STATS_LOCK:
            AGENT_STATS[(MODEL, rid)] = stats
        return cached["note"], ents, cached["trace"]
    out = VERISOAP_AGENTS.invoke({"rid": rid, "transcript": tr, "kspan_attempts": 0, "log": []},
                                 config={"recursion_limit": 60})
    note, ents = out["best_note"], out["entities"]
    final = run_all_checks(ents, note, tr)
    trace = list(out.get("trace") or []) + [_trace_entry("final", final)]
    log = out.get("log") or []
    stats = {"agent_steps": len(log),
             "agent_kspan_attempts": out.get("kspan_attempts", 1),
             "agent_entities_changed": int(bool(out.get("entities_changed"))),
             "agent_revisions": sum(1 for e in log if e["agent"] == "Reviser"),
             "agent_rejected": sum(1 for e in log if "REJECTED" in e["action"]),
             "agent_code_deletions": sum(1 for e in log if "no LLM" in e["action"])}
    if rid:
        ck_save(_mk("note_agentD_full"), rid, {"note": note, "trace": trace, "stats": stats, "log": log,
                                             "entities": [asdict(e) for e in ents]})
    with _AGENT_STATS_LOCK:
        AGENT_STATS[(MODEL, rid)] = stats
    return note, ents, trace


def show_agent_log(rid, model=None):
    """What the agents decided for one record (after it has been scored)."""
    key = _mk_for(model or MODEL, "note_agentD_full")
    d = ck_load(key, str(rid))
    if not d:
        print(f"no agent log for record {rid} (model {model or MODEL})"); return
    for i, e in enumerate(d.get("log", []), 1):
        extra = "  ".join(f"{k}={v}" for k, v in e.items() if k not in ("agent", "action"))
        print(f"{i:>2}. [{e['agent']}] {e['action']}" + (f"   ({extra})" if extra else ""))
    print(f"   -> {d.get('stats')}")


if RUN_ARM_D:
    ARMS["D_agentic"] = run_arm_D
ARM_DOC["D_agentic"] = (
    "D", "agentic VeriSOAP",
    "The same Stages A-D run by LangGraph agents that decide: the Verifier can send "
    "K-Span back to re-extract (with feedback) or switch to a deterministic UMLS "
    "re-check; Reflection keeps the best draft and rejects revisions that do not "
    "reduce the violations; the Reviser deletes unsupported claims in code or "
    "rewrites only the sections with findings.")
print(f"agentic arm D ready | arms now: {list(ARMS)}")
print(VERISOAP_AGENTS.get_graph().draw_mermaid())

---
## Staged benchmarking — one model per session

Run one model, read its numbers, fix anything wrong, then the next. Each model's results
persist to `artifacts/results_<model>.json`, so a later session accumulates instead of
restarting.

Stage A/B checkpoints are keyed by **record**, not by model, so verification is paid once
and every later model reuses it. Only Stage C/D re-run — which is also the correct
experiment, since the front end is then identical across the whole table.


In [ ]:
# ============================================================================
# Cell RUN - one model per session, results accumulate across sessions.
#
# Run ONE model, look at its numbers, fix anything wrong, then run the next.
# Each model's results are written to artifacts/results_<model>.json, so a later
# session picks up everything already finished instead of redoing it.
#
#     run_model("gpt")        # this session
#     run_model("llama")      # next session
#     compare_models()        # any time - table of everything captured so far
# ============================================================================
def _results_path(model_key):
    return ARTIFACTS / f"results_{model_key}.json"


def run_model(model_key=None, n=None, judge=True, arms=None):
    """Run every arm for ONE model and persist the result.

    Stage A/B checkpoints are keyed by record, not by model, so the expensive
    verification is computed once and reused by every later model. Only Stage C
    and D actually re-run - which is also the correct experiment, since the
    front end is then identical across the whole results table.
    """
    global MODEL
    prev = MODEL
    MODEL = model_key or MODEL
    try:
        spec = MODEL_REGISTRY[MODEL]
        print("=" * 78)
        print(f"MODEL: {MODEL}  ({spec['kind']})"
              + (f"   | paper ROUGE-L {spec['paper_rougeL']} ({spec['paper_mode']})"
                 if spec.get("paper_rougeL") else ""))
        if spec["kind"] == "plm":
            print(f"  PLM: stages A/B/D run on {EXTRACTION_MODEL}; "
                  f"PLM generates only (mode={PLM_MODE})")
        if USE_FINETUNED and not is_trained(MODEL):
            msg = (f"{MODEL} has no fine-tuned weights "
                   f"({_adapter_paths(MODEL) or 'no adapter path'}).")
            if REQUIRE_FINETUNED:
                print("!" * 74)
                print(f"!  REFUSING TO SCORE: {msg}")
                print("!  These would be BASE-model numbers recorded as fine-tuned ones,")
                print("!  which is the fastest way to poison a results table.")
                print(f"!    train_model('{MODEL}')      then run_model('{MODEL}')")
                print("!  To score the base model deliberately, set REQUIRE_FINETUNED=False")
                print("!  or USE_FINETUNED=False - results are then tagged base_model.")
                print("!" * 74)
                return {}
            print(f"  !! {msg} Running BASE - tagged base_model in the saved results.")
        print("=" * 78)

        # RECORDS is the held-out TEST split; nothing here was trained on.
        res = run_batch(records=(RECORDS[:n] if n else RECORDS), arms=arms, judge=judge)
        if not res:
            return {}
        payload = {"model": MODEL, "kind": spec["kind"],
                   "base_model": (not is_trained(MODEL)) or (not USE_FINETUNED),
                   "n_train": len(TRAIN_RECORDS), "n_test": len(TEST_RECORDS),
                   "verify_mode": VERIFY_MODE, "paper_hparams": PAPER_HPARAMS,
                   "n_records": len(RECORDS[:n] if n else RECORDS),
                   "paper_rougeL": spec.get("paper_rougeL"),
                   "results": res}
        _results_path(MODEL).write_text(json.dumps(payload, indent=1, default=str))
        print(f"\nsaved -> {_results_path(MODEL)}")
        results_table(res)
        significance_table(res)
        return res
    finally:
        MODEL = prev


def load_all_results():
    """Every model captured so far, across sessions."""
    out = {}
    for f in sorted(ARTIFACTS.glob("results_*.json")):
        try:
            d = json.loads(f.read_text())
            out[d["model"]] = d
        except Exception:
            continue
    return out


# Headline metrics for the cross-model table. Full per-arm detail stays in the
# per-model JSON; this is the view that goes in the paper.
COMPARE_KEYS = [
    ("rougeL_f1",                  "ROUGE-L"),
    ("bertscore_f1",               "BERTSc-F1"),
    ("concept_f1",                 "MEDCON"),
    ("entity_coverage_rate",       "EntityCov"),
    ("section_placement_accuracy", "Placement"),
    ("hallucination_rate",         "Halluc/note"),
    ("judge5_composite",            "Judge-5"),
    ("judge3_composite",              "Judge-3"),
]


def compare_models(arm="C_verisoap_full"):
    """One row per model, for the arm named. Run after each run_model()."""
    allr = load_all_results()
    # A table with zero measured models is still worth printing: the projected
    # block below is the whole point before training finishes.
    if not allr and not (load_projections() if "load_projections" in globals() else {}):
        print("Nothing measured and nothing projected yet.")
        return {}
    w = 12
    hdr = f"{'model':<10}{'kind':<8}" + "".join(f"{lbl:>{w}}" for _, lbl in COMPARE_KEYS)
    _l, _short, _ = ARM_DOC.get(arm, (arm, arm, ""))
    print("=" * len(hdr))
    print(f"CROSS-MODEL COMPARISON  |  arm {_l}: {_short}  ({arm})")
    print("=" * len(hdr))
    print(hdr); print("-" * len(hdr))
    for name, d in sorted(allr.items(),
                          key=lambda kv: -(kv[1].get("paper_rougeL") or 0)):
        rows = d["results"].get(arm) or next(iter(d["results"].values()), [])
        tag = ("*" if d.get("base_model") else "") + ("+" if d.get("split_unsafe") else "")
        line = f"{name + tag:<10}{d.get('kind',''):<8}"
        for key, _ in COMPARE_KEYS:
            vals = [r[key] for r in rows
                    if isinstance(r.get(key), (int, float)) and r[key] == r[key]]
            line += f"{(statistics.fmean(vals) if vals else float('nan')):>{w}.3f}"
        print(line)
    projection_block(arm, COMPARE_KEYS, w,
                     lambda m, d: f"{'~' + m:<10}{d.get('kind', ''):<8}", len(hdr))
    print("-" * len(hdr))
    print("ROUGE-L / BERTScore-F1 / MEDCON = overlap with the gold note "
          "(word, embedding, UMLS-concept).")
    print("EntityCov = verified entities reaching the note. Placement = entities in "
          "the right S/O/A/P section.")
    print("Halluc/note = unsupported claims per note, LOWER is better. "
          "Judge-5 / Judge-4 = LLM judges, mean of their axes, 1-5.")
    if any(d.get("base_model") for d in allr.values()):
        print("* = BASE model, NOT fine-tuned - not comparable to the fine-tuned rows.")
    if any(d.get("split_unsafe") for d in allr.values()):
        print("+ = fine-tuned BEFORE the train/test split existed; may have seen test")
        print("    records. Quote it as a reference point, never as a fair comparison.")
    if not allr:
        print("(no measured rows yet - every row above is projected)")
    n_tr = next((d.get("n_train") for d in allr.values() if d.get("n_train")), None)
    n_te = next((d.get("n_test") for d in allr.values() if d.get("n_test")), None)
    if n_tr:
        print(f"trained on {n_tr} records, scored on {n_te} held-out records.")
    print("K-SOAP paper ROUGE-L, for orientation only (CliniKnote, different data):")
    for name, spec in sorted(MODEL_REGISTRY.items()):
        if spec.get("paper_rougeL"):
            print(f"   {name:<10} {spec['paper_rougeL']:.3f}  ({spec['paper_mode']})")
    todo = [m for m in MODEL_REGISTRY if m not in allr]
    if todo:
        _p = load_projections() if "load_projections" in globals() else {}
        shown = [m for m in todo if m in _p]
        rest = [m for m in todo if m not in _p]
        if shown:
            print(f"\nnot measured yet, shown above as projections: {', '.join(shown)}")
        if rest:
            print(f"\nnot measured and not projected: {', '.join(rest)}")
        print("   Train them in verisoap_run_all.ipynb, then re-run this notebook -")
        print("   each measured row replaces its projected one automatically.")
    return allr


def export_all(stem="verisoap_all_models"):
    """CSV + JSON for the paper. Every row says whether it was measured."""
    import csv
    allr = load_all_results()
    rows = [{**r, "model": m, "kind": d.get("kind"), "arm": arm, "measured": True,
             "base_model": bool(d.get("base_model")),
             "split_unsafe": bool(d.get("split_unsafe"))}
            for m, d in allr.items() for arm, rs in d["results"].items() for r in rs]
    # Projections go in as ONE aggregate row per (model, arm), flagged measured=False
    # and with record_id left blank, so no spreadsheet can average them in with
    # per-record measurements by accident.
    for m, d in load_projections().items():
        for arm, agg in d.get("aggregates", {}).items():
            rows.append({**{k: v for k, v in agg.items() if not k.startswith("_")},
                         "model": m, "kind": d.get("kind"), "arm": arm,
                         "measured": False, "projected_from": d.get("anchor"),
                         "projection_basis": d.get("basis"), "record_id": ""})
    if not rows:
        print("nothing to export")
        return
    (ARTIFACTS / f"{stem}.json").write_text(json.dumps(allr, indent=1, default=str))
    keys = sorted({k for r in rows for k in r})
    with open(ARTIFACTS / f"{stem}.csv", "w", newline="") as f:
        wtr = csv.DictWriter(f, fieldnames=keys); wtr.writeheader(); wtr.writerows(rows)
    print(f"exported {len(rows)} rows across {len(allr)} models -> {ARTIFACTS/stem}.csv")


print("staged runner ready.")
print("   run_model('gpt')      # one model per session")
print("   compare_models()      # everything captured so far")
print("   export_all()          # one CSV for the paper")
print("   captured so far:", list(load_all_results()) or "none")

In [ ]:
# ============================================================================
# Cell BLOB - move results off Kaggle without losing anything.
#
# Kaggle wipes /kaggle/working between batch commits, so results have to leave
# the session. Two ways out:
#   results_blob()          prints ONE compact line -> paste it into chat
#   import_results(blob)    stores it locally, exactly as run_model() would have
#
# The blob is real JSON, so nothing is lost to terminal formatting or truncation.
# ============================================================================
import base64, zlib


def results_blob(models=None, quiet=False):
    """Compact, copy-pasteable capsule of every captured result."""
    allr = load_all_results()
    if models:
        allr = {k: v for k, v in allr.items() if k in set(models)}
    if not allr:
        print("Nothing captured yet - run run_model('...') first.")
        return ""
    raw = json.dumps(allr, separators=(",", ":"), default=str).encode()
    blob = "VERISOAP1:" + base64.b64encode(zlib.compress(raw, 9)).decode()
    if not quiet:
        print("=" * 78)
        print(f"RESULTS BLOB  ({len(allr)} model(s): {', '.join(allr)})")
        print(f"{len(raw):,} bytes -> {len(blob):,} chars. Copy the WHOLE line below.")
        print("=" * 78)
        print(blob)
        print("=" * 78)
    return blob


def import_results(blob, overwrite=False):
    """Restore a blob into artifacts/ so compare_models() sees it."""
    blob = "".join(str(blob).split())
    if not blob.startswith("VERISOAP1:"):
        print("Not a VeriSOAP blob (must start with 'VERISOAP1:').")
        return {}
    try:
        data = json.loads(zlib.decompress(base64.b64decode(blob[len("VERISOAP1:"):])))
    except Exception as ex:
        print(f"Could not decode: {type(ex).__name__}. Was the whole line copied?")
        return {}
    saved = []
    for model, payload in data.items():
        path = _results_path(model)
        if path.exists() and not overwrite:
            print(f"  {model}: already present, skipped (overwrite=True to replace)")
            continue
        path.write_text(json.dumps(payload, indent=1, default=str))
        saved.append(model)
        n = sum(len(v) for v in payload.get("results", {}).values())
        print(f"  {model}: imported ({n} rows, "
              f"{'BASE MODEL' if payload.get('base_model') else 'fine-tuned'})")
    print(f"\nimported {len(saved)}: {', '.join(saved) if saved else 'none'}")
    return data


def save_results_file(stem="verisoap_results_export"):
    """Write one JSON file to download from the Kaggle output panel."""
    allr = load_all_results()
    if not allr:
        print("nothing to save")
        return None
    f = ARTIFACTS / f"{stem}.json"
    f.write_text(json.dumps(allr, indent=1, default=str))
    print(f"saved {len(allr)} model(s) -> {f}")
    print("Download it from the Kaggle 'Output' panel, or use results_blob() to paste.")
    return f


print("results transfer ready.")
print("   results_blob()        # paste this line into chat")
print("   save_results_file()   # or download the JSON from Kaggle's Output panel")
print("   import_results(blob)  # restore on the other side")

### Fine-tune first — every model, on the same split

For a paper each model must be fine-tuned on the **same** `TRAIN_RECORDS` before any
comparison. Scoring a fine-tuned GPT against a base LLaMA measures *fine-tuned vs not*,
not model quality.

`TEST_RECORDS` are held out from the start and are the only records the results table
ever sees.


In [ ]:
# ============================================================================
# Cell TRAIN - fine-tune ONE model per session, then evaluate it.
#
# For a paper every model has to be fine-tuned on the SAME training split before
# any comparison. Comparing a fine-tuned GPT against a base LLaMA measures
# "fine-tuned vs not", not model quality.
#
#     train_model("llama")     # trains on TRAIN_RECORDS, saves the adapter
#     run_model("llama")       # scores on TEST_RECORDS, saves results
#     training_status()        # what is trained, what is left
# ============================================================================
def find_adapter(path):
    """Locate an adapter, including one restored from a previous Kaggle run.

    A Kaggle batch commit starts with an EMPTY /kaggle/working, so adapters
    trained in an earlier commit are gone. They do survive as that notebook's
    OUTPUT, which you re-attach as an input dataset - which lands them under
    /kaggle/input/<something>/. Searching there makes training resume across
    commits instead of retraining from scratch every time.
    """
    if not path:
        return None
    if os.path.isdir(path):
        return path
    base = os.path.basename(path.rstrip("/"))
    for pat in (f"/kaggle/input/*/{base}", f"/kaggle/input/*/*/{base}"):
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    return None


def _adapter_paths(model_key):
    spec = MODEL_REGISTRY[model_key]
    if spec["kind"] == "openai":
        return [spec["ft"]] if spec.get("ft") else []
    if spec["kind"] == "plm":
        return list(_plm_paths(spec).values())
    return [spec["out"]]


# ---------------------------------------------------------------------------
# COMPLETION MARKERS - why FRESH_START is gone.
#
# An adapter directory exists the moment Trainer writes its first checkpoint, so
# "the directory is there" is not the same as "this model finished training". A
# crashed run therefore looked trained, the queue skipped it, and a broken model
# went into the results table. The old fix was to delete every adapter at startup,
# which is how bart and biobart were lost.
#
# A marker settles it without deleting anything: _verisoap_done.json is written
# only after training returns AND the adapter is on disk. No marker -> retrain.
# It also records the hyperparameters that produced the adapter, so a results
# table can never silently mix two training configs.
# ---------------------------------------------------------------------------
DONE_MARKER = "_verisoap_done.json"


def _train_signature():
    """The settings that define what an adapter IS.

    Two adapters trained under different settings must never sit in the same
    results table, and on 2026-09-15 they nearly did: bart and biobart finished
    under GRAD_ACCUM=16 (eight optimizer steps) and MAX_SEQ=4096, and when those
    were corrected the completion marker would have happily declared them done.
    Recording the signature and comparing it makes a config change invalidate the
    adapter instead of silently mixing two experiments.
    """
    return {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
            "epochs": NUM_EPOCHS, "grad_accum": GRAD_ACCUM, "lr": LR,
            "max_seq": MAX_SEQ, "warmup_ratio": WARMUP_RATIO,
            "plm_lora": PLM_USE_LORA, "plm_mode": PLM_MODE,
            "train_records": len(TRAIN_RECORDS), "dataset": DATASET}


# FALSE: this notebook trains, so an adapter built under different settings must
# be rebuilt rather than reported. bart and biobart on the Hub were trained under
# GRAD_ACCUM=16 / MAX_SEQ=4096 (8-12 optimizer steps, which is a warm-up not a
# fine-tune); they carry no `signature` in their marker and will be retrained.
# Two training configs must never merge into one measured results table.
ALLOW_STALE_ADAPTERS = False


def _marker(d):
    try:
        with open(os.path.join(d, DONE_MARKER)) as f:
            return json.load(f)
    except Exception:
        return None


def _is_complete(d):
    """Finished AND trained under the settings this notebook is currently using."""
    if not (d and os.path.isdir(d)):
        return False
    m = _marker(d)
    if m is None:
        return False
    if ALLOW_STALE_ADAPTERS:
        return True
    return m.get("signature") == _train_signature()


def _stale_reason(d):
    """Why an existing adapter is being retrained. Printed, never guessed at."""
    m = _marker(d)
    if m is None:
        return "no completion marker (crashed run)"
    old, new = m.get("signature") or {}, _train_signature()
    diff = [f"{k}: {old.get(k)!r} -> {new[k]!r}" for k in new if old.get(k) != new[k]]
    return "settings changed - " + "; ".join(diff) if diff else ""


def _mark_done(model_key):
    """Stamp every adapter directory of this model as finished."""
    stamped = []
    for p in _adapter_paths(model_key):
        d = find_adapter(p)
        if not (d and os.path.isdir(d)):
            continue
        sec = os.path.basename(d.rstrip("/")).rsplit("-", 1)[-1]
        loss = LAST_TRAIN_LOSS.get(f"{model_key}:{sec}",
                                   LAST_TRAIN_LOSS.get(model_key))
        try:
            with open(os.path.join(d, DONE_MARKER), "w") as f:
                json.dump({"model": model_key,
                           "base_id": MODEL_REGISTRY[model_key]["id"],
                           "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                                         time.gmtime()),
                           "final_train_loss": loss,
                           "paper_hparams": PAPER_HPARAMS,
                           "signature": _train_signature()}, f, indent=1)
            stamped.append(os.path.basename(d.rstrip("/"))
                           + (f" (loss {loss:.3f})" if loss is not None else ""))
        except OSError as ex:
            print(f"   [marker] could not stamp {d}: {ex}")
    return stamped


def is_trained(model_key):
    """True only when every artefact exists AND carries a completion marker."""
    spec = MODEL_REGISTRY[model_key]
    if spec["kind"] == "openai":
        return bool(spec.get("ft"))
    paths = _adapter_paths(model_key)
    return bool(paths) and all(_is_complete(find_adapter(p)) for p in paths)


def _partially_trained(model_key):
    """Adapter dirs exist but are not usable: crashed, or trained under other settings."""
    if MODEL_REGISTRY[model_key]["kind"] == "openai":
        return False
    found = [find_adapter(p) for p in _adapter_paths(model_key)]
    return any(found) and not all(_is_complete(d) for d in found)


def stale_report(models=None):
    """Adapters present on disk that this run will retrain, and exactly why."""
    rows = []
    for k in (models or [m for m in MODEL_REGISTRY if MODEL_REGISTRY[m]["kind"] != "openai"]):
        for pth in _adapter_paths(k):
            d = find_adapter(pth)
            if d and not _is_complete(d):
                rows.append((k, os.path.basename(d.rstrip("/")), _stale_reason(d)))
    if rows:
        print("Adapters on disk that will be RETRAINED:")
        seen = set()
        for k, base, why in rows:
            if (k, why) in seen:
                continue
            seen.add((k, why))
            print(f"  {k:<10} {why}")
    return rows


def training_status():
    print(f"{'model':<10}{'kind':<8}{'trained':<9}{'scored':<8}artefacts")
    print("-" * 78)
    done = load_all_results()
    for k, spec in MODEL_REGISTRY.items():
        t = "YES" if is_trained(k) else ("partial" if _partially_trained(k) else "no")
        r = "YES" if k in done else "no"
        if spec["kind"] == "openai":
            where = spec.get("ft") or "-"
        else:
            where = ", ".join(
                os.path.basename(p) + ("" if os.path.isdir(p) or not find_adapter(p)
                                       else " (from /kaggle/input)")
                for p in _adapter_paths(k))
        print(f"{k:<10}{spec['kind']:<8}{t:<9}{r:<8}{where[:44]}")
    print("-" * 78)
    todo = [k for k in MODEL_REGISTRY if not is_trained(k)]
    print(f"to train: {', '.join(todo) if todo else 'none - all trained'}")
    print(f"train on {len(TRAIN_RECORDS)} records, score on {len(TEST_RECORDS)} held out")
    return todo


def train_model(model_key, force=False):
    """Fine-tune ONE model on TRAIN_RECORDS. Never touches TEST_RECORDS."""
    global MODEL
    spec = MODEL_REGISTRY[model_key]
    if is_trained(model_key) and not force:
        print(f"{model_key} already trained -> {_adapter_paths(model_key)}")
        print("   pass force=True to retrain.")
        return
    if not TRAIN_RECORDS:
        print("No training records. Attach clean_english_100.jsonl first.")
        return

    prev = MODEL
    MODEL = model_key
    try:
        print("=" * 78)
        print(f"TRAINING {model_key} ({spec['kind']}) on {len(TRAIN_RECORDS)} records")
        print(f"  hparams: r={LORA_R} alpha={LORA_ALPHA} epochs={NUM_EPOCHS} lr={LR} "
              f"(paper_hparams={PAPER_HPARAMS})")
        print(f"  TEST_RECORDS ({len(TEST_RECORDS)}) are held out and never seen here.")
        print("=" * 78)
        t0 = time.time()
        if spec["kind"] == "openai":
            path = build_openai_ft_jsonl(TRAIN_RECORDS,
                                         ARTIFACTS / f"ft_{model_key}.jsonl")
            job = launch_openai_ft(path, base_model=spec["id"], suffix=f"verisoap-{model_key}")
            print(f"\nOpenAI job {job} launched. When it finishes, put the resulting")
            print(f"model id in MODEL_REGISTRY['{model_key}']['ft'] and re-run CONFIG.")
            return job
        elif spec["kind"] == "plm":
            out = train_plm(TRAIN_RECORDS)
        else:
            out = train_qlora(TRAIN_RECORDS)
        mark = _mark_done(model_key)
        print(f"\ntrained in {(time.time()-t0)/60:.1f} min -> {out}")
        if mark:
            print(f"   marked complete: {', '.join(mark)}")
        else:
            print("   WARNING no adapter directory found to mark - treated as FAILED.")
        return out
    finally:
        MODEL = prev


def train_and_run(model_key, force=False):
    """Train then immediately score. The whole per-model session in one call."""
    train_model(model_key, force=force)
    if MODEL_REGISTRY[model_key]["kind"] == "openai" and not is_trained(model_key):
        print("OpenAI fine-tune is asynchronous - re-run run_model() once it completes.")
        return None
    return run_model(model_key)


print("staged trainer ready.")
print("   training_status()          # what is trained / scored / left")
print("   train_model('llama')       # fine-tune one model on TRAIN_RECORDS")
print("   train_and_run('llama')     # train, then score on TEST_RECORDS")

### Train everything (resumable)

Roughly **4-6 GPU hours** for the seven open models on a Kaggle T4 x2 — fits a 12h
session, but eats most of a 9h/week quota. `llama2` and `qcammel` are 13B and are the
risk: tight in 4-bit on a 16GB card.

Built to survive interruption rather than to run in one go — cheapest models first,
trained models skipped on re-run, GPU freed between models, and one failure does not
stop the queue.


In [ ]:
# ============================================================================
# Cell TRAINALL - fine-tune EVERY model, sequentially and resumably.
#
# Feasibility, honestly: on one Kaggle T4 x2 session this is roughly 4-6 GPU
# hours for the seven open models. That fits a 12h session but eats most of a
# 9h/week quota, and the two 13B models (llama2, qcammel) are the risk - they
# are tight in 4-bit on a 16GB card.
#
# So it is built to survive interruption rather than to be run in one heroic go:
#   * cheapest models first, so useful adapters exist early
#   * already-trained models are SKIPPED, so re-running resumes
#   * GPU memory is released between models or the 13B loads OOM
#   * one model failing does not stop the rest
#
#     train_all_models()                  # everything not yet trained
#     train_all_models(only=["bart"])     # just these
#     train_all_models(skip_13b=True)     # duck the two expensive ones
# ============================================================================
# ---------------------------------------------------------------------------
# Already trained on 2026-09-15 (200 train records, 4 S/O/A/P sections each):
#     bart     38.7 min   /kaggle/working/bart-soap-{S,O,A,P}
#     biobart   8.8 min   /kaggle/working/biobart-soap-{S,O,A,P}
# Both are out of the queue. To retrain either, move it back into TRAIN_ORDER.
DONE_ALREADY = ["bart", "biobart"]

# THE QUEUE. Everything is in it - is_trained() skips whatever is already done,
# so there is no hand-maintained "already trained" list to drift out of sync.
#
# bart and biobart trained once (38.7 and 8.8 min on 200 records) and were then
# lost: /kaggle/working is wiped between sessions and the old FRESH_START cell
# deleted adapters on startup. Both are back, and the marker + Hub scheme means
# that cannot happen again.
#
# Cheapest first, so a session that dies early still leaves usable adapters.
# llama2 is gated on Meta's licence; if the token cannot reach it the queue drops
# it automatically instead of failing.
TRAIN_ORDER  = ["biobart", "bart", "qwen", "llama", "openbio", "qcammel", "llama2"]

# Models that CANNOT train here, so the queue must not even try.
#
# qcammel is 13B. It OOM-ed on a T4 twice - on 2026-09-15 mid-training and on
# 2026-09-20 during the 26 GB weight load itself, before step 1. Attempting it is
# not just wasted time: the half-loaded shards were still resident afterwards and
# fed the leak that then OOM-ed the whole scoring phase. Both it and llama2 get a
# projected row, scaled from the measured models.
SKIP_TRAINING = {"qcammel"}
DONE_ALREADY = []
ALL_MODELS   = TRAIN_ORDER
BIG_MODELS  = {"llama2", "qcammel"}          # 13B - slow, and tight on a T4


def _free_gpu():
    """Release the previous model, and report DISK - biobart died on a full disk,
    not on memory, and the torch error for that ('unexpected pos N vs M') gives
    no hint that space is the cause."""
    for cache in (_HF_PIPE_HOLDER, _PLM):
        cache.clear() if isinstance(cache, dict) else None
    _cuda_purge()
    for d, f, t in _gpu_free():
        print(f"   [gpu{d}] {f:.1f} GiB free of {t:.1f} GiB")
    if PURGE_HF_CACHE:
        import glob as _g
        hub = os.path.expanduser("~/.cache/huggingface/hub")
        for d in _g.glob(os.path.join(hub, "models--*")):
            try:
                shutil.rmtree(d, ignore_errors=True)
            except Exception:
                pass
    try:
        du = shutil.disk_usage("/kaggle/working" if ON_KAGGLE else ".")
        print(f"   [disk] {du.free/2**30:.1f} GiB free of {du.total/2**30:.1f} GiB")
        if du.free < 3 * 2**30:
            print("   [disk] WARNING under 3 GiB - saves will fail. "
                  "Delete old adapters or set PLM_USE_LORA=True.")
    except Exception:
        pass


def train_all_models(only=None, skip=None, skip_13b=False, force=False,
                     include_gpt=False, time_budget_min=None):
    """Fine-tune every model in TRAIN_ORDER. Returns a per-model status dict."""
    if not TRAIN_RECORDS:
        print("No training records - attach clean_english_100.jsonl first.")
        return {}

    todo = list(only) if only else list(TRAIN_ORDER)
    if include_gpt:
        todo = ["gpt"] + todo
    todo = [m for m in todo if m not in (set(skip or ()) | set(SKIP_TRAINING))]
    if SKIP_TRAINING:
        print(f"not attempted (cannot fit this GPU): {', '.join(sorted(SKIP_TRAINING))}")
    if skip_13b:
        todo = [m for m in todo if m not in BIG_MODELS]

    print("=" * 78)
    print(f"TRAIN ALL  |  {len(TRAIN_RECORDS)} train / {len(TEST_RECORDS)} held-out test")
    print(f"hparams    |  r={LORA_R} alpha={LORA_ALPHA} epochs={NUM_EPOCHS} lr={LR} "
          f"(paper_hparams={PAPER_HPARAMS})")
    print(f"queue      |  {' -> '.join(todo)}")
    print("backup     |  " + (f"hub {HUB['repo']}" if HUB.get("write")
                              else "hub OFF - nothing will survive this session"))
    if time_budget_min:
        print(f"budget     |  stop starting new models after {time_budget_min:.0f} min")
    print("=" * 78)

    status, t_start = {}, time.time()
    for n, key in enumerate(todo, 1):
        elapsed = (time.time() - t_start) / 60
        if time_budget_min and elapsed > time_budget_min:
            status[key] = ("skipped", "time budget reached")
            print(f"\n[{n}/{len(todo)}] {key}: SKIPPED - {elapsed:.0f} min used")
            continue
        if is_trained(key) and not force:
            status[key] = ("already", "adapter present")
            print(f"\n[{n}/{len(todo)}] {key}: already trained, skipping")
            continue
        if _partially_trained(key):
            for _p in _adapter_paths(key):
                _d = find_adapter(_p)
                if _d and not _is_complete(_d):
                    print(f"\n[{n}/{len(todo)}] {key}: retraining - {_stale_reason(_d)}")
                    break

        print(f"\n[{n}/{len(todo)}] {key}"
              + ("   (13B - the slow one)" if key in BIG_MODELS else ""))
        t0 = time.time()
        try:
            _free_gpu()
            train_model(key, force=force)
            mins = (time.time() - t0) / 60
            ok = is_trained(key) or MODEL_REGISTRY[key]["kind"] == "openai"
            status[key] = ("trained" if ok else "no artefact", f"{mins:.1f} min")
            print(f"   -> {status[key][0]} in {mins:.1f} min")
            if ok:
                # Push NOW, not at the end: if the session dies during the next
                # model, this one is already safe.
                hub_push_model(key)
                hub_push_artifacts(verbose=False)
        except Exception as ex:
            status[key] = ("FAILED", f"{type(ex).__name__}: {str(ex)[:120]}")
            print(f"   -> FAILED: {type(ex).__name__}: {str(ex)[:160]}")
            if "out of memory" in str(ex).lower():
                print(f"      OOM. MAX_SEQ={MAX_SEQ}, batch=1, accum={GRAD_ACCUM}. "
                      f"Lower MAX_SEQ before retrying this model.")
            # An exception object holds its traceback, the traceback holds the
            # failing frame, and that frame still references the 4-bit model. Not
            # clearing it leaked ~2 GiB of VRAM into the next model on 2026-09-15.
            ex.__traceback__ = None
            del ex
            print("      continuing with the next model.")
        finally:
            _free_gpu()

    print("\n" + "=" * 78)
    print(f"{'model':<12}{'status':<14}detail")
    print("-" * 78)
    for k, (st, detail) in status.items():
        print(f"{k:<12}{st:<14}{detail[:50]}")
    print("-" * 78)
    print(f"total {(time.time()-t_start)/60:.1f} min")
    left = [m for m in TRAIN_ORDER if not is_trained(m)]
    print(f"still untrained: {', '.join(left) if left else 'none'}")
    if left:
        print("   Re-run this notebook - trained models are skipped, so it resumes.")
    hub_push_artifacts()
    print("=" * 78)
    return status


def run_all_models(only=None, skip=None):
    """Score every TRAINED model on the held-out test split, one at a time."""
    todo = list(only) if only else ["gpt"] + TRAIN_ORDER
    todo = [m for m in todo if m not in (set(skip or ()) | set(SKIP_TRAINING))]
    if SKIP_TRAINING:
        print(f"not attempted (cannot fit this GPU): {', '.join(sorted(SKIP_TRAINING))}")
    done = {}
    for key in todo:
        if not is_trained(key):
            print(f"\n{key}: not trained yet - skipping (train_model('{key}') first)")
            continue
        try:
            done[key] = run_model(key)
        except Exception as ex:
            print(f"\n{key}: scoring FAILED - {type(ex).__name__}: {str(ex)[:140]}")
    print("\n")
    compare_models()
    return done


print("train_all_models() / run_all_models() ready.")
print("   train_all_models()                # everything untrained, resumable")
print("   train_all_models(skip_13b=True)   # skip llama2 + qcammel")
print("   run_all_models()                  # score everything trained")

---
## Medical coding — SNOMED CT / ICD-10-CM / CPT

MedCodER-style coding, ported from `audio2soap.ipynb`: the 194-entry SNOMED keyword table and
the 25-entry CPT table come across **verbatim**.

One upgrade over the original: Stage B has already resolved a UMLS CUI and preferred term for
each entity, so coding now tries the **UMLS preferred term** as well as the raw span. "tummy
ache" never keyword-matches the SNOMED table, but its preferred term *Abdominal pain* does.

Negated entities are never coded — you do not bill for a symptom the patient denied.

In [ ]:
# ============================================================================
# Cell MC.1 - coding tables.  PORTED VERBATIM from audio2soap.ipynb.
#             194 SNOMED concepts, 25 CPT codes.
# ============================================================================
SNOMED_TABLE = [
    {"id":"422587007","t":"Nausea","c":"R11.0","kw":["nausea","nauseated","nauseous","feeling sick"]},
    {"id":"422400008","t":"Vomiting","c":"R11.10","kw":["vomiting","vomit","throw up","throwing up","emesis"]},
    {"id":"73879007","t":"Nausea and vomiting","c":"R11.2","kw":["nausea and vomiting","nausea with vomiting"]},
    {"id":"62315008","t":"Diarrhea","c":"R19.7","kw":["diarrhea","diarrhoea","loose stools","watery stools","diadea"]},
    {"id":"14760008","t":"Constipation","c":"K59.00","kw":["constipation","constipated"]},
    {"id":"21522001","t":"Abdominal pain","c":"R10.9","kw":["abdominal pain","belly pain","stomach pain","stomachache","abdominal tenderness"]},
    {"id":"73063007","t":"Abdominal cramps","c":"R10.84","kw":["cramps","crampy","cramping","abdominal cramps","stomach cramps"]},
    {"id":"698065002","t":"Heartburn","c":"R12","kw":["heartburn","acid reflux","pyrosis"]},
    {"id":"235365009","t":"GERD","c":"K21.0","kw":["gerd","gastroesophageal reflux","reflux disease"]},
    {"id":"235856003","t":"Gastroenteritis","c":"K52.9","kw":["gastroenteritis","stomach flu","stomach bug","food poisoning"]},
    {"id":"4556007","t":"Gastritis","c":"K29.70","kw":["gastritis"]},
    {"id":"64226004","t":"Dysphagia","c":"R13.10","kw":["dysphagia","difficulty swallowing","trouble swallowing"]},
    {"id":"271681002","t":"Loss of appetite","c":"R63.0","kw":["loss of appetite","not eating","not hungry","decreased appetite","anorexia"]},
    {"id":"249497008","t":"Bloating","c":"R14.0","kw":["bloating","bloated","distended","abdominal distension"]},
    {"id":"162607003","t":"Blood in stool","c":"K92.1","kw":["blood in stool","blood in stools","blood in your stools","rectal bleeding","bloody stool","melena","hematochezia"]},
    {"id":"422823003","t":"Hematemesis","c":"K92.0","kw":["blood in vomit","hematemesis","vomiting blood"]},
    {"id":"197456007","t":"IBS","c":"K58.9","kw":["irritable bowel","ibs"]},
    {"id":"396332003","t":"Appendicitis","c":"K35.80","kw":["appendicitis"]},
    {"id":"34000006","t":"Crohn disease","c":"K50.90","kw":["crohn","crohn's"]},
    {"id":"64766004","t":"Ulcerative colitis","c":"K51.90","kw":["ulcerative colitis","colitis"]},
    {"id":"44054006","t":"Bowel obstruction","c":"K56.60","kw":["bowel obstruction","intestinal obstruction"]},
    {"id":"60728008","t":"Jaundice","c":"R17","kw":["jaundice","yellow skin","icterus","yellow eyes"]},

    {"id":"29857009","t":"Chest pain","c":"R07.9","kw":["chest pain","chest tightness","tight feeling in my chest","pressure on my chest","chest pressure"]},
    {"id":"80313002","t":"Palpitations","c":"R00.2","kw":["palpitations","heart racing","racing of the heart","heart pounding","beating faster","heart fluttering","thumping"]},
    {"id":"3424008","t":"Tachycardia","c":"R00.0","kw":["tachycardia","fast heart rate","rapid heart rate"]},
    {"id":"48867003","t":"Bradycardia","c":"R00.1","kw":["bradycardia","slow heart rate"]},
    {"id":"42343007","t":"Heart failure","c":"I50.9","kw":["heart failure","chf","congestive heart failure","fluid overload"]},
    {"id":"22298006","t":"Myocardial infarction","c":"I21.9","kw":["heart attack","myocardial infarction","mi","stemi","nstemi"]},
    {"id":"194828000","t":"Angina","c":"I20.9","kw":["angina","angina pectoris"]},
    {"id":"38341003","t":"Hypertension","c":"I10","kw":["high blood pressure","hypertension","elevated bp","elevated blood pressure","blood pressure high"]},
    {"id":"45007003","t":"Hypotension","c":"I95.9","kw":["low blood pressure","hypotension"]},
    {"id":"49436004","t":"Atrial fibrillation","c":"I48.91","kw":["atrial fibrillation","afib","a-fib","irregular heartbeat"]},
    {"id":"233604007","t":"Pericarditis","c":"I30.9","kw":["pericarditis","pericardial"]},
    {"id":"253273006","t":"Heart murmur","c":"R01.1","kw":["heart murmur","murmur"]},
    {"id":"59282003","t":"Pulmonary edema","c":"J81.0","kw":["fluid in lungs","pulmonary edema","fluid in the lungs"]},

    {"id":"49727002","t":"Cough","c":"R05.9","kw":["cough","coughing","persistent cough","dry cough","productive cough"]},
    {"id":"267036007","t":"Dyspnea","c":"R06.00","kw":["shortness of breath","difficulty breathing","trouble breathing","short of breath","breathless","dyspnea","winded","gasping for air"]},
    {"id":"56018004","t":"Wheezing","c":"R06.2","kw":["wheeze","wheezing"]},
    {"id":"48409008","t":"Orthopnea","c":"R06.01","kw":["orthopnea","can't breathe lying flat","breathe lying flat","prop up with pillows","pillows at night"]},
    {"id":"70572006","t":"Stridor","c":"R06.1","kw":["stridor","noisy breathing"]},
    {"id":"195967001","t":"Asthma","c":"J45.909","kw":["asthma","asthmatic","reactive airway"]},
    {"id":"13645005","t":"COPD","c":"J44.1","kw":["copd","chronic obstructive","emphysema"]},
    {"id":"233604007","t":"Pneumonia","c":"J18.9","kw":["pneumonia"]},
    {"id":"233703007","t":"Pulmonary embolism","c":"I26.99","kw":["pulmonary embolism","pe","blood clot in lung"]},
    {"id":"18165001","t":"Pleural effusion","c":"J90","kw":["pleural effusion","fluid around lungs"]},
    {"id":"65124004","t":"Crackles","c":"R09.89","kw":["crackles","rales","crepitations","bibasilar crackles"]},
    {"id":"267038008","t":"Edema","c":"R60.0","kw":["swelling","edema","oedema","swollen","pitting edema","ankle swelling","swelling in my ankles","swelling in ankles","leg swelling"]},
    {"id":"87433001","t":"Hemoptysis","c":"R04.2","kw":["coughing blood","hemoptysis","blood in sputum","bringing up blood"]},
    {"id":"68235000","t":"Nasal congestion","c":"R09.81","kw":["congestion","stuffy nose","blocked nose","runny nose","nasal congestion","rhinorrhea"]},
    {"id":"162397003","t":"Sore throat","c":"R07.0","kw":["sore throat","throat pain","pharyngitis","scratchy throat"]},
    {"id":"301354004","t":"Mucus production","c":"R09.3","kw":["mucus","phlegm","sputum","clear mucus"]},

    {"id":"25064002","t":"Headache","c":"R51.9","kw":["headache","headaches","head pain","cephalgia"]},
    {"id":"37796009","t":"Migraine","c":"G43.909","kw":["migraine","migraines"]},
    {"id":"404640003","t":"Dizziness","c":"R42","kw":["dizziness","dizzy","lightheaded","lightheadedness","vertigo","room spinning"]},
    {"id":"271594007","t":"Syncope","c":"R55","kw":["syncope","fainted","fainting","faint","passed out","loss of consciousness","lost consciousness","blacked out"]},
    {"id":"62106007","t":"Numbness","c":"R20.0","kw":["numbness","numb","tingling","paresthesia","pins and needles"]},
    {"id":"26079004","t":"Tremor","c":"R25.1","kw":["tremor","tremors","shaking","shaky"]},
    {"id":"91175000","t":"Seizure","c":"R56.9","kw":["seizure","seizures","convulsion","convulsions","fit","fits"]},
    {"id":"230690007","t":"Stroke","c":"I63.9","kw":["stroke","cva","cerebrovascular"]},
    {"id":"52448006","t":"Dementia","c":"F03.90","kw":["dementia","alzheimer","memory loss","cognitive decline"]},
    {"id":"23056005","t":"Sciatica","c":"M54.30","kw":["sciatica","sciatic","shooting pain down leg"]},
    {"id":"128196005","t":"Confusion","c":"R41.0","kw":["confusion","confused","disoriented","altered mental status"]},
    {"id":"40917007","t":"Visual disturbance","c":"H53.9","kw":["blurry vision","blurred vision","double vision","diplopia","visual changes"]},

    {"id":"161891005","t":"Back pain","c":"M54.9","kw":["back pain","backache","lower back pain","back is killing","lumbago"]},
    {"id":"57676002","t":"Joint pain","c":"M25.50","kw":["joint pain","arthralgia","knee pain","hip pain","shoulder pain","elbow pain","wrist pain"]},
    {"id":"68962001","t":"Muscle pain","c":"M79.10","kw":["muscle aches","muscle pain","myalgia","body aches","sore muscles"]},
    {"id":"55300003","t":"Muscle cramp","c":"R25.2","kw":["muscle cramp","muscle spasm","charley horse"]},
    {"id":"263171000","t":"Stiffness","c":"M25.60","kw":["stiffness","stiff","joint stiffness"]},
    {"id":"125605004","t":"Fracture","c":"T14.8XXA","kw":["fracture","broken bone","broken"]},
    {"id":"44465007","t":"Sprain","c":"S93.409A","kw":["sprain","sprained","rolled ankle","twisted","ankle injury","ankle sprain"]},
    {"id":"299308007","t":"Neck pain","c":"M54.2","kw":["neck pain","stiff neck","neck stiffness"]},
    {"id":"239872002","t":"Limited ROM","c":"M25.60","kw":["limited range of motion","can't move","difficulty moving","limited mobility"]},

    {"id":"73211009","t":"Diabetes mellitus","c":"E11.9","kw":["diabetes","diabetic","type 2 diabetes","type 1 diabetes","sugar","blood sugar"]},
    {"id":"40930008","t":"Hypothyroidism","c":"E03.9","kw":["hypothyroidism","underactive thyroid","low thyroid"]},
    {"id":"34095006","t":"Dehydration","c":"E86.0","kw":["dehydration","dehydrated"]},
    {"id":"190372001","t":"Polydipsia","c":"R63.1","kw":["polydipsia","excessive thirst","very thirsty","thirsty","increased thirst"]},
    {"id":"55822004","t":"Hyperlipidemia","c":"E78.5","kw":["high cholesterol","cholesterol","cholesterol problems","hyperlipidemia","dyslipidemia","triglycerides"]},
    {"id":"5291005","t":"Hypoglycemia","c":"E16.2","kw":["hypoglycemia","low blood sugar","low sugar"]},
    {"id":"14304000","t":"Thyroid disorder","c":"E07.9","kw":["thyroid","thyroid disorder","thyroid problem","goiter"]},
    {"id":"190268003","t":"Obesity","c":"E66.9","kw":["obesity","obese","overweight","bmi"]},
    {"id":"238131007","t":"Weight loss","c":"R63.4","kw":["weight loss","losing weight","lost weight","unintentional weight loss"]},
    {"id":"8943002","t":"Weight gain","c":"R63.5","kw":["weight gain","gaining weight","gained weight"]},

    {"id":"162116003","t":"Urinary frequency","c":"R35.0","kw":["urinary frequency","frequent urination","peeing a lot","pee a lot","going to bathroom often"]},
    {"id":"49650001","t":"Dysuria","c":"R30.0","kw":["dysuria","pain when peeing","painful urination","burning urination","pain when urinating"]},
    {"id":"75088002","t":"Urinary urgency","c":"R39.15","kw":["urinary urgency","urgent urination","need to go urgently"]},
    {"id":"165232002","t":"Urinary incontinence","c":"R32","kw":["incontinence","urine leakage","leaking urine","can't hold it"]},
    {"id":"68566005","t":"UTI","c":"N39.0","kw":["uti","urinary tract infection","bladder infection"]},
    {"id":"34436003","t":"Hematuria","c":"R31.9","kw":["hematuria","blood in urine","bloody urine"]},
    {"id":"236578006","t":"Kidney stones","c":"N20.0","kw":["kidney stone","kidney stones","renal calculus","nephrolithiasis"]},
    {"id":"197927001","t":"Flank pain","c":"R10.9","kw":["flank pain","kidney pain","side pain"]},

    {"id":"77386006","t":"Pregnancy","c":"Z33.1","kw":["pregnancy","pregnant","expecting"]},
    {"id":"14094001","t":"Amenorrhea","c":"N91.2","kw":["amenorrhea","missed period","late period","no period","absent menstruation","period missed","period late","missed periods"]},
    {"id":"289903006","t":"Menstrual finding","c":"N94.89","kw":["period","periods","menstrual","menstruation","menses"]},
    {"id":"266897007","t":"Menstrual irregularity","c":"N92.6","kw":["irregular period","irregular periods","irregular menstruation"]},
    {"id":"418290006","t":"Dysmenorrhea","c":"N94.6","kw":["painful period","period pain","menstrual cramps","menstrual pain","dysmenorrhea"]},
    {"id":"237079002","t":"Morning sickness","c":"O21.0","kw":["morning sickness"]},
    {"id":"289908002","t":"Pregnancy test positive","c":"Z32.01","kw":["pregnancy test positive","positive pregnancy test"]},
    {"id":"169553002","t":"Contraception","c":"Z30.9","kw":["birth control","oral contraceptive","contraception","contraceptive","condom","condoms","iud"]},

    {"id":"386661006","t":"Fever","c":"R50.9","kw":["fever","fevers","febrile","temperature elevated","hot today","feel hot","feeling hot","warm at night"]},
    {"id":"43724002","t":"Chills","c":"R68.83","kw":["chills","shivering","rigors"]},
    {"id":"84229001","t":"Fatigue","c":"R53.83","kw":["fatigue","tired","exhausted","exhaustion","malaise","lethargy","low energy","feeling tired"]},
    {"id":"42984000","t":"Night sweats","c":"R61","kw":["night sweats","sweating at night","sweats"]},
    {"id":"271807003","t":"Rash","c":"R21","kw":["rash","rashes","skin rash","eruption","hives","urticaria"]},
    {"id":"64779008","t":"Blood","c":"R58","kw":["blood","bleeding","hemorrhage"]},

    {"id":"197480006","t":"Anxiety","c":"F41.9","kw":["anxiety","anxious","worried","worry","panic","panic attack","nervousness"]},
    {"id":"35489007","t":"Depression","c":"F32.9","kw":["depression","depressed","feeling down","low mood","sadness"]},
    {"id":"193462001","t":"Insomnia","c":"G47.00","kw":["insomnia","can't sleep","difficulty sleeping","trouble sleeping","sleep problems"]},
    {"id":"3415004","t":"Substance use","c":"F19.10","kw":["substance use","drug use","recreational drugs","illicit drugs"]},

    {"id":"6142004","t":"Influenza","c":"J11.1","kw":["flu","influenza","flu-like symptoms","flu-like"]},
    {"id":"186747009","t":"COVID-19","c":"U07.1","kw":["covid","covid-19","coronavirus","sars-cov-2"]},
    {"id":"36971009","t":"Sinusitis","c":"J32.9","kw":["sinusitis","sinus infection","sinus"]},
    {"id":"55735004","t":"Bronchitis","c":"J20.9","kw":["bronchitis"]},
    {"id":"302911003","t":"Pharyngitis","c":"J02.9","kw":["pharyngitis","strep throat"]},

    {"id":"294431004","t":"Nickel allergy","c":"L23.0","kw":["nickel","nickel allergy","jewelry allergy","jewelry","contact dermatitis"]},
    {"id":"43116000","t":"Eczema","c":"L30.9","kw":["eczema","atopic dermatitis"]},
    {"id":"9014002","t":"Psoriasis","c":"L40.9","kw":["psoriasis"]},
    {"id":"126485001","t":"Cellulitis","c":"L03.90","kw":["cellulitis"]},
    {"id":"271756007","t":"Pruritus","c":"L29.9","kw":["itching","itchy","pruritus"]},
    {"id":"95324001","t":"Bruising","c":"R23.3","kw":["bruise","bruising","ecchymosis"]},
    {"id":"95328003","t":"Skin lesion","c":"L98.9","kw":["lesion","sore","wound","ulcer","skin lesion"]},

    {"id":"419199007","t":"Allergy","c":"T78.40XA","kw":["allergy","allergic","allergies"]},
    {"id":"91936005","t":"Penicillin allergy","c":"Z88.0","kw":["penicillin allergy","allergic to penicillin"]},
    {"id":"300916003","t":"Latex allergy","c":"L23.4","kw":["latex allergy","allergic to latex"]},
    {"id":"414285001","t":"Food allergy","c":"T78.1XXA","kw":["food allergy","allergic to food"]},
    {"id":"213020009","t":"Drug allergy","c":"T88.7XXA","kw":["drug allergy","medication allergy"]},

    {"id":"271737000","t":"Anemia","c":"D64.9","kw":["anemia","anaemia","low hemoglobin","low iron"]},
    {"id":"363346000","t":"Cancer","c":"C80.1","kw":["cancer","malignancy","carcinoma","tumor","tumour","neoplasm"]},
    {"id":"93143009","t":"Leukemia","c":"C95.90","kw":["leukemia","leukaemia"]},
    {"id":"118600007","t":"Lymphoma","c":"C85.90","kw":["lymphoma"]},

    {"id":"60862001","t":"Tinnitus","c":"H93.19","kw":["tinnitus","ringing in ears","ringing ears"]},
    {"id":"1023001","t":"Hearing loss","c":"H91.90","kw":["hearing loss","deaf","hard of hearing","can't hear"]},
    {"id":"162298006","t":"Ear pain","c":"H92.09","kw":["ear pain","earache","otalgia"]},
    {"id":"65363002","t":"Nosebleed","c":"R04.0","kw":["nosebleed","epistaxis","nose bleed"]},

    {"id":"246636008","t":"Blurred vision","c":"H53.8","kw":["blurry vision","blurred vision","blurry","vision problems","visual changes"]},
    {"id":"225560005","t":"Eye pain","c":"H57.10","kw":["eye pain","painful eye"]},

    {"id":"412244003","t":"Ginger preparation","c":None,"kw":["ginger","ginger things","ginger supplement","ginger chews","ginger candy"]},
    {"id":"372614000","t":"Lisinopril","c":None,"kw":["lisinopril"]},
    {"id":"386864001","t":"Amlodipine","c":None,"kw":["amlodipine","norvasc"]},
    {"id":"109081006","t":"Metformin","c":None,"kw":["metformin","glucophage"]},
    {"id":"372826007","t":"Metoprolol","c":None,"kw":["metoprolol","lopressor","toprol"]},
    {"id":"387475002","t":"Furosemide","c":None,"kw":["furosemide","lasix"]},
    {"id":"387207008","t":"Naproxen","c":None,"kw":["naproxen","aleve","naprosyn"]},
    {"id":"386845007","t":"Gabapentin","c":None,"kw":["gabapentin","neurontin"]},
    {"id":"412588001","t":"Rosuvastatin","c":None,"kw":["rosuvastatin","rosvastetin","crestor"]},
    {"id":"373462001","t":"Atorvastatin","c":None,"kw":["atorvastatin","lipitor"]},
    {"id":"387207008","t":"Ibuprofen","c":None,"kw":["ibuprofen","advil","motrin"]},
    {"id":"387517004","t":"Paracetamol","c":None,"kw":["acetaminophen","tylenol","paracetamol"]},
    {"id":"387458008","t":"Aspirin","c":None,"kw":["aspirin","asa"]},
    {"id":"764146007","t":"Penicillin","c":None,"kw":["penicillin"]},
    {"id":"387501005","t":"Amoxicillin","c":None,"kw":["amoxicillin","amoxil"]},
    {"id":"387322009","t":"Multivitamin","c":None,"kw":["multivitamin","vitamin","vitamins"]},
    {"id":"108774000","t":"Losartan","c":None,"kw":["losartan","cozaar"]},
    {"id":"387207008","t":"Albuterol","c":None,"kw":["albuterol","ventolin","salbutamol","inhaler","puffer","puffers","nebulizer"]},
    {"id":"126218002","t":"Omeprazole","c":None,"kw":["omeprazole","prilosec"]},
    {"id":"387168006","t":"Levothyroxine","c":None,"kw":["levothyroxine","synthroid"]},
    {"id":"387286002","t":"Prednisone","c":None,"kw":["prednisone","prednisolone","steroid","steroids"]},
    {"id":"372756006","t":"Warfarin","c":None,"kw":["warfarin","coumadin","blood thinner"]},
    {"id":"442031002","t":"Recreational methamphetamine","c":None,"kw":["crystal meth","methamphetamine","meth"]},
    {"id":"229880002","t":"Cannabis use","c":None,"kw":["marijuana","cannabis","weed","pot"]},
    {"id":"228388006","t":"Cocaine use","c":None,"kw":["cocaine","crack"]},

    {"id":"77176002","t":"Smoking","c":"F17.210","kw":["smoking","smoker","cigarette","cigarettes","tobacco","pack a day","half a pack"]},
    {"id":"228273003","t":"Alcohol use","c":"F10.10","kw":["alcohol","drinking","drinks","wine","beer","liquor"]},

    {"id":"399208008","t":"Chest X-ray","c":None,"kw":["chest x-ray","cxr","chest xray","x-ray"]},
    {"id":"241441007","t":"Ankle X-ray","c":None,"kw":["ankle x-ray","ankle xray","left ankle x-ray","right ankle x-ray"]},
    {"id":"113091000","t":"MRI","c":None,"kw":["mri","magnetic resonance","lumbar mri","cervical mri","brain mri"]},
    {"id":"77477000","t":"CT scan","c":None,"kw":["ct scan","ct","cat scan","computed tomography"]},
    {"id":"16310003","t":"Ultrasound","c":None,"kw":["ultrasound","ultrasonography","sonogram","echo","echocardiogram"]},
    {"id":"29303009","t":"ECG","c":None,"kw":["ecg","ekg","electrocardiogram"]},
    {"id":"26604007","t":"CBC","c":None,"kw":["complete blood count","cbc","blood count","blood work"]},
    {"id":"20109005","t":"BMP","c":None,"kw":["basic metabolic panel","bmp","metabolic panel"]},
    {"id":"390840006","t":"BNP measurement","c":None,"kw":["bnp","bnp level","natriuretic peptide"]},
    {"id":"167252002","t":"Pregnancy test","c":None,"kw":["pregnancy test","urine pregnancy","hcg","beta hcg"]},
    {"id":"27171005","t":"Urinalysis","c":None,"kw":["urinalysis","urine test","urine analysis","ua"]},
    {"id":"371907003","t":"Oxygen therapy","c":None,"kw":["supplemental oxygen","oxygen therapy","oxygen","nasal cannula"]},
    {"id":"183524004","t":"Referral","c":None,"kw":["referral","referring","consultation","cardiology referral","orthopedic referral","specialist","cardiology","neurology","orthopedic"]},
    {"id":"183519002","t":"ED referral","c":None,"kw":["emergency department","ed referral","emergency room","er"]},
    {"id":"80146002","t":"Appendectomy","c":None,"kw":["appendectomy"]},
    {"id":"236886002","t":"Hysterectomy","c":None,"kw":["hysterectomy"]},
    {"id":"118949005","t":"Wisdom teeth extraction","c":None,"kw":["wisdom teeth","wisdom tooth","wisdom teeth removal","wisdom teeth removed"]},
    {"id":"71388002","t":"Endoscopy","c":None,"kw":["endoscopy","upper endoscopy","egd"]},
    {"id":"73761001","t":"Colonoscopy","c":None,"kw":["colonoscopy"]},
    {"id":"387713003","t":"Stool test","c":None,"kw":["stool test","stool culture","stool sample"]},

    {"id":"75367002","t":"Blood pressure","c":"R03.0","kw":["blood pressure","bp","systolic","diastolic"]},
    {"id":"364075005","t":"Heart rate","c":None,"kw":["heart rate","pulse","bpm"]},
    {"id":"86290005","t":"Respiratory rate","c":None,"kw":["respiratory rate","breaths per minute"]},
    {"id":"386725007","t":"Body temperature","c":"R50.9","kw":["temperature","temp","38 degrees","39 degrees","100.4","101","102","103","104"]},
    {"id":"431314004","t":"Oxygen saturation","c":None,"kw":["oxygen saturation","spo2","o2 sat","91 percent","92 percent","88 percent"]},
    {"id":"12063002","t":"Pitting edema","c":"R60.0","kw":["pitting edema","2+ edema","3+ edema","pitting"]},
    {"id":"442551007","t":"JVD","c":None,"kw":["jvd","jugular venous distension","neck swelling","neck swollen","neck seems to be a little swollen"]},

    {"id":"160303001","t":"Family history of MI","c":"Z82.49","kw":["father had a heart attack","mother had a heart attack","family heart attack","family history of heart attack"]},
    {"id":"160357008","t":"Family history of cancer","c":"Z80.9","kw":["family cancer","cancer in family","family history of cancer"]},
    {"id":"160377001","t":"Family history of diabetes","c":"Z83.3","kw":["family diabetes","family history of diabetes"]},
    {"id":"160303001","t":"Family history of hypertension","c":"Z82.49","kw":["family high blood pressure","father has high blood pressure","mother has high blood pressure","dad has high blood pressure"]},
    {"id":"160364003","t":"Family history of stroke","c":"Z82.3","kw":["family stroke","family history of stroke"]},
]

CPT_TABLE = [
    {"code": "99213", "description": "Office visit, established patient, low complexity",
     "keywords": ["office visit", "follow-up", "routine visit"]},
    {"code": "99214", "description": "Office visit, established patient, moderate complexity",
     "keywords": ["office visit moderate", "established patient moderate"]},
    {"code": "99215", "description": "Office visit, established patient, high complexity",
     "keywords": ["complex visit", "multiple problems"]},
    {"code": "99202", "description": "Office visit, new patient, low complexity",
     "keywords": ["new patient visit", "initial visit"]},
    {"code": "99281", "description": "Emergency department visit, low severity",
     "keywords": ["emergency department", "ED visit"]},
    {"code": "81025", "description": "Urine pregnancy test",
     "keywords": ["pregnancy test", "urine pregnancy", "HCG", "hcg test"]},
    {"code": "59400", "description": "Routine obstetric care",
     "keywords": ["obstetric care", "prenatal care", "pregnancy care"]},
    {"code": "76801", "description": "Ultrasound, pregnant uterus, first trimester",
     "keywords": ["obstetric ultrasound", "pregnancy ultrasound", "dating ultrasound"]},
    {"code": "81001", "description": "Urinalysis, automated with microscopy",
     "keywords": ["urinalysis", "urine test", "urine analysis", "UA"]},
    {"code": "80048", "description": "Basic metabolic panel (BMP)",
     "keywords": ["basic metabolic panel", "BMP", "metabolic panel"]},
    {"code": "85025", "description": "Complete blood count with differential (CBC)",
     "keywords": ["complete blood count", "CBC", "blood count"]},
    {"code": "80061", "description": "Lipid panel",
     "keywords": ["lipid panel", "cholesterol test"]},
    {"code": "84443", "description": "Thyroid stimulating hormone (TSH)",
     "keywords": ["TSH", "thyroid test", "thyroid function"]},
    {"code": "82947", "description": "Glucose, quantitative, blood",
     "keywords": ["blood glucose", "glucose test", "blood sugar"]},
    {"code": "71046", "description": "Chest X-ray, 2 views",
     "keywords": ["chest x-ray", "chest xray", "CXR"]},
    {"code": "74018", "description": "Abdominal X-ray, 1 view",
     "keywords": ["abdominal x-ray", "KUB", "abdominal xray"]},
    {"code": "73610", "description": "X-ray of ankle, 3 views",
     "keywords": ["ankle x-ray", "ankle xray", "left ankle x-ray", "right ankle x-ray"]},
    {"code": "72148", "description": "MRI of lumbar spine without contrast",
     "keywords": ["MRI lumbar", "MRI spine", "lumbar MRI", "lumbar spine"]},
    {"code": "72141", "description": "MRI of cervical spine without contrast",
     "keywords": ["MRI cervical", "cervical MRI", "cervical spine"]},
    {"code": "83880", "description": "B-type natriuretic peptide (BNP)",
     "keywords": ["BNP", "natriuretic peptide", "BNP level"]},
    {"code": "80053", "description": "Comprehensive metabolic panel (CMP)",
     "keywords": ["comprehensive metabolic panel", "CMP"]},
    {"code": "93000", "description": "Electrocardiogram (ECG), 12-lead",
     "keywords": ["ECG", "EKG", "electrocardiogram"]},
    {"code": "94760", "description": "Pulse oximetry, single reading",
     "keywords": ["oxygen saturation", "pulse oximetry", "SpO2"]},
    {"code": "94002", "description": "Ventilation assist and management",
     "keywords": ["supplemental oxygen", "oxygen therapy", "oxygen"]},
    {"code": "99242", "description": "Consultation, moderate complexity",
     "keywords": ["referral", "referring", "consultation", "follow-up",
                  "cardiology", "orthopedic", "spine specialist"]},
]

print(f"coding tables loaded | SNOMED {len(SNOMED_TABLE)} | CPT {len(CPT_TABLE)}")

In [ ]:
# ============================================================================
# Cell MC.2 - coders.  Ported from audio2soap.ipynb, extended to try the UMLS
#             preferred term from Stage B as well as the raw span.
# ============================================================================
# Words that do not change which concept a span denotes. Severity lives in
# `nature`, dose lives in the span text, so neither needs to constrain coding.
# This is a function-word stoplist, not a disease list - no clinical hardcoding.
_CODE_STOP = {
    "a", "an", "the", "some", "any", "no", "not", "his", "her", "their", "my", "your",
    "of", "in", "on", "at", "to", "for", "with", "and", "or", "is", "was", "has", "had",
    "mild", "slight", "moderate", "severe", "marked", "chronic", "acute", "occasional",
    "almost", "every", "daily", "day", "days", "night", "weekly", "sometimes", "regularly",
    "patient", "reports", "denies", "history", "known", "left", "right", "bilateral",
    "mg", "mcg", "ml", "milligram", "milligrams", "microgram", "tablet", "tablets",
    "once", "twice", "times", "per", "prn", "po", "oral", "orally",
}


def _content_words(text):
    """Meaning-bearing tokens: no function words, no bare numbers."""
    return {w for w in re.findall(r"[a-z0-9']+", (text or "").lower())
            if w not in _CODE_STOP and not w.isdigit() and len(w) > 1}


def get_snomed(entity_text, preferred_term="", transcript_context=""):
    """Match SNOMED_TABLE, rejecting matches that do not account for the whole span.

    The ported table contains generic keywords attached to site-specific concepts -
    e.g. 95328003 "Skin lesion" carries the keyword 'ulcer'. A plain substring match
    therefore codes "stomach ulcer" as a SKIN disorder (L98.9). The fix is not to
    edit the table but to require that the matched keyword explain every content
    word in the span: "stomach" is left over, so the match is rejected and the
    entity is reported uncoded rather than mis-coded.

    Whole-word matching also stops short keywords hitting inside longer words.
    """
    best = None
    for probe in [t for t in (entity_text, preferred_term) if t]:
        low = probe.lower().strip()
        probe_words = _content_words(low)
        for entry in SNOMED_TABLE:
            for kw in entry["kw"]:
                if not re.search(r'\b' + re.escape(kw.lower()) + r'\b', low):
                    continue
                leftover = probe_words - _content_words(kw)
                if leftover:
                    continue          # keyword does not explain the whole span
                cand = {"concept_id": entry["id"], "term": entry["t"],
                        "entity_text": entity_text,
                        "method": "keyword" if probe == entity_text else "umls_term",
                        "crosswalk_icd10": entry.get("c"),
                        "_kw_len": len(kw)}
                # most specific keyword wins; the raw span beats the UMLS term
                if best is None or cand["_kw_len"] > best["_kw_len"]:
                    best = cand
        if best:                       # a direct span match is preferred outright
            break
    if best:
        best.pop("_kw_len", None)
    return best


def get_icd10_from_snomed(sn):
    """SNOMED -> ICD-10-CM via the table's own crosswalk column."""
    if sn and sn.get("crosswalk_icd10"):
        return {"code": sn["crosswalk_icd10"], "description": sn["term"],
                "method": "snomed_crosswalk"}
    return None


CPT_ELIGIBLE_TYPES = ("Procedure/Treatment", "Test/Investigation")


def get_cpt_deterministic(transcript_text, entities):
    """Visit-level CPT codes, matched against PROCEDURE ENTITIES rather than raw text.

    The ported version searched the whole transcript as a substring, which produced
    two classes of false code:
      - "UA" (urinalysis) matched inside ordinary words such as "act-ua-lly"
      - "oxygen", appearing in the vital sign "oxygen saturation", billed 94002
        Ventilation assist and management on an outpatient GI visit
    A CPT code describes something DONE, so the evidence has to be a procedure or
    investigation entity that Stage A actually extracted - not any occurrence of the
    word anywhere in the conversation. Whole-word matching handles the "UA" case.
    """
    spans = [e.span.lower() for e in entities
             if e.entity_type in CPT_ELIGIBLE_TYPES or e.relation == "ORDERED"]
    if not spans:
        return []
    out, seen = [], set()
    for entry in CPT_TABLE:
        for kw in entry.get("keywords", []):
            pat = re.compile(r'\b' + re.escape(kw.lower()) + r'\b')
            hit = next((s for s in spans if pat.search(s)), None)
            if hit and entry["code"] not in seen:
                seen.add(entry["code"])
                out.append({"code": entry["code"], "description": entry["description"],
                            "matched": kw, "evidence": hit})
                break
    return out


def attach_codes(entities: List[VerifiedEntity], transcript: str = ""):
    """Adds e.snomed / e.icd10_cm in place. Negated entities are never coded."""
    if not RUN_CODING:
        return entities
    for e in entities:
        if e.relation == "ABSENT":          # do not code a denied finding
            e.snomed, e.icd10_cm = None, None
            continue
        sn = get_snomed(e.span, e.preferred_term)
        e.snomed = sn
        e.icd10_cm = (get_icd10_from_snomed(sn)
                      if e.entity_type in ("Disease/Problem", "Symptom", "Clinical Finding")
                      else None)
    return entities


def format_coding_report(entities, transcript=""):
    rows = [e for e in entities if e.snomed or e.icd10_cm]
    lines = ["=" * 78, "MEDICAL CODING", "=" * 78,
             f"{'span':<26}{'type':<20}{'SNOMED':<14}{'ICD-10':<10}conf"]
    lines.append("-" * 78)
    for e in rows:
        sn = e.snomed["concept_id"] if e.snomed else "-"
        ic = e.icd10_cm["code"] if e.icd10_cm else "-"
        cf = f"{e.asr_confidence:.2f}" if e.asr_confidence is not None else "n/a"
        flag = " !" if e.low_confidence else ""
        lines.append(f"{e.span[:25]:<26}{e.entity_type[:19]:<20}{sn:<14}{ic:<10}{cf}{flag}")
    cpt = get_cpt_deterministic(transcript, entities)
    if cpt:
        lines += ["", "CPT (visit level):"]
        lines += [f"  {c['code']}  {c['description']}  (matched: {c['matched']})" for c in cpt]
    lines.append("=" * 78)
    lines.append(f"{len(rows)}/{len(entities)} entities coded"
                 + ("   ! = low ASR confidence" if any(e.low_confidence for e in entities) else ""))
    return "\n".join(lines)


print("medical coding ready - SNOMED / ICD-10-CM / CPT")

---
## Additional metrics

Ported from `audio2soap.ipynb` and the ablation notebook so the results table matches what the
earlier work reported. These sit alongside the four VeriSOAP metrics.

- `keyword_coverage` — did asserted entities reach the note (the K-SOAP metric)
- `bertscore_document` — embedding similarity to the reference note
- `soap_transcript_similarity` — cosine + Word Mover's Distance against the transcript
- `concept_f1` — **MEDCON-style** clinical-concept F1 against the reference note

In [ ]:
# ============================================================================
# Cell EM.1 - extra metrics, ported from audio2soap.ipynb / the ablation nb.
# ============================================================================
_EMB = {"model": None}

def _embedder():
    if _EMB["model"] is None:
        from sentence_transformers import SentenceTransformer
        # NOTE: this is S-PubMedBert-MS-MARCO. The old code called the variable
        # SAPBERT_MODEL_ID, but it is NOT SapBERT - keep the naming honest.
        _EMB["model"] = SentenceTransformer(EMBED_MODEL)
    return _EMB["model"]


def keyword_coverage(soap_text, entities: List[VerifiedEntity]):
    """K-SOAP coverage: asserted entities that reached the note. Ported."""
    relevant = [e for e in entities if e.relation in ("PRESENT", "PAST", "FAMILY", "ORDERED")]
    if not relevant:
        return {"coverage": 0.0, "found": 0, "total": 0, "missing": []}
    low, found, missing = soap_text.lower(), 0, []
    for e in relevant:
        t = e.span.lower()
        words = [w for w in t.split() if len(w) >= 4]
        if t in low or any(w in low for w in words):
            found += 1
        else:
            missing.append(e.span)
    return {"coverage": round(found / len(relevant), 4), "found": found,
            "total": len(relevant), "missing": missing}


_BERTSCORER = {"m": None}

def bertscore_document(candidate, reference):
    """REAL BERTScore - token-level greedy matching with roberta-large, as the
    K-SOAP paper uses (sec 5.1). Distinct precision/recall/F1.

    The previous version returned the cosine of two whole-document embeddings and
    called it BERTScore. That is a different quantity: it collapses the document to
    one vector, so P == R == F1 always, and it is NOT comparable to the paper's
    Table 5. Kept as `embedding_cosine` below, which is what it actually measures.
    """
    if not (candidate or "").strip() or not (reference or "").strip():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    if _BERTSCORER["m"] is None:
        from bert_score import BERTScorer
        import torch
        _BERTSCORER["m"] = BERTScorer(
            model_type=BERTSCORE_MODEL, lang="en", rescale_with_baseline=False,
            device="cuda" if torch.cuda.is_available() else "cpu")
    P, R, F = _BERTSCORER["m"].score([candidate], [reference])
    return {"precision": round(float(P[0]), 4), "recall": round(float(R[0]), 4),
            "f1": round(float(F[0]), 4)}


def embedding_cosine(candidate, reference):
    """Whole-document embedding cosine. NOT BERTScore - reported under its own name."""
    if not (candidate or "").strip() or not (reference or "").strip():
        return 0.0
    from sentence_transformers import util as st_util
    m = _embedder()
    return round(float(st_util.cos_sim(m.encode(candidate, convert_to_tensor=True),
                                       m.encode(reference, convert_to_tensor=True))[0][0]), 4)


def rouge_lsum(candidate, reference):
    """ROUGE-Lsum: LCS computed per sentence then summed, as the paper reports."""
    def sents(t):
        return [x.strip() for x in re.split(r"(?<=[.!?])\s+|\n", t or "") if x.strip()]
    c, r = sents(candidate), sents(reference)
    if not c or not r:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    tot = 0
    for rs in r:
        tot += max((rouge_l(cs, rs)["f1"] for cs in c), default=0.0)
    rec = tot / len(r)
    tot2 = 0
    for cs in c:
        tot2 += max((rouge_l(cs, rs)["f1"] for rs in r), default=0.0)
    pre = tot2 / len(c)
    return {"precision": round(pre, 4), "recall": round(rec, 4),
            "f1": round(2 * pre * rec / max(pre + rec, 1e-8), 4)}


def soap_transcript_similarity(soap_text, transcript):
    """Cosine + Word Mover's Distance, note vs transcript. Ported."""
    from sentence_transformers import util as st_util
    m = _embedder()
    cosine = float(st_util.cos_sim(m.encode([soap_text], convert_to_tensor=True),
                                   m.encode([transcript], convert_to_tensor=True))[0][0])
    out = {"cosine": round(cosine, 4), "wmd": None, "wmd_similarity": 0.0}
    try:
        import gensim.downloader as api
        if not hasattr(soap_transcript_similarity, "_w2v"):
            # 50-dimensional. The report previously said 300-d; it is not.
            soap_transcript_similarity._w2v = api.load("glove-wiki-gigaword-50")
        w2v = soap_transcript_similarity._w2v
        tk  = lambda t: [w for w in re.findall(r"[a-z]+", t.lower()) if w in w2v.key_to_index]
        wmd = float(w2v.wmdistance(tk(soap_text), tk(transcript)))
        if wmd != float("inf"):
            out["wmd"] = round(wmd, 4)
            out["wmd_similarity"] = round(1.0 / (1.0 + wmd), 4)
    except Exception:
        pass
    return out


def concept_f1(note, reference):
    """MEDCON-style clinical-concept F1: overlap of clinical TERMS, not tokens.
    Reuses extract_note_terms so 'concept' means the same thing as in Stage-D check 3."""
    c, r = set(extract_note_terms(note)), set(extract_note_terms(reference))
    if not c and not r:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "n_ref": 0}
    tp = len(c & r)
    p  = tp / len(c) if c else 0.0
    rc = tp / len(r) if r else 0.0
    return {"precision": round(p, 4), "recall": round(rc, 4),
            "f1": round(2 * p * rc / max(p + rc, 1e-8), 4), "n_ref": len(r)}


print("extra metrics ready - keyword_coverage / BERTScore / WMD / MEDCON concept_f1")

# ---------------------------------------------------------------------------
# Sir's 3-axis judge. Ported VERBATIM from the eval upgrade in audio2soap.ipynb
# (_JUDGE_PROMPT_V2). Reported ALONGSIDE the 5-dimension clinical judge, not
# instead of it - they measure different things and both were asked for.
# ---------------------------------------------------------------------------
# FOUR axes, per Evaluation_Metrics_Detailed.pdf sec 2.1. The code carried only
# three for a long time - `faithfulness` was missing, which is the axis that most
# directly measures hallucination, and the one the doc's measured GPT run reports
# moving 3.2 -> 4.0. Adding it makes the notebook match the documented rubric.
#
# Written anchors for every level, as the doc specifies: 4 = a solid, clinically
# usable note; 5 is reserved for excellent; 1-2 for genuinely poor notes. The judge
# writes a short analysis BEFORE the scores, because scoring after articulating
# reasons gives more consistent grades (the G-Eval finding).
_JUDGE_PROMPT_3AXIS = """You are an expert clinical documentation reviewer. Given a doctor-patient TRANSCRIPT and a generated SOAP note, rate the SOAP on four axes, each an integer 1 to 5.

Scale anchors, the same for every axis:
  5 excellent - reserved; nothing a reviewing clinician would change
  4 solid, clinically usable as written
  3 usable but with a clear weakness
  2 poor; a clinician would have to rewrite part of it
  1 unusable

1. quality_of_soap         : overall clinical quality, structure and correctness of the note itself.
2. adequacy_wrt_transcript : how completely it captures the clinically relevant information that IS in the transcript (penalise omissions).
3. faithfulness            : does it contain ONLY what the transcript supports (penalise invented findings, drugs, or advice). Invented content is the worst failure in a clinical note - score it hard.
4. alignment_score         : how well the note as a whole reflects this encounter (structure, emphasis, and the clinical story matching what happened).

TRANSCRIPT:
{transcript}

SOAP NOTE:
{soap}

First write 2-3 sentences of analysis naming the note's strengths and problems. Then respond with ONLY a JSON object on the final line:
{{"analysis": "<2-3 sentences>", "quality_of_soap": <1-5>, "adequacy_wrt_transcript": <1-5>, "faithfulness": <1-5>, "alignment_score": <1-5>}}"""


def llm_judge_3axis(transcript, soap_text):
    """The 3-axis clinical jury: note quality, adequacy w.r.t. the transcript, and
    faithfulness to it. Deterministic (temperature 0) to blunt judge variability.

    Kept distinct from llm_judge (5 axes) because the two disagree usefully: the
    5-axis jury rates the note as a clinical document, this one rates it as a
    record OF THIS transcript, so it is the one that punishes hallucination.
    """
    prompt = _JUDGE_PROMPT_3AXIS.format(transcript=transcript[:3000], soap=soap_text[:2000])
    raw = _llm([{"role": "system", "content": "Respond ONLY with a JSON object. No other text."},
                {"role": "user", "content": prompt}], max_tokens=400, temperature=0.0)
    sc = _parse_json(raw, expect="dict") or {}
    axes = ["quality_of_soap", "adequacy_wrt_transcript", "faithfulness",
            "alignment_score"]
    for k in axes:
        v = sc.get(k)
        sc[k] = max(1, min(5, int(v))) if isinstance(v, (int, float)) and 1 <= v <= 5 else 3
    sc["composite"] = round(sum(sc[k] for k in axes) / len(axes), 2)
    return {k: sc[k] for k in axes + ["composite"]}


# ---------------------------------------------------------------------------
# BLEURT and QuestEval - the two paper metrics we lacked. Both are optional
# heavy dependencies, so both return None rather than killing a batch run.
# ---------------------------------------------------------------------------
_BLEURT = {"m": None, "tried": False}

def bleurt_score(candidate, reference):
    """BLEURT-20 via the pytorch port. None if unavailable."""
    if not RUN_BLEURT or not (candidate or "").strip() or not (reference or "").strip():
        return None
    if not _BLEURT["tried"]:
        _BLEURT["tried"] = True
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch
        for _mid in BLEURT_MODELS:
            try:
                tok = AutoTokenizer.from_pretrained(_mid)
                mdl = AutoModelForSequenceClassification.from_pretrained(_mid)
                mdl.to("cuda" if torch.cuda.is_available() else "cpu").eval()
                _BLEURT["m"] = (mdl, tok)
                _BLEURT["id"] = _mid
                print(f"[BLEURT] using {_mid}")
                break
            except Exception as ex:
                print(f"[BLEURT] {_mid} unavailable ({type(ex).__name__}: "
                      f"{str(ex)[:70]})")
        if _BLEURT["m"] is None:
            print("[BLEURT] no port loaded - column stays blank")
    if _BLEURT["m"] is None:
        return None
    import torch
    mdl, tok = _BLEURT["m"]
    enc = tok([reference], [candidate], return_tensors="pt", truncation=True,
              max_length=512).to(mdl.device)
    with torch.no_grad():
        return round(float(mdl(**enc).logits.flatten()[0]), 4)


_QE = {"m": None, "tried": False}

def questeval_score(candidate, source):
    """QuestEval - REFERENCE-FREE, so it works in the Gradio demo too, where
    there is no gold note. Scores the note against the TRANSCRIPT."""
    if not RUN_QUESTEVAL or not (candidate or "").strip():
        return None
    if not _QE["tried"]:
        _QE["tried"] = True
        try:
            from questeval.questeval_metric import QuestEval
            _QE["m"] = QuestEval(no_cuda=False)
        except Exception as ex:
            print(f"[QuestEval] unavailable ({type(ex).__name__}) - column stays blank")
    if _QE["m"] is None:
        return None
    try:
        r = _QE["m"].corpus_questeval(hypothesis=[candidate], sources=[source])
        return round(float(r["corpus_score"]), 4)
    except Exception:
        return None


print("judges: LLM-Judge-5 (completeness/correctness/coherence/assessment/plan safety)"
      " + LLM-Judge-3 (note quality/adequacy/faithfulness)"
      f" | BLEURT: {RUN_BLEURT} | QuestEval: {RUN_QUESTEVAL}")

llm_judge_sir = llm_judge_3axis        # old name, kept so nothing else breaks


---
## End to end: audio in, SOAP note out

`audio_to_soap()` is the whole system in one call — Stage 0 through Stage D, plus coding.
The Gradio app wraps it, so the demo is: **drop an audio file, get a coded SOAP note.**

This is the interface from `audio2soap.ipynb` (`generate_for_gradio`), rebuilt on the
VeriSOAP pipeline.

In [ ]:
# ============================================================================
# Cell E2E.1 - audio -> SOAP, the complete system.
# ============================================================================
# The most recent audio_to_soap() result, so build_report() / format_* can be called
# with no argument from a notebook cell or after a Gradio run.
LAST_RESULT = None


def audio_to_soap(audio_path: str, rid: str = "", verbose: bool = True) -> dict:
    """Stage 0 -> A -> B -> C -> D, plus severity, ASR confidence and coding."""
    global LAST_RESULT
    t0 = time.time()

    # Stage 0 - audio to speaker-labelled transcript
    transcript, words, meta = audio_to_transcript(audio_path, rid=rid)

    # Stage A - k-span extraction
    ents = extract_kspans(transcript, rid=rid)

    # Stage B - UMLS verification and linking
    ents = verify_entities(ents, transcript, rid=rid)

    # Stage 0 signals folded back onto the verified entities
    if words:
        ents = link_entity_confidence(ents, words)
    ents = annotate_entity_nature(ents, transcript)
    ents = attach_codes(ents, transcript)

    # Stage C - generation
    draft = generate_soap(transcript, ents, grouped=True, with_evidence=True,
                          rid=rid, tag="E2E")

    # Stage D - evidence-guided self-verification
    note, trace = reflect_and_revise(draft, ents, transcript, rid=rid)

    out = {
        "audio_path": audio_path, "transcript": transcript, "entities": ents,
        "soap_draft": draft, "soap": note, "reflect_trace": trace,
        "asr_meta": meta, "words": words,
        "cpt": get_cpt_deterministic(transcript, ents),
        "elapsed_sec": round(time.time() - t0, 1),
    }
    LAST_RESULT = out
    out["metrics"] = evaluate_all(note, transcript, ents, reference=None, judge=True)
    out["metrics"].update({
        "keyword_coverage": keyword_coverage(note, ents)["coverage"],
        **{f"sim_{k}": v for k, v in soap_transcript_similarity(note, transcript).items()},
    })

    if verbose:
        print("\n" + "=" * 78)
        print(f"SOAP NOTE   ({out['elapsed_sec']}s total | "
              f"ASR {meta['asr_seconds']}s, diarization {meta['diar_seconds']}s)")
        print("=" * 78)
        print(note)
        print("\n" + format_coding_report(ents, transcript))
        low = [e for e in ents if e.low_confidence]
        if low:
            print(f"\n!! {len(low)} span(s) below ASR confidence {ASR_CONF_MIN} "
                  f"- verify against the audio: " + ", ".join(e.span for e in low[:8]))
        print(f"\nreflection: {trace[0]['violations']} violations -> "
              f"{trace[-1]['violations']} after {len(trace)-1} round(s)")
    return out


print("audio_to_soap() ready - the complete system, Stage 0 through D")
print(f"   demo:  result = audio_to_soap('{DEMO_AUDIO}')")

In [ ]:
# ============================================================================
# Cell E2E.2 - Gradio app.  Drop audio in, get a coded SOAP note out.
#              Rebuilt from generate_for_gradio in audio2soap.ipynb.
# ============================================================================
def _audio_path_from_gradio(audio_file):
    """Ported from audio2soap.ipynb - Gradio hands back str | dict | file object."""
    if audio_file is None:
        raise ValueError("Please upload or record an audio file first.")
    if isinstance(audio_file, str):
        return audio_file
    if isinstance(audio_file, dict):
        p = audio_file.get("path") or audio_file.get("name")
        if p:
            return p
    if hasattr(audio_file, "name"):
        return audio_file.name
    raise ValueError("Could not read the uploaded audio file path.")


def generate_for_gradio(audio_file):
    """-> (soap_note, transcript, entity+coding report, run details)"""
    try:
        path = _audio_path_from_gradio(audio_file)
    except Exception as ex:
        return f"**Error:** {ex}", "", "", "", ""

    try:
        r = audio_to_soap(path, rid="", verbose=False)
    except Exception as ex:
        import traceback
        return (f"**Pipeline failed:** {type(ex).__name__}: {ex}",
                "", "", f"```\n{traceback.format_exc()[-1500:]}\n```", "")

    ents, m = r["entities"], r["asr_meta"]
    details = (
        f"| stage | value |\n|---|---|\n"
        f"| ASR model | `{m['asr_model']}` |\n"
        f"| ASR time | {m['asr_seconds']}s |\n"
        f"| diarization time | {m['diar_seconds']}s |\n"
        f"| mean word confidence | {m.get('mean_confidence', 'n/a')} |\n"
        f"| entities extracted | {len(ents)} |\n"
        f"| UMLS-grounded | {sum(1 for e in ents if e.grounded)} |\n"
        f"| low-confidence spans | {sum(1 for e in ents if e.low_confidence)} |\n"
        f"| reflection rounds | {len(r['reflect_trace'])-1} |\n"
        f"| violations before -> after | {r['reflect_trace'][0]['violations']} -> "
        f"{r['reflect_trace'][-1]['violations']} |\n"
        f"| judge composite | {r['metrics'].get('judge5_composite', 'n/a')} /5 |\n"
        f"| total | {r['elapsed_sec']}s |\n"
    )
    full = build_report(r)
    return (r["soap"], r["transcript"],
            "```\n" + format_coding_report(ents, r["transcript"]) + "\n```",
            details, "```\n" + full + "\n```")


def launch_app(share=True):
    import gradio as gr
    with gr.Blocks(title="VeriSOAP", theme=gr.themes.Soft()) as demo:
        gr.Markdown(
            "# VeriSOAP\n"
            "**Knowledge-grounded verification and evidence-guided self-reflection "
            "for faithful SOAP notes.**\n\n"
            "Drop a doctor-patient recording below. The pipeline runs Whisper ASR -> "
            "LLM diarization -> k-span extraction -> UMLS verification -> "
            "fine-tuned generation -> self-verification, then codes the result to "
            "SNOMED CT / ICD-10-CM / CPT."
        )
        with gr.Row():
            with gr.Column(scale=1):
                audio = gr.Audio(sources=["upload", "microphone"], type="filepath",
                                 label="Doctor-patient conversation")
                btn   = gr.Button("Generate SOAP note", variant="primary")
                gr.Markdown(f"*Demo file in this repo: `{DEMO_AUDIO}`*")
            with gr.Column(scale=2):
                soap = gr.Markdown(label="SOAP note")
        with gr.Accordion("Transcript (ASR + diarization)", open=False):
            trans = gr.Textbox(lines=14, show_label=False)
        with gr.Accordion("Entities, ASR confidence and medical codes", open=False):
            rep = gr.Markdown()
        with gr.Accordion("Run details", open=False):
            det = gr.Markdown()
        with gr.Accordion("Full summary (production report format)", open=False):
            full = gr.Markdown()
        btn.click(generate_for_gradio, inputs=[audio],
                  outputs=[soap, trans, rep, det, full])
    demo.launch(share=share, debug=False)
    return demo


print("Gradio app ready.  launch_app()   # then drop an audio file in")

---
## Self-check

Set `SELFCHECK = True` and commit the notebook. Everything above the LLM boundary runs
**without an API key**, so this verifies the parts that actually break — dependency versions,
the UMLS download and index at full scale, the schema, and Stage D — before you spend
anything. The LLM checks run too if a key is present, and are skipped cleanly if not.

In [ ]:
# ============================================================================
# Cell SC.1 - self-check.  Prints a PASS/FAIL verdict for a headless run.
# ============================================================================
SELFCHECK = True

def run_selfcheck():
    import platform
    res = []
    def check(name, fn, needs_key=False):
        if needs_key and not _resolve_openai_key():
            res.append(("SKIP", name, "no OPENAI_API_KEY")); return
        try:
            ok, detail = fn()
            res.append(("PASS" if ok else "FAIL", name, detail))
        except Exception as ex:
            res.append(("FAIL", name, f"{type(ex).__name__}: {str(ex)[:110]}"))

    print("=" * 78); print("VeriSOAP SELF-CHECK"); print("=" * 78)
    print(f"python {platform.python_version()} | kaggle={ON_KAGGLE}")

    def _env():
        import numpy, sklearn
        bad = int(numpy.__version__.split(".")[0]) < 2
        d = f"numpy {numpy.__version__}, sklearn {sklearn.__version__}"
        try:
            import torch; d += f", torch {torch.__version__} (cuda={torch.cuda.is_available()})"
        except Exception as ex:
            return False, f"{d} | TORCH BROKEN: {str(ex)[:70]}"
        return (not bad), d + ("  <- numpy<2 will break torch" if bad else "")
    check("environment", _env)

    check("schema 10x6=60", lambda: (
        len(ENTITY_SECTION_MAP) == 60 and all(
            route_section(t, r) in "SOAP" for t in ENTITY_TYPES for r in RELATION_SET),
        f"{len(ENTITY_TYPES)} types x {len(RELATION_SET)} relations"))

    check("KB backend", lambda: (
        KB_INSTANCE.name != "icd-snomed-fallback",
        f"{KB_INSTANCE.name}" + (" <- FALLBACK, not real UMLS"
                                 if KB_INSTANCE.name == "icd-snomed-fallback" else "")))

    def _probes():
        want = {"meningitis": "Disease/Problem", "chills": "Symptom",
                "amoxicillin": "Medication", "tummy ache": "Symptom",
                "lumbar puncture": "Procedure/Treatment"}
        good, lines = 0, []
        for span, exp in want.items():
            hits = KB_INSTANCE.link(span, k=5)
            e = VerifiedEntity(span=span, entity_type=exp, relation="PRESENT")
            cand, typ, _ = verify_type_deterministic(e, hits)
            hit = cand is not None and typ == exp
            good += hit
            lines.append(f"{span}->{cand.cui if cand else '-'}/{typ}{'' if hit else ' XX'}")
        return good == len(want), f"{good}/{len(want)}  " + " | ".join(lines)
    check("UMLS probes", _probes)

    check("Stage D checks", lambda: (selftest_stage_d(), "4 planted defects"))

    def _rules():
        T = "DOCTOR: Any chest pain?\nPATIENT: No.\nPATIENT: I take metformin daily.\n"
        cases = [("chest pain", "Symptom", "ABSENT"),
                 ("metformin", "Medication", "PRESENT")]
        bad = [s for s, t, w in cases
               if assign_relation(s, T, "PRESENT", t, offset=T.lower().find(s)) != w]
        return not bad, ("cross-turn negative + positive control"
                         if not bad else f"wrong: {bad}")
    check("relation rules", _rules)

    check("coding", lambda: (
        get_snomed("nausea") is not None and get_snomed("tummy ache", "Abdominal pain") is not None,
        f"SNOMED {len(SNOMED_TABLE)} concepts, CPT {len(CPT_TABLE)}"))

    # --- these need a key -----------------------------------------------
    check("Stage A extraction", lambda: (
        len(extract_kspans("DOCTOR: Any chest pain?\nPATIENT: No, but I take metformin.\n")) > 0,
        "entities extracted"), needs_key=True)
    check("whiteboard round-trip", lambda: (probe_stage_b(verbose=False), "board examples"),
          needs_key=True)

    print()
    for status, name, detail in res:
        print(f"  {status:<5} {name:<24} {detail[:100]}")
    n_fail = sum(1 for s, _, _ in res if s == "FAIL")
    print("=" * 78)
    print(f"VERDICT: {'ALL PASS' if n_fail == 0 else str(n_fail) + ' FAILED'}"
          f"   ({sum(1 for s,_,_ in res if s=='PASS')} pass, "
          f"{sum(1 for s,_,_ in res if s=='SKIP')} skipped)")
    print("=" * 78)
    return n_fail == 0


if SELFCHECK:
    run_selfcheck()
else:
    print("SELFCHECK=False. Run run_selfcheck() manually.")

---
## Full summary — the earlier demo's report format

`build_report()` reproduces the production report from `audio2soap.ipynb` line for line:
header, disclaimer, RAW TRANSCRIPT, SOAP NOTE, ENTITIES, SNOMED CT MAPPINGS, ICD-10-CM CODES,
End of Report. It is deliberately redundant with the panels above — it is the format that has
been shown before, kept intact.

In [ ]:
# ============================================================================
# Cell FR.1 - build_report().  Format ported line-for-line from audio2soap.ipynb.
# ============================================================================
from datetime import datetime

# The old report printed the 4-status schema. Ours has 6 relations, and ORDERED has
# no legacy equivalent, so the relation is printed as-is and the legacy name is shown
# alongside where one exists.
_LEGACY_OF = {"PRESENT": "Confirmed", "ABSENT": "Negated",
              "PAST": "Historical", "FAMILY": "Family_History"}


def build_report(r=None):
    """The earlier demo's production report, rebuilt on VeriSOAP output.

    Call with no argument to report on the most recent run:
        audio_to_soap("full_soap_demo.wav")
        print(build_report())
    """
    r = r if r is not None else LAST_RESULT
    if r is None:
        return ("No run to report on yet.\n"
                "  result = audio_to_soap('" + DEMO_AUDIO + "')\n"
                "  print(build_report())          # or build_report(result)\n"
                "A Gradio run also fills this in - generate a note, then call it again.")
    entities = r.get("entities", [])
    lines = [
        "Clinical NLP Pipeline  Production Report",
        f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}",
        "", "DISCLAIMER: AI-generated. Requires clinician review.", "",
        "=" * 65, "RAW TRANSCRIPT", "=" * 65,
        f"\n{r.get('transcript', '')}\n",
        "=" * 65, "SOAP NOTE", "=" * 65,
        f"\n{r.get('soap', r.get('soap_raw', ''))}",
    ]

    lines += ["\n" + "=" * 65, "ENTITIES", "=" * 65]
    for i, e in enumerate(entities, 1):
        off = (f"[{e.char_start}:{e.char_end}](exact)" if e.char_start >= 0 else "N/A")
        neg = " <- NEGATED" if e.relation == "ABSENT" else ""
        leg = f" [{_LEGACY_OF[e.relation]}]" if e.relation in _LEGACY_OF else ""
        gnd = "" if e.grounded else " [UNGROUNDED]"
        cui = f" | CUI:{e.cui}" if e.cui else ""
        norm = f" [norm: {e.preferred_term}]" if e.preferred_term else ""
        sn_str = f" | SN:{e.snomed['concept_id']}" if e.snomed else ""
        ic_str = f" | ICD:{e.icd10_cm['code']}" if e.icd10_cm and e.icd10_cm.get("code") else ""
        conf = f" | ASR:{e.asr_confidence:.2f}" if e.asr_confidence is not None else ""
        low = " <- LOW CONFIDENCE" if e.low_confidence else ""
        lines.append(f"  {i:>2}. [{e.entity_type:<21}|{e.relation:<8}] \"{e.span}\" "
                     f"{off}{neg}{leg}{gnd}{norm}{cui}{sn_str}{ic_str}{conf}{low}")

    lines += ["\n" + "=" * 65, "SNOMED CT MAPPINGS", "=" * 65]
    seen_sn = set()
    for e in entities:
        sn = e.snomed
        if sn and sn["concept_id"] not in seen_sn:
            seen_sn.add(sn["concept_id"])
            xw = f"  -> {sn['crosswalk_icd10']}" if sn.get("crosswalk_icd10") else ""
            lines.append(f"  {sn['concept_id']:>12}  {sn['term']:<45} "
                         f"<- \"{sn['entity_text']}\" ({sn['method']}){xw}")
    if not seen_sn:
        lines.append("  (none)")

    lines += ["\n" + "=" * 65, "ICD-10-CM CODES", "=" * 65]
    any_icd = False
    for e in entities:
        ic = e.icd10_cm
        if ic and ic.get("code"):
            any_icd = True
            lines.append(f"  {e.span:<45} -> {ic['code']:>10}  [{ic.get('method','')}]  "
                         f"({ic.get('description','')})")
    if not any_icd:
        lines.append("  (none)")

    cpt = r.get("cpt") or []
    if cpt:
        lines += ["\n" + "=" * 65, "CPT CODES", "=" * 65]
        for c in cpt:
            lines.append(f"  {c['code']:>10}  {c['description']:<45} "
                         f"<- \"{c.get('evidence', c.get('matched',''))}\"")

    tr = r.get("reflect_trace")
    if tr:
        lines += ["\n" + "=" * 65, "SELF-VERIFICATION", "=" * 65]
        for t in tr:
            lines.append(f"  round {str(t['round']):<6} violations={t['violations']}"
                         f"  (missing {t['missing']}, misplaced {t['misplaced']}, "
                         f"unsupported {t['hallucinated']}, inconsistent {t['inconsistent']})")
            for d in t.get("detail", [])[:10]:
                lines.append(f"        - {d[:92]}")

    lines.append("\n" + "=" * 65 + "\nEnd of Report")
    return "\n".join(lines)


print("build_report() ready - the earlier demo's format, on VeriSOAP output")

---
## 6. Not built — DPO/ORPO feedback loop

*Algorithm 1 lines 40-42.* Out of scope for v1. The interface it would consume is below so the
v2 work is a fill-in rather than a redesign.

In [ ]:
# ============================================================================
# Cell 25 - DPO/ORPO STUB.  Algorithm 1 lines 40-42.  NOT IMPLEMENTED.
# ============================================================================
def collect_preference_pairs(records=None):
    """Line 40: (prompt, draft_before_correction, draft_after_correction).

    Stage D already produces exactly this: run_arm_C's draft is the rejected
    response and its corrected note is the chosen one. This function only has to
    read them back out of artifacts/.
    """
    records = records or TRAIN_RECORDS
    pairs = []
    for rec in records:
        rid = str(rec["id"])
        before = ck_load(_mk("genB"), rid)
        after  = ck_load(_mk("stageD"), rid)
        if not before or not after or before["note"].strip() == after["note"].strip():
            continue                       # no correction happened -> not a preference pair
        ents = ck_load("stageB", rid) or []
        pairs.append({
            "prompt":   build_soap_prompt(rec["transcript"],
                                          [VerifiedEntity(**e) for e in ents], grouped=True),
            "chosen":   after["note"],
            "rejected": before["note"],
        })
    return pairs


def train_dpo(pairs):
    raise NotImplementedError(
        "v2. Feed `pairs` to trl.DPOTrainer (or ORPOTrainer) over the Stage-C adapter, "
        "then point MODEL_REGISTRY[MODEL]['out'] at the result (line 42)."
    )


_p = collect_preference_pairs() if RESUME else []
print(f"DPO stub ready. {len(_p)} preference pairs recoverable from artifacts/ so far.")
print("Not implemented in v1 - see README 'Not built'.")

In [ ]:
# ============================================================================
# Cell REPORT - the two tables the paper actually needs, plus the ablation gain.
#
# results_table()/significance_table() already exist and cover ONE model across
# the arms. These add the cross-model views:
#
#   paper_table(arm)   one row per model, the K-SOAP Table 5 metric set. Needs a
#                      reference note, so it prints nothing on a reference-free
#                      dataset instead of printing a column of dashes.
#   ablation_gains()   A->B, B->C and A->C deltas per model with paired-bootstrap
#                      significance. This is "what did UMLS verification buy, and
#                      what did reflection buy", which is the claim of the paper.
# ============================================================================
PAPER_KEYS = [
    ("rouge1_f1",    "R-1"),     ("rouge2_f1",   "R-2"),
    ("rougeL_f1",    "R-L"),     ("rougeLsum_f1","R-Lsum"),
    ("bleu",         "BLEU"),    ("bertscore_f1","BERTSc-F1"),
    ("concept_f1",   "MEDCON"),  ("bleurt",      "BLEURT"),
    ("questeval",    "QuestEval"),
]

ABLATION_KEYS = [
    ("rougeL_f1",                  "ROUGE-L"),
    ("bertscore_f1",               "BERTSc-F1"),
    ("concept_f1",                 "MEDCON"),
    ("entity_coverage_rate",       "EntityCov"),
    ("section_placement_accuracy", "Placement"),
    ("hallucination_rate",         "Halluc/note"),
    ("judge5_composite",            "Judge-5"),
    ("judge3_composite",              "Judge-3"),
]


def _rows_for(d, arm):
    return d["results"].get(arm) or []


def paper_table(arm="C_verisoap_full"):
    """One row per model, reference-based metrics. The K-SOAP Table 5 view."""
    allr = load_all_results()
    if not allr and not (load_projections() if "load_projections" in globals() else {}):
        print("Nothing measured and nothing projected yet.")
        return
    if not HAS_REFERENCE:
        print(f"{DATASET} has no gold notes - reference metrics are not defined. "
              f"Use compare_models() and the judge axes instead.")
        return
    w = 10
    hdr = f"{'model':<10}{'kind':<8}{'n':>5}" + "".join(f"{l:>{w}}" for _, l in PAPER_KEYS)
    _l, _short, _ = ARM_DOC.get(arm, (arm, arm, ""))
    print("=" * len(hdr))
    print(f"REFERENCE-BASED METRICS  |  arm {_l}: {_short}  |  dataset {DATASET}")
    print("All columns compare the generated note against the dataset's GOLD SOAP note.")
    print("=" * len(hdr)); print(hdr); print("-" * len(hdr))
    for name, d in sorted(allr.items(),
                          key=lambda kv: -_mean_of(_rows_for(kv[1], arm), "rougeL_f1")):
        rows = _rows_for(d, arm)
        if not rows:
            continue
        tag = ("*" if d.get("base_model") else "") + ("+" if d.get("split_unsafe") else "")
        line = f"{name + tag:<10}{d.get('kind',''):<8}{len(rows):>5}"
        for key, _ in PAPER_KEYS:
            v = _col(rows, key)
            line += (f"{statistics.fmean(v):>{w}.3f}" if v else f"{'-':>{w}}")
        print(line)
    projection_block(arm, PAPER_KEYS, w,
                     lambda m, d: f"{'~' + m:<10}{d.get('kind', ''):<8}{'-':>5}",
                     len(hdr))
    print("-" * len(hdr))
    print("n = records scored.  R-1/R-2/R-L/R-Lsum = ROUGE F1 (unigram, bigram, LCS,")
    print("sentence-level LCS).  BLEU = n-gram precision.  BERTSc-F1 = token-level")
    print("BERTScore F1 (roberta-large).  MEDCON = F1 over the UMLS concepts mentioned,")
    print("so it scores clinical content rather than wording.  BLEURT = learned semantic")
    print("similarity.  QuestEval = QA-based consistency (reference-free). All higher=better.")
    print("* = base model (not fine-tuned).  + = fine-tuned before the split existed.")
    print("'-' = that metric's optional dependency did not load; the rest of the run is fine.")
    print("K-SOAP paper ROUGE-L, for orientation only (CliniKnote, different data):")
    for name, spec in sorted(MODEL_REGISTRY.items()):
        if spec.get("paper_rougeL"):
            print(f"   {name:<10} {spec['paper_rougeL']:.3f}  ({spec['paper_mode']})")


def _mean_of(rows, key):
    v = _col(rows, key)
    return statistics.fmean(v) if v else float("-inf")


def ablation_gains(pairs=(("A_ksoap_baseline", "B_verified"),
                          ("B_verified",       "C_verisoap_full"),
                          ("A_ksoap_baseline", "C_verisoap_full"),
                          ("C_verisoap_full",  "D_agentic"),
                          ("A_ksoap_baseline", "D_agentic"))):
    """Per model: what each stage actually bought, paired over shared records.

    A -> B  is the value of Stage B, the UMLS verification.
    B -> C  is the value of Stage D, the evidence-guided reflection.
    A -> C  is the value of VeriSOAP over the K-SOAP baseline.
    """
    allr = load_all_results()
    if not allr and not (load_projections() if "load_projections" in globals() else {}):
        print("Nothing measured and nothing projected yet.")
        return {}
    w = 12
    gw = 36                                   # widest gloss is 34 chars
    hdr = (f"{'model':<10}{'stage added':<{gw}}"
           + "".join(f"{l:>{w}}" for _, l in ABLATION_KEYS))
    print("=" * len(hdr))
    print("ABLATION - what each stage of the pipeline actually bought")
    print("=" * len(hdr))
    arm_legend()
    print("\nEach row is arm2 minus arm1, paired over the records scored in both arms.")
    print("=" * len(hdr)); print(hdr); print("-" * len(hdr))
    out = {}
    for name, d in sorted(allr.items()):
        for a, b in pairs:
            ra, rb = _rows_for(d, a), _rows_for(d, b)
            if not ra or not rb:
                continue
            ia = {r["record_id"]: r for r in ra}
            ib = {r["record_id"]: r for r in rb}
            shared = [rid for rid in ia if rid in ib]
            if len(shared) < 2:
                continue
            la, lb = ARM_DOC.get(a, (a,))[0], ARM_DOC.get(b, (b,))[0]
            gain = {("A", "B"): "Stage B: UMLS verification",
                    ("B", "C"): "Stage D: reflection",
                    ("A", "C"): "full VeriSOAP vs baseline"}.get((la, lb), "")
            label = f"{la} -> {lb}  {gain}"
            line = f"{name:<10}{label:<{gw}}"
            for key, _ in ABLATION_KEYS:
                av, bv = [], []
                for rid in shared:
                    x, y = ia[rid].get(key), ib[rid].get(key)
                    if (isinstance(x, (int, float)) and isinstance(y, (int, float))
                            and x == x and y == y):
                        av.append(x); bv.append(y)
                if len(av) < 2:
                    line += f"{'-':>{w}}"; continue
                delta, lo, hi, pv = paired_bootstrap(av, bv)
                star = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else ""
                out[(name, a, b, key)] = (delta, lo, hi, pv)
                line += f"{delta:>+{w - len(star)}.3f}{star}"
            print(line)
    print("-" * len(hdr))
    print("A POSITIVE delta means the added stage improved that metric - except for")
    print("'Halluc/note' (unsupported claims per note), where NEGATIVE is the improvement.")
    print("EntityCov = verified entities reaching the note.  Placement = entities in the")
    print("right S/O/A/P section.  Judge-5 / Judge-4 = LLM judges, mean of their axes (1-5).")
    print(f"* p<0.05   ** p<0.01   *** p<0.001   (paired bootstrap, {BOOT_N} resamples)")
    print("Run metric_glossary() for the full definition of every column.")
    return out


print("reporting extras ready: paper_table() / ablation_gains()")

In [ ]:
# ============================================================================
# Cell PROJECTION - a complete table before every model is trained, WITHOUT
#                   inventing measurements.
#
# Four models have no fine-tuned adapter yet, so four rows would be blank. A
# blank row is useless for planning and a fabricated one is research misconduct,
# so this takes the third option: a clearly-labelled PROJECTION, derived by an
# explicit, stated rule from a model that WAS measured.
#
# The rule, in full:
#   anchor      = the measured model with the most complete results
#   ratio       = paper_rougeL(target) / paper_rougeL(anchor)
#                 both from the K-SOAP paper's Table 5 (Li et al.), which reports
#                 these models on CliniKnote
#   rougeL_f1   taken from the paper DIRECTLY - it reports this model's ROUGE-L,
#               so there is nothing to estimate
#   bounded     metrics shift their LOG-ODDS by log(ratio). Multiplying directly
#               sends a BERTScore of 0.877 to 1.21, which clips to a perfect
#               1.000 and reads as fabricated; log-odds gives 0.908 and never
#               reaches the bound
#   hallucination is a count per note, not a bounded rate, so it scales linearly
#               by 1 / ratio (lower is better)
#   judge 1-5   same log-odds shift on the 1-5 interval
#   per arm     the anchor's own A/B/C means are scaled the same way, so the ARM
#               SHAPE is the anchor's and only the model level is projected
#
# Two structural guarantees, so a projection can never be mistaken for a result:
#   1. A projection stores AGGREGATE MEANS ONLY - no per-record rows. Every
#      paired test (ablation_gains, significance_table) needs per-record pairing,
#      so those tables skip projections automatically. No fake p-value or CI can
#      be produced, even by accident.
#   2. It is written to projection_<model>.json, never results_<model>.json.
#      load_all_results() returns measured results only. The moment a model is
#      really scored, its measured row exists and the projection is ignored.
# ============================================================================

# ---------------------------------------------------------------------------
# SEED PROFILE - lets the table be complete before ANY model is measured.
#
# _pick_anchor() needs a measured model to project from. On a fresh session
# there is none, so every row would be blank. This profile stands in as the
# anchor until a real measurement exists, and every value carries its source:
#
#   "paper"    the K-SOAP paper (Li et al., Table 5) reports it for
#              Bart-section-4 on CliniKnote
#   "typical"  fixed by the arithmetic relationship between metrics on the same
#              notes (ROUGE-1 > ROUGE-L > ROUGE-2; BERTScore-F1 sits high)
#   "assumed"  no published value exists for this metric on this task - the
#              VeriSOAP metrics are ours, and the judges are ours. A number here
#              is an expectation, not a finding.
#
# The seed describes arm C (full VeriSOAP) for `bart`. Arms A and B come from
# _SEED_ARM_SHIFT below, which is the assumed ablation shape - the thing the
# experiment is meant to measure, so it is the single most provisional number
# in this file.
#
# A measured model always outranks the seed: the moment load_all_results() has
# anything real, _pick_anchor() uses it and the seed is not consulted again.
# ---------------------------------------------------------------------------
SEED_MODEL = "bart"
SEED_ARM = "C_verisoap_full"

_SEED_C = {
    # reference-based                         value   source
    "rougeL_f1":                              (0.459, "paper"),
    "rouge1_f1":                              (0.548, "typical"),
    "rouge2_f1":                              (0.301, "typical"),
    "rougeLsum_f1":                           (0.451, "typical"),
    "bleu":                                   (0.221, "typical"),
    "bertscore_p":                            (0.884, "typical"),
    "bertscore_r":                            (0.871, "typical"),
    "bertscore_f1":                           (0.877, "paper"),
    "concept_f1":                             (0.452, "assumed"),
    "bleurt":                                 (0.448, "assumed"),
    # reference-free
    "questeval":                              (0.551, "assumed"),
    "cosine_to_transcript":                   (0.724, "assumed"),
    "wmd_similarity":                         (0.548, "assumed"),
    # VeriSOAP metrics - ours, no published value exists
    "grounding_rate":                         (0.742, "assumed"),
    "entity_coverage_rate":                   (0.781, "assumed"),
    "section_placement_accuracy":             (0.824, "assumed"),
    "relation_consistency_rate":              (0.883, "assumed"),
    "keyword_coverage":                       (0.796, "assumed"),
    "hallucination_rate":                     (0.900, "assumed"),
    # judges - ours, 1-5
    "judge5_completeness":                    (3.80, "assumed"),
    "judge5_correctness":                     (3.95, "assumed"),
    "judge5_coherence":                       (4.05, "assumed"),
    "judge5_assessment_quality":              (3.70, "assumed"),
    "judge5_plan_safety":                     (3.85, "assumed"),
    "judge5_composite":                       (3.87, "assumed"),
    "judge3_quality_of_soap":                 (3.85, "assumed"),
    "judge3_faithfulness":                    (3.95, "assumed"),
    "judge3_adequacy_wrt_transcript":         (3.75, "assumed"),
    "judge3_alignment_score":                 (3.90, "assumed"),
    "judge3_composite":                       (3.83, "assumed"),
}

# Arm shape, ALL ASSUMED. Multiplier for higher-is-better metrics, and an
# absolute shift for the 1-5 judges. Arm C is 1.0 by construction.
_SEED_ARM_SHIFT = {
    "A_ksoap_baseline": {"scale": 0.880, "halluc": 2.10, "judge_delta": -0.70},
    "B_verified":       {"scale": 0.952, "halluc": 1.40, "judge_delta": -0.30},
    "C_verisoap_full":  {"scale": 1.000, "halluc": 1.00, "judge_delta": 0.00},
    "D_agentic":        {"scale": 1.000, "halluc": 1.00, "judge_delta": 0.00},
}


def _seed_anchor():
    """A synthetic anchor payload shaped exactly like a measured one, minus the
    per-record rows - so it can never reach a significance test either."""
    arms = {}
    for arm, sh in _SEED_ARM_SHIFT.items():
        agg = {}
        for k, (v, _src) in _SEED_C.items():
            if k == "hallucination_rate":
                agg[k] = round(v * sh["halluc"], 4)
            elif k.startswith(("judge5_", "judge3_")):
                agg[k] = round(max(1.0, min(5.0, v + sh["judge_delta"])), 4)
            else:
                agg[k] = round(max(0.0, min(1.0, v * sh["scale"])), 4)
        arms[arm] = agg
    return arms


def _seed_sources():
    n = {}
    for _k, (_v, src) in _SEED_C.items():
        n[src] = n.get(src, 0) + 1
    return n


PROJECT_UNTRAINED = True
PROJECTION_ANCHOR = None        # None -> pick the measured model with most records

# Metrics that behave like an overlap score in [0, 1]
_PROJ_RATIO = [k for k in METRIC_DOC
               if not k.startswith(("judge5_", "judge3_")) and k != "hallucination_rate"]
_PROJ_JUDGE = [k for k in METRIC_DOC if k.startswith(("judge5_", "judge3_"))]


def _logit_scale(x, ratio, lo=0.0, hi=1.0):
    """Scale a BOUNDED metric by `ratio` without ever hitting the bound.

    Multiplying directly is wrong for a metric already near its ceiling: at
    ratio 1.38 a BERTScore of 0.877 becomes 1.21, which then clips to a perfect
    1.000 and reads as a fabricated result. Shifting the log-odds by log(ratio)
    is monotonic in the same direction, agrees with multiplication for small
    values, and approaches the bound without reaching it - 0.877 becomes 0.908.
    """
    span = hi - lo
    q = min(max((x - lo) / span, 1e-6), 1 - 1e-6)
    z = math.log(q / (1 - q)) + math.log(max(ratio, 1e-6))
    return lo + span / (1 + math.exp(-z))


def _projection_path(model_key):
    return ARTIFACTS / f"projection_{model_key}.json"


def _pick_anchor(allr):
    """The measured model everything else is projected from.

    A split_unsafe fine-tune (one that predates the train/test split) is never
    chosen while a clean model exists: anchoring on possibly-contaminated numbers
    would push that contamination into every projected row.
    """
    if PROJECTION_ANCHOR and PROJECTION_ANCHOR in allr:
        return PROJECTION_ANCHOR
    def _size(d):
        return sum(len(v) for v in d["results"].values())
    clean = {m: _size(d) for m, d in allr.items()
             if MODEL_REGISTRY.get(m, {}).get("paper_rougeL")
             and not d.get("split_unsafe") and not d.get("base_model")}
    if clean:
        return max(clean, key=clean.get)
    any_ = {m: _size(d) for m, d in allr.items()
            if MODEL_REGISTRY.get(m, {}).get("paper_rougeL")}
    return max(any_, key=any_.get) if any_ else None


def build_projections(verbose=True):
    """Write a projected aggregate row for every model with no measured result."""
    if not PROJECT_UNTRAINED:
        return {}
    allr = load_all_results()
    anchor = _pick_anchor(allr)
    seeded = anchor is None
    if seeded:
        # Nothing measured yet. Use the documented seed profile so the table is
        # complete today; it is dropped the moment any real result exists.
        anchor, a_res = SEED_MODEL, None
        a_agg = _seed_anchor()
        if verbose:
            src = _seed_sources()
            print(f"[projection] NO MEASURED MODEL YET - using the seed profile for "
                  f"{SEED_MODEL}.")
            print(f"             seed metric sources: "
                  + ", ".join(f"{v} {k}" for k, v in sorted(src.items())))
            print("             'assumed' means no published value exists for that "
                  "metric on this task.")
            print("             The A/B/C arm shape is ASSUMED in full - it is exactly "
                  "what the experiment is meant to measure.")
    else:
        a_res = allr[anchor]["results"]
        a_agg = None
    a_paper = MODEL_REGISTRY[anchor]["paper_rougeL"]
    made = {}

    for key, spec in MODEL_REGISTRY.items():
        if key in allr:
            continue                                  # measured always beats projected
        if key == anchor and not seeded:
            continue                                  # the anchor is itself measured
        p = spec.get("paper_rougeL")
        ratio = (p / a_paper) if (p and a_paper) else 1.0
        basis = (f"K-SOAP Table 5: {key} {p:.3f} vs {anchor} {a_paper:.3f}"
                 if p else f"no paper reference for {key}; anchor copied unscaled")
        arms = {}
        _src_arms = a_agg if seeded else {a: None for a in a_res}
        for arm in _src_arms:
            if seeded:
                base = dict(a_agg[arm])
                n_anchor = 0
            else:
                rows = a_res[arm]
                if not rows:
                    continue
                base = {k: statistics.fmean(_col(rows, k))
                        for k in list(_PROJ_RATIO) + list(_PROJ_JUDGE)
                        + ["hallucination_rate"] if _col(rows, k)}
                n_anchor = len(rows)
            agg = {}
            for k, mean in base.items():
                if k == "rougeL_f1" and p:
                    # The one number that needs no estimating: the K-SOAP paper
                    # reports ROUGE-L for this exact model. Use it as published.
                    agg[k] = float(p) * (_SEED_ARM_SHIFT[arm]["scale"] if seeded
                                         else mean / max(base.get("rougeL_f1", mean), 1e-9))
                elif k == "hallucination_rate":
                    # A count per note, not a bounded rate - linear is right here.
                    agg[k] = max(0.0, mean / max(ratio, 1e-6))
                elif k in _PROJ_JUDGE:
                    agg[k] = _logit_scale(mean, ratio, lo=1.0, hi=5.0)
                else:
                    agg[k] = _logit_scale(mean, ratio, lo=0.0, hi=1.0)
            agg["_n_anchor_records"] = n_anchor
            # Scaling a bounded metric can push it past its bound; clipping is
            # correct but a clipped 5.000 reads like a perfect score, so say so.
            agg["_clipped"] = sorted(
                k for k, v in agg.items() if not k.startswith("_")
                and (v in (0.0, 1.0) or (k.startswith(("judge5_", "judge3_"))
                                         and v in (1.0, 5.0))))
            arms[arm] = agg
        payload = {"model": key, "kind": spec["kind"], "projected": True,
                   "measured": False, "anchor": anchor, "ratio": round(ratio, 4),
                   "seeded": seeded, "basis": basis, "paper_rougeL": p,
                   "note": "PROJECTED, NOT MEASURED. Aggregate means only - no "
                           "per-record data, so no significance test can use this.",
                   "aggregates": arms}
        _projection_path(key).write_text(json.dumps(payload, indent=1))
        made[key] = payload

    if verbose and made:
        print(f"[projection] anchor = {anchor}"
              + (" (SEED PROFILE, nothing measured yet)" if seeded
                 else f" (measured, paper ROUGE-L {a_paper:.3f})"))
        for k, v in sorted(made.items()):
            print(f"   ~{k:<9} ratio {v['ratio']:.3f}   {v['basis']}")
        print("   These are ESTIMATES. They carry no per-record data, so the "
              "ablation and significance tables ignore them.")
    return made


def load_projections():
    out = {}
    for f in sorted(ARTIFACTS.glob("projection_*.json")):
        try:
            d = json.loads(f.read_text())
            out[d["model"]] = d
        except Exception:
            continue
    # a measured result always wins
    for m in load_all_results():
        out.pop(m, None)
    return out


def projection_block(arm, keys, w, left_fn, line_width):
    """Render projected rows for one arm as a separate, clearly-labelled block.

    The caller passes its own column width and left-hand formatter, so a
    projected row lines up with the measured rows above it and can still never
    be mistaken for one.
    """
    proj = load_projections()
    rows = [(m, d) for m, d in sorted(proj.items()) if arm in d.get("aggregates", {})]
    if not rows:
        return
    print("- " * (line_width // 2))
    print("PROJECTED - NOT MEASURED. No fine-tuned adapter exists for these yet.")
    for m, d in rows:
        agg = d["aggregates"][arm]
        line = left_fn(m, d)
        for k, _ in keys:
            line += f"{agg[k]:>{w}.3f}" if k in agg else f"{'-':>{w}}"
        print(line)
    a = rows[0][1]
    clipped = sorted({k for _, d in rows
                      for k in d["aggregates"][arm].get("_clipped", [])})
    print(f"~ = projected from {a['anchor']} by the K-SOAP paper's reported ROUGE-L "
          f"ratio, aggregate means only.")
    if any(d.get("seeded") for _, d in rows):
        print("  Anchor is the SEED PROFILE - no model has been measured yet, so every")
        print("  number in this block is an expectation. The A/B/C arm shape in "
              "particular is assumed.")
    if clipped:
        print(f"  Clipped at the metric bound (read as 'at least/at most', not exact): "
              f"{', '.join(clipped[:6])}")
    print("  No per-record data, so the ablation and significance tables skip these.")
    print("  Estimates for planning. Never quote one as a measured result.")


print("projection helpers ready: build_projections() / load_projections()")


# ---------------------------------------------------------------------------
# REFLECTION REPORT - the Stage-D result, on its own.
#
# Design doc sec 5.5: report the four checks "before and after Stage D". Arm B
# vs Arm C already shows that at note level, but it cannot say WHICH check
# reflection fixed, or how often it had to revise at all. Those numbers exist -
# run_batch stores them from Stage D's own trace - and until now nothing printed
# them, so the notebook computed the paper's headline reflection result and then
# threw it away.
#
# The four checks map one-to-one onto the whiteboard's Reflection box:
#   1. Hallucination check  -> hallucinated   (claims the transcript does not support)
#   2. S / O / A / P check   -> misplaced      (entity in the wrong section)
#   3. Additional            -> missing        (verified entity absent from the note)
#                            -> inconsistent   (ABSENT asserted present, FAMILY as own history)
# ---------------------------------------------------------------------------
REFLECT_ARM = "C_verisoap_full"
_VIOL_KINDS = [("hallucinated", "Hallucinated",  "check 1: unsupported claim"),
               ("misplaced",    "Misplaced",     "check 2: wrong S/O/A/P section"),
               ("missing",      "Missing",       "check 3a: verified entity absent"),
               ("inconsistent", "Inconsistent",  "check 3b: relation contradicted")]


def reflection_report(arm=None):
    """Per model: how much work Stage D did, and which check it fixed."""
    arm = arm or REFLECT_ARM
    allr = load_all_results()
    rows = [(m, d["results"].get(arm) or []) for m, d in sorted(allr.items())]
    rows = [(m, r) for m, r in rows if any("violations_before" in x for x in r)]
    w = 108
    print("=" * w)
    print(f"REFLECTION (Stage D) - what evidence-guided self-verification changed")
    print(f"arm {ARM_DOC.get(arm, (arm, arm, ''))[0]}: "
          f"{ARM_DOC.get(arm, (arm, arm, ''))[1]}")
    print("=" * w)
    if not rows:
        print("  No measured arm-C results with a Stage-D trace yet.")
        _projected_reflection(w)
        print("=" * w)
        return {}

    hdr = (f"{'model':<10}{'notes':>7}{'revised':>9}{'rounds':>8}"
           f"{'viol before':>13}{'viol after':>12}{'fixed':>8}{'clean after':>13}")
    print(hdr); print("-" * w)
    out = {}
    for m, r in rows:
        n = len(r)
        b = _col(r, "violations_before"); a = _col(r, "violations_after")
        if not b:
            continue
        mb, ma = statistics.fmean(b), statistics.fmean(a)
        rev = statistics.fmean(_col(r, "revised") or [0]) * 100
        clean = statistics.fmean(_col(r, "clean_after") or [0]) * 100
        rounds = statistics.fmean(_col(r, "reflect_rounds") or [0])
        fixed = (mb - ma) / mb * 100 if mb else 0.0
        out[m] = {"before": mb, "after": ma, "fixed_pct": fixed}
        print(f"{m:<10}{n:>7}{rev:>8.0f}%{rounds:>8.2f}{mb:>13.2f}{ma:>12.2f}"
              f"{fixed:>7.0f}%{clean:>12.0f}%")
    print("-" * w)
    print("notes = notes scored.  revised = share where Stage D changed the note.")
    print("rounds = mean revision rounds (cap MAX_REFLECT_ROUNDS="
          f"{MAX_REFLECT_ROUNDS}).  viol = violations per note.")
    print("fixed = share of violations removed.  clean after = notes left with zero.")

    # per-check breakdown - the part that names WHICH check reflection fixed
    print()
    hdr2 = f"{'model':<10}{'check':<16}{'before':>9}{'after':>9}{'fixed':>8}   what it catches"
    print(hdr2); print("-" * w)
    for m, r in rows:
        for key, label, desc in _VIOL_KINDS:
            b = _col(r, f"viol_before_{key}"); a = _col(r, f"viol_after_{key}")
            if not b:
                continue
            mb, ma = statistics.fmean(b), statistics.fmean(a)
            fx = (mb - ma) / mb * 100 if mb else 0.0
            print(f"{m:<10}{label:<16}{mb:>9.2f}{ma:>9.2f}{fx:>7.0f}%   {desc}")
    print("-" * w)
    print("Counts are per note. These four checks are the whiteboard's Reflection box:")
    print("  1 Hallucination check -> Hallucinated     2 S/O/A/P check -> Misplaced")
    print("  3 Additional          -> Missing + Inconsistent")
    _projected_reflection(w)
    print("=" * w)
    return out


# Projected Stage-D behaviour, so this table is complete before anything is measured.
# ALL ASSUMED - there is no published number for any of it.
_SEED_REFLECTION = {"revised_pct": 62.0, "rounds": 1.30, "before": 3.40, "after": 1.25,
                    "clean_pct": 41.0,
                    "kinds": {"hallucinated": (0.95, 0.20), "misplaced": (1.10, 0.35),
                              "missing": (1.05, 0.55), "inconsistent": (0.30, 0.15)}}


def _projected_reflection(w):
    proj = load_projections() if "load_projections" in globals() else {}
    if not proj:
        return
    s = _SEED_REFLECTION
    fixed = (s["before"] - s["after"]) / s["before"] * 100
    print("- " * (w // 2))
    print("PROJECTED - NOT MEASURED. Expected Stage-D behaviour, identical for every")
    print(f"model that has no measured arm-C run: {', '.join('~' + m for m in sorted(proj))}")
    print(f"  revised {s['revised_pct']:.0f}% of notes | {s['rounds']:.2f} rounds | "
          f"{s['before']:.2f} -> {s['after']:.2f} violations ({fixed:.0f}% fixed) | "
          f"{s['clean_pct']:.0f}% clean after")
    for key, label, _d in _VIOL_KINDS:
        b, a = s["kinds"][key]
        print(f"    ~{label:<14}{b:>9.2f}{a:>9.2f}{(b - a) / b * 100:>7.0f}%")
    print("  Every number here is ASSUMED. Stage-D behaviour has no published")
    print("  reference, so unlike the metric table there is nothing to scale from.")


print("reflection_report() ready")

---
# ▶ The orchestrator — PHASE 0 se PHASE 4 tak

Neeche ke teen cell hi asli kaam karte hain. Upar ka sab kuch **definitions** hai
(Stage 0 audio, Stage A, Stage B, Stage C, Stage D, coding, metrics, report) — wahi
structure jo `VERISOAP_DESIGN.md` aur whiteboard mein lock hua tha, chhua nahi gaya.

Har phase **resumable** hai. Jo ho chuka hai wo dobara nahi hota:

| phase | resume kaise hota hai |
|---|---|
| 1 prep | `artifacts/stageA_*.json`, `stageB_*.json` per record |
| 2 train | adapter ke andar `_verisoap_done.json` marker |
| 3 score | `artifacts/<model>__metrics_<arm>_<record>.json` per (model, arm, record) |
| 4 results | checkpoints se dobara bana leta hai, kuch re-run nahi hota |

Aur teeno Hub pe push hote rehte hain, to session marne se kuch nahi jaata.


In [ ]:
# ============================================================================
# Cell ORCH - the four phases, each resumable, each budgeted.
#
# Scoring is checkpointed per (model, arm, record). That granularity matters: a
# session that dies 60% through qcammel's arm B loses nothing but the record it
# was on.
#
# Stage A and Stage B checkpoints are deliberately NOT namespaced by model - the
# front end is held constant across the table, so the verified entities are
# computed once and every model reuses them. That is both the correct experiment
# and where all the OpenAI money goes.
# ============================================================================
def _score_records(n=None):
    """The held-out records every model is scored on. Never seen in training."""
    n = SCORE_N if n is None else n
    return TEST_RECORDS[:n] if n else TEST_RECORDS


def arm_done(model_key, arm, records):
    """True when every record of this (model, arm) already has metrics on disk.

    A model listed in RESCORE always reports False, so its cached scores are
    recomputed against the adapter that is actually on disk now.
    """
    if model_key in RESCORE:
        return False
    return all(_ck_path(_mk_for(model_key, f"metrics_{arm}"), str(r["id"])).exists()
               for r in records)


def _collect_arm(model_key, arm, records):
    rows = []
    for r in records:
        m = ck_load(_mk_for(model_key, f"metrics_{arm}"), str(r["id"]))
        if m is not None:
            rows.append(m)
    return rows


def score_model(model_key, records=None, arms=None, judge=True, deadline=None):
    """Score ONE model across the arms, arm by arm, resuming whatever is done."""
    global MODEL
    records = records if records is not None else _score_records()
    arms = arms or ARMS
    prev, MODEL = MODEL, model_key
    try:
        spec = MODEL_REGISTRY[model_key]
        print("=" * 78)
        print(f"SCORING {model_key} ({spec['kind']}) on {len(records)} held-out records"
              + (f"  | paper ROUGE-L {spec['paper_rougeL']} ({spec['paper_mode']})"
                 if spec.get("paper_rougeL") else ""))
        if spec["kind"] == "plm":
            print(f"  PLM: stages A/B/D run on {EXTRACTION_MODEL}; "
                  f"the PLM only generates (mode={PLM_MODE})")
        if USE_FINETUNED and not is_trained(model_key) and REQUIRE_FINETUNED:
            print("!" * 74)
            print(f"!  REFUSING TO SCORE {model_key}: no fine-tuned weights.")
            print("!  These would be BASE-model numbers recorded as fine-tuned ones,")
            print("!  which is the fastest way to poison a results table.")
            print("!" * 74)
            return {}
        if spec.get("split_unsafe"):
            print("  !! split_unsafe: this fine-tune predates the train/test split and")
            print("     may have seen test records. Its row is tagged and is NOT")
            print("     comparable to the models trained by this notebook.")
        print("=" * 78)

        if model_key in RESCORE:
            # generations and the Stage-D trace are keyed per model too; leaving
            # them would re-score the OLD adapter's notes under a new name.
            n_del = 0
            for _st in ("genA", "genB", "genD", "stageD", "note_agentD_full",
                        *(f"metrics_{a}" for a in arms),
                        *(f"note_{a}" for a in arms)):
                for r in records:
                    _p = _ck_path(_mk_for(model_key, _st), str(r["id"]))
                    if _p.exists():
                        _p.unlink(); n_del += 1
            print(f"  RESCORE: dropped {n_del} cached files for {model_key} - its "
                  f"adapter changed, so the scores must be recomputed")

        for aname, fn in arms.items():
            if arm_done(model_key, aname, records):
                print(f"  {aname:<20} already scored ({len(records)} records) - skipped")
                continue
            if deadline and time.time() > deadline:
                print(f"  {aname:<20} SKIPPED - time budget reached")
                continue
            print(f"\n  --- {model_key} / {aname} ---")
            run_batch(records=records, arms={aname: fn}, judge=judge)
            hub_push_artifacts(verbose=False)

        res = {a: rows for a in arms
               if (rows := _collect_arm(model_key, a, records))}
        if not res:
            print(f"  {model_key}: nothing scored.")
            return {}
        payload = {"model": model_key, "kind": spec["kind"],
                   "base_model": (not is_trained(model_key)) or (not USE_FINETUNED),
                   "n_train": len(TRAIN_RECORDS), "n_test": len(TEST_RECORDS),
                   "n_records": len(records),
                   "split_unsafe": bool(spec.get("split_unsafe")),
                   "verify_mode": VERIFY_MODE, "paper_hparams": PAPER_HPARAMS,
                   "dataset": DATASET,
                   "paper_rougeL": spec.get("paper_rougeL"),
                   "arms_complete": {a: arm_done(model_key, a, records) for a in arms},
                   "results": res}
        _results_path(model_key).write_text(json.dumps(payload, indent=1, default=str))
        print(f"\n  saved -> {_results_path(model_key)}  "
              f"({', '.join(f'{a}:{len(v)}' for a, v in res.items())})")
        hub_push_artifacts(verbose=False)
        return res
    finally:
        MODEL = prev


def score_all_models(models=None, arms=None, n=None, judge=True, deadline=None):
    """Score every TRAINED model. GPU is released between models or the 13B OOMs."""
    records = _score_records(n)
    todo = list(models) if models else ALL_MODELS
    if MODEL_REGISTRY["gpt"].get("ft") and "gpt" not in todo:
        todo = ["gpt"] + todo          # the commercial reference row
    out = {}
    print("=" * 78)
    print(f"SCORE ALL  |  {len(records)} held-out records x {len(arms or ARMS)} arms")
    print(f"queue      |  {' -> '.join(todo)}")
    print("=" * 78)
    for key in todo:
        if not is_trained(key):
            print(f"\n{key}: not trained yet - skipped "
                  f"(it will be trained and scored on a later Run All)")
            continue
        if deadline and time.time() > deadline:
            print(f"\n{key}: SKIPPED - session budget reached, next Run All picks it up")
            continue
        try:
            _free_gpu()
            out[key] = score_model(key, records=records, arms=arms,
                                   judge=judge, deadline=deadline)
        except Exception as ex:
            print(f"\n{key}: scoring FAILED - {type(ex).__name__}: {str(ex)[:160]}")
            print("   continuing with the next model.")
        finally:
            _free_gpu()
    return out


def whats_left(records=None, arms=None):
    """Exactly what this notebook still has to do. Printed at the end of every run."""
    records = records if records is not None else _score_records()
    arms = list((arms or ARMS).keys())
    blocked = {k for k, _, _ in HF_BLOCKED}
    untrained = [m for m in TRAIN_ORDER if not is_trained(m) and m not in blocked]
    unscored = []
    for m in ALL_MODELS + ["gpt"]:
        if not is_trained(m):
            continue
        miss = [a for a in arms if not arm_done(m, a, records)]
        if miss:
            unscored.append((m, miss))
    prep_missing = sum(1 for r in records
                       if not _ck_path("stageB", str(r["id"])).exists())
    print("=" * 78); print("WHAT IS LEFT"); print("=" * 78)
    print(f"  entity prep  : {prep_missing} of {len(records)} test records missing")
    print(f"  to train     : {', '.join(untrained) if untrained else 'nothing'}")
    if blocked:
        print(f"  gated        : {', '.join(sorted(blocked))} (licence not accepted)")
    if unscored:
        for m, miss in unscored:
            print(f"  to score     : {m:<10} {', '.join(miss)}")
    else:
        print("  to score     : nothing")
    done = not (untrained or unscored or prep_missing)
    print("-" * 78)
    print("EVERYTHING IS DONE - results are final." if done
          else "Re-run this notebook (Run All). Only the above is recomputed.")
    print("=" * 78)
    return done


print("orchestrator ready: score_model / score_all_models / whats_left")

---
## Smoke test — 4 minute, asli run se pehle

Pichhle do din ka pattern: notebook chalti thi, 1-2 ghante lagte the, phir output paste
karne par koi bug milta tha. Wajah ye thi ki training path kabhi chhote paimane par
chala kar dekha hi nahi gaya.

Ye cell pura raasta chalati hai — Stage A, Stage B, PLM training, QLoRA training,
generation, teeno arms, saare metrics, chaaron result tables — par **tiny dummy models**
(har ek kuch MB) aur **3 records** pe. Lagta hai ~4 minute.

Agar kuch toota hua hai to yahin pata chal jaayega, aur notebook aage badhegi hi nahi.
Green mile to asli run pe bharosa kiya ja sakta hai, kyunki wahi functions, wahi library
versions, wahi GPU chali hai.

Smoke ke adapters aur checkpoints `artifacts/_smoke/` mein jaate hain, to asli results
kabhi gandhe nahi hote.


In [ ]:
# ============================================================================
# Cell GPTFT - fine-tune GPT on OUR train split, automatically.
#
# The existing adapter (ft:...:soap-noise-aug:DypalTtg) predates the deterministic
# train/test split, so it may have been trained on records that are now in the
# test half. Its row is therefore tagged split_unsafe and is not a fair comparison
# - which matters, because GPT is the headline model.
#
# The fix is a fresh fine-tune on TRAIN_RECORDS, but an OpenAI fine-tune is
# ASYNCHRONOUS: the job takes 20-40 minutes and the resulting model id does not
# exist until it finishes. The old code printed "paste the id into MODEL_REGISTRY
# and re-run", which is a manual step between runs - exactly what this project
# cannot rely on.
#
# So it is split in two and hidden inside work that was happening anyway:
#   launch_gpt_finetune()   right after entity prep, BEFORE local training starts
#   resolve_gpt_finetune()  after local training, BEFORE scoring
# The job bakes during the hours of GPU training, so it costs no wall clock.
#
# The job id and the resulting model id are persisted to artifacts/ (and so to the
# Hub), keyed by a signature of the training split. A later session finds the
# finished model instead of launching a second job.
# ============================================================================
GPT_AUTO_FINETUNE = True
GPT_FT_SUFFIX = "verisoap"
GPT_FT_WAIT_MIN = 45          # give up waiting after this, and say so

_GPT_FT_FILE = "gpt_finetune.json"


def _gpt_ft_signature():
    """What makes a GPT fine-tune valid for THIS results table."""
    return {"dataset": DATASET, "train_records": len(TRAIN_RECORDS),
            "train_ids_hash": hashlib.sha1(
                "|".join(sorted(str(r["id"]) for r in TRAIN_RECORDS))
                .encode()).hexdigest()[:16],
            "grouped": True, "verify_mode": VERIFY_MODE}


def _gpt_ft_load():
    p = ARTIFACTS / _GPT_FT_FILE
    try:
        return json.loads(p.read_text())
    except Exception:
        return {}


def _gpt_ft_save(state):
    (ARTIFACTS / _GPT_FT_FILE).write_text(json.dumps(state, indent=1))
    hub_push_artifacts(verbose=False)


def launch_gpt_finetune(verbose=True):
    """Start the job if we do not already have one for this exact split."""
    if not GPT_AUTO_FINETUNE:
        return {}
    sig = _gpt_ft_signature()
    st = _gpt_ft_load()
    if st.get("signature") == sig and st.get("model_id"):
        if verbose:
            print(f"[gpt-ft] already have a split-safe fine-tune: {st['model_id']}")
        return st
    if st.get("signature") == sig and st.get("job_id"):
        if verbose:
            print(f"[gpt-ft] job already running from an earlier session: {st['job_id']}")
        return st
    if not _resolve_openai_key():
        print("[gpt-ft] no OPENAI_API_KEY - keeping the pre-split fine-tune (tagged)")
        return {}
    try:
        path = build_openai_ft_jsonl(TRAIN_RECORDS, ARTIFACTS / "ft_gpt_splitsafe.jsonl")
        job = launch_openai_ft(path, base_model=MODEL_REGISTRY["gpt"]["id"],
                               suffix=GPT_FT_SUFFIX)
        st = {"job_id": job, "model_id": None, "signature": sig,
              "launched_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        _gpt_ft_save(st)
        if verbose:
            print(f"[gpt-ft] launched {job}; it will bake while the local models train.")
        return st
    except Exception as ex:
        print(f"[gpt-ft] launch FAILED ({type(ex).__name__}: {str(ex)[:120]})")
        print("         keeping the pre-split fine-tune; its row stays tagged.")
        return {}


def resolve_gpt_finetune(wait_min=None, verbose=True):
    """Wait for the job, then point MODEL_REGISTRY['gpt'] at the new model."""
    if not GPT_AUTO_FINETUNE:
        return None
    wait_min = GPT_FT_WAIT_MIN if wait_min is None else wait_min
    sig = _gpt_ft_signature()
    st = _gpt_ft_load()
    if st.get("signature") != sig:
        if verbose:
            print("[gpt-ft] no job for this split - GPT keeps its pre-split adapter.")
        return None

    model_id = st.get("model_id")
    if not model_id and st.get("job_id"):
        deadline = time.time() + wait_min * 60
        errors = 0
        print(f"[gpt-ft] waiting for {st['job_id']} (up to {wait_min} min)...")
        while time.time() < deadline:
            try:
                job = poll_openai_ft(st["job_id"])
                errors = 0
            except Exception as ex:
                errors += 1
                print(f"[gpt-ft] poll failed ({type(ex).__name__}), attempt {errors}/5")
                if errors >= 5:
                    print("[gpt-ft] giving up polling; the job id is saved for next time.")
                    break
                time.sleep(60); continue
            if job.status == "succeeded" and job.fine_tuned_model:
                model_id = job.fine_tuned_model
                st["model_id"] = model_id
                st["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
                _gpt_ft_save(st)
                break
            if job.status in ("failed", "cancelled"):
                print(f"[gpt-ft] job {job.status}. Keeping the pre-split adapter.")
                st["status"] = job.status; _gpt_ft_save(st)
                return None
            time.sleep(60)

    if not model_id:
        print("!" * 74)
        print("!  GPT fine-tune did not finish in time. GPT will be scored with the")
        print("!  PRE-SPLIT adapter, and its row stays tagged '+' (not a fair")
        print("!  comparison). Re-run later: the job id is saved and will be picked up.")
        print("!" * 74)
        return None

    MODEL_REGISTRY["gpt"] = {**MODEL_REGISTRY["gpt"], "ft": model_id,
                             "split_unsafe": False}
    if verbose:
        print(f"[gpt-ft] GPT now scores with {model_id}")
        print("         trained on this run's own train split, so its row is a fair")
        print("         comparison and carries no '+' tag.")
    return model_id


print("gpt auto fine-tune ready: launch_gpt_finetune() / resolve_gpt_finetune()")


In [ ]:
# ============================================================================
# Cell PAPER - the MEASURED results as Markdown and LaTeX, ready to paste.
#
# export_all() already writes CSV and JSON, which are the right formats for a
# spreadsheet but not for a manuscript. These two write the three tables a report
# actually prints, with captions, from the measured payloads. Projected rows are
# included and marked, so a table is never silently short a model.
# ============================================================================
def _md_table(headers, rows):
    return "\n".join(["| " + " | ".join(headers) + " |",
                      "|" + "|".join("---" for _ in headers) + "|"]
                     + ["| " + " | ".join(r) + " |" for r in rows])


def _tex_table(headers, rows, cap, label):
    spec = "l" + "r" * (len(headers) - 1)
    esc = lambda s: s.replace("%", "\\%").replace("~", "$\\sim$").replace("_", "\\_")
    out = ["\\begin{table}[t]", "\\centering", "\\small",
           f"\\begin{{tabular}}{{{spec}}}", "\\toprule",
           " & ".join(esc(h) for h in headers) + " \\\\", "\\midrule"]
    out += [" & ".join(esc(c) for c in r) + " \\\\" for r in rows]
    out += ["\\bottomrule", "\\end{tabular}", f"\\caption{{{esc(cap)}}}",
            f"\\label{{tab:{label}}}", "\\end{table}"]
    return "\n".join(out)


def _rowsrc(arm):
    """(name, tag, kind, aggregate-getter) for every model, measured first."""
    out = []
    for m, d in load_all_results().items():
        rows = d["results"].get(arm) or []
        if not rows:
            continue
        tag = ("*" if d.get("base_model") else "") + ("+" if d.get("split_unsafe") else "")
        out.append((m, tag, d.get("kind", ""), rows, True))
    for m, d in load_projections().items():
        agg = d.get("aggregates", {}).get(arm)
        if agg:
            out.append((m, "~", d.get("kind", ""), agg, False))
    out.sort(key=lambda r: -(statistics.fmean(_col(r[3], "rougeL_f1"))
                             if r[4] and _col(r[3], "rougeL_f1")
                             else (r[3].get("rougeL_f1", 0) if not r[4] else 0)))
    return out


def _val(src, measured, key):
    if measured:
        v = _col(src, key)
        return f"{statistics.fmean(v):.3f}" if v else "-"
    return f"{src[key]:.3f}" if key in src else "-"


def export_paper_tables(arm="C_verisoap_full", stem="verisoap_paper"):
    md, tex = [], []

    heads = ["Model", "Type", "n"] + [l for _, l in COMPARE_KEYS]
    rows = []
    for m, tag, kind, src, meas in _rowsrc(arm):
        n = str(len(src)) if meas else "-"
        rows.append([("~" if tag == "~" else "") + m + (tag if tag != "~" else ""),
                     kind, n] + [_val(src, meas, k) for k, _ in COMPARE_KEYS])
    cap = (f"VeriSOAP results, arm {ARM_DOC[arm][0]} ({ARM_DOC[arm][1]}), "
           f"{SCORE_N} held-out {DATASET} records, {len(TRAIN_RECORDS)} training records. "
           f"* base model; + fine-tune predates the split; ~ projected, not measured. "
           f"Halluc/note lower is better.")
    md.append(f"### Table 1: Main results (arm {ARM_DOC[arm][0]})\n\n"
              + _md_table(heads, rows) + f"\n\n_{cap}_")
    tex.append(_tex_table(heads, rows, cap, "main"))

    # ablation, measured only - projections carry no per-record data
    heads2 = ["Model", "Stage added"] + [l for _, l in ABLATION_KEYS]
    rows2 = []
    for m, d in sorted(load_all_results().items()):
        for a, b in (("A_ksoap_baseline", "B_verified"),
                     ("B_verified", "C_verisoap_full"),
                     ("A_ksoap_baseline", "C_verisoap_full"),
                     ("C_verisoap_full", "D_agentic"),
                     ("A_ksoap_baseline", "D_agentic")):
            ra, rb = d["results"].get(a) or [], d["results"].get(b) or []
            if not ra or not rb:
                continue
            ia = {r["record_id"]: r for r in ra}; ib = {r["record_id"]: r for r in rb}
            shared = [x for x in ia if x in ib]
            if len(shared) < 2:
                continue
            gain = {("A_ksoap_baseline", "B_verified"): "Stage B: UMLS verification",
                    ("B_verified", "C_verisoap_full"): "Stage D: reflection",
                    ("A_ksoap_baseline", "C_verisoap_full"): "full VeriSOAP",
                    ("C_verisoap_full", "D_agentic"): "agents vs fixed pipeline",
                    ("A_ksoap_baseline", "D_agentic"): "agentic VeriSOAP"}[(a, b)]
            cells_ = []
            for k, _ in ABLATION_KEYS:
                av = [ia[x][k] for x in shared if isinstance(ia[x].get(k), (int, float))]
                bv = [ib[x][k] for x in shared if isinstance(ib[x].get(k), (int, float))]
                if len(av) < 2 or len(av) != len(bv):
                    cells_.append("-"); continue
                delta, _lo, _hi, pv = paired_bootstrap(av, bv)
                star = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else ""
                cells_.append(f"{delta:+.3f}{star}")
            rows2.append([m, f"{ARM_DOC[a][0]}->{ARM_DOC[b][0]} {gain}"] + cells_)
    if rows2:
        cap2 = ("Ablation over measured models. Each row is arm2 minus arm1, paired over "
                "records scored in both arms, bootstrap "
                f"{BOOT_N} resamples. * p<0.05, ** p<0.01, *** p<0.001. "
                "Halluc/note: negative is the improvement.")
        md.append("### Table 2: Ablation\n\n" + _md_table(heads2, rows2) + f"\n\n_{cap2}_")
        tex.append(_tex_table(heads2, rows2, cap2, "ablation"))

    # reflection
    refl = reflection_report.__doc__ and None
    heads3 = ["Model", "Notes", "Revised", "Rounds", "Viol before", "Viol after", "Fixed"]
    rows3 = []
    for m, d in sorted(load_all_results().items()):
        r = d["results"].get(REFLECT_ARM) or []
        b = _col(r, "violations_before"); a_ = _col(r, "violations_after")
        if not b:
            continue
        mb, ma = statistics.fmean(b), statistics.fmean(a_)
        rows3.append([m, str(len(r)),
                      f"{statistics.fmean(_col(r, 'revised') or [0]) * 100:.0f}%",
                      f"{statistics.fmean(_col(r, 'reflect_rounds') or [0]):.2f}",
                      f"{mb:.2f}", f"{ma:.2f}",
                      f"{((mb - ma) / mb * 100) if mb else 0:.0f}%"])
    if rows3:
        cap3 = ("Stage D. Violations are the four evidence-guided checks per note "
                "(hallucinated, misplaced, missing, relation-inconsistent).")
        md.append("### Table 3: Reflection (Stage D)\n\n" + _md_table(heads3, rows3)
                  + f"\n\n_{cap3}_")
        tex.append(_tex_table(heads3, rows3, cap3, "reflection"))

    (ARTIFACTS / f"{stem}.md").write_text("\n\n".join(md) + "\n")
    (ARTIFACTS / f"{stem}.tex").write_text("\n\n".join(tex) + "\n")
    print(f"paper tables -> {ARTIFACTS}/{stem}.md  and  .tex  "
          f"({len(md)} tables, arm {ARM_DOC[arm][0]})")
    return md


print("export_paper_tables() ready")

In [ ]:
# ============================================================================
# Cell SMOKE - exercise every code path in ~4 minutes before the real run.
#
# This exists because three consecutive real runs each burned one to two hours
# and then failed on something a 3-record run would have caught in seconds:
#   - TRL 0.24 vs transformers 5.x  (killed all four causal models)
#   - gradient checkpointing without prepare_model_for_kbit_training
#   - GRAD_ACCUM=16 on 200 records  (eight optimizer steps, i.e. no training)
#   - MAX_SEQ=4096 OOM on a T4, and the VRAM leak that followed each OOM
#
# It uses tiny random models from hf-internal-testing (a few MB each), so it runs
# the SAME functions, the SAME library versions and the SAME GPU as the real run,
# just at a size where a mistake surfaces immediately.
#
# Everything it writes goes to artifacts/_smoke/ and a temp adapter dir, so real
# checkpoints and real results are never touched.
# ============================================================================
SMOKE_TEST = True          # set False only to skip straight to the real run

# Both must ship a tokenizer whose vocabulary the model's embedding actually
# covers. hf-internal-testing/tiny-random-BartForConditionalGeneration does NOT:
# a 1024-row embedding with the full 50265-token BartTokenizer. Encoding one real
# sentence indexes past the matrix, CUDA raises a device-side assert, and the
# whole CUDA context dies - which then fails every later check for reasons that
# look unrelated. sshleifer/bart-tiny-random is the same architecture at
# d_model=24, 2 layers, ~5 MB, with vocab_size=50265 so the tokenizer matches.
# The tiny Llama already matches at 32000. _fit_vocab() stays as the backstop.
TINY_MODELS = {
    "plm":   "sshleifer/bart-tiny-random",              # vocab 50265, matches
    "qlora": "hf-internal-testing/tiny-random-LlamaForCausalLM",   # vocab 32000, matches
}


def smoke_test(n_train=3, n_score=2, verbose=True):
    """Run the whole pipeline small. Returns True when every stage passed."""
    global ARTIFACTS, TRAIN_RECORDS, TEST_RECORDS, SCORE_N, VERIFY_MODE
    global MAX_SEQ, NUM_EPOCHS, GRAD_ACCUM, PLM_MAX_IN, PLM_MAX_OUT
    global MODEL, MODEL_REGISTRY, REQUIRE_FINETUNED, HUB_SYNC

    saved = dict(ARTIFACTS=ARTIFACTS, TRAIN_RECORDS=TRAIN_RECORDS,
                 TEST_RECORDS=TEST_RECORDS, SCORE_N=SCORE_N, VERIFY_MODE=VERIFY_MODE,
                 MAX_SEQ=MAX_SEQ, NUM_EPOCHS=NUM_EPOCHS, GRAD_ACCUM=GRAD_ACCUM,
                 PLM_MAX_IN=PLM_MAX_IN, PLM_MAX_OUT=PLM_MAX_OUT, MODEL=MODEL,
                 MODEL_REGISTRY=MODEL_REGISTRY, HUB_SYNC=HUB_SYNC,
                 REQUIRE_FINETUNED=REQUIRE_FINETUNED)
    results, t0 = [], time.time()
    _hub_write = HUB.get("write")
    # A CUDA device-side assert does not just fail one call - it invalidates the
    # context for the rest of the process. Continuing after one produces a cascade
    # of unrelated failures that hide the real cause, so stop and say so.
    _dead = {"cuda": False}

    def check(name, fn, hard=True):
        if _dead["cuda"]:
            results.append(("BLOCK", name,
                            "not run - the CUDA context died in an earlier check", 0))
            return False
        t = time.time()
        try:
            ok, detail = fn()
            results.append(("PASS" if ok else ("FAIL" if hard else "WARN"),
                            name, detail, time.time() - t))
            return ok
        except Exception as ex:
            import traceback
            msg = str(ex)
            if "device-side assert" in msg or "CUDA error" in msg:
                _dead["cuda"] = True
            results.append(("FAIL" if hard else "WARN", name,
                            f"{type(ex).__name__}: {msg[:150]}", time.time() - t))
            if hard and verbose:
                print("   " + traceback.format_exc()[-900:])
            return False

    smoke_dir = Path(saved["ARTIFACTS"]) / "_smoke"
    if smoke_dir.exists():
        shutil.rmtree(smoke_dir, ignore_errors=True)
    smoke_dir.mkdir(parents=True, exist_ok=True)
    out_dir = str(smoke_dir / "adapters")
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 78)
    print(f"SMOKE TEST - {n_train} train / {n_score} score records, tiny models")
    print("=" * 78)
    try:
        # shrink everything. Same code, small numbers.
        ARTIFACTS = smoke_dir
        TRAIN_RECORDS = saved["TRAIN_RECORDS"][:n_train]
        TEST_RECORDS = saved["TEST_RECORDS"][:n_score]
        SCORE_N, MAX_SEQ = n_score, 256
        NUM_EPOCHS, GRAD_ACCUM = 1, 1
        PLM_MAX_IN, PLM_MAX_OUT = 128, 64
        VERIFY_MODE = "deterministic"      # Stage B without 80 LLM calls per record
        REQUIRE_FINETUNED, HUB_SYNC = True, False
        # hub_push_* gate on HUB["write"], not HUB_SYNC, so throwaway smoke
        # artifacts would otherwise land in the real store.
        _hub_write, HUB["write"] = HUB.get("write"), False
        MODEL_REGISTRY = dict(saved["MODEL_REGISTRY"])
        MODEL_REGISTRY["bart"] = {**MODEL_REGISTRY["bart"], "id": TINY_MODELS["plm"],
                                  "out": f"{out_dir}/tinybart-soap"}
        MODEL_REGISTRY["qwen"] = {**MODEL_REGISTRY["qwen"], "id": TINY_MODELS["qlora"],
                                  "out": f"{out_dir}/tinyqwen-soap-qlora"}

        # ---- 1. Stage A + B ------------------------------------------------
        def _prep():
            pairs = _prep_records(TRAIN_RECORDS + TEST_RECORDS, True, " smoke")
            n_ent = sum(len(e) for _, e in pairs)
            return (len(pairs) == n_train + n_score and n_ent > 0,
                    f"{len(pairs)} records, {n_ent} verified entities")
        if not check("Stage A + B (extract + UMLS verify)", _prep):
            raise RuntimeError("entity prep failed - nothing downstream can run")

        # ---- 2. PLM training ------------------------------------------------
        def _plm():
            global MODEL
            MODEL = "bart"
            paths = train_plm(TRAIN_RECORDS)
            _mark_done("bart")
            loss = [v for k, v in LAST_TRAIN_LOSS.items() if k.startswith("bart")]
            return (is_trained("bart"),
                    f"{len(paths)} section adapters, loss "
                    + (f"{statistics.fmean(loss):.2f}" if loss else "n/a"))
        check("PLM fine-tune (4 sections, LoRA, marker)", _plm)

        # ---- 3. QLoRA training ---------------------------------------------
        def _qlora():
            global MODEL
            MODEL = "qwen"
            out = train_qlora(TRAIN_RECORDS)
            _mark_done("qwen")
            return (is_trained("qwen"),
                    f"adapter at {os.path.basename(str(out))}, loss "
                    + (f"{LAST_TRAIN_LOSS.get('qwen', float('nan')):.2f}"))
        check("QLoRA fine-tune (4-bit, grad ckpt, marker)", _qlora)

        # ---- 4. generation + all three arms + metrics -----------------------
        def _score():
            res = score_model("bart", records=TEST_RECORDS, judge=False)
            arms_ok = [a for a, rows in res.items() if rows]
            keys = sorted({k for rows in res.values() for r in rows for k in r})
            return (len(arms_ok) == len(ARMS) and "rougeL_f1" in keys,
                    f"arms {len(arms_ok)}/{len(ARMS)}, {len(keys)} metrics incl "
                    + ", ".join(k for k in ("rougeL_f1", "bertscore_f1", "concept_f1",
                                            "entity_coverage_rate") if k in keys))
        scored = check("Generation -> arms A/B/C -> metrics", _score)

        # ---- 5. resume: a second pass must recompute nothing ----------------
        def _resume():
            # glob through the same key builder, or a SCORING_VERSION bump
            # silently makes this count zero and the check fails for the wrong reason
            _pat = _mk_for("bart", "metrics_") + "*"
            before = len(list(smoke_dir.glob(_pat)))
            calls = {"n": 0}
            orig = globals()["run_batch"]
            def counting(*a, **k):
                calls["n"] += 1
                return orig(*a, **k)
            globals()["run_batch"] = counting
            try:
                score_model("bart", records=TEST_RECORDS, judge=False)
            finally:
                globals()["run_batch"] = orig
            return (calls["n"] == 0 and before > 0,
                    f"{before} cached metric files, {calls['n']} recomputations")
        if scored:
            check("Resume (re-running recomputes nothing)", _resume)

        # ---- 6. the four result tables --------------------------------------
        def _tables():
            import io, contextlib
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                metric_glossary()
                for a in ARMS:
                    compare_models(arm=a)
                    paper_table(arm=a)
                ablation_gains()
                allr = load_all_results()
                for m in allr:
                    if len(allr[m]["results"]) >= 2:
                        results_table(allr[m]["results"])
                        significance_table(allr[m]["results"])
                export_all()
            out = buf.getvalue()
            need = ["GLOSSARY", "CROSS-MODEL COMPARISON", "ABLATION",
                    "PIPELINE ARMS COMPARED", "PAIRWISE ARM COMPARISONS"]
            missing = [w for w in need if w not in out]
            return (not missing, f"{len(out.splitlines())} lines"
                    + (f", MISSING {missing}" if missing else ", all 4 tables rendered"))
        if scored:
            check("Result tables + CSV export", _tables)

    except Exception as ex:
        results.append(("FAIL", "smoke test aborted", f"{type(ex).__name__}: {ex}", 0))
    finally:
        for k, v in saved.items():
            globals()[k] = v
        HUB["write"] = _hub_write
        LAST_TRAIN_LOSS.clear()
        try:
            _free_gpu()
        except Exception:
            pass

    print()
    for st, name, detail, secs in results:
        print(f"  {st:<5} {name:<44} {secs:>5.0f}s  {detail[:70]}")
    n_fail = sum(1 for s, *_ in results if s == "FAIL")
    if _dead["cuda"]:
        print()
        print("  The CUDA context was destroyed by a device-side assert, so every")
        print("  check after that point is BLOCK, not a separate problem. Fix the")
        print("  first FAIL above and the rest will run.")
    print("=" * 78)
    print(f"SMOKE: {'ALL PASS' if n_fail == 0 else str(n_fail) + ' FAILED'} "
          f"in {(time.time() - t0) / 60:.1f} min")
    print("=" * 78)
    return n_fail == 0


if SMOKE_TEST:
    SMOKE_OK = smoke_test()
    if not SMOKE_OK:
        raise SystemExit(
            "Smoke test FAILED - stopping before the real run so no time is wasted. "
            "Paste the block above; the failing stage is named in it.")
else:
    SMOKE_OK = True
    print("SMOKE_TEST=False - skipped.")


In [ ]:
# ============================================================================
# PREFLIGHT - fail in ten seconds, not three hours in.
# ============================================================================
def preflight():
    ok, warn = True, []
    # IPython catches SystemExit and simply ends the cell, so a failed smoke test
    # does NOT stop a Run All by itself. Every later cell re-checks the gate.
    if not SMOKE_OK:
        print("=" * 78)
        print("SMOKE TEST FAILED - preflight skipped, nothing will run.")
        print("=" * 78)
        return False
    recs = _score_records()
    print("=" * 78); print("PREFLIGHT"); print("=" * 78)

    print(f"  dataset        : {DATASET}  |  {len(TRAIN_RECORDS)} train / "
          f"{len(TEST_RECORDS)} test, scoring on {len(recs)}")
    if not TRAIN_RECORDS:
        print("    FAIL - no training records"); ok = False

    # entity prep: shared by every model, so it is paid once
    need = {str(r["id"]) for r in TRAIN_RECORDS} | {str(r["id"]) for r in recs}
    have = sum(1 for rid in need if _ck_path("stageB", rid).exists())
    missing = len(need) - have
    _spec, _active = _pipeline_spec()
    print(f"  entity prep    : {have}/{len(need)} records checkpointed "
          f"({_active}, VERIFY_MODE={VERIFY_MODE})")
    if missing and _spec["kind"] == "openai":
        if _resolve_openai_key():
            calls = missing * (81 if VERIFY_MODE == "llm_evidence" else 1)
            print(f"    OPENAI_API_KEY found - {missing} records, ~{calls:,} calls, "
                  f"~{calls / (RECORD_WORKERS * N_WORKERS) * 0.8 / 60:.0f} min")
        else:
            print("    FAIL - no OpenAI key. Put OPENAI_API_KEY=sk-... in the .env file")
            print("           next to this notebook (never paste keys into the notebook).")
            ok = False
    elif not missing:
        print("    all restored - Stage A/B needs no OpenAI calls")
    # Prep is not the only consumer: generation for gpt, Stage-D reflection and
    # both judges all call the API during scoring, on every run.
    if not _resolve_openai_key():
        print("    FAIL - scoring needs the OpenAI key too (reflection + both judges),")
        print("           even when entity prep is fully restored. See above.")
        ok = False

    blocked = {k for k, _, _ in HF_BLOCKED}
    trainable = [m for m in TRAIN_ORDER if m not in blocked]
    print(f"  HF access      : {len(trainable)}/{len(TRAIN_ORDER)} downloadable")
    for k, mid, c in HF_BLOCKED:
        print(f"    gated: {k} ({mid}) HTTP {c} - accept the licence to include it")
    if not trainable:
        print("    FAIL - no model is downloadable"); ok = False

    try:
        import torch
        if torch.cuda.is_available():
            free, tot = torch.cuda.mem_get_info()
            print(f"  GPU            : {torch.cuda.get_device_name(0)} "
                  f"{free / 2**30:.1f}/{tot / 2**30:.1f} GiB free")
        else:
            print("  GPU            : NONE - install a CUDA build of PyTorch (setup.sh) and pick the SOAP (GPU) kernel")
            ok = False
    except Exception as e:
        print(f"  GPU            : torch unavailable ({type(e).__name__})"); ok = False

    try:
        du = shutil.disk_usage("/kaggle/working" if ON_KAGGLE else ".")
        print(f"  disk           : {du.free / 2**30:.1f} GiB free of "
              f"{du.total / 2**30:.1f} GiB (PURGE_HF_CACHE={PURGE_HF_CACHE}, "
              f"PLM_USE_LORA={PLM_USE_LORA})")
        if du.free < 8 * 2**30:
            warn.append("under 8 GiB free - a 15 GiB model download may not fit")
    except Exception:
        pass

    if HUB.get("write"):
        print(f"  backup         : ON -> {HUB['repo']}")
    else:
        print(f"  backup         : OFF - {HUB.get('why', 'no write token')}")
        if ON_KAGGLE:      # locally everything is already on disk
            warn.append("Hub backup OFF: nothing survives this session. Fix the token.")

    trained = [m for m in TRAIN_ORDER if is_trained(m)]
    part = [m for m in TRAIN_ORDER if _partially_trained(m)]
    if trained: print(f"  already trained: {', '.join(trained)}")
    if part:
        print(f"  will retrain   : {', '.join(part)}")
        stale_report(part)

    # How much training actually happens. Eight optimizer steps is a warm-up, not
    # a fine-tune, and that is exactly what GRAD_ACCUM=16 gave on 200 records.
    _dev = 2 if 'torch' in dir() else 1
    try:
        import torch as _t
        _dev = max(1, _t.cuda.device_count())
    except Exception:
        pass
    _steps = max(1, int(len(TRAIN_RECORDS) / (1 * GRAD_ACCUM * _dev))) * NUM_EPOCHS
    print(f"  optimizer steps: ~{_steps} per causal model "
          f"({len(TRAIN_RECORDS)} records, accum {GRAD_ACCUM}, {_dev} GPU, "
          f"{NUM_EPOCHS} epochs), MAX_SEQ={MAX_SEQ}")
    if _steps < 40:
        warn.append(f"only ~{_steps} optimizer steps - the adapter will barely move. "
                    f"Lower GRAD_ACCUM or raise TRAIN_LIMIT.")

    # honest estimate for what is actually left
    EST_TRAIN = {"biobart": 10, "bart": 40, "qwen": 45, "llama": 45,
                 "openbio": 45, "qcammel": 85, "llama2": 85}
    EST_SCORE = {"gpt": 12, "biobart": 25, "bart": 25}      # per model, all arms
    to_train = [m for m in trainable if not is_trained(m)]
    t_min = sum(EST_TRAIN.get(m, 45) for m in to_train)
    s_min = 0
    for m in ALL_MODELS + ["gpt"]:
        miss = [a for a in ARMS if not arm_done(m, a, recs)]
        if miss:
            per = EST_SCORE.get(m, 0.8 * len(recs))         # ~48 s/record for causal
            s_min += per * (len(miss) / max(1, len(ARMS)))
    print(f"  estimate       : ~{missing * 0.9:.0f} min prep + ~{t_min} min train "
          f"+ ~{s_min:.0f} min score = ~{(missing * 0.9 + t_min + s_min) / 60:.1f} h")
    print(f"  budgets        : train {TRAIN_BUDGET_MIN} min | score {SCORE_BUDGET_MIN} min "
          f"| hard stop {SESSION_HARD_MIN} min")

    for w in warn:
        print(f"  WARNING        : {w}")
    print("=" * 78)
    print("READY - the next cell runs everything." if ok
          else "NOT READY - fix the FAILs above. Nothing was run.")
    print("=" * 78)
    return ok


PREFLIGHT_OK = preflight()

In [ ]:
# ============================================================================
# RUN.  Train everything, score everything, project what cannot be trained.
# ============================================================================
if not SMOKE_OK:
    raise SystemExit("Smoke test fail - kuch nahi chala. Upar SMOKE block dekho.")
if not PREFLIGHT_OK:
    raise SystemExit("Preflight fail - upar dekho. Kuch nahi chala.")

T0 = time.time()
_mins = lambda: (time.time() - T0) / 60
_blocked = {k for k, _, _ in HF_BLOCKED}
SCORE_RECORDS = _score_records()

if _blocked:
    print(f"Gated, train nahi honge: {', '.join(sorted(_blocked))}")
    print("Inki row PROJECTED aayegi, measured models se scale hokar.\n")

# ---------------------------------------------------------------- PHASE 1
print("\n" + "#" * 78)
print("# PHASE 1 - entity prep (Stage A k-spans -> Stage B UMLS verification)")
print("#" * 78)
_need_train = (ONLY_MODELS is None or LAUNCH_GPT_FINETUNE
               or any(m != "gpt" for m in ONLY_MODELS))     # training records need entity prep
_seen, _targets = set(), []
for _r in (list(TRAIN_RECORDS) if _need_train else []) + list(SCORE_RECORDS):
    if str(_r["id"]) not in _seen:
        _seen.add(str(_r["id"])); _targets.append(_r)
_missing = [r for r in _targets if not _ck_path("stageB", str(r["id"])).exists()]
if _missing:
    print(f"{len(_missing)} of {len(_targets)} records need prep.")
    _prep_records(_missing, grouped=True, label=" all")
    hub_push_artifacts()
else:
    print(f"all {len(_targets)} records already prepped - nothing to do.")
kb_health_report()

# ---------------------------------------------------------------- PHASE 1b
# Launch the OpenAI job NOW so it bakes during the hours of GPU training below.
print("\n" + "#" * 78)
print("# PHASE 1b - launch the split-safe GPT fine-tune (async, no GPU)")
print("#" * 78)
GPT_FT = launch_gpt_finetune() if LAUNCH_GPT_FINETUNE else None

# ---------------------------------------------------------------- PHASE 2
print("\n" + "#" * 78)
print("# PHASE 2 - fine-tune every model on the SAME training split")
print("#" * 78)
_train_only = None if ONLY_MODELS is None else [m for m in ONLY_MODELS if m != "gpt"]
if _train_only == []:
    print("ONLY_MODELS has no local model - training skipped")
    TRAIN_STATUS = {}
else:
    TRAIN_STATUS = train_all_models(
        only=_train_only,
        skip=_blocked,
        time_budget_min=max(5, TRAIN_BUDGET_MIN - _mins()),
    )

# ---------------------------------------------------------------- PHASE 2b
print("\n" + "#" * 78)
print("# PHASE 2b - collect the GPT fine-tune before scoring")
print("#" * 78)
GPT_FT_MODEL = resolve_gpt_finetune() if LAUNCH_GPT_FINETUNE else None

# ---------------------------------------------------------------- PHASE 2c
# Training holds a lot of GPU state. Scoring then has to fit four PLM section
# models plus BLEURT, roberta-large and the biomedical embedder - about 4 GiB of
# metric models on top of the generator. On 2026-09-20 the leak left 2.81 MiB
# free and every single record OOM-ed, so the state is now reset and MEASURED
# before a single record is scored.
print("\n" + "#" * 78)
print("# PHASE 2c - release training state before scoring")
print("#" * 78)
_free_gpu()
_cuda_purge()
_vram = _gpu_free()
for _d, _f, _t in _vram:
    print(f"  gpu{_d}: {_f:.1f} GiB free of {_t:.1f} GiB")
_worst = min([f for _, f, _ in _vram], default=99.0)
# 5.0 was too low: on 2026-09-20 the purge reached 5.16 GiB, the gate passed, and
# scoring then failed on a 12 MiB allocation for all 50 records. Scoring needs the
# four PLM section models plus BLEURT, roberta-large and the embedder resident at
# once. 8 GiB is the honest floor.
if _vram and _worst < 8.0:
    print("!" * 74)
    print(f"!  Only {_worst:.1f} GiB free - scoring will OOM. STOPPING.")
    print("!  Scoring needs the four PLM section models plus BLEURT, roberta-large")
    print("!  and the biomedical embedder resident at once - about 6 GiB, and a")
    print("!  12 MiB allocation already failed at 5.16 GiB free.")
    print("!  Training state did not release. Restart the kernel and Run All again:")
    print("!  every trained adapter is on disk, so PHASE 2 skips them and scoring")
    print("!  starts on a clean GPU.")
    print("!" * 74)
    raise SystemExit("insufficient GPU memory for scoring - start a fresh notebook")
elif _vram:
    print(f"  {_worst:.1f} GiB free on the tightest device - enough for scoring.")

# ---------------------------------------------------------------- PHASE 3
print("\n" + "#" * 78)
print(f"# PHASE 3 - score every trained model on {len(SCORE_RECORDS)} held-out "
      f"records, arms {list(ARMS)}")
print("#" * 78)
SCORE_STATUS = score_all_models(models=ONLY_MODELS, n=SCORE_N, deadline=T0 + SCORE_BUDGET_MIN * 60)

# ---------------------------------------------------------------- PHASE 4
print("\n" + "#" * 78)
print("# PHASE 4 - results")
print("#" * 78)

# 4a. projections for whatever could not be trained (gated models), scaled from
#     the best MEASURED model - not from the seed, because we now have real data.
print()
PROJECTIONS = build_projections()

# 4b. the glossary, so every table below stands on its own
metric_glossary()

# 4c. cross-model headline table, once per arm
for _arm in ARMS:
    print()
    compare_models(arm=_arm)

# 4d. the K-SOAP Table 5 view: reference-based metrics only
for _arm in ARMS:
    print()
    paper_table(arm=_arm)

# 4e. what each stage bought. MEASURED MODELS ONLY.
print()
ABLATION = ablation_gains()

# 4f. Stage D on its own: how often reflection revised, and which check it fixed.
print()
REFLECTION = reflection_report()

# 4g. the full per-arm table for every measured model
_allr = load_all_results()
for _m in sorted(_allr):
    _res = _allr[_m]["results"]
    if len(_res) < 2:
        continue
    print("\n" + "=" * 78)
    print(f"ARMS {' / '.join(ARM_DOC[a][0] for a in ARMS)}  -  {_m}   (measured)")
    print("=" * 78)
    _prev_MODEL, MODEL = MODEL, _m
    try:
        results_table(_res)
        significance_table(_res)
    finally:
        MODEL = _prev_MODEL

export_all()
for _arm in ARMS:
    export_paper_tables(arm=_arm, stem=f"verisoap_paper_{ARM_DOC[_arm][0]}")
hub_push_artifacts()

# 4h. what the agents did (arm D): re-extractions, tool switches, accepted / rejected revisions
if "D_agentic" in ARMS:
    for _m, _d in sorted(_allr.items()):
        _rows = _d["results"].get("D_agentic") or []
        if not _rows:
            continue
        _avg = lambda k: sum(r.get(k, 0) or 0 for r in _rows) / len(_rows)
        print(f"\nAGENTS ({_m}, {len(_rows)} records): steps {_avg('agent_steps'):.1f} | "
              f"K-Span attempts {_avg('agent_kspan_attempts'):.2f} | entity set changed "
              f"{_avg('agent_entities_changed'):.0%} | revisions {_avg('agent_revisions'):.2f} | "
              f"rejected {_avg('agent_rejected'):.2f} | code deletions {_avg('agent_code_deletions'):.2f}")
    print("  per-record decisions: show_agent_log('<record id>', model='gpt')")

# ---------------------------------------------------------------- summary
print("\n" + "#" * 78)
print(f"# SESSION DONE in {_mins():.0f} min")
print("#" * 78)
training_status()
print()
ALL_DONE = whats_left(records=SCORE_RECORDS)

print(f"\n  measured  : {', '.join(sorted(_allr)) if _allr else 'none'}"
      f"   ({len(SCORE_RECORDS)} records x {len(ARMS)} arms)")
_proj = load_projections()
print(f"  projected : {', '.join(sorted(_proj)) if _proj else 'none'}")
if GPT_FT_MODEL:
    print(f"  gpt model : {GPT_FT_MODEL}  (split-safe, no '+' tag)")
else:
    print(f"  gpt model : {MODEL_REGISTRY['gpt']['ft']}  (PRE-SPLIT, row tagged '+')")
print(f"  exported  : {ARTIFACTS}/verisoap_all_models.csv  and  .json")
if HUB.get("write"):
    print(f"\n  Sab {HUB['repo']} pe backed up hai.")
if not ALL_DONE:
    print("\n  Something is left - just Run All again; finished work is skipped.")
else:
    print("\n  Sab khatam. Ye numbers paper mein jaa sakte hain.")